### Nama : Hazel Novero Winata
### NIM  : 2802519114
### FINAL PROJECT MODEL DEPLOYMENT

### Link Vidio: https://drive.google.com/drive/folders/1irfbmjds6OJg06BBivk-rkVUnxBles2Z?usp=sharing

# DATA UNDERSTANDING

Dataset yang digunakan merupakan data historis nasabah pada sebuah institusi keuangan. Data ini memuat berbagai informasi mengenai profil ekonomi, kepemilikan fasilitas kredit, jumlah pinjaman, riwayat pembayaran, tingkat keterlambatan, jumlah utang, penggunaan kredit, serta perilaku pembayaran nasabah. Informasi tersebut akan digunakan untuk memahami karakteristik nasabah dan membangun model machine learning yang dapat menilai performa kredit.

Dataset terdiri atas **25.000 baris dan 29 kolom**, dengan **11.254 nasabah unik** yang diidentifikasi melalui kolom `Customer_ID`. Satu nasabah dapat muncul lebih dari satu kali karena setiap baris merepresentasikan kondisi kredit nasabah pada bulan tertentu. Periode pencatatan data mencakup delapan bulan, yaitu Januari hingga Agustus. Oleh karena itu, satu baris data bukan selalu satu nasabah yang berbeda, melainkan satu catatan bulanan dari seorang nasabah.

Variabel yang akan diprediksi adalah `Credit_Score`. Variabel ini memiliki tiga kelas, yaitu:

* `Good`, yang menunjukkan performa kredit nasabah tergolong baik.
* `Standard`, yang menunjukkan performa kredit nasabah berada pada tingkat normal atau menengah.
* `Poor`, yang menunjukkan performa kredit nasabah tergolong buruk atau memiliki risiko lebih tinggi.

Distribusi target terdiri atas **13.282 data `Standard`**, **7.268 data `Poor`**, dan **4.450 data `Good`**. Karena target memiliki lebih dari dua kelas, permasalahan ini termasuk dalam **multiclass classification**. Distribusi kelas juga tidak sepenuhnya seimbang karena kelas `Standard` memiliki jumlah data paling besar, sedangkan kelas `Good` memiliki jumlah paling sedikit.

Secara umum, fitur-fitur pada dataset dapat dikelompokkan menjadi beberapa kategori utama. Kelompok pertama adalah **kolom identitas dan administratif**, yang terdiri dari `Unnamed: 0`, `ID`, `Customer_ID`, `Name`, dan `SSN`. Kolom-kolom ini berfungsi sebagai identitas data maupun identitas nasabah dan tidak memiliki hubungan langsung dengan performa kredit sehingga tidak akan digunakan sebagai fitur prediksi. Namun, `Customer_ID` tetap akan dimanfaatkan untuk mengidentifikasi nasabah yang sama pada periode yang berbeda.

Kelompok kedua adalah **profil demografis dan pekerjaan nasabah**, yang terdiri dari `Age` dan `Occupation`. Fitur-fitur ini menggambarkan karakteristik dasar nasabah yang berpotensi memengaruhi kondisi keuangan dan kemampuan pengelolaan kredit.

Kelompok ketiga adalah **informasi pendapatan dan kondisi keuangan**, yang terdiri dari `Annual_Income`, `Monthly_Inhand_Salary`, `Amount_invested_monthly`, dan `Monthly_Balance`. Fitur-fitur ini menggambarkan kemampuan finansial nasabah, mulai dari pendapatan tahunan, pendapatan bulanan bersih, jumlah investasi yang dilakukan setiap bulan, hingga saldo dana yang masih tersisa setelah berbagai kewajiban finansial dipenuhi.

Kelompok keempat adalah **kepemilikan fasilitas kredit dan pinjaman**, yang terdiri dari `Num_Bank_Accounts`, `Num_Credit_Card`, `Num_of_Loan`, `Type_of_Loan`, `Interest_Rate`, `Credit_Mix`, dan `Changed_Credit_Limit`. Kelompok ini menjelaskan jumlah rekening bank, jumlah kartu kredit, jumlah dan jenis pinjaman yang dimiliki, tingkat bunga yang dikenakan, kualitas kombinasi kredit yang dimiliki nasabah, serta perubahan batas kredit yang terjadi dari waktu ke waktu.

Kelompok kelima adalah **riwayat pembayaran dan perilaku kredit**, yang terdiri dari `Delay_from_due_date`, `Num_of_Delayed_Payment`, `Num_Credit_Inquiries`, `Payment_of_Min_Amount`, dan `Payment_Behaviour`. Fitur-fitur ini menggambarkan kedisiplinan nasabah dalam memenuhi kewajiban kreditnya, seperti jumlah keterlambatan pembayaran, frekuensi keterlambatan, jumlah pemeriksaan kredit, kebiasaan membayar jumlah minimum tagihan, serta pola pengeluaran dan pembayaran yang dilakukan.

Kelompok keenam adalah **informasi utang dan pemanfaatan kredit**, yang terdiri dari `Outstanding_Debt`, `Credit_Utilization_Ratio`, `Credit_History_Age`, dan `Total_EMI_per_month`. Fitur-fitur ini menggambarkan total utang yang masih dimiliki nasabah, tingkat penggunaan fasilitas kredit, lamanya riwayat kredit yang dimiliki, serta total cicilan bulanan yang harus dibayarkan.

Selain fitur-fitur tersebut, terdapat pula kolom `Month` yang menunjukkan periode pencatatan data. Kolom ini penting karena dataset bersifat longitudinal, sehingga memungkinkan seorang nasabah memiliki beberapa catatan pada bulan yang berbeda dengan kondisi kredit yang berubah dari waktu ke waktu.

Hasil pemeriksaan awal menunjukkan bahwa dataset masih memiliki beberapa permasalahan kualitas data, seperti missing value, nilai kosong terselubung, fitur numerik yang tersimpan sebagai tipe data kategorikal, karakter tambahan berupa underscore, serta sejumlah nilai yang tidak realistis. Oleh karena itu, sebelum proses pemodelan dilakukan, data perlu melalui tahapan pembersihan, validasi nilai, konversi tipe data, imputasi missing value, encoding fitur kategorikal, dan penanganan nilai ekstrem.

Tujuan akhir dari analisis ini adalah membangun beberapa model machine learning untuk memprediksi apakah seorang nasabah memiliki `Credit_Score` **Good, Standard, atau Poor**. Setiap model akan dibandingkan menggunakan metrik evaluasi yang sesuai untuk klasifikasi multikelas, sehingga dapat dipilih model terbaik untuk digunakan pada proses deployment.


In [ ]:
## Library-library yang kira kira akan saya gunakan

# STANDARD LIBRARY
import json
import logging
import os
import re
import time
import warnings

from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


# DATA MANIPULATION
import numpy as np
import pandas as pd


# DATA VISUALIZATION
import matplotlib.pyplot as plt
import seaborn as sns

# SCIKIT-LEARN BASE CLASSES FOR OOP
from sklearn.base import BaseEstimator, TransformerMixin, clone


# PREPROCESSING AND PIPELINE
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    RobustScaler,
    StandardScaler,
    label_binarize
)


# DATA SPLITTING AND CROSS-VALIDATION
from sklearn.model_selection import (
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold,
    cross_validate
)


# CLASS IMBALANCE HANDLING
from sklearn.utils.class_weight import (
    compute_class_weight,
    compute_sample_weight
)


# MACHINE LEARNING MODELS
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


# MODEL EVALUATION
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score
)


# FEATURE IMPORTANCE
from sklearn.inspection import permutation_importance

# HYPERPARAMETER OPTIMIZATION
import optuna
from optuna.samplers import TPESampler

# MLFLOW EXPERIMENT TRACKING
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

# MODEL SERIALIZATION
import joblib



In [ ]:
# GLOBAL CONFIGURATION
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}"
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)



Konfigurasi global saya gunakan untuk menjaga konsistensi dan keterbacaan proses analisis. `RANDOM_STATE` ditetapkan sebesar 42 agar proses yang melibatkan pengacakan dapat menghasilkan output yang relatif sama ketika dijalankan ulang. Pengaturan tampilan pandas digunakan agar seluruh kolom dan baris tertentu dapat ditampilkan dengan format angka yang lebih rapi. Peringatan bertipe `FutureWarning` disembunyikan agar output notebook tidak terlalu penuh, sedangkan logging digunakan untuk mencatat informasi proses secara terstruktur. 


## Function Cek Missing Value

In [ ]:
def missing_value_summary(df):
    
    missing_df = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        )
    })

    missing_df = (
        missing_df[
            missing_df["missing_count"] > 0
        ]
        .sort_values(
            by="missing_count",
            ascending=False
        )
        .reset_index()
        .rename(
            columns={"index": "column"}
        )
    )

    return missing_df


def show_missing_rows(df, column):
    
    return df[
        df[column].isna()
    ]

## Function Cek Outlier

In [ ]:
def show_outlier_rows(df, column):

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    print(f"Column         : {column}")
    print(f"Lower Bound    : {lower_bound:,.4f}")
    print(f"Upper Bound    : {upper_bound:,.4f}")

    outlier_df = df[
        (
            df[column] < lower_bound
        ) |
        (
            df[column] > upper_bound
        )
    ].copy()

    print(f"Total Outlier  : {len(outlier_df):,}")

    return outlier_df.sort_values(
        by=column,
        ascending=False
    )

In [ ]:
# Membaca dataset
df = pd.read_csv("data_C.csv")

df

# Function Untuk Cek Missing Value dan Outlier

# A. Exploratory Data Analysis Secara Umum

## 1. df.info()

In [ ]:
df.info()

Berdasarkan output `df.info()`, dataset terdiri atas **25.000 baris dan 29 kolom** dengan penggunaan memori sekitar **5,5 MB**. Secara umum, dataset memiliki kombinasi tipe data `int64`, `float64`, dan `object`.

Hasil pemeriksaan awal menunjukkan bahwa beberapa kolom masih memiliki tipe data yang belum sesuai dengan karakteristik sebenarnya. Contohnya, kolom seperti `Age`, `Annual_Income`, `Num_of_Loan`, `Num_of_Delayed_Payment`, dan `Changed_Credit_Limit` masih terbaca sebagai `object`, padahal secara konsep seharusnya berupa data numerik. Kondisi ini kemungkinan disebabkan oleh adanya karakter tambahan, nilai khusus, atau format data yang tidak konsisten.

Selain itu, beberapa kolom memiliki missing value, seperti `Name`, `Monthly_Inhand_Salary`, `Type_of_Loan`, `Num_of_Delayed_Payment`, `Num_Credit_Inquiries`, dan `Monthly_Balance`. Missing value tersebut perlu dianalisis lebih lanjut karena tidak seluruh nilai kosong memiliki penyebab yang sama. Sebagai contoh, nilai kosong pada `Type_of_Loan` dapat terjadi karena nasabah memang tidak memiliki pinjaman, sedangkan missing value pada `Monthly_Inhand_Salary` menunjukkan informasi pendapatan bulanan yang tidak tersedia.

Dataset juga memiliki beberapa kolom identitas, seperti `ID`, `Customer_ID`, `Name`, `SSN`, dan `Unnamed: 0`. Kolom-kolom tersebut tidak akan langsung digunakan sebagai fitur prediksi karena dapat menyebabkan model menghafal identitas nasabah dan tidak memberikan informasi yang relevan untuk menilai performa kredit. Namun, `Customer_ID` tetap akan digunakan untuk mengidentifikasi catatan dari nasabah yang sama pada periode yang berbeda.

Pada tahap selanjutnya, akan dilakukan pemeriksaan yang lebih mendalam terhadap missing value, nilai tidak valid, kesesuaian tipe data, data duplikat, distribusi target, dan karakteristik setiap fitur. Setelah itu, data akan melalui proses pembersihan, konversi tipe data, imputasi, encoding fitur kategorikal, serta penanganan nilai ekstrem sebelum digunakan dalam proses training model machine learning.


## 2. NUMERICAL DESCRIPTIVE STATISTICS

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns

df[numeric_columns].describe().T

Berdasarkan statistik deskriptif fitur numerik, kolom `Unnamed: 0` tidak akan menjadi fokus analisis karena hanya merupakan indeks atau nomor urut data yang ditambahkan saat proses penyimpanan dataset. Nilai minimum, maksimum, maupun rata-ratanya tidak memiliki makna bisnis yang berkaitan dengan performa kredit nasabah sehingga kolom ini akan dihapus pada tahap preprocessing.

Beberapa fitur menunjukkan indikasi adanya nilai yang tidak wajar. Sebagai contoh, `Num_Bank_Accounts` memiliki nilai maksimum 1.789 rekening dan bahkan terdapat nilai minimum -1. Demikian pula pada `Num_Credit_Card` yang memiliki nilai maksimum 1.498 kartu kredit, serta `Interest_Rate` yang mencapai 5.797. Nilai-nilai tersebut tidak realistis dalam konteks dunia nyata dan mengindikasikan adanya data corruption atau kesalahan pencatatan yang perlu ditangani pada tahap pembersihan data.

Kondisi serupa juga terlihat pada `Num_Credit_Inquiries` yang memiliki nilai maksimum 2.592, padahal median hanya sebesar 6. Perbedaan yang sangat jauh antara nilai maksimum dan median menunjukkan adanya outlier ekstrem yang dapat memengaruhi proses pelatihan model apabila tidak ditangani dengan tepat.

Pada fitur `Total_EMI_per_month`, nilai maksimum mencapai 82.331 sementara median hanya sebesar 68,25. Hal ini menunjukkan distribusi data yang sangat menceng ke kanan (right-skewed) dan kemungkinan terdapat sejumlah kecil observasi dengan nilai yang sangat besar dibandingkan mayoritas data.

Selain itu, terdapat beberapa kolom yang memiliki missing value, seperti `Monthly_Inhand_Salary` dengan 21.164 data terisi dari total 25.000 data, serta `Monthly_Balance` dan `Num_Credit_Inquiries` yang juga memiliki sejumlah nilai kosong. Missing value tersebut perlu dianalisis lebih lanjut sebelum dilakukan proses imputasi.

Sementara itu, beberapa fitur masih menunjukkan rentang nilai yang relatif masuk akal. Sebagai contoh, `Credit_Utilization_Ratio` memiliki nilai antara 20 hingga 50 dengan median sebesar 32,31, sedangkan `Delay_from_due_date` memiliki median keterlambatan pembayaran sebesar 18 hari. Namun demikian, keberadaan nilai negatif pada `Delay_from_due_date` perlu ditinjau lebih lanjut untuk memastikan apakah nilai tersebut merepresentasikan pembayaran sebelum jatuh tempo atau merupakan anomali data.

Secara keseluruhan, statistik deskriptif awal menunjukkan bahwa dataset masih memerlukan proses data cleaning yang cukup signifikan, terutama untuk menangani missing value, memperbaiki tipe data yang belum sesuai, serta mengidentifikasi dan menangani nilai-nilai ekstrem yang tidak realistis sebelum digunakan dalam proses pemodelan machine learning.


## 3.CATEGORICAL DESCRIPTIVE STATISTICS

In [ ]:
categorical_columns = df.select_dtypes(
    exclude="number"
).columns

df[categorical_columns].describe().T

Berdasarkan statistik deskriptif fitur kategorikal, terlihat bahwa dataset memiliki beberapa karakteristik yang menarik untuk diperhatikan sebelum proses pemodelan dilakukan. Kolom `Customer_ID` menunjukkan terdapat **11.254 nasabah unik** dari total **25.000 observasi**, yang mengindikasikan bahwa satu nasabah dapat muncul pada beberapa bulan yang berbeda. Hal ini sesuai dengan karakteristik dataset yang bersifat longitudinal atau time-series per pelanggan.

Kolom `Month` memiliki **8 kategori unik**, yaitu periode Januari hingga Agustus, dengan bulan Februari menjadi periode yang paling banyak muncul sebanyak **3.203 observasi**. Distribusi ini menunjukkan bahwa data relatif tersebar merata pada setiap bulan.

Selain itu, ditemukan beberapa nilai yang mengindikasikan adanya **hidden missing value** atau data yang tidak valid. Sebagai contoh, nilai `#F%$D@*&8` pada kolom `SSN` muncul sebanyak **1.385 kali**, nilai `_______` pada kolom `Occupation` muncul sebanyak **1.765 kali**, nilai `_` pada kolom `Changed_Credit_Limit` muncul sebanyak **518 kali**, serta nilai `__10000__` pada kolom `Amount_invested_monthly` muncul sebanyak **1.049 kali**. Nilai-nilai tersebut perlu ditangani lebih lanjut pada tahap preprocessing karena secara konsep tidak merepresentasikan informasi yang valid.

Beberapa kolom yang secara konsep seharusnya bersifat numerik juga masih terbaca sebagai tipe data kategorikal, seperti `Age`, `Annual_Income`, `Num_of_Loan`, `Num_of_Delayed_Payment`, dan `Outstanding_Debt`. Hal ini terlihat dari banyaknya nilai unik yang dimiliki masing-masing kolom. Oleh karena itu, kolom-kolom tersebut perlu dibersihkan dan dikonversi ke tipe data numerik sebelum digunakan dalam proses pemodelan.

Pada variabel target `Credit_Score`, kelas yang paling banyak muncul adalah `Standard` dengan **13.282 observasi**, diikuti oleh `Poor` dan `Good`. Hal ini menunjukkan bahwa distribusi target tidak sepenuhnya seimbang sehingga perlu diperhatikan pada tahap evaluasi model agar performa model tidak hanya baik pada kelas mayoritas.

Secara keseluruhan, hasil analisis awal pada fitur kategorikal menunjukkan bahwa dataset masih memerlukan proses data cleaning, khususnya untuk menangani hidden missing value, memperbaiki tipe data yang belum sesuai, serta mempersiapkan fitur kategorikal agar dapat digunakan secara optimal dalam proses machine learning.


## 4. Cek Duplikat

### a. duplikat keseluruhan per row

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Jumlah baris duplikat: {duplicate_count:,}")

Data ini bersih dari data duplikat

### b. Cek ID duplikat

In [ ]:
duplicate_id = df["ID"].duplicated().sum()

print(f"Jumlah ID duplikat: {duplicate_id:,}")

Jumlah `ID` sama dengan jumlah baris pada dataset, sehingga tidak ditemukan duplikasi pada kolom tersebut. Hal ini menunjukkan bahwa setiap `ID` merepresentasikan satu catatan unik untuk seorang nasabah pada bulan tertentu. Meskipun satu nasabah dapat muncul beberapa kali dalam dataset, setiap kemunculan tersebut merepresentasikan periode pencatatan yang berbeda sehingga tetap memiliki `ID` yang unik.


In [ ]:
duplicate_customer_month = df.duplicated(
    subset=["Customer_ID", "Month"]
).sum()

print(
    f"Jumlah kombinasi Customer_ID dan Month yang duplikat: "
    f"{duplicate_customer_month:,}"
)

### c. Cek Customer_ID dan Month duplikat

Berdasarkan hasil pemeriksaan, tidak ditemukan duplikasi pada kombinasi `Customer_ID` dan `Month`. Hal ini menunjukkan bahwa setiap nasabah hanya memiliki satu catatan untuk setiap bulan dalam dataset. Dengan demikian, tidak terdapat pencatatan ganda untuk nasabah yang sama pada periode yang sama, sehingga kualitas data pada aspek ini dapat dianggap baik dan konsisten. Hasil ini juga mengindikasikan bahwa setiap baris data merepresentasikan kondisi kredit nasabah yang unik pada suatu bulan tertentu, sehingga tidak diperlukan proses penghapusan duplikasi berdasarkan kombinasi `Customer_ID` dan `Month`.


# B. Exploratory Data Analysis Khusus

## 1. Mengubah Tipe Data yang Tidak Sesuai 

Secara umum, proses konversi tipe data merupakan bagian dari tahap preprocessing. Namun, pada penelitian ini beberapa kolom akan dikonversi terlebih dahulu ke tipe data yang sesuai sebelum visualisasi dilakukan. Hal ini bertujuan agar hubungan antar variabel dapat dianalisis dan divisualisasikan dengan lebih akurat pada tahap Exploratory Data Analysis (EDA). Oleh karena itu, konversi tipe data pada tahap ini dilakukan untuk keperluan eksplorasi dan pemahaman data, sedangkan proses pembersihan dan transformasi data secara menyeluruh tetap akan dilakukan pada tahap preprocessing.


Beberapa kolom pada dataset masih memiliki tipe data yang belum sesuai dengan makna sebenarnya. Oleh karena itu, konversi tipe data akan dilakukan agar setiap variabel dapat dianalisis, divisualisasikan, dan diproses oleh model machine learning dengan tepat.

Kolom yang akan dikonversi adalah:

* `Age` dari `object` menjadi numerik karena merepresentasikan usia nasabah.
* `Annual_Income` dari `object` menjadi `float` karena merepresentasikan pendapatan tahunan.
* `Num_of_Loan` dari `object` menjadi numerik karena merepresentasikan jumlah pinjaman aktif.
* `Num_of_Delayed_Payment` dari `object` menjadi numerik karena merepresentasikan jumlah pembayaran yang terlambat.
* `Changed_Credit_Limit` dari `object` menjadi `float` karena merepresentasikan perubahan batas kredit.
* `Outstanding_Debt` dari `object` menjadi `float` karena merepresentasikan jumlah utang yang belum dibayar.
* `Amount_invested_monthly` dari `object` menjadi `float` karena merepresentasikan jumlah investasi bulanan.
* `Credit_History_Age` dari `object` menjadi numerik dalam satuan total bulan. Nilai tahun dan bulan akan dikonversi menggunakan perhitungan `tahun × 12 + bulan`.
* `Num_Credit_Inquiries` dapat diubah dari `float` menjadi integer setelah seluruh missing value selesai ditangani karena secara konsep kolom ini merupakan data hitungan.

Kolom kategorikal seperti `Month`, `Occupation`, `Type_of_Loan`, `Credit_Mix`, `Payment_of_Min_Amount`, `Payment_Behaviour`, dan `Credit_Score` tetap dipertahankan sebagai data kategorikal. Kolom identitas seperti `ID`, `Customer_ID`, `Name`, dan `SSN` juga tidak memerlukan konversi menjadi numerik karena tidak digunakan sebagai fitur numerik dalam pemodelan.

Konversi awal untuk kebutuhan EDA akan dilakukan pada salinan dataframe agar data mentah tetap terjaga. Seluruh proses konversi yang digunakan dalam training nantinya akan diterapkan kembali secara konsisten melalui class preprocessing berbasis OOP.


## Namun, karena masih terdapat banyak nilai dengan format yang tidak konsisten, nilai ekstrem, serta nilai yang tidak realistis secara konteks, proses konversi tipe data tidak dapat langsung dilakukan. Oleh karena itu, setiap kolom perlu diperiksa dan dibersihkan terlebih dahulu melalui tahap data preprocessing agar nilai yang tidak valid dapat diidentifikasi dan ditangani dengan tepat sebelum digunakan dalam analisis lebih lanjut.


# C. Data Preprocessing: Pemeriksaan dan Perbaikan Format, Nilai Ekstrem, dan Anomali Data


In [ ]:
df.info()

## 1. Kolom Teknis dan Identitas

### a. Unnamed: 0

In [ ]:
df["Unnamed: 0"].value_counts().sort_index()

setiap nilainya unik dari 0 sampai 24.999, hasil value_counts() akan menunjukkan frekuensi 1 untuk setiap nilai. Kolom ini hanya merupakan indeks hasil penyimpanan CSV sehingga nantinya akan dihapus.

### b. ID

In [ ]:
df["ID"].value_counts()

In [ ]:
print(f"Jumlah baris     : {len(df):,}")
print(f"Jumlah ID unik   : {df['ID'].nunique():,}")
print(f"Jumlah ID duplikat: {df['ID'].duplicated().sum():,}")

Perubahan datatype ID tidak akan memengaruhi apa apa dan tidak ada format yang anomali di kolom ini

### c. Customer ID

In [ ]:
df["Customer_ID"].value_counts()

In [ ]:
customer_id_pattern = r"^CUS_0x[0-9a-fA-F]+$"

customer_id_format = pd.DataFrame({
    "Customer_ID": df["Customer_ID"],
    "Valid_Format": df["Customer_ID"]
    .astype("string")
    .str.fullmatch(customer_id_pattern, na=False)
})

print(
    f"Jumlah format valid   : "
    f"{customer_id_format['Valid_Format'].sum():,}"
)

print(
    f"Jumlah format invalid : "
    f"{(~customer_id_format['Valid_Format']).sum():,}"
)

display(
    customer_id_format[
        ~customer_id_format["Valid_Format"]
    ]
)

Perubahan datatype ID tidak akan memengaruhi apa apa dan tidak ada format yang anomali di kolom ini

### d. Name

In [ ]:
df["Name"].value_counts(dropna=False)

In [ ]:
print(f"Jumlah nama unik : {df['Name'].nunique():,}")
print(f"Jumlah missing   : {df['Name'].isna().sum():,}")

Perubahan datatype ID tidak akan memengaruhi apa apa dan tidak ada format yang anomali di kolom ini

### e. SSN

In [ ]:
df["SSN"].value_counts(dropna=False)

Perubahan datatype ID tidak akan memengaruhi apa apa dan terdapat format yang anomali di kolom ini (asumsi awal saya itu adalah misisng value)

## 2. Kolom Waktu dan Profil Pelanggan

### a. Month

In [ ]:
df["Month"].value_counts()

Kolom ini Aman

###  b. Age

In [ ]:
age_counts = (
    df["Age"]
    .value_counts(dropna=False)
    .rename_axis("Age")
    .reset_index(name="Count")
)

age_counts["Age_Numeric"] = pd.to_numeric(
    age_counts["Age"]
    .astype("string")
    .str.replace("_", "", regex=False),
    errors="coerce"
)

age_counts = (
    age_counts
    .sort_values(
        by="Age_Numeric",
        ascending=True
    )
    .reset_index(drop=True)
)

display(age_counts)

In [ ]:
# CLEAN AGE FORMAT
df["Age"] = (
    df["Age"]
    .astype("string")
    .str.strip()
    .str.replace("_", "", regex=False)
)

df["Age"] = pd.to_numeric(
    df["Age"],
    errors="coerce"
).astype("Int64")

print(f"Tipe data Age setelah konversi: {df['Age'].dtype}")
print(f"Jumlah missing Age            : {df['Age'].isna().sum():,}")

In [ ]:
# VALUE COUNT AGE AFTER CONVERSION
age_counts = (
    df["Age"]
    .value_counts(dropna=False)
    .rename_axis("Age")
    .reset_index(name="Count")
    .sort_values(
        by="Age",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(age_counts)

Pemeriksaan awal kolom `Age` dilakukan menggunakan `value_counts()` untuk melihat seluruh nilai dan frekuensinya. Hasilnya menunjukkan adanya format yang tidak konsisten, seperti `14_` dan `35_`, serta nilai yang tidak masuk akal, seperti `-500`, `1817`, dan `8697`. Oleh karena itu, karakter underscore dihapus terlebih dahulu, lalu tipe data `Age` diubah dari `object` menjadi numerik. Setelah format diperbaiki, nilai usia di bawah 1 tahun atau di atas 80 tahun ditandai sebagai tidak valid dan diperiksa lebih lanjut menggunakan catatan dari `Customer_ID` yang sama pada bulan lain.


In [ ]:
# CHECK INVALID AGE VALUES
invalid_age = df[
    (df["Age"] < 1)
    | (df["Age"] > 80)
].copy()

print(f"Jumlah Age < 1    : {(df['Age'] < 1).sum():,}")
print(f"Jumlah Age > 80   : {(df['Age'] > 80).sum():,}")
print(f"Total tidak valid : {len(invalid_age):,}")

display(invalid_age)

In [ ]:
# CHECK VALID AGE REFERENCES
valid_age_mask = df["Age"].between(1, 80)

invalid_age_mask = (
    df["Age"].isna()
    | (df["Age"] < 1)
    | (df["Age"] > 80)
)

valid_age_reference = (
    df.loc[valid_age_mask]
    .groupby("Customer_ID")["Age"]
    .agg(
        Valid_Age_Count="count",
        Valid_Age_Values=lambda values: sorted(
            values.dropna().unique().tolist()
        ),
        Suggested_Age=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_age_check = (
    df.loc[
        invalid_age_mask,
        ["ID", "Customer_ID", "Month", "Age"]
    ]
    .merge(
        valid_age_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_age_check["Has_Valid_Age_Reference"] = (
    invalid_age_check["Valid_Age_Count"].notna()
)

print(f"Total usia tidak valid: {len(invalid_age_check):,}")

print(
    "Memiliki referensi usia valid dari bulan lain: "
    f"{invalid_age_check['Has_Valid_Age_Reference'].sum():,}"
)

print(
    "Tidak memiliki referensi usia valid: "
    f"{(~invalid_age_check['Has_Valid_Age_Reference']).sum():,}"
)

display(invalid_age_check)

In [ ]:
display(
    invalid_age_check[
        invalid_age_check["Has_Valid_Age_Reference"]
    ]
)

In [ ]:
display(
    invalid_age_check[
        ~invalid_age_check["Has_Valid_Age_Reference"]
    ]
)

Berdasarkan hasil pemeriksaan, terdapat **702 baris** dengan nilai `Age` yang tidak valid, yaitu nilai usia di bawah 1 tahun atau di atas 80 tahun. Nilai seperti `-500`, `1817`, `2756`, `6579`, hingga `8200` tidak mungkin merepresentasikan usia nasabah yang sebenarnya.

Dari seluruh nilai usia yang tidak valid tersebut, hasil pemeriksaan dibagi menjadi dua kelompok:

- **592 baris atau sekitar 84,33%** memiliki referensi usia valid dari catatan bulan lain milik `Customer_ID` yang sama.
- **110 baris atau sekitar 15,67%** tidak memiliki referensi usia valid dari nasabah yang sama.

Untuk 592 baris yang memiliki referensi usia valid, nilai usia dapat diperbaiki menggunakan catatan milik nasabah yang sama pada bulan lain.

Beberapa contohnya adalah:

- `Customer_ID = CUS_ANON_033` memiliki nilai usia `-500` pada bulan Mei. Pada catatan bulan lainnya, usia nasabah tersebut tercatat konsisten sebesar `41`, sehingga nilai `-500` dapat diperbaiki menjadi `41`.

- `Customer_ID = CUS_ANON_031` memiliki nilai usia `2756` pada bulan Juli. Catatan lain dari nasabah yang sama menunjukkan usia valid sebesar `20`, sehingga nilai tersebut dapat diperbaiki menjadi `20`.

- `Customer_ID = CUS_ANON_022` memiliki nilai usia `2318` pada bulan Januari. Pada bulan lainnya, usia nasabah tersebut tercatat sebesar `15`, sehingga nilai yang tidak valid dapat diganti menjadi `15`.

Beberapa nasabah memiliki lebih dari satu nilai usia valid yang berbeda satu tahun.

Contohnya:

- `Customer_ID = CUS_ANON_011` memiliki nilai usia tidak valid sebesar `1817`, sedangkan nilai usia valid pada bulan lainnya adalah `37` dan `38`.

- `Customer_ID = CUS_ANON_030` memiliki nilai usia tidak valid sebesar `-500`, sedangkan nilai usia valid pada bulan lainnya adalah `47` dan `48`.

Perbedaan satu tahun masih dapat dianggap wajar karena periode pencatatan berlangsung dari Januari hingga Agustus. Nasabah mungkin berulang tahun selama periode tersebut. Oleh karena itu, nilai pengganti pada kasus seperti ini sebaiknya mempertimbangkan urutan bulan pencatatan dan tidak hanya menggunakan modus secara otomatis.

Sementara itu, 110 baris tidak memiliki referensi usia valid karena seluruh catatan usia milik nasabah tersebut juga tidak masuk akal.

Contohnya:

- `Customer_ID = CUS_ANON_035` memiliki nilai usia `-500` pada bulan Juni dan `8200` pada bulan Februari. Karena kedua nilainya tidak valid, tidak tersedia usia yang dapat digunakan sebagai referensi.

- `Customer_ID = CUS_ANON_026` memiliki nilai usia `3463` dan tidak memiliki catatan usia valid pada bulan lainnya.

Strategi penanganan kolom `Age` adalah:

1. Mempertahankan kolom `Age` karena masih berpotensi memberikan informasi yang relevan terhadap performa kredit nasabah.
2. Mengganti nilai usia tidak valid yang memiliki referensi dengan usia valid dari `Customer_ID` yang sama.
3. Mempertimbangkan urutan bulan apabila nasabah memiliki dua nilai usia valid yang berbeda satu tahun.
4. Mengubah nilai usia yang tidak memiliki referensi menjadi missing value.
5. Mengimputasi missing value yang tersisa menggunakan nilai median yang dipelajari hanya dari data training.

Pendekatan ini lebih baik daripada menghapus seluruh kolom `Age` atau menghapus 702 baris karena sebagian besar nilai yang tidak valid masih dapat dipulihkan tanpa kehilangan banyak informasi.

In [ ]:
# REPAIR INVALID AGE VALUES

month_order = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8
}


def repair_age_values(data):
    data = data.copy()

    data["_Month_Number"] = data["Month"].map(month_order)

    valid_age_mask = data["Age"].between(1, 80)

    invalid_age_mask = (
        data["Age"].isna()
        | (data["Age"] < 1)
        | (data["Age"] > 80)
    )

    total_invalid = invalid_age_mask.sum()

    def find_customer_age(row):
        customer_valid_ages = data.loc[
            (data["Customer_ID"] == row["Customer_ID"])
            & valid_age_mask,
            ["Age", "_Month_Number"]
        ].dropna()

        if customer_valid_ages.empty:
            return np.nan

        customer_valid_ages = customer_valid_ages.copy()

        customer_valid_ages["Month_Distance"] = (
            customer_valid_ages["_Month_Number"]
            - row["_Month_Number"]
        ).abs()

        customer_valid_ages["Is_Future_Month"] = (
            customer_valid_ages["_Month_Number"]
            > row["_Month_Number"]
        ).astype(int)

        nearest_record = customer_valid_ages.sort_values(
            by=[
                "Month_Distance",
                "Is_Future_Month",
                "_Month_Number"
            ]
        ).iloc[0]

        return nearest_record["Age"]

    data.loc[invalid_age_mask, "Age"] = (
        data.loc[invalid_age_mask]
        .apply(find_customer_age, axis=1)
    )

    repaired_count = (
        data.loc[invalid_age_mask, "Age"]
        .notna()
        .sum()
    )

    unresolved_count = (
        data.loc[invalid_age_mask, "Age"]
        .isna()
        .sum()
    )

    data["Age"] = (
        data["Age"]
        .round()
        .astype("Int64")
    )

    data = data.drop(columns="_Month_Number")

    repair_summary = pd.DataFrame({
        "Description": [
            "Total usia tidak valid",
            "Berhasil diperbaiki dari Customer_ID yang sama",
            "Tidak memiliki referensi usia valid"
        ],
        "Count": [
            total_invalid,
            repaired_count,
            unresolved_count
        ]
    })

    return data, repair_summary

In [ ]:
df, age_repair_summary = repair_age_values(df)

display(age_repair_summary)

Fungsi `repair_age_values()` digunakan untuk menangani nilai usia yang tidak valid setelah format dan tipe data kolom `Age` selesai diperbaiki pada tahap sebelumnya. Nilai usia dianggap valid apabila berada pada rentang 1 sampai 80 tahun, sedangkan nilai di bawah 1, di atas 80, atau missing value akan ditandai sebagai data yang perlu diperbaiki.

Urutan bulan dari Januari sampai Agustus terlebih dahulu diubah menjadi angka melalui kolom sementara `_Month_Number`. Nilai ini digunakan untuk menentukan catatan bulan yang paling dekat dengan bulan tempat usia tidak valid ditemukan.

Untuk setiap baris dengan usia tidak valid, fungsi mencari nilai usia valid dari `Customer_ID` yang sama pada bulan lain. Jika terdapat beberapa referensi, fungsi menghitung jarak antarbulan dan memilih catatan dengan jarak waktu paling dekat. Apabila terdapat dua bulan dengan jarak yang sama, catatan dari bulan sebelumnya akan diprioritaskan.

Sebagai contoh, apabila usia tidak valid ditemukan pada bulan Mei, sedangkan nasabah yang sama memiliki usia valid pada bulan April dan Agustus, maka usia pada bulan April akan dipilih karena memiliki jarak waktu yang lebih dekat.

Apabila nasabah tidak memiliki satu pun catatan usia yang valid, fungsi akan menghasilkan nilai `NaN`. Nilai tersebut tidak langsung diisi karena proses imputasi median akan dilakukan setelah pembagian data training dan testing.

Setelah proses perbaikan selesai, kolom `Age` dibulatkan dan disimpan menggunakan tipe data `Int64`, sehingga tetap dapat menampung bilangan bulat dan missing value. Kolom sementara `_Month_Number` kemudian dihapus.

Fungsi menghasilkan dua output, yaitu dataframe yang nilai usianya telah diperbaiki dan dataframe ringkasan yang menampilkan jumlah usia tidak valid, jumlah yang berhasil diperbaiki menggunakan riwayat nasabah yang sama, serta jumlah yang masih belum memiliki referensi usia valid.


In [ ]:
# CHECK AGE AFTER REPAIR

print(f"Tipe data Age          : {df['Age'].dtype}")
print(f"Jumlah missing Age     : {df['Age'].isna().sum():,}")
print(f"Jumlah Age di bawah 1  : {(df['Age'] < 1).sum():,}")
print(f"Jumlah Age di atas 80  : {(df['Age'] > 80).sum():,}")

display(
    df[
        df["Age"].isna()
    ][
        ["ID", "Customer_ID", "Month", "Age"]
    ]
)

Untuk umur yang kalau di di indonesia belum mungkin melakukan proses kredit(<21) nanti akan di cek ulang saat melihat hubungan antar variabell, yaitu bagian history_credit_age dengan age

### c. Occupation

In [ ]:
occupation_counts = (
    df["Occupation"]
    .value_counts(dropna=False)
    .rename_axis("Occupation")
    .reset_index(name="Count")
    .sort_values("Occupation")
    .reset_index(drop=True)
)

display(occupation_counts)

In [ ]:
# CHECK VALID OCCUPATION FROM THE SAME CUSTOMER
invalid_occupation_mask = (
    df["Occupation"].isna()
    | df["Occupation"]
    .astype("string")
    .eq("_______")
    .fillna(False)
)

valid_occupation_mask = ~invalid_occupation_mask

valid_occupation_reference = (
    df.loc[valid_occupation_mask]
    .groupby("Customer_ID")["Occupation"]
    .agg(
        Valid_Occupation_Count="count",
        Valid_Occupation_Values=lambda values: sorted(
            values.dropna().unique().tolist()
        ),
        Suggested_Occupation=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_occupation_check = (
    df.loc[
        invalid_occupation_mask,
        ["ID", "Customer_ID", "Month", "Occupation"]
    ]
    .merge(
        valid_occupation_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_occupation_check[
    "Has_Valid_Occupation_Reference"
] = (
    invalid_occupation_check[
        "Valid_Occupation_Count"
    ].notna()
)

print(
    f"Total Occupation tidak valid: "
    f"{len(invalid_occupation_check):,}"
)

print(
    "Memiliki referensi Occupation valid dari bulan lain: "
    f"{invalid_occupation_check['Has_Valid_Occupation_Reference'].sum():,}"
)

print(
    "Tidak memiliki referensi Occupation valid: "
    f"{(~invalid_occupation_check['Has_Valid_Occupation_Reference']).sum():,}"
)

display(invalid_occupation_check)

In [ ]:
display(
    invalid_occupation_check[
        invalid_occupation_check[
            "Has_Valid_Occupation_Reference"
        ]
    ]
)

In [ ]:
display(
    invalid_occupation_check[
        ~invalid_occupation_check[
            "Has_Valid_Occupation_Reference"
        ]
    ]
)

In [ ]:
# CHECK OCCUPATION CONSISTENCY PER CUSTOMER
occupation_consistency = (
    df.loc[valid_occupation_mask]
    .groupby("Customer_ID")["Occupation"]
    .nunique()
)

print(
    "Customer_ID dengan lebih dari satu Occupation valid: "
    f"{(occupation_consistency > 1).sum():,}"
)

In [ ]:
# FILL OCCUPATION FROM THE SAME CUSTOMER
invalid_occupation_mask = (
    df["Occupation"].isna()
    | df["Occupation"]
    .astype("string")
    .eq("_______")
    .fillna(False)
)

df.loc[
    invalid_occupation_mask,
    "Occupation"
] = pd.NA

occupation_reference = (
    df.dropna(subset=["Occupation"])
    .groupby("Customer_ID")["Occupation"]
    .agg(lambda values: values.mode().iloc[0])
)

fillable_mask = (
    df["Occupation"].isna()
    & df["Customer_ID"].isin(occupation_reference.index)
)

df.loc[
    fillable_mask,
    "Occupation"
] = (
    df.loc[fillable_mask, "Customer_ID"]
    .map(occupation_reference)
)

print(
    f"Occupation berhasil diisi dari Customer_ID yang sama: "
    f"{fillable_mask.sum():,}"
)

print(
    f"Occupation yang masih missing: "
    f"{df['Occupation'].isna().sum():,}"
)


Pemeriksaan awal kolom `Occupation` dilakukan menggunakan `value_counts()` untuk melihat seluruh kategori pekerjaan beserta jumlah kemunculannya. Hasil pemeriksaan menunjukkan terdapat 15 kategori pekerjaan yang valid, yaitu `Accountant`, `Architect`, `Developer`, `Doctor`, `Engineer`, `Entrepreneur`, `Journalist`, `Lawyer`, `Manager`, `Mechanic`, `Media_Manager`, `Musician`, `Scientist`, `Teacher`, dan `Writer`.



Selain kategori pekerjaan yang valid, ditemukan nilai `_______` sebanyak **1.765 baris**. Nilai tersebut tidak merepresentasikan jenis pekerjaan tertentu sehingga dianggap sebagai **hidden missing value**, yaitu nilai kosong yang tidak terdeteksi sebagai `NaN` oleh pandas karena masih tersimpan dalam bentuk teks.



Sebelum nilai tersebut langsung diimputasi, dilakukan pemeriksaan berdasarkan `Customer_ID`. Pemeriksaan ini bertujuan untuk mengetahui apakah nasabah yang memiliki nilai `Occupation = _______` mempunyai catatan pekerjaan yang valid pada bulan lain. Pemeriksaan tersebut dapat dilakukan karena satu nasabah dapat muncul beberapa kali pada periode bulan yang berbeda.



Sebagai contoh:



| Customer_ID | Month    | Occupation |

| ----------- | -------- | ---------- |

| CUS_001     | January  | Engineer   |

| CUS_001     | February | `_______`  |

| CUS_001     | March    | Engineer   |



Pada contoh tersebut, nilai pekerjaan bulan Februari dapat diperbaiki menjadi `Engineer` karena catatan bulan lain dari nasabah yang sama menunjukkan pekerjaan yang konsisten.



Hasil pemeriksaan terhadap **1.765 nilai pekerjaan yang tidak valid** menunjukkan bahwa:



* **1.506 baris** memiliki referensi pekerjaan valid dari `Customer_ID` yang sama pada bulan lain.

* **259 baris** tidak memiliki referensi pekerjaan valid dari nasabah yang sama.



Selanjutnya, dilakukan pemeriksaan konsistensi pekerjaan untuk memastikan bahwa satu `Customer_ID` tidak memiliki beberapa kategori pekerjaan valid yang berbeda. Hasil pemeriksaan menunjukkan bahwa tidak terdapat nasabah yang memiliki lebih dari satu pekerjaan valid. Dengan demikian, pekerjaan yang tercatat pada bulan lain dapat digunakan sebagai referensi secara lebih aman.



Nilai `_______` kemudian terlebih dahulu diubah menjadi missing value. Setelah itu, dibuat pemetaan antara `Customer_ID` dan pekerjaan valid yang dimiliki oleh masing-masing nasabah. Pemetaan tersebut hanya digunakan untuk mengisi baris yang memiliki nilai pekerjaan tidak valid atau kosong.



Sebanyak **1.506 nilai `Occupation` berhasil diperbaiki** menggunakan pekerjaan valid dari `Customer_ID` yang sama. Setelah proses tersebut selesai, masih terdapat **259 nilai `Occupation` yang kosong** karena nasabah tersebut tidak memiliki catatan pekerjaan valid pada bulan lain.



Nilai pekerjaan yang masih kosong tidak langsung dipaksakan menjadi salah satu pekerjaan tertentu. Pada tahap preprocessing berikutnya, nilai tersebut dapat dipertahankan sebagai kategori `Unknown` agar model dapat membedakan nasabah yang informasi pekerjaannya tidak tersedia, tanpa mengasumsikan bahwa nasabah tersebut memiliki pekerjaan tertentu.




## 3. Kolom Pendapatan dan Kondisi Arus Kas

### a. Annual Income

In [ ]:
# CHECK ANNUAL INCOME DATA TYPE
print(f"Tipe data Annual_Income: {df['Annual_Income'].dtype}")

In [ ]:
# CHECK SPECIAL CHARACTERS IN ANNUAL INCOME

special_characters = (
    df["Annual_Income"]
    .astype("string")
    .str.findall(r"[^0-9.\-]")
    .explode()
    .value_counts()
)

display(special_characters)

In [ ]:
underscore_count = (
    df["Annual_Income"]
    .astype("string")
    .str.contains("_", regex=False, na=False)
    .sum()
)

print(f"Jumlah Annual_Income dengan underscore: {underscore_count:,}")

In [ ]:
# CLEAN ANNUAL INCOME FORMAT
df["Annual_Income"] = (
    df["Annual_Income"]
    .astype("string")
    .str.strip()
    .str.replace("_", "", regex=False)
)

df["Annual_Income"] = pd.to_numeric(
    df["Annual_Income"],
    errors="coerce"
)

# CHECK RESULT

underscore_count = (
    df["Annual_Income"]
    .astype("string")
    .str.contains("_", regex=False, na=False)
    .sum()
)

print(f"Tipe data Annual_Income      : {df['Annual_Income'].dtype}")
print(f"Jumlah underscore tersisa    : {underscore_count:,}")
print(f"Jumlah missing setelah proses: {df['Annual_Income'].isna().sum():,}")

display(
    df[["Annual_Income"]].head()
)

### b. Monthly_Inhand_Salary

In [ ]:
# VALUE COUNT MONTHLY INHAND SALARY
monthly_salary_counts = (
    df["Monthly_Inhand_Salary"]
    .value_counts(dropna=False)
    .rename_axis("Monthly_Inhand_Salary")
    .reset_index(name="Count")
    .sort_values(
        by="Monthly_Inhand_Salary",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(monthly_salary_counts)

Kolom ini aman untuk segi format dan tipe data, sisa missing value

### c. Ammount_invested_month

In [ ]:
# CHECK DATA TYPE

df["Amount_invested_monthly"].dtype

In [ ]:
# VALUE COUNT AMOUNT INVESTED MONTHLY

amount_invested_counts = (
    df["Amount_invested_monthly"]
    .value_counts(dropna=False)
    .rename_axis("Amount_invested_monthly")
    .reset_index(name="Count")
)

display(amount_invested_counts)

In [ ]:
# CHECK __10000__ REFERENCE FROM THE SAME CUSTOMER
invalid_investment_mask = (
    df["Amount_invested_monthly"]
    .astype("string")
    .eq("__10000__")
)

valid_investment_reference = (
    df.loc[~invalid_investment_mask]
    .groupby("Customer_ID")["Amount_invested_monthly"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda x: sorted(
            pd.to_numeric(x, errors="coerce")
            .dropna()
            .unique()
            .tolist()
        )
    )
    .reset_index()
)

investment_check = (
    df.loc[
        invalid_investment_mask,
        ["ID", "Customer_ID", "Month", "Amount_invested_monthly"]
    ]
    .merge(
        valid_investment_reference,
        on="Customer_ID",
        how="left"
    )
)

investment_check["Has_Valid_Reference"] = (
    investment_check["Valid_Count"].notna()
)

print(f"Total __10000__ : {len(investment_check):,}")

print(
    f"Memiliki referensi valid : "
    f"{investment_check['Has_Valid_Reference'].sum():,}"
)

print(
    f"Tidak memiliki referensi : "
    f"{(~investment_check['Has_Valid_Reference']).sum():,}"
)

display(investment_check)

In [ ]:
# REPAIR AMOUNT INVESTED MONTHLY
column = "Amount_invested_monthly"

raw_investment = df[column].astype("string")

invalid_investment_mask = raw_investment.eq("__10000__")

investment_numeric = pd.to_numeric(
    raw_investment
    .mask(invalid_investment_mask)
    .str.replace("_", "", regex=False),
    errors="coerce"
)

customer_investment_stats = (
    pd.DataFrame({
        "Customer_ID": df["Customer_ID"],
        "Investment": investment_numeric
    })
    .groupby("Customer_ID")["Investment"]
    .agg(["count", "mean"])
)

reference_count = df["Customer_ID"].map(
    customer_investment_stats["count"]
)

reference_mean = df["Customer_ID"].map(
    customer_investment_stats["mean"]
)

fillable_mask = (
    invalid_investment_mask
    & reference_count.gt(0)
)

investment_numeric.loc[fillable_mask] = (
    reference_mean.loc[fillable_mask]
)

df[column] = investment_numeric.astype("float64")

repair_summary = pd.DataFrame({
    "Description": [
        "Total nilai __10000__",
        "Memiliki lebih dari satu referensi valid",
        "Memiliki satu referensi valid",
        "Berhasil diperbaiki",
        "Tidak memiliki referensi valid",
        "Total missing setelah perbaikan"
    ],
    "Count": [
        invalid_investment_mask.sum(),
        (invalid_investment_mask & reference_count.gt(1)).sum(),
        (invalid_investment_mask & reference_count.eq(1)).sum(),
        fillable_mask.sum(),
        (invalid_investment_mask & reference_count.eq(0)).sum(),
        df[column].isna().sum()
    ]
})

display(repair_summary)

print(f"Tipe data setelah perbaikan: {df[column].dtype}")

Kode tersebut digunakan untuk membersihkan dan memperbaiki kolom `Amount_invested_monthly`, khususnya nilai `__10000__` yang dianggap sebagai data tidak valid. Nilai tersebut terlebih dahulu diubah menjadi missing value, karakter underscore dihapus, lalu kolom dikonversi menjadi numerik.

Setelah itu, data dikelompokkan berdasarkan `Customer_ID` untuk menghitung jumlah dan rata-rata nilai investasi valid milik setiap nasabah. Nilai `__10000__` kemudian diganti menggunakan rata-rata investasi valid dari nasabah yang sama. Jika hanya terdapat satu referensi, nilai tersebut digunakan langsung.

Nilai yang tidak memiliki referensi tetap menjadi missing value dan akan ditangani pada tahap preprocessing berikutnya. Terakhir, kolom diubah menjadi tipe `float64` dan dibuat ringkasan jumlah nilai yang berhasil diperbaiki, tidak memiliki referensi, serta total missing value yang masih tersisa.


Hasil perbaikan menunjukkan bahwa terdapat **1.049 nilai `__10000__`** pada kolom `Amount_invested_monthly` yang dianggap sebagai nilai tidak valid. Dari jumlah tersebut, **524 baris** memiliki lebih dari satu referensi investasi valid dari `Customer_ID` yang sama, sedangkan **345 baris** memiliki satu referensi valid.

Secara keseluruhan, sebanyak **869 nilai berhasil diperbaiki** menggunakan rata-rata investasi valid milik nasabah yang sama. Sementara itu, **180 nilai tidak memiliki referensi valid**, sehingga tetap menjadi missing value.

Setelah proses perbaikan, total missing value pada kolom ini menjadi **1.314 baris**, yang terdiri atas **1.134 missing value asli** dan **180 nilai `__10000__` yang tidak dapat diperbaiki**. Missing value tersebut akan ditangani pada tahap preprocessing berikutnya menggunakan metode imputasi yang dipelajari dari data training.


In [ ]:
# VALUE COUNT AMOUNT INVESTED MONTHLY AFTER REPAIR

amount_invested_counts = (
    df["Amount_invested_monthly"]
    .value_counts(dropna=False)
    .rename_axis("Amount_invested_monthly")
    .reset_index(name="Count")
    .sort_values(
        by="Amount_invested_monthly",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(amount_invested_counts)

Sudah aman, sisa missing value aja nanti kita bersihkan di prepocessing selanjutnya

### d. Monthly Balance

In [ ]:
# CHECK DATA TYPE
df["Monthly_Balance"].dtype

In [ ]:
# VALUE COUNT MONTHLY BALANCE
monthly_balance_counts = (
    df["Monthly_Balance"]
    .value_counts(dropna=False)
    .rename_axis("Monthly_Balance")
    .reset_index(name="Count")
    .sort_values(
        by="Monthly_Balance",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(monthly_balance_counts)

Kolom ini bersih selain dari adanya missing value

### e. Total_EMI_per_month

In [ ]:
# VALUE COUNT TOTAL EMI PER MONTH
total_emi_counts = (
    df["Total_EMI_per_month"]
    .value_counts(dropna=False)
    .rename_axis("Total_EMI_per_month")
    .reset_index(name="Count")
    .sort_values(
        by="Total_EMI_per_month",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(total_emi_counts)

In [ ]:
print(f"Data Type     : {df['Total_EMI_per_month'].dtype}")
print(f"Missing Value : {df['Total_EMI_per_month'].isna().sum():,}")

In [ ]:
# CHECK MIN AND MAX TOTAL EMI PER MONTH

print(f"Minimum : {df['Total_EMI_per_month'].min():,.4f}")
print(f"Maximum : {df['Total_EMI_per_month'].max():,.4f}")

In [ ]:
# CHECK TOTAL EMI DISTRIBUTION

df["Total_EMI_per_month"].describe(
    percentiles=[0.90, 0.95, 0.99]
)

Kolom Total_EMI_per_month tidak memiliki masalah format karena sudah bertipe numerik. Namun, terdapat banyak nilai ekstrem, yang mana nanti akan kita cek bersamaan dengan melihat hubungannya dengan variabel lain seperti Annual Income, Monthly Inhand Salary, Num of Loan, dll.

## 4. Kolom Kepmelikian Fasilitas Kredit

### a. Num_Bank_Account

In [ ]:
# VALUE COUNT NUM BANK ACCOUNTS

bank_account_counts = (
    df["Num_Bank_Accounts"]
    .value_counts(dropna=False)
    .rename_axis("Num_Bank_Accounts")
    .reset_index(name="Count")
    .sort_values(
        by="Num_Bank_Accounts",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(bank_account_counts)

In [ ]:
# CHECK VALID BANK ACCOUNT REFERENCE
invalid_bank_mask = df["Num_Bank_Accounts"] < 0
valid_bank_mask = df["Num_Bank_Accounts"] >= 0

bank_account_reference = (
    df.loc[valid_bank_mask]
    .groupby("Customer_ID")["Num_Bank_Accounts"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(
            values.unique().tolist()
        ),
        Suggested_Value=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_bank_check = (
    df.loc[
        invalid_bank_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Num_Bank_Accounts"
        ]
    ]
    .merge(
        bank_account_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_bank_check["Has_Valid_Reference"] = (
    invalid_bank_check["Valid_Count"].notna()
)

print(
    f"Total nilai negatif: "
    f"{len(invalid_bank_check):,}"
)

print(
    f"Memiliki referensi valid: "
    f"{invalid_bank_check['Has_Valid_Reference'].sum():,}"
)

print(
    f"Tidak memiliki referensi valid: "
    f"{(~invalid_bank_check['Has_Valid_Reference']).sum():,}"
)

display(invalid_bank_check)

In [ ]:
# REPAIR NEGATIVE NUM BANK ACCOUNTS
valid_bank_reference = (
    df.loc[df["Num_Bank_Accounts"] >= 0]
    .groupby("Customer_ID")["Num_Bank_Accounts"]
    .agg(lambda values: values.mode().iloc[0])
)

invalid_bank_mask = df["Num_Bank_Accounts"] < 0
invalid_indices = df.index[invalid_bank_mask]

reference_values = (
    df.loc[invalid_indices, "Customer_ID"]
    .map(valid_bank_reference)
)

fillable_indices = reference_values[
    reference_values.notna()
].index

unresolved_indices = reference_values[
    reference_values.isna()
].index

df["Num_Bank_Accounts"] = (
    df["Num_Bank_Accounts"]
    .astype("Int64")
)

# Mengisi nilai negatif yang memiliki referensi valid
df.loc[
    fillable_indices,
    "Num_Bank_Accounts"
] = reference_values.loc[fillable_indices].astype("Int64")

# Mengubah nilai negatif tanpa referensi menjadi missing value
df.loc[
    unresolved_indices,
    "Num_Bank_Accounts"
] = pd.NA

print(f"Berhasil diisi dari Customer_ID yang sama : {len(fillable_indices):,}")
print(f"Diubah menjadi missing value              : {len(unresolved_indices):,}")
print(f"Nilai negatif yang masih tersisa          : {(df['Num_Bank_Accounts'] < 0).sum():,}")
print(f"Total missing Num_Bank_Accounts            : {df['Num_Bank_Accounts'].isna().sum():,}")

Pemeriksaan terhadap kolom `Num_Bank_Accounts` menunjukkan bahwa nilai `0` masih dapat dianggap valid karena dapat merepresentasikan nasabah yang tidak memiliki rekening bank aktif pada periode pencatatan. Namun, ditemukan **5 baris dengan nilai `-1`**, yang dianggap tidak valid karena jumlah rekening bank tidak mungkin bernilai negatif.

Sebelum nilai tersebut diubah, dilakukan pemeriksaan berdasarkan `Customer_ID` untuk mengetahui apakah nasabah yang sama memiliki jumlah rekening yang valid pada bulan lain. Hasilnya menunjukkan bahwa hanya **1 dari 5 baris** yang memiliki referensi valid, yaitu nasabah dengan nilai rekening `0` pada catatan lainnya. Nilai `-1` pada baris tersebut kemudian diperbaiki menjadi `0`.

Sementara itu, **4 baris lainnya tidak memiliki referensi jumlah rekening yang valid** dari nasabah yang sama. Oleh karena itu, nilai tersebut tidak dipaksakan menjadi angka tertentu, tetapi diubah menjadi missing value agar dapat ditangani pada tahap imputasi berikutnya.

Setelah proses perbaikan:

* 1 nilai negatif berhasil diperbaiki menggunakan referensi `Customer_ID` yang sama;
* 4 nilai negatif diubah menjadi missing value;
* tidak terdapat lagi nilai negatif pada kolom `Num_Bank_Accounts`;
* missing value yang tersisa akan diimputasi menggunakan nilai yang dipelajari hanya dari data training.

Permasalahan kolom ini sisa satu, yaitu banyak nilai tidak masuk akal

In [ ]:
# CHECK NUM BANK ACCOUNTS PERCENTILES

bank_account_percentiles = (
    df["Num_Bank_Accounts"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 1.00]
    )
    .reset_index()
)

bank_account_percentiles.columns = [
    "Percentile",
    "Num_Bank_Accounts"
]

bank_account_percentiles["Percentile"] *= 100

bank_account_percentiles["Increase"] = (
    bank_account_percentiles["Num_Bank_Accounts"].diff()
)

display(bank_account_percentiles)



In [ ]:
# CHECK VALUES AROUND NUM BANK ACCOUNTS BOUNDARY

display(
    df.loc[
        df["Num_Bank_Accounts"].between(0, 30),
        "Num_Bank_Accounts"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Num_Bank_Accounts")
    .reset_index(name="Count")
)

In [ ]:
# CHECK NUM BANK ACCOUNTS BOUNDARY

bank_account_boundary = 10

print(
    "Jumlah Num_Bank_Accounts 0–10:",
    df["Num_Bank_Accounts"].between(0, bank_account_boundary).sum()
)

print(
    "Jumlah Num_Bank_Accounts > 10:",
    (df["Num_Bank_Accounts"] > bank_account_boundary).sum()
)

print(
    "Jumlah missing Num_Bank_Accounts:",
    df["Num_Bank_Accounts"].isna().sum()
)

In [ ]:
# CHECK VALID NUM BANK ACCOUNTS REFERENCE

bank_account_boundary = 10

invalid_bank_account_mask = (
    df["Num_Bank_Accounts"] > bank_account_boundary
)

valid_bank_account_mask = (
    df["Num_Bank_Accounts"]
    .between(0, bank_account_boundary)
)

bank_account_reference = (
    df.loc[valid_bank_account_mask]
    .groupby("Customer_ID")["Num_Bank_Accounts"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(values.unique().tolist()),
        Suggested_Value=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_bank_account_check = (
    df.loc[
        invalid_bank_account_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Num_Bank_Accounts"
        ]
    ]
    .merge(
        bank_account_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_bank_account_check["Has_Valid_Reference"] = (
    invalid_bank_account_check["Valid_Count"].notna()
)

print(
    "Total Num_Bank_Accounts > 10:",
    len(invalid_bank_account_check)
)

print(
    "Memiliki referensi valid 0–10:",
    invalid_bank_account_check["Has_Valid_Reference"].sum()
)

print(
    "Tidak memiliki referensi valid:",
    (~invalid_bank_account_check["Has_Valid_Reference"]).sum()
)

display(invalid_bank_account_check)

In [ ]:
# REPAIR NUM BANK ACCOUNTS EXTREME VALUES

valid_bank_account_reference = (
    df.loc[df["Num_Bank_Accounts"].between(0, 10)]
    .groupby("Customer_ID")["Num_Bank_Accounts"]
    .agg(lambda values: values.mode().iloc[0])
)

suggested_bank_account_values = (
    df.loc[invalid_bank_account_mask, "Customer_ID"]
    .map(valid_bank_account_reference)
)

fillable_bank_indices = suggested_bank_account_values[
    suggested_bank_account_values.notna()
].index

unresolved_bank_indices = suggested_bank_account_values[
    suggested_bank_account_values.isna()
].index

df.loc[
    fillable_bank_indices,
    "Num_Bank_Accounts"
] = suggested_bank_account_values.loc[fillable_bank_indices]

df.loc[
    unresolved_bank_indices,
    "Num_Bank_Accounts"
] = pd.NA

df["Num_Bank_Accounts"] = (
    df["Num_Bank_Accounts"]
    .round()
    .astype("Int64")
)

print(
    "Berhasil diperbaiki dari Customer_ID yang sama:",
    len(fillable_bank_indices)
)

print(
    "Diubah menjadi missing value:",
    len(unresolved_bank_indices)
)

print(
    "Num_Bank_Accounts < 0 yang masih tersisa:",
    (df["Num_Bank_Accounts"] < 0).sum()
)

print(
    "Num_Bank_Accounts > 10 yang masih tersisa:",
    (df["Num_Bank_Accounts"] > 10).sum()
)

print(
    "Total missing Num_Bank_Accounts:",
    df["Num_Bank_Accounts"].isna().sum()
)

In [ ]:
# CHECK FINAL NUM BANK ACCOUNTS

df["Num_Bank_Accounts"].value_counts(
    dropna=False
).sort_index()

### b. Num_Credit_Card

In [ ]:
# VALUE COUNT NUM CREDIT CARD
credit_card_counts = (
    df["Num_Credit_Card"]
    .value_counts(dropna=False)
    .rename_axis("Num_Credit_Card")
    .reset_index(name="Count")
    .sort_values(
        by="Num_Credit_Card",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(credit_card_counts)

In [ ]:
# CHECK NUM CREDIT CARD PERCENTILES

credit_card_percentiles = (
    df["Num_Credit_Card"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 1.00]
    )
    .reset_index()
)

credit_card_percentiles.columns = [
    "Percentile",
    "Num_Credit_Card"
]

credit_card_percentiles["Percentile"] *= 100

credit_card_percentiles["Increase"] = (
    credit_card_percentiles["Num_Credit_Card"].diff()
)

display(credit_card_percentiles)

In [ ]:
# CHECK VALUES AROUND NUM CREDIT CARD BOUNDARY

display(
    df.loc[
        df["Num_Credit_Card"].between(0, 30),
        "Num_Credit_Card"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Num_Credit_Card")
    .reset_index(name="Count")
)

In [ ]:
df[df["Num_Credit_Card"] > 11]

In [ ]:
# VALUE COUNT NUM CREDIT CARD ABOVE 25

credit_card_above_25_counts = (
    df.loc[
        df["Num_Credit_Card"] > 25,
        "Num_Credit_Card"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Num_Credit_Card")
    .reset_index(name="Count")
)

display(credit_card_above_25_counts)

In [ ]:
# CHECK VALID NUM CREDIT CARD REFERENCE

credit_card_boundary = 25

invalid_credit_card_mask = (
    df["Num_Credit_Card"] > credit_card_boundary
)

valid_credit_card_mask = (
    df["Num_Credit_Card"]
    .between(0, credit_card_boundary)
)

credit_card_reference = (
    df.loc[valid_credit_card_mask]
    .groupby("Customer_ID")["Num_Credit_Card"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(values.unique().tolist()),
        Suggested_Value=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_credit_card_check = (
    df.loc[
        invalid_credit_card_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Num_Credit_Card"
        ]
    ]
    .merge(
        credit_card_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_credit_card_check["Has_Valid_Reference"] = (
    invalid_credit_card_check["Valid_Count"].notna()
)

print(
    "Total Num_Credit_Card > 25:",
    len(invalid_credit_card_check)
)

print(
    "Memiliki referensi valid 0–25:",
    invalid_credit_card_check["Has_Valid_Reference"].sum()
)

print(
    "Tidak memiliki referensi valid:",
    (~invalid_credit_card_check["Has_Valid_Reference"]).sum()
)

display(invalid_credit_card_check)

In [ ]:
# REPAIR NUM CREDIT CARD VALUES

valid_credit_card_reference = (
    df.loc[df["Num_Credit_Card"].between(0, 25)]
    .groupby("Customer_ID")["Num_Credit_Card"]
    .agg(lambda values: values.mode().iloc[0])
)

suggested_credit_card_values = (
    df.loc[invalid_credit_card_mask, "Customer_ID"]
    .map(valid_credit_card_reference)
)

fillable_credit_card_indices = suggested_credit_card_values[
    suggested_credit_card_values.notna()
].index

unresolved_credit_card_indices = suggested_credit_card_values[
    suggested_credit_card_values.isna()
].index

# Mengisi nilai ekstrem yang memiliki referensi valid
df.loc[
    fillable_credit_card_indices,
    "Num_Credit_Card"
] = suggested_credit_card_values.loc[fillable_credit_card_indices]

# Mengubah nilai ekstrem tanpa referensi menjadi missing value
df.loc[
    unresolved_credit_card_indices,
    "Num_Credit_Card"
] = pd.NA

df["Num_Credit_Card"] = (
    df["Num_Credit_Card"]
    .round()
    .astype("Int64")
)

print(
    "Berhasil diperbaiki dari Customer_ID yang sama:",
    len(fillable_credit_card_indices)
)

print(
    "Diubah menjadi missing value:",
    len(unresolved_credit_card_indices)
)

print(
    "Num_Credit_Card > 25 yang masih tersisa:",
    (df["Num_Credit_Card"] > 25).sum()
)

print(
    "Total missing Num_Credit_Card:",
    df["Num_Credit_Card"].isna().sum()
)

In [ ]:
# CHECK FINAL NUM CREDIT CARD

df["Num_Credit_Card"].value_counts(
    dropna=False
).sort_index()

Kolom `Num_Credit_Card` menunjukkan jumlah kartu kredit yang dimiliki oleh setiap nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat variasi nilai dan frekuensi kemunculannya.

Dari pemeriksaan awal, kolom `Num_Credit_Card` sudah berbentuk numerik sehingga tidak memerlukan pembersihan format seperti penghapusan underscore atau konversi dari teks ke angka. Selanjutnya, dilakukan pemeriksaan terhadap kemungkinan nilai yang tidak masuk akal, seperti nilai negatif atau nilai yang terlalu ekstrem.

Nilai negatif pada kolom ini tidak valid karena jumlah kartu kredit tidak mungkin kurang dari nol. Berdasarkan pemeriksaan, tidak ditemukan nilai negatif pada kolom `Num_Credit_Card`, sehingga tidak diperlukan penanganan khusus untuk nilai negatif. Nilai `0` tetap dipertahankan karena masih memungkinkan jika nasabah tidak memiliki kartu kredit.

Setelah itu, distribusi nilai dianalisis menggunakan persentil untuk menentukan batas investigasi berdasarkan pola data. Hasil analisis persentil menunjukkan bahwa mayoritas data berada pada rentang yang relatif kecil. Nilai tengah atau persentil ke-50 berada di sekitar `5`, persentil ke-75 berada di sekitar `7`, persentil ke-90 berada di sekitar `9`, dan persentil tinggi sampai sekitar persentil ke-99,5 masih berada di sekitar `10`.

Namun, nilai maksimum mencapai `1.498`, sehingga terdapat lonjakan yang sangat besar dari kelompok utama data menuju nilai ekstrem. Hal ini menunjukkan bahwa sebagian kecil nilai pada kolom ini tidak mengikuti pola distribusi utama.

Nilai `11`, `15`, `16`, `18`, dan `25` kemudian diperiksa lebih lanjut. Meskipun nilai tersebut jarang muncul, nilai-nilai tersebut masih mungkin terjadi dalam konteks dunia nyata karena seseorang masih mungkin memiliki jumlah kartu kredit yang cukup banyak. Oleh karena itu, proses pembersihan tidak dibuat terlalu agresif dengan langsung menganggap nilai di atas `10` sebagai salah.

Berdasarkan pertimbangan tersebut, nilai `25` digunakan sebagai batas investigasi untuk kolom `Num_Credit_Card`. Batas ini bukan batas maksimum universal jumlah kartu kredit, melainkan batas yang dipilih berdasarkan pola distribusi dataset. Nilai sampai `25` masih dipertahankan, sedangkan nilai lebih dari `25` ditandai sebagai nilai ekstrem yang perlu diperiksa lebih lanjut.

Pemeriksaan terhadap nilai `Num_Credit_Card > 25` menunjukkan bahwa nilai setelah batas tersebut langsung melonjak ke angka seperti `37`, `44`, `56`, kemudian ratusan hingga `1.498`. Sebagian besar nilai tersebut hanya muncul satu kali dan tidak membentuk pola distribusi yang natural. Oleh karena itu, nilai di atas `25` dianggap sebagai indikasi anomali.

Nilai ekstrem tersebut tidak langsung dihapus. Setiap baris dengan `Num_Credit_Card > 25` diperiksa berdasarkan `Customer_ID` untuk mengetahui apakah nasabah yang sama memiliki nilai jumlah kartu kredit yang valid pada bulan lain.

Nilai valid yang digunakan sebagai referensi adalah nilai pada rentang `0–25`. Dari catatan valid milik nasabah yang sama, dibuat beberapa informasi pendukung:

* `Valid_Count`, yaitu jumlah catatan valid yang tersedia;
* `Valid_Values`, yaitu daftar nilai valid yang dimiliki nasabah;
* `Suggested_Value`, yaitu nilai yang paling sering muncul atau modus;
* `Has_Valid_Reference`, yaitu penanda apakah nilai ekstrem memiliki referensi valid.

Modus digunakan sebagai nilai pengganti karena `Num_Credit_Card` merupakan fitur jumlah atau data diskrit. Penggunaan modus lebih sesuai dibandingkan rata-rata karena rata-rata dapat menghasilkan nilai desimal, sedangkan jumlah kartu kredit seharusnya berupa bilangan bulat.

Nilai `Num_Credit_Card > 25` yang memiliki referensi valid dari `Customer_ID` yang sama kemudian diperbaiki menggunakan `Suggested_Value`. Sementara itu, nilai ekstrem yang tidak memiliki referensi valid tidak dipaksakan untuk diisi menggunakan nilai tertentu, tetapi diubah menjadi missing value.

Setelah proses perbaikan selesai, kolom `Num_Credit_Card` dikonversi kembali menjadi tipe data `Int64`. Tipe data ini dipilih karena jumlah kartu kredit harus berupa bilangan bulat, tetapi kolom tetap perlu mendukung missing value.

Pemeriksaan akhir dilakukan untuk memastikan bahwa:

* tidak terdapat nilai negatif pada kolom `Num_Credit_Card`;
* nilai ekstrem di atas `25` sudah ditangani;
* nilai yang memiliki referensi valid berhasil diperbaiki;
* nilai tanpa referensi valid telah diubah menjadi missing value;
* tipe data akhir kolom menjadi `Int64`.

Missing value yang masih tersisa tidak langsung diisi pada tahap eksplorasi data. Nilai tersebut akan ditangani pada pipeline preprocessing menggunakan metode imputasi yang hanya dipelajari dari data training agar tidak terjadi kebocoran informasi dari data testing.

Secara keseluruhan, proses penanganan kolom `Num_Credit_Card` dilakukan melalui tahapan berikut:

1. Memeriksa distribusi awal menggunakan `value_counts()`.
2. Memeriksa apakah terdapat nilai negatif.
3. Mempertahankan nilai `0` karena masih mungkin secara konteks.
4. Menganalisis persentil untuk memahami pola distribusi data.
5. Mengidentifikasi lonjakan ekstrem dari kelompok utama menuju nilai maksimum yang sangat besar.
6. Memeriksa nilai-nilai langka seperti `11`, `15`, `16`, `18`, dan `25`.
7. Menetapkan `25` sebagai batas investigasi yang tidak terlalu agresif.
8. Mengidentifikasi seluruh nilai `Num_Credit_Card > 25`.
9. Mencari referensi valid dari `Customer_ID` yang sama pada rentang `0–25`.
10. Menggunakan modus sebagai nilai pengganti.
11. Memperbaiki nilai ekstrem yang memiliki referensi valid.
12. Mengubah nilai ekstrem tanpa referensi valid menjadi missing value.
13. Mengonversi tipe data akhir menjadi `Int64`.
14. Melakukan pemeriksaan akhir terhadap nilai negatif, nilai ekstrem, missing value, dan tipe data.

Metode ini dipilih agar proses pembersihan tidak terlalu agresif dan tidak menghapus nilai yang masih mungkin benar secara konteks. Penentuan batas dilakukan berdasarkan distribusi dataset, sedangkan proses perbaikannya memanfaatkan riwayat bulanan dari nasabah yang sama.


### c. Interest Rate

In [ ]:
# VALUE COUNT INTEREST RATE
interest_rate_counts = (
    df["Interest_Rate"]
    .value_counts(dropna=False)
    .rename_axis("Interest_Rate")
    .reset_index(name="Count")
    .sort_values(
        by="Interest_Rate",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(interest_rate_counts)

In [ ]:
# ANALYZE INTEREST RATE PERCENTILES

percentile_levels = [
    0.50,
    0.75,
    0.90,
    0.95,
    0.96,
    0.97,
    0.98,
    0.99,
    1.00
]

interest_rate_percentiles = (
    df["Interest_Rate"]
    .quantile(percentile_levels)
    .reset_index()
)

interest_rate_percentiles.columns = [
    "Percentile",
    "Interest_Rate"
]

interest_rate_percentiles["Percentile"] = (
    interest_rate_percentiles["Percentile"] * 100
)

interest_rate_percentiles["Increase"] = (
    interest_rate_percentiles["Interest_Rate"].diff()
)

display(interest_rate_percentiles)

In [ ]:
# COMPARE PERCENTILE 97 AND 98
percentile_97 = df["Interest_Rate"].quantile(0.97)
percentile_98 = df["Interest_Rate"].quantile(0.98)

print(f"Persentil ke-97 : {percentile_97:.2f}")
print(f"Persentil ke-98 : {percentile_98:.2f}")
print(
    "Kenaikan nilai : "
    f"{percentile_98 - percentile_97:.2f}"
)

In [ ]:
# SET INTEREST RATE BOUNDARY FROM PERCENTILE
interest_rate_boundary = df[
    "Interest_Rate"
].quantile(0.97)

print(
    "Batas berdasarkan persentil ke-97:",
    interest_rate_boundary
)

Saya menggunakan nilai 34 sebagai batas nilai ekstrem karena bukan karena 34 digunakan sebagai batas maksimum suku bunga secara universal. Melainkan batas tersebut ditentukan berdasarkan pola distribusi dataset, karena nilai 1 sampai 34 membentuk kelompok utama yang berurutan dan mencakup sekitar 97,86% data. Setelah nilai 34 terdapat lonjakan langsung menuju 73 dan nilai-nilai berikutnya tersebar hingga ribuan. Oleh karena itu, nilai Interest_Rate > 34 ditandai sebagai indikasi anomali untuk diperiksa menggunakan catatan Customer_ID yang sama, bukan langsung dianggap salah tanpa pemeriksaan.

In [ ]:
# CHECK INTEREST RATE ABOVE PERCENTILE BOUNDARY
normal_interest_rate = df[
    df["Interest_Rate"] <= interest_rate_boundary
].copy()

extreme_interest_rate = df[
    df["Interest_Rate"] > interest_rate_boundary
].copy()

print(
    f"Jumlah Interest_Rate <= {interest_rate_boundary:.0f}: "
    f"{len(normal_interest_rate):,}"
)

print(
    f"Jumlah Interest_Rate > {interest_rate_boundary:.0f}: "
    f"{len(extreme_interest_rate):,}"
)

print(
    "Persentase nilai di atas batas: "
    f"{len(extreme_interest_rate) / len(df) * 100:.2f}%"
)

In [ ]:
# CHECK VALID INTEREST RATE REFERENCE
interest_rate_boundary = 34

invalid_interest_rate_mask = (
    df["Interest_Rate"] > interest_rate_boundary
)

valid_interest_rate_mask = (
    df["Interest_Rate"].between(
        1,
        interest_rate_boundary,
        inclusive="left"
    )
)

interest_rate_reference = (
    df.loc[valid_interest_rate_mask]
    .groupby("Customer_ID")["Interest_Rate"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(
            values.unique().tolist()
        ),
        Suggested_Value=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_interest_rate_check = (
    df.loc[
        invalid_interest_rate_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Interest_Rate"
        ]
    ]
    .merge(
        interest_rate_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_interest_rate_check["Has_Valid_Reference"] = (
    invalid_interest_rate_check["Valid_Count"].notna()
)

print(
    "Total Interest_Rate > 34: "
    f"{len(invalid_interest_rate_check):,}"
)

print(
    "Memiliki referensi valid < 34: "
    f"{invalid_interest_rate_check['Has_Valid_Reference'].sum():,}"
)

print(
    "Tidak memiliki referensi valid: "
    f"{(~invalid_interest_rate_check['Has_Valid_Reference']).sum():,}"
)

display(invalid_interest_rate_check)

In [ ]:
# REPAIR INTEREST RATE VALUES
interest_rate_boundary = 34

invalid_interest_rate_mask = (
    df["Interest_Rate"] > interest_rate_boundary
)

valid_interest_rate_reference = (
    df.loc[
        df["Interest_Rate"].between(
            1,
            interest_rate_boundary,
            inclusive="left"
        )
    ]
    .groupby("Customer_ID")["Interest_Rate"]
    .agg(lambda values: values.mode().iloc[0])
)

suggested_interest_rate = (
    df.loc[invalid_interest_rate_mask, "Customer_ID"]
    .map(valid_interest_rate_reference)
)

fillable_indices = suggested_interest_rate[
    suggested_interest_rate.notna()
].index

unresolved_indices = suggested_interest_rate[
    suggested_interest_rate.isna()
].index

# Imputasi menggunakan referensi valid Customer_ID yang sama
df.loc[
    fillable_indices,
    "Interest_Rate"
] = suggested_interest_rate.loc[fillable_indices]

# Nilai tanpa referensi valid menjadi missing value
df.loc[
    unresolved_indices,
    "Interest_Rate"
] = pd.NA

df["Interest_Rate"] = (
    df["Interest_Rate"]
    .round()
    .astype("Int64")
)

print(
    "Berhasil diimputasi dari Customer_ID yang sama : "
    f"{len(fillable_indices):,}"
)

print(
    "Diubah menjadi missing value                  : "
    f"{len(unresolved_indices):,}"
)

print(
    "Interest_Rate > 34 yang masih tersisa         : "
    f"{(df['Interest_Rate'] > 34).sum():,}"
)

print(
    "Total missing Interest_Rate                   : "
    f"{df['Interest_Rate'].isna().sum():,}"
)

In [ ]:
# CHECK FINAL INTEREST RATE
display(
    df.loc[
        fillable_indices,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Interest_Rate"
        ]
    ].head(20)
)

display(
    df.loc[
        unresolved_indices,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Interest_Rate"
        ]
    ].head(20)
)

Kolom `Interest_Rate` menunjukkan tingkat suku bunga yang dimiliki oleh setiap nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat frekuensi kemunculan setiap nilai. Hasil pemeriksaan menunjukkan bahwa mayoritas nilai `Interest_Rate` berada pada rentang yang relatif kecil, tetapi terdapat beberapa nilai sangat besar hingga mencapai ratusan dan ribuan.

Karena tidak terdapat batas maksimum suku bunga yang dapat langsung diterapkan secara universal, penentuan batas anomali dilakukan berdasarkan distribusi internal dataset. Analisis persentil digunakan untuk melihat perubahan nilai pada beberapa titik distribusi, seperti persentil ke-90, 95, 97, 98, dan 99.

Hasil analisis menunjukkan bahwa persentil ke-97 masih memiliki nilai sekitar `34`, sedangkan persentil ke-98 mengalami peningkatan yang sangat besar. Lonjakan tersebut menunjukkan adanya pemisahan yang kuat antara kelompok nilai utama dan kelompok nilai ekstrem. Oleh karena itu, nilai `34` digunakan sebagai batas investigasi berdasarkan pola distribusi dataset, bukan sebagai batas maksimum suku bunga secara universal.

Setelah batas ditentukan, data dibagi menjadi dua kelompok:

* `Interest_Rate` sampai dengan `34` dianggap sebagai kelompok utama.
* `Interest_Rate > 34` dianggap sebagai nilai ekstrem yang perlu diperiksa lebih lanjut.

Hasil pemeriksaan menunjukkan bahwa terdapat **534 baris** dengan nilai `Interest_Rate > 34`. Nilai tersebut tidak langsung dihapus atau diganti karena masih terdapat kemungkinan untuk memperbaikinya menggunakan riwayat nasabah yang sama.

Selanjutnya, setiap nilai ekstrem diperiksa berdasarkan `Customer_ID`. Sistem mencari catatan lain dari nasabah yang sama pada bulan berbeda yang memiliki nilai `Interest_Rate` valid, yaitu pada rentang:

```text
1 <= Interest_Rate < 34
```

Dari catatan valid tersebut, dibuat beberapa informasi pendukung:

* `Valid_Count`, yaitu jumlah catatan valid milik nasabah yang sama;
* `Valid_Values`, yaitu daftar nilai suku bunga valid yang ditemukan;
* `Suggested_Value`, yaitu nilai yang paling sering muncul atau modus;
* `Has_Valid_Reference`, yaitu penanda apakah nilai ekstrem memiliki referensi yang dapat digunakan.

Hasil pemeriksaan menunjukkan bahwa:

* **453 dari 534 baris** memiliki referensi valid dari `Customer_ID` yang sama;
* **81 baris** tidak memiliki referensi valid;
* nilai yang memiliki referensi diperbaiki menggunakan `Suggested_Value`;
* nilai yang tidak memiliki referensi diubah menjadi missing value.

Modus digunakan karena `Interest_Rate` pada dataset tersimpan sebagai nilai diskrit. Nilai yang paling sering muncul pada riwayat nasabah dianggap lebih representatif dibandingkan rata-rata yang dapat menghasilkan nilai desimal.

Setelah proses perbaikan, kolom `Interest_Rate` dikonversi menjadi tipe data `Int64`. Tipe data ini dipilih agar nilai tetap berupa bilangan bulat, tetapi tetap dapat menyimpan missing value.

Pemeriksaan akhir menunjukkan bahwa:

* **453 nilai ekstrem berhasil diperbaiki** menggunakan referensi nasabah yang sama;
* **81 nilai diubah menjadi missing value**;
* tidak terdapat lagi nilai `Interest_Rate > 34`;
* total missing value pada kolom `Interest_Rate` menjadi **81 baris**.

Missing value yang tersisa tidak langsung diisi pada tahap ini. Nilai tersebut akan ditangani pada preprocessing model menggunakan metode imputasi yang dipelajari hanya dari data training untuk mencegah kebocoran informasi dari data testing.


### d. Num_of_Loan

In [ ]:
df["Num_of_Loan"].dtype

In [ ]:
# VALUE COUNT NUM OF LOAN
num_of_loan_counts = (
    df["Num_of_Loan"]
    .value_counts(dropna=False)
    .rename_axis("Num_of_Loan")
    .reset_index(name="Count")
)

display(num_of_loan_counts)

In [ ]:
# CHECK NUM OF LOAN FORMAT
loan_text = df["Num_of_Loan"].astype("string")

print("Mengandung underscore :", loan_text.str.contains("_", na=False).sum())
print("Mengandung tanda minus:", loan_text.str.contains("-", na=False).sum())
print(
    "Mengandung karakter lain:",
    loan_text.str.contains(r"[^0-9_-]", regex=True, na=False).sum()
)

In [ ]:
display(
    df.loc[
        loan_text.str.contains(r"[_-]", regex=True, na=False),
        ["ID", "Customer_ID", "Month", "Num_of_Loan"]
    ]
)

In [ ]:
display(
    loan_text[
        loan_text.str.contains(r"[^0-9]", regex=True, na=False)
    ].value_counts()
)

In [ ]:
# CLEAN NUM OF LOAN FORMAT
df["Num_of_Loan"] = (
    df["Num_of_Loan"]
    .astype("string")
    .str.strip()
    .str.replace("_", "", regex=False)
)

df["Num_of_Loan"] = pd.to_numeric(
    df["Num_of_Loan"],
    errors="coerce"
).astype("Int64")

In [ ]:
df["Num_of_Loan"].value_counts(dropna=False).sort_index()

In [ ]:
# DISPLAY NUM OF LOAN PERCENTILES

valid_num_of_loan_for_percentile = df.loc[
    df["Num_of_Loan"] >= 0,
    "Num_of_Loan"
]

loan_percentiles = (
    valid_num_of_loan_for_percentile
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 1.00]
    )
    .reset_index()
)

loan_percentiles.columns = [
    "Percentile",
    "Num_of_Loan"
]

loan_percentiles["Percentile"] *= 100
loan_percentiles["Increase"] = loan_percentiles["Num_of_Loan"].diff()

display(loan_percentiles)

In [ ]:
# CHECK NUM OF LOAN BOUNDARY

num_of_loan_boundary = 9

print(
    "Jumlah Num_of_Loan 0–9:",
    df["Num_of_Loan"].between(0, num_of_loan_boundary).sum()
)

print(
    "Jumlah Num_of_Loan < 0:",
    (df["Num_of_Loan"] < 0).sum()
)

print(
    "Jumlah Num_of_Loan > 9:",
    (df["Num_of_Loan"] > num_of_loan_boundary).sum()
)

In [ ]:
# CHECK VALID NUM OF LOAN REFERENCE

num_of_loan_boundary = 9

invalid_loan_mask = (
    (df["Num_of_Loan"] < 0)
    | (df["Num_of_Loan"] > num_of_loan_boundary)
    | df["Num_of_Loan"].isna()
)

valid_loan_mask = df["Num_of_Loan"].between(
    0,
    num_of_loan_boundary
)

loan_reference = (
    df.loc[valid_loan_mask]
    .groupby("Customer_ID")["Num_of_Loan"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(values.unique().tolist()),
        Suggested_Value=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_loan_check = (
    df.loc[
        invalid_loan_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Num_of_Loan"
        ]
    ]
    .merge(
        loan_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_loan_check["Has_Valid_Reference"] = (
    invalid_loan_check["Valid_Count"].notna()
)

print("Total Num_of_Loan tidak valid:", len(invalid_loan_check))
print(
    "Memiliki referensi valid 0–9:",
    invalid_loan_check["Has_Valid_Reference"].sum()
)
print(
    "Tidak memiliki referensi valid:",
    (~invalid_loan_check["Has_Valid_Reference"]).sum()
)

display(invalid_loan_check)

In [ ]:
# REPAIR NUM OF LOAN VALUES

valid_loan_reference = (
    df.loc[df["Num_of_Loan"].between(0, 9)]
    .groupby("Customer_ID")["Num_of_Loan"]
    .agg(lambda values: values.mode().iloc[0])
)

suggested_loan_values = (
    df.loc[invalid_loan_mask, "Customer_ID"]
    .map(valid_loan_reference)
)

fillable_indices = suggested_loan_values[
    suggested_loan_values.notna()
].index

unresolved_indices = suggested_loan_values[
    suggested_loan_values.isna()
].index

df.loc[
    fillable_indices,
    "Num_of_Loan"
] = suggested_loan_values.loc[fillable_indices]

df.loc[
    unresolved_indices,
    "Num_of_Loan"
] = pd.NA

df["Num_of_Loan"] = (
    df["Num_of_Loan"]
    .round()
    .astype("Int64")
)

print(
    "Berhasil diperbaiki dari Customer_ID yang sama:",
    len(fillable_indices)
)

print(
    "Diubah menjadi missing value:",
    len(unresolved_indices)
)

print(
    "Num_of_Loan < 0 yang masih tersisa:",
    (df["Num_of_Loan"] < 0).sum()
)

print(
    "Num_of_Loan > 9 yang masih tersisa:",
    (df["Num_of_Loan"] > 9).sum()
)

print(
    "Total missing Num_of_Loan:",
    df["Num_of_Loan"].isna().sum()
)

In [ ]:
df["Num_of_Loan"].value_counts(dropna=False).sort_index()

Kolom `Num_of_Loan` menunjukkan jumlah pinjaman yang dimiliki oleh setiap nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat variasi nilai dan frekuensi kemunculannya. Dari pemeriksaan tersebut, ditemukan bahwa kolom masih bertipe `object` karena beberapa nilai mengandung karakter tambahan seperti underscore (`_`) dan tanda negatif.

Pemeriksaan format kemudian dilakukan tanpa langsung mengubah data. Tujuannya adalah mengetahui jenis ketidakkonsistenan yang terdapat pada kolom, seperti:

* nilai yang mengandung underscore;
* nilai yang mengandung tanda negatif;
* nilai yang mengandung karakter selain angka;
* kemungkinan missing value.

Hasil pemeriksaan menunjukkan bahwa terdapat **1.204 nilai yang mengandung underscore** dan **977 nilai yang mengandung tanda negatif**. Tidak ditemukan karakter lain di luar angka, underscore, dan tanda negatif.

Karakter underscore hanya merupakan masalah format penulisan sehingga dapat dihapus tanpa mengubah makna angka. Setelah underscore dan spasi tambahan dibersihkan, kolom dikonversi menjadi tipe data numerik `Int64`.

Tanda negatif tidak dihapus menggunakan fungsi absolut karena langkah tersebut dapat mengubah nilai tidak valid menjadi nilai positif yang terlihat valid. Sebagai contoh, nilai `-100` akan berubah menjadi `100`, padahal nilai tersebut kemungkinan merupakan penanda data tidak valid, bukan menunjukkan bahwa nasabah mempunyai 100 pinjaman. Oleh karena itu, nilai negatif tidak dianggap sebagai nilai valid dan tidak digunakan sebagai referensi dalam proses perbaikan.

Setelah proses pembersihan format selesai, `value_counts()` kembali digunakan untuk memastikan bahwa:

* underscore sudah tidak ditemukan;
* nilai sudah tersimpan dalam format numerik;
* nilai negatif masih dapat terlihat sebagai nilai tidak valid yang perlu ditangani;
* distribusi jumlah pinjaman dapat dianalisis dengan lebih jelas.

Selanjutnya, analisis persentil dilakukan untuk menentukan batas maksimum berdasarkan distribusi internal dataset. Analisis ini tidak langsung menetapkan batas berdasarkan asumsi, tetapi membandingkan perubahan nilai pada beberapa titik persentil.

Sebagai pengujian tambahan, analisis persentil juga dilakukan dengan memperhatikan keberadaan nilai negatif. Hasilnya tetap menunjukkan bahwa kelompok utama data berada pada nilai paling banyak `9`. Dengan demikian, batas `9` bukan terbentuk akibat perubahan nilai negatif menjadi positif, tetapi berasal dari pola distribusi asli kolom `Num_of_Loan`.

Hasil analisis persentil pada nilai yang valid menunjukkan bahwa:

* persentil ke-50 bernilai `3`;
* persentil ke-75 bernilai `6`;
* persentil ke-90 bernilai `8`;
* persentil ke-95 bernilai `9`;
* setelah persentil ke-95 terjadi lonjakan nilai yang sangat besar.

Berdasarkan hasil tersebut, nilai `9` digunakan sebagai batas investigasi untuk kolom `Num_of_Loan`. Batas ini bukan batas universal mengenai jumlah pinjaman yang dapat dimiliki seseorang, melainkan batas yang diperoleh dari distribusi dataset karena sekitar 95% data berada pada nilai paling banyak 9.

Pemeriksaan berdasarkan batas tersebut menunjukkan bahwa terdapat **23.892 baris** dengan `Num_of_Loan` berada pada rentang `0` sampai `9`. Rentang ini dianggap sebagai kelompok nilai valid karena sesuai dengan pola utama distribusi data.

Sementara itu, nilai yang dianggap tidak valid terdiri dari:

* nilai negatif, karena jumlah pinjaman tidak mungkin kurang dari nol;
* nilai lebih dari `9`, karena berada di luar kelompok utama data dan menunjukkan lonjakan yang tidak wajar.

Secara total, terdapat **1.108 baris** yang perlu diperiksa lebih lanjut, yaitu gabungan dari nilai negatif dan nilai `Num_of_Loan > 9`.

Nilai tidak valid tersebut tidak langsung dihapus karena masih terdapat kemungkinan bahwa nilainya dapat diperbaiki menggunakan catatan nasabah yang sama pada bulan lain. Oleh karena itu, setiap baris yang tidak valid diperiksa berdasarkan `Customer_ID`.

Untuk setiap nasabah, dicari catatan bulan lain yang memiliki nilai `Num_of_Loan` valid pada rentang `0–9`. Dari catatan tersebut, dibuat beberapa informasi pendukung:

* `Valid_Count`, yaitu jumlah catatan valid yang tersedia;
* `Valid_Values`, yaitu daftar nilai valid yang dimiliki nasabah;
* `Suggested_Value`, yaitu nilai yang paling sering muncul atau modus;
* `Has_Valid_Reference`, yaitu penanda apakah nilai tidak valid memiliki referensi valid.

Modus digunakan sebagai nilai pengganti karena `Num_of_Loan` merupakan data jumlah atau diskrit. Penggunaan modus juga mencegah munculnya nilai desimal yang tidak sesuai dengan makna jumlah pinjaman.

Nilai tidak valid yang mempunyai referensi valid dari `Customer_ID` yang sama kemudian diganti menggunakan `Suggested_Value`. Sementara itu, nilai yang tidak mempunyai referensi valid tidak diisi secara sembarangan, tetapi diubah menjadi missing value.

Hasil proses perbaikan menunjukkan bahwa:

* **926 nilai** berhasil diperbaiki menggunakan referensi dari `Customer_ID` yang sama;
* **182 nilai** tidak memiliki referensi valid dan diubah menjadi missing value;
* tidak terdapat lagi nilai `Num_of_Loan < 0`;
* tidak terdapat lagi nilai `Num_of_Loan > 9`;
* total missing value pada kolom `Num_of_Loan` menjadi **182 baris**.

Setelah proses perbaikan selesai, kolom dikonversi kembali menjadi tipe data `Int64`. Tipe ini digunakan karena jumlah pinjaman harus berupa bilangan bulat, tetapi kolom tetap perlu mendukung missing value.

Pemeriksaan akhir dilakukan menggunakan `value_counts()` dan perhitungan kondisi untuk memastikan bahwa:

* nilai dengan format underscore sudah tidak ada;
* nilai negatif sudah tidak digunakan sebagai data valid;
* nilai `Num_of_Loan > 9` sudah tidak tersisa;
* nilai yang mempunyai referensi berhasil diperbaiki;
* nilai tanpa referensi telah diubah menjadi missing value;
* kolom telah memiliki tipe data `Int64`.

Missing value yang masih tersisa tidak langsung diisi pada tahap eksplorasi. Nilai tersebut akan ditangani pada pipeline preprocessing menggunakan metode imputasi yang hanya dipelajari dari data training agar tidak terjadi kebocoran informasi dari data testing.



### e. Type_of_Loan

In [ ]:
type_of_loan_counts = (
    df["Type_of_Loan"]
    .value_counts(dropna=False)
    .rename_axis("Type_of_Loan")
    .reset_index(name="Count")
)

display(type_of_loan_counts)

In [ ]:
print("Jumlah nilai unik:", df["Type_of_Loan"].nunique(dropna=False))

In [ ]:
# CHECK BASE LOAN TYPES
loan_types = (
    df["Type_of_Loan"]
    .dropna()
    .str.replace(", and ", ", ", regex=False)
    .str.replace(" and ", ", ", regex=False)
    .str.split(", ")
    .explode()
    .str.strip()
)

loan_type_counts = (
    loan_types
    .value_counts()
    .rename_axis("Loan_Type")
    .reset_index(name="Count")
)

display(loan_type_counts)

In [ ]:
# CHECK MISSING TYPE OF LOAN
missing_type_loan = df["Type_of_Loan"].isna()

print(
    "Missing dengan Num_of_Loan = 0:",
    (
        missing_type_loan
        & df["Num_of_Loan"].eq(0)
    ).sum()
)

print(
    "Missing dengan Num_of_Loan > 0:",
    (
        missing_type_loan
        & df["Num_of_Loan"].gt(0)
    ).sum()
)

print(
    "Missing karena Num_of_Loan juga kosong:",
    (
        missing_type_loan
        & df["Num_of_Loan"].isna()
    ).sum()
)

Kolom `Type_of_Loan` menunjukkan jenis pinjaman yang dimiliki oleh setiap nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat variasi nilai dan frekuensi kemunculan setiap kombinasi jenis pinjaman.

Hasil pemeriksaan menunjukkan bahwa satu baris dapat berisi lebih dari satu jenis pinjaman, misalnya:

```text
Personal Loan, Student Loan, and Mortgage Loan
```

Selain itu, terdapat kemungkinan jenis pinjaman yang sama muncul lebih dari satu kali dalam satu baris. Contohnya:

```text
Student Loan, Mortgage Loan, and Student Loan
```

Setelah itu, jumlah nilai unik diperiksa menggunakan `nunique()`. Hasilnya menunjukkan bahwa kolom `Type_of_Loan` memiliki **5.372 nilai unik**.

Jumlah nilai unik yang sangat besar bukan berarti terdapat 5.372 jenis pinjaman yang berbeda. Banyaknya nilai unik terjadi karena kolom ini menyimpan berbagai kombinasi jenis pinjaman dalam satu teks. Perbedaan urutan, jumlah pinjaman, dan pengulangan jenis pinjaman menyebabkan satu kombinasi dianggap sebagai kategori yang berbeda.

Sebagai contoh, kombinasi berikut akan terbaca sebagai nilai kategorikal yang berbeda:

```text
Personal Loan, Student Loan, and Mortgage Loan
Student Loan, Personal Loan, and Mortgage Loan
Personal Loan, Personal Loan, and Student Loan
```

Oleh karena itu, kolom `Type_of_Loan` tidak tepat apabila langsung diproses menggunakan one-hot encoding berdasarkan 5.372 kombinasi tersebut. Langkah tersebut akan menghasilkan terlalu banyak fitur, meningkatkan dimensi data, dan membuat model lebih sulit diproses.

Pemeriksaan lebih lanjut dilakukan dengan memisahkan setiap jenis pinjaman yang terdapat dalam satu baris. Pemisahan dilakukan menggunakan tanda koma dan kata `and`, kemudian setiap jenis pinjaman dihitung secara terpisah.

Hasil pemeriksaan menunjukkan bahwa meskipun terdapat ribuan kombinasi, jenis dasar pinjaman hanya terdiri dari:

* `Auto Loan`
* `Credit-Builder Loan`
* `Debt Consolidation Loan`
* `Home Equity Loan`
* `Mortgage Loan`
* `Payday Loan`
* `Personal Loan`
* `Student Loan`
* `Not Specified`

Dengan demikian, kolom `Type_of_Loan` merupakan **multi-label categorical feature**, yaitu satu kolom kategorikal yang dapat mengandung beberapa label sekaligus dalam satu baris.

Pada tahap preprocessing berikutnya, kolom ini akan diubah menjadi beberapa fitur numerik yang menunjukkan jumlah kemunculan setiap jenis pinjaman. Penggunaan jumlah dipilih karena satu nasabah dapat memiliki lebih dari satu pinjaman dengan jenis yang sama.

Contoh transformasinya adalah sebagai berikut:

| Auto_Loan | Student_Loan | Mortgage_Loan | Payday_Loan |
| --------: | -----------: | ------------: | ----------: |
|         1 |            2 |             1 |           0 |

Contoh tersebut menunjukkan bahwa nasabah memiliki:

* 1 `Auto Loan`;
* 2 `Student Loan`;
* 1 `Mortgage Loan`;
* tidak memiliki `Payday Loan`.

Sebelum melakukan transformasi, missing value pada `Type_of_Loan` juga perlu diperiksa bersama kolom `Num_of_Loan`.

Penanganannya direncanakan sebagai berikut:

* Jika `Type_of_Loan` kosong dan `Num_of_Loan = 0`, kondisi tersebut menunjukkan bahwa nasabah memang tidak memiliki pinjaman.
* Jika `Type_of_Loan` kosong tetapi `Num_of_Loan > 0`, jenis pinjaman nasabah tidak diketahui.
* Jika `Num_of_Loan` juga missing, informasi tersebut akan ditangani melalui proses imputasi pada pipeline preprocessing.




### f. Credit_Mix

In [ ]:
credit_mix_counts = (
    df["Credit_Mix"]
    .value_counts(dropna=False)
    .rename_axis("Credit_Mix")
    .reset_index(name="Count")
)

display(credit_mix_counts)

In [ ]:
# CHECK VALID CREDIT MIX REFERENCE
invalid_credit_mix_mask = (
    df["Credit_Mix"].isna()
    | df["Credit_Mix"].eq("_")
)

valid_credit_mix_mask = df["Credit_Mix"].isin(
    ["Standard", "Bad", "Good"]
)

credit_mix_reference = (
    df.loc[valid_credit_mix_mask]
    .groupby("Customer_ID")["Credit_Mix"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(values.unique().tolist()),
        Suggested_Value=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_credit_mix_check = (
    df.loc[
        invalid_credit_mix_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Credit_Mix"
        ]
    ]
    .merge(
        credit_mix_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_credit_mix_check["Has_Valid_Reference"] = (
    invalid_credit_mix_check["Valid_Count"].notna()
)

print(
    "Total Credit_Mix '_' atau missing:",
    len(invalid_credit_mix_check)
)

print(
    "Memiliki referensi valid:",
    invalid_credit_mix_check["Has_Valid_Reference"].sum()
)

print(
    "Tidak memiliki referensi valid:",
    (~invalid_credit_mix_check["Has_Valid_Reference"]).sum()
)

display(invalid_credit_mix_check)

In [ ]:
# REPAIR CREDIT MIX VALUES
valid_credit_mix_reference = (
    df.loc[
        df["Credit_Mix"].isin(["Standard", "Bad", "Good"])
    ]
    .groupby("Customer_ID")["Credit_Mix"]
    .agg(lambda values: values.mode().iloc[0])
)

invalid_credit_mix_mask = (
    df["Credit_Mix"].isna()
    | df["Credit_Mix"].eq("_")
)

suggested_credit_mix = (
    df.loc[invalid_credit_mix_mask, "Customer_ID"]
    .map(valid_credit_mix_reference)
)

fillable_indices = suggested_credit_mix[
    suggested_credit_mix.notna()
].index

unresolved_indices = suggested_credit_mix[
    suggested_credit_mix.isna()
].index

# Mengisi berdasarkan Customer_ID yang sama
df.loc[
    fillable_indices,
    "Credit_Mix"
] = suggested_credit_mix.loc[fillable_indices]

# Mengubah nilai tanpa referensi menjadi missing value
df.loc[
    unresolved_indices,
    "Credit_Mix"
] = pd.NA

print(
    "Berhasil diperbaiki dari Customer_ID yang sama:",
    len(fillable_indices)
)

print(
    "Diubah menjadi missing value:",
    len(unresolved_indices)
)

print(
    "Nilai '_' yang masih tersisa:",
    df["Credit_Mix"].eq("_").sum()
)

print(
    "Total missing Credit_Mix:",
    df["Credit_Mix"].isna().sum()
)

In [ ]:
credit_mix_final = (
    df["Credit_Mix"]
    .value_counts(dropna=False)
    .rename_axis("Credit_Mix")
    .reset_index(name="Count")
)

display(credit_mix_final)

Kolom `Credit_Mix` menunjukkan komposisi atau kualitas kredit yang dimiliki oleh setiap nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat kategori yang tersedia beserta frekuensi kemunculannya.

Hasil pemeriksaan menunjukkan distribusi berikut:

* `Standard` sebanyak **9.170 baris**;
* `Good` sebanyak **6.123 baris**;
* `Bad` sebanyak **4.781 baris**;
* simbol `_` sebanyak **4.926 baris**.

Kategori yang dianggap valid pada kolom ini adalah `Standard`, `Good`, dan `Bad`. Sementara itu, simbol `_` tidak merepresentasikan kategori kredit yang sebenarnya sehingga dianggap sebagai nilai tidak valid.

Nilai `_` tidak langsung dihapus atau diganti menggunakan kategori yang paling sering muncul pada seluruh dataset. Sebelum melakukan perubahan, setiap baris dengan nilai `_` diperiksa berdasarkan `Customer_ID` untuk mengetahui apakah nasabah yang sama memiliki nilai `Credit_Mix` valid pada bulan lain.

Nilai referensi yang digunakan hanya berasal dari tiga kategori valid, yaitu:

* `Standard`;
* `Good`;
* `Bad`.

Untuk setiap `Customer_ID`, dibuat beberapa informasi pendukung:

* `Valid_Count`, yaitu jumlah catatan valid dari nasabah yang sama;
* `Valid_Values`, yaitu daftar kategori valid yang pernah tercatat;
* `Suggested_Value`, yaitu kategori yang paling sering muncul atau modus;
* `Has_Valid_Reference`, yaitu penanda apakah nilai `_` memiliki referensi valid.

Hasil pemeriksaan menunjukkan bahwa dari **4.926 baris** dengan nilai `_`:

* **3.959 baris** memiliki referensi `Credit_Mix` yang valid dari `Customer_ID` yang sama;
* **967 baris** tidak memiliki referensi valid.

Nilai `Credit_Mix` yang memiliki referensi valid kemudian diperbaiki menggunakan `Suggested_Value` dari nasabah yang sama. Modus digunakan karena `Credit_Mix` merupakan fitur kategorikal. Kategori yang paling sering muncul pada riwayat bulanan nasabah dianggap lebih representatif dibandingkan memilih kategori secara acak.

Sementara itu, nilai `_` yang tidak memiliki referensi valid tidak dipaksakan menjadi `Standard`, `Good`, maupun `Bad`. Sebanyak **967 baris** tersebut diubah menjadi missing value agar dapat ditangani secara lebih aman pada tahap preprocessing berikutnya.

Setelah proses perbaikan, pemeriksaan akhir dilakukan untuk memastikan bahwa:

* **3.959 nilai** berhasil diperbaiki menggunakan riwayat nasabah yang sama;
* **967 nilai** diubah menjadi missing value;
* simbol `_` sudah tidak tersisa pada kolom `Credit_Mix`;
* kategori yang tersisa hanya `Standard`, `Good`, `Bad`, dan missing value;
* total missing value pada kolom `Credit_Mix` menjadi **967 baris**.

Missing value yang masih tersisa tidak langsung diisi pada tahap eksplorasi. Nilai tersebut akan ditangani pada pipeline preprocessing menggunakan metode imputasi kategorikal yang hanya dipelajari dari data training. Pendekatan ini dilakukan untuk mencegah kebocoran informasi dari data testing.




### g. Changed_Credit_Limit

In [ ]:
changed_credit_limit_counts = (
    df["Changed_Credit_Limit"]
    .value_counts(dropna=False)
    .rename_axis("Changed_Credit_Limit")
    .reset_index(name="Count")
)

display(changed_credit_limit_counts)

In [ ]:
df["Changed_Credit_Limit"].dtype

In [ ]:
changed_credit_limit_numeric = pd.to_numeric(
    df["Changed_Credit_Limit"].replace("_", pd.NA),
    errors="coerce"
)

print("Minimum:", changed_credit_limit_numeric.min())
print("Maximum:", changed_credit_limit_numeric.max())

Pemeriksaan nilai minimum dan maksimum pada kolom `Changed_Credit_Limit` dilakukan setelah nilai sementara dikonversi ke format numerik. Hasil pemeriksaan menunjukkan bahwa nilai minimum adalah `-6,48`, sedangkan nilai maksimum adalah `35,82`.

Rentang tersebut masih dianggap masuk akal karena kolom `Changed_Credit_Limit` menunjukkan perubahan batas kredit, bukan jumlah batas kredit yang dimiliki nasabah. Oleh karena itu, nilai pada kolom ini dapat berupa negatif, nol, maupun positif.

Nilai negatif seperti `-6,48` menunjukkan bahwa batas kredit nasabah mengalami penurunan, sedangkan nilai positif seperti `35,82` menunjukkan bahwa batas kredit mengalami peningkatan. Nilai `0` dapat menunjukkan bahwa tidak terdapat perubahan batas kredit.

Berbeda dengan fitur jumlah seperti `Num_of_Loan` atau `Num_Bank_Accounts`, tanda negatif pada kolom ini memiliki makna dan tidak boleh dihapus atau diubah menjadi nilai positif. Selain itu, nilai maksimum `35,82` tidak menunjukkan lonjakan hingga ratusan atau ribuan sehingga belum terdapat indikasi kuat bahwa nilai tersebut merupakan data rusak.

Dengan demikian, nilai numerik pada rentang `-6,48` sampai `35,82` akan dipertahankan. 


In [ ]:
# CHECK VALID CHANGED CREDIT LIMIT REFERENCE
changed_credit_numeric = pd.to_numeric(
    df["Changed_Credit_Limit"].replace("_", pd.NA),
    errors="coerce"
)

invalid_changed_credit_mask = changed_credit_numeric.isna()
valid_changed_credit_mask = changed_credit_numeric.notna()

changed_credit_reference = (
    pd.DataFrame({
        "Customer_ID": df["Customer_ID"],
        "Changed_Credit_Limit": changed_credit_numeric
    })
    .loc[valid_changed_credit_mask]
    .groupby("Customer_ID")["Changed_Credit_Limit"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(values.unique().tolist()),
        Suggested_Value="mean"
    )
    .reset_index()
)

invalid_changed_credit_check = (
    df.loc[
        invalid_changed_credit_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Changed_Credit_Limit"
        ]
    ]
    .merge(
        changed_credit_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_changed_credit_check["Has_Valid_Reference"] = (
    invalid_changed_credit_check["Valid_Count"].notna()
)

print(
    "Total '_' atau missing:",
    len(invalid_changed_credit_check)
)

print(
    "Memiliki referensi valid dari bulan lain:",
    invalid_changed_credit_check["Has_Valid_Reference"].sum()
)

print(
    "Tidak memiliki referensi valid:",
    (~invalid_changed_credit_check["Has_Valid_Reference"]).sum()
)

display(invalid_changed_credit_check)

In [ ]:
# REPAIR CHANGED CREDIT LIMIT
customer_credit_stats = (
    pd.DataFrame({
        "Customer_ID": df["Customer_ID"],
        "Changed_Credit_Limit": changed_credit_numeric
    })
    .dropna(subset=["Changed_Credit_Limit"])
    .groupby("Customer_ID")["Changed_Credit_Limit"]
    .agg(["count", "mean"])
)

invalid_mask = changed_credit_numeric.isna()

reference_count = (
    df.loc[invalid_mask, "Customer_ID"]
    .map(customer_credit_stats["count"])
)

reference_value = (
    df.loc[invalid_mask, "Customer_ID"]
    .map(customer_credit_stats["mean"])
)

fillable_indices = reference_value[
    reference_value.notna()
].index

unresolved_indices = reference_value[
    reference_value.isna()
].index

# Jika hanya satu referensi, mean sama dengan nilai tersebut.
# Jika lebih dari satu referensi, digunakan rata-rata.
changed_credit_numeric.loc[
    fillable_indices
] = reference_value.loc[fillable_indices]

changed_credit_numeric.loc[
    unresolved_indices
] = pd.NA

df["Changed_Credit_Limit"] = changed_credit_numeric.astype("float64")

print(
    "Berhasil diisi dari Customer_ID yang sama:",
    len(fillable_indices)
)

print(
    "Diubah menjadi missing value:",
    len(unresolved_indices)
)

print(
    "Total missing Changed_Credit_Limit:",
    df["Changed_Credit_Limit"].isna().sum()
)

print(
    "Tipe data akhir:",
    df["Changed_Credit_Limit"].dtype
)

Kolom `Changed_Credit_Limit` menunjukkan perubahan batas kredit yang dialami oleh setiap nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat variasi nilai dan frekuensi kemunculannya.

Hasil pemeriksaan menunjukkan bahwa kolom ini masih bertipe `object`. Hal tersebut terjadi karena selain berisi angka desimal, terdapat simbol `_` yang tidak dapat langsung dibaca sebagai nilai numerik.

Selanjutnya, tipe data kolom diperiksa menggunakan `dtype`. Karena kolom masih bertipe `object`, nilai dikonversi sementara menjadi numerik untuk memeriksa rentang minimum dan maksimum tanpa langsung mengubah DataFrame utama.

Hasil pemeriksaan menunjukkan bahwa:

* nilai minimum adalah `-6,48`;
* nilai maksimum adalah `35,82`.

Rentang tersebut masih dianggap masuk akal karena `Changed_Credit_Limit` menunjukkan perubahan batas kredit, bukan jumlah batas kredit yang dimiliki nasabah.

Nilai negatif seperti `-6,48` dapat menunjukkan bahwa batas kredit mengalami penurunan, sedangkan nilai positif seperti `35,82` menunjukkan bahwa batas kredit mengalami peningkatan. Nilai `0` dapat menunjukkan tidak adanya perubahan batas kredit.

Berbeda dengan kolom jumlah seperti `Num_of_Loan` atau `Num_Bank_Accounts`, nilai negatif pada kolom ini memiliki makna sehingga tidak boleh dihapus atau diubah menjadi nilai positif. Selain itu, rentang nilai tidak menunjukkan lonjakan hingga ratusan atau ribuan sehingga belum terdapat indikasi kuat adanya outlier ekstrem.

Permasalahan utama pada kolom ini adalah simbol `_` atau nilai yang tidak dapat dikonversi menjadi angka. Simbol tersebut kemudian diperlakukan sebagai missing value.

Setelah nilai numerik dan nilai tidak valid dipisahkan, dilakukan pemeriksaan berdasarkan `Customer_ID`. Tujuannya adalah mengetahui apakah nasabah dengan nilai `_` atau missing value memiliki catatan `Changed_Credit_Limit` yang valid pada bulan lain.

Hasil pemeriksaan menunjukkan bahwa:

* terdapat **518 baris** dengan simbol `_` atau missing value;
* **442 baris** memiliki referensi valid dari `Customer_ID` yang sama;
* **76 baris** tidak memiliki referensi valid.

Untuk setiap nasabah yang memiliki referensi, dihitung jumlah catatan valid dan nilai rata-ratanya. Aturan pengisian yang digunakan adalah:

* jika hanya terdapat satu referensi valid, nilai tersebut langsung digunakan;
* jika terdapat lebih dari satu referensi valid, digunakan rata-rata seluruh nilai valid milik nasabah yang sama.

Penggunaan fungsi rata-rata dapat menangani kedua kondisi tersebut sekaligus. Apabila hanya terdapat satu nilai, hasil rata-ratanya tetap sama dengan nilai tersebut. Apabila terdapat beberapa nilai, hasilnya menjadi rata-rata seluruh catatan valid.

Sebanyak **442 nilai** kemudian berhasil diperbaiki menggunakan referensi dari `Customer_ID` yang sama. Sementara itu, **76 nilai** yang tidak mempunyai referensi valid tetap dijadikan missing value agar tidak diisi menggunakan angka tanpa dasar yang jelas.

Setelah proses perbaikan selesai, kolom `Changed_Credit_Limit` dikonversi menjadi tipe data `float64` karena kolom tersebut mengandung nilai desimal dan masih perlu mendukung missing value.

Pemeriksaan akhir menunjukkan bahwa:

* simbol `_` sudah tidak digunakan sebagai nilai;
* **442 nilai** berhasil diperbaiki menggunakan riwayat nasabah yang sama;
* **76 nilai** tetap menjadi missing value;
* nilai negatif yang valid tetap dipertahankan;
* tipe data akhir kolom menjadi `float64`;
* total missing value yang tersisa adalah **76 baris**.

Missing value yang masih tersisa tidak langsung diisi pada tahap eksplorasi. Nilai tersebut akan ditangani pada pipeline preprocessing menggunakan metode imputasi yang hanya dipelajari dari data training agar tidak terjadi kebocoran informasi dari data testing.



### h. Num_Credit_Inquiries

In [ ]:
num_credit_inquiries_counts = (
    df["Num_Credit_Inquiries"]
    .value_counts(dropna=False)
    .rename_axis("Num_Credit_Inquiries")
    .reset_index(name="Count")
    .sort_values(
        by="Num_Credit_Inquiries",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(num_credit_inquiries_counts)

In [ ]:
# CHECK NUM CREDIT INQUIRIES PERCENTILES
inquiry_percentiles = (
    df["Num_Credit_Inquiries"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 1.00]
    )
    .reset_index()
)

inquiry_percentiles.columns = [
    "Percentile",
    "Num_Credit_Inquiries"
]

inquiry_percentiles["Percentile"] *= 100

inquiry_percentiles["Increase"] = (
    inquiry_percentiles["Num_Credit_Inquiries"].diff()
)

display(inquiry_percentiles)

In [ ]:
# CHECK VALUES AROUND UPPER BOUNDARY
values_around_boundary = (
    df.loc[
        df["Num_Credit_Inquiries"].between(10, 100),
        "Num_Credit_Inquiries"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("Num_Credit_Inquiries")
    .reset_index(name="Count")
)

display(values_around_boundary)

In [ ]:
# CHECK NUM CREDIT INQUIRIES BOUNDARY
print(
    "Jumlah Num_Credit_Inquiries 0–17:",
    df["Num_Credit_Inquiries"].between(0, 17).sum()
)

print(
    "Jumlah Num_Credit_Inquiries > 17:",
    (df["Num_Credit_Inquiries"] > 17).sum()
)

print(
    "Jumlah missing:",
    df["Num_Credit_Inquiries"].isna().sum()
)

In [ ]:
# CHECK VALID NUM CREDIT INQUIRIES REFERENCE
invalid_inquiry_mask = df["Num_Credit_Inquiries"] > 17
valid_inquiry_mask = df["Num_Credit_Inquiries"].between(0, 17)

inquiry_reference = (
    df.loc[valid_inquiry_mask]
    .groupby("Customer_ID")["Num_Credit_Inquiries"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(values.unique().tolist()),
        Suggested_Value="mean"
    )
    .reset_index()
)

invalid_inquiry_check = (
    df.loc[
        invalid_inquiry_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Num_Credit_Inquiries"
        ]
    ]
    .merge(
        inquiry_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_inquiry_check["Has_Valid_Reference"] = (
    invalid_inquiry_check["Valid_Count"].notna()
)

print(
    "Total Num_Credit_Inquiries > 17:",
    len(invalid_inquiry_check)
)

print(
    "Memiliki referensi valid 0–17:",
    invalid_inquiry_check["Has_Valid_Reference"].sum()
)

print(
    "Tidak memiliki referensi valid:",
    (~invalid_inquiry_check["Has_Valid_Reference"]).sum()
)

display(invalid_inquiry_check)

In [ ]:
# REPAIR NUM CREDIT INQUIRIES
valid_inquiry_mask = df["Num_Credit_Inquiries"].between(0, 17)
invalid_inquiry_mask = df["Num_Credit_Inquiries"] > 17

customer_inquiry_stats = (
    df.loc[valid_inquiry_mask]
    .groupby("Customer_ID")["Num_Credit_Inquiries"]
    .agg(
        Valid_Count="count",
        Mean_Value="mean"
    )
)

invalid_customer_ids = df.loc[
    invalid_inquiry_mask,
    "Customer_ID"
]

reference_count = invalid_customer_ids.map(
    customer_inquiry_stats["Valid_Count"]
)

reference_mean = invalid_customer_ids.map(
    customer_inquiry_stats["Mean_Value"]
)

fillable_indices = reference_mean[
    reference_mean.notna()
].index

unresolved_indices = reference_mean[
    reference_mean.isna()
].index

df.loc[
    fillable_indices,
    "Num_Credit_Inquiries"
] = reference_mean.loc[fillable_indices]

df.loc[
    unresolved_indices,
    "Num_Credit_Inquiries"
] = pd.NA

df["Num_Credit_Inquiries"] = pd.to_numeric(
    df["Num_Credit_Inquiries"],
    errors="coerce"
).astype("float64")

print(
    "Berhasil diperbaiki dari Customer_ID yang sama:",
    len(fillable_indices)
)

print(
    "Diubah menjadi missing value:",
    len(unresolved_indices)
)

print(
    "Nilai > 17 yang masih tersisa:",
    (df["Num_Credit_Inquiries"] > 17).sum()
)

print(
    "Total missing Num_Credit_Inquiries:",
    df["Num_Credit_Inquiries"].isna().sum()
)

Kolom `Num_Credit_Inquiries` menunjukkan jumlah permintaan pemeriksaan riwayat kredit yang dilakukan terhadap seorang nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat distribusi nilai dan mengidentifikasi kemungkinan adanya nilai yang tidak wajar.

Setelah distribusi awal diperoleh, dilakukan analisis persentil untuk memahami pola sebaran data secara lebih rinci. Tujuan analisis ini adalah menentukan batas investigasi berdasarkan distribusi aktual dataset, bukan berdasarkan asumsi atau aturan yang ditetapkan secara subjektif.

Hasil analisis persentil menunjukkan bahwa:

* persentil ke-50 bernilai `6`;
* persentil ke-75 bernilai `9`;
* persentil ke-90 bernilai `12`;
* persentil ke-95 bernilai `13`;
* persentil ke-97 bernilai `15`;
* persentil ke-98 bernilai `16`;
* persentil ke-99 melonjak menjadi sekitar `1.130`.

Pola tersebut menunjukkan bahwa sebagian besar data berada pada rentang yang relatif kecil, kemudian terjadi lonjakan yang sangat besar setelah persentil ke-98. Untuk memastikan batas yang lebih tepat, nilai unik di sekitar batas atas diperiksa kembali.

Hasil pemeriksaan menunjukkan bahwa nilai masih muncul secara berurutan hingga `17`, sedangkan nilai berikutnya mulai menunjukkan lonjakan yang tidak wajar. Oleh karena itu, nilai `17` digunakan sebagai batas investigasi untuk kolom `Num_Credit_Inquiries`.

Berdasarkan hasil tersebut, data dibagi menjadi dua kelompok:

* `Num_Credit_Inquiries` antara `0` sampai `17` dianggap sebagai kelompok utama yang masih masuk akal;
* `Num_Credit_Inquiries > 17` dianggap sebagai nilai anomali yang perlu diperiksa lebih lanjut.

Nilai di atas `17` tidak langsung dihapus atau diubah menjadi missing value. Sebelum dilakukan perbaikan, setiap baris diperiksa berdasarkan `Customer_ID` untuk mengetahui apakah nasabah yang sama memiliki nilai `Num_Credit_Inquiries` yang valid pada bulan lain.

Nilai valid yang digunakan sebagai referensi hanya berasal dari rentang:

```text
0 <= Num_Credit_Inquiries <= 17
```

Untuk setiap `Customer_ID`, dihitung jumlah referensi valid dan rata-rata nilai valid yang dimiliki. Pendekatan yang digunakan adalah:

* apabila hanya terdapat satu nilai valid, nilai tersebut digunakan sebagai pengganti;
* apabila terdapat lebih dari satu nilai valid, digunakan rata-rata seluruh nilai valid milik nasabah yang sama.

Secara matematis, penggunaan rata-rata dapat menangani kedua kondisi tersebut sekaligus karena rata-rata dari satu nilai akan menghasilkan nilai yang sama.

Nilai `Num_Credit_Inquiries > 17` yang memiliki referensi valid kemudian diperbaiki menggunakan rata-rata nilai valid dari `Customer_ID` yang sama. Sementara itu, nilai yang tidak memiliki referensi valid tidak dipaksakan untuk diisi menggunakan nilai umum dari seluruh dataset, tetapi diubah menjadi missing value.

Setelah proses perbaikan selesai, kolom dikonversi menjadi tipe data `float64`. Tipe data ini dipilih karena penggunaan rata-rata dapat menghasilkan nilai desimal dan kolom tetap perlu mendukung missing value.

Pemeriksaan akhir dilakukan untuk memastikan bahwa:

* nilai ekstrem di atas `17` sudah ditangani;
* nilai yang memiliki referensi valid berhasil diperbaiki;
* nilai tanpa referensi valid telah diubah menjadi missing value;
* tipe data kolom telah berubah menjadi `float64`;
* tidak terdapat lagi nilai yang melampaui batas investigasi yang ditetapkan.

Missing value yang masih tersisa tidak langsung diimputasi pada tahap eksplorasi data. Nilai tersebut akan ditangani pada pipeline preprocessing menggunakan metode imputasi yang dipelajari hanya dari data training agar tidak terjadi kebocoran informasi dari data testing.



## 5. Kolom Riwayat Pembayaran dan Utang

### a. Delay_from_due_date

In [ ]:
delay_from_due_date_counts = (
    df["Delay_from_due_date"]
    .value_counts(dropna=False)
    .rename_axis("Delay_from_due_date")
    .reset_index(name="Count")
    .sort_values(
        by="Delay_from_due_date",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(delay_from_due_date_counts)

Kolom `Delay_from_due_date` menunjukkan selisih hari antara tanggal pembayaran dan tanggal jatuh tempo. Nilai negatif menunjukkan bahwa pembayaran dilakukan sebelum tanggal jatuh tempo, nilai nol menunjukkan pembayaran tepat waktu, sedangkan nilai positif menunjukkan keterlambatan pembayaran.

Pemeriksaan distribusi nilai menunjukkan bahwa rentang data berada pada interval `-5` hingga `67`. Nilai negatif hanya ditemukan pada rentang `-5` sampai `-1`, yang masih masuk akal karena menunjukkan pembayaran dilakukan 1–5 hari lebih awal dari tanggal jatuh tempo.

Selain itu, distribusi nilai terlihat cukup konsisten tanpa adanya lonjakan ekstrem atau nilai yang tidak wajar. Nilai maksimum sebesar `67` hari juga masih dapat terjadi pada kasus keterlambatan pembayaran kredit.

Berdasarkan hasil pemeriksaan tersebut, tidak ditemukan indikasi kesalahan format maupun anomali yang memerlukan perbaikan. Oleh karena itu, seluruh nilai pada kolom `Delay_from_due_date` dipertahankan dan tidak dilakukan proses imputasi maupun koreksi data.

### b. Num_of_Delayed_Payment

In [ ]:
num_delayed_payment_counts = (
    df["Num_of_Delayed_Payment"]
    .value_counts(dropna=False)
    .rename_axis("Num_of_Delayed_Payment")
    .reset_index(name="Count")
    .sort_values(
        by="Num_of_Delayed_Payment",
        ascending=True,
        na_position="last"
    )
    .reset_index(drop=True)
)

display(num_delayed_payment_counts)

In [ ]:
print("Tipe data :", df["Num_of_Delayed_Payment"].dtype)

In [ ]:
# CLEAN NUM OF DELAYED PAYMENT
df["Num_of_Delayed_Payment"] = (
    df["Num_of_Delayed_Payment"]
    .astype("string")
    .str.strip()
    .str.replace("_", "", regex=False)
)

df["Num_of_Delayed_Payment"] = pd.to_numeric(
    df["Num_of_Delayed_Payment"],
    errors="coerce"
)

# Nilai negatif tidak valid untuk jumlah pembayaran terlambat
df.loc[
    df["Num_of_Delayed_Payment"] < 0,
    "Num_of_Delayed_Payment"
] = pd.NA

df["Num_of_Delayed_Payment"] = (
    df["Num_of_Delayed_Payment"]
    .astype("Int64")
)

print("Tipe data :", df["Num_of_Delayed_Payment"].dtype)
print("Missing   :", df["Num_of_Delayed_Payment"].isna().sum())
print("Negatif   :", (df["Num_of_Delayed_Payment"] < 0).sum())

In [ ]:
# CHECK NUM OF DELAYED PAYMENT PERCENTILES
delayed_payment_percentiles = (
    df["Num_of_Delayed_Payment"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 1.00]
    )
    .reset_index()
)

delayed_payment_percentiles.columns = [
    "Percentile",
    "Num_of_Delayed_Payment"
]

delayed_payment_percentiles["Percentile"] *= 100

delayed_payment_percentiles["Increase"] = (
    delayed_payment_percentiles["Num_of_Delayed_Payment"].diff()
)

display(delayed_payment_percentiles)

In [ ]:
# CHECK VALUES ABOVE 27
display(
    df.loc[
        df["Num_of_Delayed_Payment"] > 27,
        "Num_of_Delayed_Payment"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("Num_of_Delayed_Payment")
    .reset_index(name="Count")
)

In [ ]:
# CHECK NATURAL BOUNDARY
display(
    df.loc[
        df["Num_of_Delayed_Payment"].between(20, 40),
        "Num_of_Delayed_Payment"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("Num_of_Delayed_Payment")
    .reset_index(name="Count")
)

In [ ]:
# CHECK VALID NUM OF DELAYED PAYMENT REFERENCE
num_delayed_payment_boundary = 28

invalid_delayed_payment_mask = (
    df["Num_of_Delayed_Payment"] > num_delayed_payment_boundary
)

valid_delayed_payment_mask = (
    df["Num_of_Delayed_Payment"]
    .between(0, num_delayed_payment_boundary)
)

delayed_payment_reference = (
    df.loc[valid_delayed_payment_mask]
    .groupby("Customer_ID")["Num_of_Delayed_Payment"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(values.unique().tolist()),
        Suggested_Value=lambda values: values.mean()
    )
    .reset_index()
)

invalid_delayed_payment_check = (
    df.loc[
        invalid_delayed_payment_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Num_of_Delayed_Payment"
        ]
    ]
    .merge(
        delayed_payment_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_delayed_payment_check["Has_Valid_Reference"] = (
    invalid_delayed_payment_check["Valid_Count"].notna()
)

print(
    "Total Num_of_Delayed_Payment > 28:",
    len(invalid_delayed_payment_check)
)

print(
    "Memiliki referensi valid 0–28:",
    invalid_delayed_payment_check["Has_Valid_Reference"].sum()
)

print(
    "Tidak memiliki referensi valid:",
    (~invalid_delayed_payment_check["Has_Valid_Reference"]).sum()
)

display(invalid_delayed_payment_check)

In [ ]:
# REPAIR NUM OF DELAYED PAYMENT
df["Num_of_Delayed_Payment"] = df["Num_of_Delayed_Payment"].astype("float64")

suggested_delayed_payment = (
    df.loc[invalid_delayed_payment_mask, "Customer_ID"]
    .map(
        delayed_payment_reference
        .set_index("Customer_ID")["Suggested_Value"]
    )
)

suggested_delayed_payment = pd.to_numeric(
    suggested_delayed_payment,
    errors="coerce"
)

fillable_indices = suggested_delayed_payment[
    suggested_delayed_payment.notna()
].index

unresolved_indices = suggested_delayed_payment[
    suggested_delayed_payment.isna()
].index

df.loc[
    fillable_indices,
    "Num_of_Delayed_Payment"
] = suggested_delayed_payment.loc[fillable_indices]

df.loc[
    unresolved_indices,
    "Num_of_Delayed_Payment"
] = np.nan

print(
    "Berhasil diperbaiki dari Customer_ID yang sama:",
    len(fillable_indices)
)

print(
    "Diubah menjadi missing value:",
    len(unresolved_indices)
)

print(
    "Nilai > 28 yang masih tersisa:",
    (df["Num_of_Delayed_Payment"] > 28).sum()
)

print(
    "Total missing Num_of_Delayed_Payment:",
    df["Num_of_Delayed_Payment"].isna().sum()
)

print(
    "Tipe data akhir:",
    df["Num_of_Delayed_Payment"].dtype
)

### c. Oustanding Debt

In [ ]:
# VALUE COUNT OUTSTANDING DEBT
outstanding_debt_counts = (
    df["Outstanding_Debt"]
    .value_counts(dropna=False)
    .rename_axis("Outstanding_Debt")
    .reset_index(name="Count")
)

display(outstanding_debt_counts)

In [ ]:
# CHECK DATA TYPE
print("Tipe data :", df["Outstanding_Debt"].dtype)
print("Missing   :", df["Outstanding_Debt"].isna().sum())

In [ ]:
# CHECK OUTSTANDING DEBT FORMAT
outstanding_debt_text = df["Outstanding_Debt"].astype("string")

print("Mengandung underscore :", outstanding_debt_text.str.contains("_", na=False).sum())
print("Mengandung tanda minus:", outstanding_debt_text.str.contains("-", na=False).sum())
print(
    "Mengandung karakter lain:",
    outstanding_debt_text.str.contains(r"[^0-9.\-_]", regex=True, na=False).sum()
)

In [ ]:
# CLEAN OUTSTANDING DEBT FORMAT
df["Outstanding_Debt"] = (
    df["Outstanding_Debt"]
    .astype("string")
    .str.strip()
    .str.replace("_", "", regex=False)
)

df["Outstanding_Debt"] = pd.to_numeric(
    df["Outstanding_Debt"],
    errors="coerce"
).astype("float64")

print("Tipe data :", df["Outstanding_Debt"].dtype)
print("Missing   :", df["Outstanding_Debt"].isna().sum())
print("Minimum   :", df["Outstanding_Debt"].min())
print("Maximum   :", df["Outstanding_Debt"].max())

Kolom `Outstanding_Debt` menunjukkan jumlah utang yang masih harus dibayar oleh setiap nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat variasi nilai yang terdapat pada kolom tersebut.

Selanjutnya, tipe data kolom diperiksa menggunakan `dtype`. Hasil pemeriksaan menunjukkan bahwa kolom masih bertipe `object`, sehingga diperlukan pemeriksaan format sebelum kolom dapat digunakan sebagai variabel numerik.

Pemeriksaan format dilakukan untuk mengidentifikasi kemungkinan adanya karakter tambahan seperti underscore (`_`), spasi, tanda baca yang tidak sesuai, maupun karakter non-numerik lainnya. Hasil pemeriksaan menunjukkan bahwa beberapa nilai mengandung underscore (`_`) yang menyebabkan kolom tidak dapat langsung dikenali sebagai data numerik.

Karakter underscore tidak memiliki makna numerik sehingga dianggap sebagai masalah format penulisan. Oleh karena itu, seluruh underscore dihapus dan nilai kemudian dikonversi menjadi tipe numerik menggunakan `pd.to_numeric()`.

Setelah proses pembersihan format selesai, kolom berhasil dikonversi menjadi tipe data `float64`. Pemeriksaan lanjutan menunjukkan bahwa:

* tidak terdapat missing value setelah proses konversi;
* seluruh nilai berhasil diubah menjadi format numerik;
* tidak ditemukan nilai yang gagal dikonversi.

Langkah berikutnya adalah memeriksa rentang nilai menggunakan nilai minimum dan maksimum. Hasil pemeriksaan menunjukkan bahwa:

* nilai minimum adalah `0,34`;
* nilai maksimum adalah `4.997,10`.

Rentang tersebut masih dianggap masuk akal karena `Outstanding_Debt` merepresentasikan jumlah utang yang masih harus dibayar oleh nasabah. Nilai minimum yang mendekati nol menunjukkan bahwa sebagian nasabah hampir tidak memiliki utang yang tersisa atau berada pada tahap akhir pelunasan. Sementara itu, nilai maksimum sebesar `4.997,10` masih berada pada skala ribuan dan tidak menunjukkan lonjakan ekstrem yang mengindikasikan kesalahan pencatatan data.

Selain itu, seluruh nilai bersifat positif. Kondisi ini sesuai dengan karakteristik utang yang secara logis tidak dapat bernilai negatif. Nilai desimal juga masih wajar karena jumlah utang dapat mengandung satuan pecahan.

Berdasarkan hasil pemeriksaan tersebut, tidak ditemukan indikasi kesalahan format maupun anomali numerik yang memerlukan perbaikan lebih lanjut. Oleh karena itu, seluruh nilai pada kolom `Outstanding_Debt` dipertahankan setelah proses pembersihan format dan konversi tipe data.



### d. Credit Utilization Ratio

In [ ]:
credit_utilization_ratio_counts = (
    df["Credit_Utilization_Ratio"]
    .value_counts(dropna=False)
    .rename_axis("Credit_Utilization_Ratio")
    .reset_index(name="Count")
)

display(credit_utilization_ratio_counts)

In [ ]:
print("Tipe data :", df["Credit_Utilization_Ratio"].dtype)
print("Missing   :", df["Credit_Utilization_Ratio"].isna().sum())
print("Minimum   :", df["Credit_Utilization_Ratio"].min())
print("Maximum   :", df["Credit_Utilization_Ratio"].max())

Kolom ini sudah aman dan bersih

### e. Credit History Age

In [ ]:
credit_history_age_counts = (
    df["Credit_History_Age"]
    .value_counts(dropna=False)
    .rename_axis("Credit_History_Age")
    .reset_index(name="Count")
)

display(credit_history_age_counts)

In [ ]:
print("Tipe data :", df["Credit_History_Age"].dtype)
print("Missing   :", df["Credit_History_Age"].isna().sum())

In [ ]:
# CONVERT CREDIT HISTORY AGE TO MONTHS
credit_history_text = df["Credit_History_Age"].astype("string")

years = pd.to_numeric(
    credit_history_text.str.extract(r"(\d+)\s+Years?")[0],
    errors="coerce"
)

months = pd.to_numeric(
    credit_history_text.str.extract(r"(\d+)\s+Months?")[0],
    errors="coerce"
)

df["Credit_History_Age"] = (
    years * 12 + months
).astype("Int64")

print("Tipe data :", df["Credit_History_Age"].dtype)
print("Missing   :", df["Credit_History_Age"].isna().sum())
print("Minimum   :", df["Credit_History_Age"].min())
print("Maximum   :", df["Credit_History_Age"].max())

Kolom `Credit_History_Age` menunjukkan lamanya riwayat kredit yang dimiliki oleh setiap nasabah. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat variasi nilai yang terdapat pada kolom tersebut.

Hasil pemeriksaan menunjukkan bahwa kolom masih berbentuk teks dengan format seperti:

* `22 Years and 1 Months`
* `15 Years and 11 Months`
* `3 Years and 5 Months`

Format tersebut belum dapat langsung digunakan dalam proses analisis maupun pemodelan karena masih berupa kombinasi teks dan angka.

Selanjutnya, tipe data kolom diperiksa menggunakan `dtype`. Hasil pemeriksaan menunjukkan bahwa kolom masih bertipe `object`, sehingga diperlukan proses transformasi ke bentuk numerik.

Untuk mempermudah analisis, seluruh nilai pada kolom `Credit_History_Age` diubah menjadi satuan bulan. Proses ini dilakukan dengan mengekstrak informasi tahun dan bulan dari setiap nilai, kemudian menghitung total bulan menggunakan rumus:

`Total Bulan = (Tahun × 12) + Bulan`

Sebagai contoh:

* `22 Years and 1 Months` → 265 bulan
* `15 Years and 11 Months` → 191 bulan
* `3 Years and 5 Months` → 41 bulan

Setelah proses konversi selesai, kolom berhasil diubah menjadi tipe data numerik `Int64`. Tipe data ini dipilih karena nilai yang dihasilkan berupa bilangan bulat dan tetap mendukung keberadaan missing value.

Pemeriksaan lanjutan menunjukkan bahwa:

* tipe data akhir menjadi `Int64`;
* terdapat **2.240 missing value**;
* nilai minimum sebesar **2 bulan**;
* nilai maksimum sebesar **404 bulan**.

Nilai minimum sebesar 2 bulan masih masuk akal karena dapat merepresentasikan nasabah yang baru memiliki riwayat kredit. Sementara itu, nilai maksimum sebesar 404 bulan setara dengan sekitar **33,7 tahun** riwayat kredit.

Perhitungannya adalah:

`404 ÷ 12 = 33,67 tahun`

Nilai maksimum tersebut masih realistis karena seseorang dapat memiliki riwayat kredit selama puluhan tahun sejak pertama kali membuka fasilitas kredit. Oleh karena itu, nilai maksimum yang ditemukan tidak menunjukkan indikasi anomali maupun kesalahan pencatatan data.

Selain itu, tidak ditemukan nilai negatif setelah proses konversi. Seluruh nilai yang berhasil dikonversi juga berada pada rentang yang masih logis untuk usia riwayat kredit seorang nasabah.



### f. Payment_of_Min_Ammount

In [ ]:
payment_min_amount_counts = (
    df["Payment_of_Min_Amount"]
    .value_counts(dropna=False)
    .rename_axis("Payment_of_Min_Amount")
    .reset_index(name="Count")
)

display(payment_min_amount_counts)

Kolom ini aman dan bersih

## 6. Prilaku Pembayaran

### a. Payment_Behaviour

In [ ]:
payment_behaviour_counts = (
    df["Payment_Behaviour"]
    .value_counts(dropna=False)
    .rename_axis("Payment_Behaviour")
    .reset_index(name="Count")
)

display(payment_behaviour_counts)

In [ ]:
# CHECK VALID PAYMENT BEHAVIOUR REFERENCE
invalid_payment_behaviour_mask = (
    df["Payment_Behaviour"].eq("!@9#%8")
    | df["Payment_Behaviour"].isna()
)

valid_payment_behaviour_mask = (
    ~invalid_payment_behaviour_mask
)

payment_behaviour_reference = (
    df.loc[valid_payment_behaviour_mask]
    .groupby("Customer_ID")["Payment_Behaviour"]
    .agg(
        Valid_Count="count",
        Valid_Values=lambda values: sorted(values.unique().tolist()),
        Suggested_Value=lambda values: values.mode().iloc[0]
    )
    .reset_index()
)

invalid_payment_behaviour_check = (
    df.loc[
        invalid_payment_behaviour_mask,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Payment_Behaviour"
        ]
    ]
    .merge(
        payment_behaviour_reference,
        on="Customer_ID",
        how="left"
    )
)

invalid_payment_behaviour_check["Has_Valid_Reference"] = (
    invalid_payment_behaviour_check["Valid_Count"].notna()
)

print(
    "Total nilai tidak valid:",
    len(invalid_payment_behaviour_check)
)

print(
    "Memiliki referensi valid:",
    invalid_payment_behaviour_check["Has_Valid_Reference"].sum()
)

print(
    "Tidak memiliki referensi valid:",
    (~invalid_payment_behaviour_check["Has_Valid_Reference"]).sum()
)

display(invalid_payment_behaviour_check)

In [ ]:
# REPAIR PAYMENT BEHAVIOUR VALUES
valid_payment_behaviour_reference = (
    df.loc[valid_payment_behaviour_mask]
    .groupby("Customer_ID")["Payment_Behaviour"]
    .agg(lambda values: values.mode().iloc[0])
)

suggested_payment_behaviour = (
    df.loc[invalid_payment_behaviour_mask, "Customer_ID"]
    .map(valid_payment_behaviour_reference)
)

fillable_indices = suggested_payment_behaviour[
    suggested_payment_behaviour.notna()
].index

unresolved_indices = suggested_payment_behaviour[
    suggested_payment_behaviour.isna()
].index

df.loc[
    fillable_indices,
    "Payment_Behaviour"
] = suggested_payment_behaviour.loc[fillable_indices]

df.loc[
    unresolved_indices,
    "Payment_Behaviour"
] = pd.NA

print(
    "Berhasil diperbaiki dari Customer_ID yang sama:",
    len(fillable_indices)
)

print(
    "Diubah menjadi missing value:",
    len(unresolved_indices)
)

print(
    "Nilai '!@9#%8' yang masih tersisa:",
    df["Payment_Behaviour"].eq("!@9#%8").sum()
)

print(
    "Total missing Payment_Behaviour:",
    df["Payment_Behaviour"].isna().sum()
)

Kolom `Payment_Behaviour` menunjukkan pola perilaku pembayaran nasabah berdasarkan kombinasi tingkat pengeluaran dan nilai pembayaran. Pemeriksaan awal dilakukan menggunakan `value_counts()` untuk melihat kategori yang tersedia dan frekuensi kemunculannya.

Hasil pemeriksaan menunjukkan bahwa sebagian besar nilai pada kolom ini berada dalam kategori yang valid, seperti:

* `Low_spent_Small_value_payments`
* `Low_spent_Medium_value_payments`
* `Low_spent_Large_value_payments`
* `High_spent_Small_value_payments`
* `High_spent_Medium_value_payments`
* `High_spent_Large_value_payments`

Namun, ditemukan juga nilai tidak valid berupa `!@9#%8`. Nilai tersebut tidak merepresentasikan kategori perilaku pembayaran yang sebenarnya, sehingga perlu ditangani sebelum data digunakan untuk pemodelan.

Nilai `!@9#%8` tidak langsung dihapus atau diganti dengan kategori paling sering dari seluruh dataset. Sebelum melakukan perbaikan, setiap baris dengan nilai tidak valid diperiksa berdasarkan `Customer_ID` untuk mengetahui apakah nasabah yang sama memiliki nilai `Payment_Behaviour` valid pada bulan lain.

Untuk setiap `Customer_ID`, dibuat beberapa informasi pendukung:

* `Valid_Count`, yaitu jumlah catatan valid milik nasabah yang sama;
* `Valid_Values`, yaitu daftar kategori valid yang pernah tercatat;
* `Suggested_Value`, yaitu kategori yang paling sering muncul atau modus;
* `Has_Valid_Reference`, yaitu penanda apakah nilai tidak valid memiliki referensi valid dari bulan lain.

Modus digunakan sebagai nilai pengganti karena `Payment_Behaviour` merupakan fitur kategorikal. Kategori yang paling sering muncul pada riwayat bulanan nasabah dianggap lebih representatif dibandingkan memilih kategori secara acak atau menggunakan kategori dominan dari seluruh dataset.

Nilai `Payment_Behaviour` yang memiliki referensi valid dari `Customer_ID` yang sama kemudian diperbaiki menggunakan `Suggested_Value`. Sementara itu, nilai yang tidak memiliki referensi valid tidak dipaksakan menjadi kategori tertentu, tetapi diubah menjadi missing value.

Setelah proses perbaikan selesai, dilakukan pemeriksaan ulang untuk memastikan bahwa:

* nilai tidak valid `!@9#%8` sudah tidak tersisa;
* nilai yang memiliki referensi valid berhasil diperbaiki;
* nilai tanpa referensi valid telah diubah menjadi missing value;
* kategori yang tersisa hanya kategori pembayaran valid dan missing value.

Hasil akhir menunjukkan bahwa simbol `!@9#%8` sudah berhasil dihapus dari kolom `Payment_Behaviour`. Missing value yang tersisa tidak langsung diisi pada tahap eksplorasi data. Nilai tersebut akan ditangani pada pipeline preprocessing menggunakan metode imputasi kategorikal yang hanya dipelajari dari data training agar tidak terjadi kebocoran informasi dari data testing.



# D. Exploratory Data Analysis Lanjutann

## 1. Melihat Hubungan Antar Kolom dan Lansung Melakukan Perbaikan Kalau Ada Kesalahan Data

### Setup EDA hubungan antar kolom

In [ ]:
eda = df.copy()

eda = eda.replace(
    [np.inf, -np.inf],
    np.nan
)

month_order = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8
}

eda["Month_Number"] = eda["Month"].map(month_order)

numeric_cols = [
    "Age",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Delay_from_due_date",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Num_Credit_Inquiries",
    "Outstanding_Debt",
    "Credit_Utilization_Ratio",
    "Credit_History_Age",
    "Total_EMI_per_month",
    "Amount_invested_monthly",
    "Monthly_Balance"
]

for col in numeric_cols:
    if col in eda.columns:
        eda[col] = pd.to_numeric(eda[col], errors="coerce")

print("Shape data EDA:", eda.shape)
print("Jumlah Customer_ID unik:", eda["Customer_ID"].nunique())
print("Total missing values:", eda.isna().sum().sum())

### Hubungan Annual_Income dengan Monthly_Inhand_Salary 

In [ ]:
# SETUP INCOME-SALARY RELATIONSHIP EDA

income_salary_eda = df.copy()

month_order = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8
}

income_salary_eda["Month_Number"] = (
    income_salary_eda["Month"]
    .map(month_order)
)

income_salary_eda["Annual_Income"] = pd.to_numeric(
    income_salary_eda["Annual_Income"],
    errors="coerce"
)

income_salary_eda["Monthly_Inhand_Salary"] = pd.to_numeric(
    income_salary_eda["Monthly_Inhand_Salary"],
    errors="coerce"
)

print("Shape data:", income_salary_eda.shape)
print("Missing Annual_Income:", income_salary_eda["Annual_Income"].isna().sum())
print("Missing Monthly_Inhand_Salary:", income_salary_eda["Monthly_Inhand_Salary"].isna().sum())

#### a. Distribusi Annual Income

In [ ]:
# CHECK ANNUAL INCOME DISTRIBUTION

annual_income_distribution = (
    income_salary_eda["Annual_Income"]
    .dropna()
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.97,
            0.98,
            0.99,
            0.995
        ]
    )
)

annual_income_distribution

Output ini menunjukkan distribusi kolom `Annual_Income` setelah formatnya dibersihkan dan dikonversi menjadi numerik. Jumlah data valid pada kolom ini adalah 25.000 baris, sehingga tidak terdapat missing value pada `Annual_Income`.

Secara umum, mayoritas nilai `Annual_Income` masih berada pada rentang yang cukup wajar. Nilai median adalah sekitar 37.313,32, persentil ke-75 sekitar 72.712,20, persentil ke-90 sekitar 114.015,36, dan persentil ke-99 sekitar 179.078,28. Ini menunjukkan bahwa 99% data masih berada di bawah sekitar 179 ribu.

Namun, terdapat perbedaan yang sangat besar antara persentil ke-99 dan persentil ke-99,5. Nilai persentil ke-99,5 langsung melonjak menjadi sekitar 11.469.299,38, sedangkan nilai maksimum mencapai 24.160.009,00. Lonjakan ini menunjukkan bahwa sebagian kecil data memiliki nilai `Annual_Income` yang sangat ekstrem dan jauh berbeda dari pola mayoritas data.

Hal ini juga terlihat dari nilai mean sebesar 162.340,41 yang jauh lebih tinggi dibandingkan median sebesar 37.313,32. Perbedaan besar antara mean dan median menunjukkan bahwa rata-rata tertarik ke atas oleh nilai-nilai ekstrem. Selain itu, standar deviasi juga sangat besar, yaitu sekitar 1.343.255,49, yang semakin memperkuat indikasi adanya outlier ekstrem.

Berdasarkan output ini, kolom `Annual_Income` tidak memiliki masalah missing value, tetapi memiliki masalah outlier yang sangat kuat. Oleh karena itu, kolom ini perlu dianalisis lebih lanjut dengan membandingkannya terhadap `Monthly_Inhand_Salary` dan riwayat `Customer_ID` yang sama. Jika nilai `Annual_Income` ekstrem tidak konsisten dengan gaji bulanan maupun catatan bulanan customer tersebut, maka nilai tersebut dapat dipertimbangkan untuk diperbaiki menggunakan referensi valid dari customer yang sama atau dijadikan missing value sebelum tahap preprocessing model.

#### b. Percentile jump Annual Income

In [ ]:
# CHECK ANNUAL INCOME PERCENTILE JUMP
annual_income_percentiles = (
    income_salary_eda["Annual_Income"]
    .dropna()
    .quantile([
        0.50,
        0.75,
        0.90,
        0.95,
        0.97,
        0.98,
        0.99,
        0.995,
        1.00
    ])
    .reset_index()
)

annual_income_percentiles.columns = [
    "Percentile",
    "Annual_Income"
]

annual_income_percentiles["Percentile"] = (
    annual_income_percentiles["Percentile"] * 100
)

annual_income_percentiles["Increase"] = (
    annual_income_percentiles["Annual_Income"]
    .diff()
)

annual_income_percentiles

Output percentile pada kolom `Annual_Income` menunjukkan adanya indikasi outlier ekstrem yang sangat kuat. Pada bagian mayoritas data, nilai `Annual_Income` masih meningkat secara bertahap. Median berada di sekitar 37.313,32, persentil ke-75 sekitar 72.712,20, persentil ke-90 sekitar 114.015,36, dan persentil ke-99 sekitar 179.078,28.

Namun, setelah persentil ke-99, terlihat lonjakan yang sangat besar. Nilai persentil ke-99,5 langsung naik menjadi sekitar 11.469.299,38, padahal persentil ke-99 hanya sekitar 179.078,28. Kenaikannya mencapai lebih dari 11 juta. Nilai maksimum bahkan mencapai 24.160.009,00.

Lonjakan ini menunjukkan bahwa sebagian kecil nilai `Annual_Income` sangat berbeda dari pola mayoritas data. Jika dibandingkan dengan distribusi sebelumnya, nilai sampai persentil ke-99 masih berada pada rentang yang relatif masuk akal, sedangkan nilai setelah itu melonjak terlalu jauh ke jutaan hingga puluhan juta.

Dengan demikian, kolom `Annual_Income` memiliki indikasi outlier ekstrem. Nilai ekstrem tersebut tidak langsung dihapus, tetapi perlu diperiksa lebih lanjut dengan membandingkannya terhadap `Monthly_Inhand_Salary` dan riwayat bulanan dari `Customer_ID` yang sama. Jika nilai `Annual_Income` ekstrem tidak konsisten dengan gaji bulanan dan catatan customer yang sama, maka nilai tersebut dapat dipertimbangkan untuk diperbaiki menggunakan referensi customer yang sama atau dijadikan missing value sebelum masuk ke pipeline preprocessing.

#### c. Distribusi Monthly Inhand Salary

In [ ]:
# CHECK MONTHLY INHAND SALARY DISTRIBUTION

monthly_salary_distribution = (
    income_salary_eda["Monthly_Inhand_Salary"]
    .dropna()
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.97,
            0.98,
            0.99,
            0.995
        ]
    )
)

monthly_salary_distribution

Output ini menunjukkan distribusi kolom `Monthly_Inhand_Salary` setelah hanya mengambil data yang tidak missing. Jumlah data valid yang dianalisis adalah 21.164 baris.

Nilai rata-rata `Monthly_Inhand_Salary` adalah sekitar 4.189,51, sedangkan median atau persentil ke-50 adalah sekitar 3.071,42. Karena mean lebih besar daripada median, distribusi gaji bulanan sedikit condong ke kanan, tetapi tidak menunjukkan lonjakan ekstrem yang terlalu besar.

Nilai minimum adalah 303,65 dan nilai maksimum adalah 15.204,63. Pada bagian atas distribusi, persentil ke-95 berada di sekitar 10.872,25, persentil ke-99 sekitar 13.943,25, dan persentil ke-99,5 sekitar 14.490,78. Nilai maksimum 15.204,63 masih cukup dekat dengan persentil ke-99,5, sehingga tidak terlihat lonjakan yang sangat tidak wajar.

Dengan demikian, kolom `Monthly_Inhand_Salary` secara distribusi terlihat cukup stabil dan tidak menunjukkan indikasi outlier ekstrem yang kuat. Kolom ini lebih aman dibandingkan `Annual_Income`, karena nilai-nilai tinggi pada `Monthly_Inhand_Salary` masih meningkat secara bertahap dan tidak melonjak ke angka yang sangat jauh dari pola mayoritas data.

#### d. Percentile jump Monthly Inhand Salary

In [ ]:
# CHECK MONTHLY INHAND SALARY PERCENTILE JUMP

monthly_salary_percentiles = (
    income_salary_eda["Monthly_Inhand_Salary"]
    .dropna()
    .quantile([
        0.50,
        0.75,
        0.90,
        0.95,
        0.97,
        0.98,
        0.99,
        0.995,
        1.00
    ])
    .reset_index()
)

monthly_salary_percentiles.columns = [
    "Percentile",
    "Monthly_Inhand_Salary"
]

monthly_salary_percentiles["Percentile"] = (
    monthly_salary_percentiles["Percentile"] * 100
)

monthly_salary_percentiles["Increase"] = (
    monthly_salary_percentiles["Monthly_Inhand_Salary"]
    .diff()
)

monthly_salary_percentiles

Hasil percentile pada kolom `Monthly_Inhand_Salary` menunjukkan bahwa distribusi gaji bulanan masih relatif stabil dan tidak menunjukkan lonjakan ekstrem. Nilai median berada di sekitar 3.071,42, sedangkan persentil ke-95 berada di sekitar 10.872,25. Pada bagian atas distribusi, kenaikan dari persentil ke-95 sampai maksimum masih bertahap, yaitu dari 10.872,25 menjadi 15.204,63.

Tidak terlihat lonjakan yang terlalu drastis antara persentil tinggi seperti P99, P99.5, dan nilai maksimum. Nilai maksimum 15.204,63 juga masih cukup dekat dengan P99.5 sebesar 14.490,78. Hal ini menunjukkan bahwa `Monthly_Inhand_Salary` tidak memiliki indikasi outlier ekstrem yang kuat.

Dengan demikian, kolom `Monthly_Inhand_Salary` secara distribusi terlihat lebih aman dibandingkan `Annual_Income`. Masalah utama pada hubungan income-salary lebih kuat berasal dari nilai ekstrem pada `Annual_Income`, bukan dari `Monthly_Inhand_Salary`.

### e. Buat kolom hubungan income-salary

In [ ]:
# CREATE INCOME-SALARY RELATIONSHIP FEATURES

salary_relation_data = income_salary_eda.dropna(
    subset=[
        "Annual_Income",
        "Monthly_Inhand_Salary"
    ]
).copy()

salary_relation_data["Expected_Monthly_Salary"] = (
    salary_relation_data["Annual_Income"] / 12
)

salary_relation_data["Salary_Ratio"] = (
    salary_relation_data["Monthly_Inhand_Salary"]
    / salary_relation_data["Expected_Monthly_Salary"]
)

salary_relation_data["Salary_Difference_Percentage"] = (
    (
        salary_relation_data["Monthly_Inhand_Salary"]
        - salary_relation_data["Expected_Monthly_Salary"]
    ).abs()
    / salary_relation_data["Expected_Monthly_Salary"]
    * 100
)

salary_relation_data = salary_relation_data.replace(
    [np.inf, -np.inf],
    np.nan
)

salary_relation_data = salary_relation_data.dropna(
    subset=[
        "Expected_Monthly_Salary",
        "Salary_Ratio",
        "Salary_Difference_Percentage"
    ]
).copy()

print("Jumlah data valid untuk hubungan income-salary:", len(salary_relation_data))

Kode ini digunakan untuk membuat fitur tambahan yang membantu mengecek konsistensi hubungan antara `Annual_Income` dan `Monthly_Inhand_Salary`.

Pertama, data disimpan ke dalam variabel `salary_relation_data` dengan hanya mengambil baris yang memiliki nilai lengkap pada kolom `Annual_Income` dan `Monthly_Inhand_Salary`. Hal ini dilakukan karena perhitungan hubungan kedua kolom membutuhkan kedua nilai tersebut tersedia.

Kemudian dibuat kolom `Expected_Monthly_Salary`, yaitu estimasi gaji bulanan berdasarkan `Annual_Income / 12`. Kolom ini digunakan sebagai pembanding terhadap `Monthly_Inhand_Salary` aktual.

Selanjutnya, dibuat kolom `Salary_Ratio`, yaitu perbandingan antara `Monthly_Inhand_Salary` aktual dengan `Expected_Monthly_Salary`. Jika nilainya mendekati 1, berarti gaji bulanan aktual cukup konsisten dengan pendapatan tahunan.

Selain itu, dibuat kolom `Salary_Difference_Percentage` untuk menghitung persentase selisih antara `Monthly_Inhand_Salary` aktual dan `Expected_Monthly_Salary`. Nilai ini membantu mengidentifikasi baris yang memiliki perbedaan sangat besar antara pendapatan tahunan dan gaji bulanan.

Nilai `inf` dan `-inf` kemudian diganti menjadi missing value, karena nilai tersebut dapat muncul jika terjadi pembagian dengan nilai nol atau tidak valid. Setelah itu, baris yang masih memiliki missing value pada kolom hasil perhitungan dihapus sementara agar analisis hubungan income-salary hanya menggunakan data yang valid.

Output `Jumlah data valid untuk hubungan income-salary` menunjukkan jumlah baris yang dapat digunakan untuk menganalisis konsistensi antara `Annual_Income` dan `Monthly_Inhand_Salary`.

#### f. Ringkasan hubungan income-salary

In [ ]:
# CHECK INCOME-SALARY RELATIONSHIP SUMMARY

salary_relation_summary = (
    salary_relation_data[
        [
            "Annual_Income",
            "Monthly_Inhand_Salary",
            "Expected_Monthly_Salary",
            "Salary_Ratio",
            "Salary_Difference_Percentage"
        ]
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

salary_relation_summary

Berdasarkan output hubungan antara `Annual_Income` dan `Monthly_Inhand_Salary`, kedua kolom secara umum masih memiliki hubungan yang cukup konsisten pada mayoritas data. Hal ini terlihat dari median `Annual_Income` sebesar 37.367,34, sehingga `Expected_Monthly_Salary` atau `Annual_Income / 12` menjadi sekitar 3.113,95. Nilai ini sangat dekat dengan median `Monthly_Inhand_Salary` sebesar 3.071,42.

Selain itu, median `Salary_Ratio` sebesar 0,9976 menunjukkan bahwa pada sebagian besar data, `Monthly_Inhand_Salary` hampir sama dengan estimasi gaji bulanan dari `Annual_Income / 12`. Median `Salary_Difference_Percentage` juga hanya sekitar 4,14%, sehingga perbedaan antara gaji bulanan aktual dan estimasi gaji bulanan masih relatif kecil.

Namun, pada bagian maksimum terlihat perbedaan yang sangat besar. Nilai maksimum `Annual_Income` mencapai 24.160.009, sehingga `Expected_Monthly_Salary` menjadi 2.013.334,08. Sementara itu, nilai maksimum `Monthly_Inhand_Salary` hanya 15.204,63. Hal ini menyebabkan nilai minimum `Salary_Ratio` sangat kecil, yaitu 0,0004, dan nilai maksimum `Salary_Difference_Percentage` mencapai 99,96%.

Dengan demikian, mayoritas hubungan antara `Annual_Income` dan `Monthly_Inhand_Salary` masih valid, tetapi terdapat sebagian kecil data dengan `Annual_Income` yang sangat ekstrem dan tidak sebanding dengan `Monthly_Inhand_Salary`. Jadi, masalah utama lebih kuat berasal dari outlier pada `Annual_Income`, bukan dari `Monthly_Inhand_Salary`.

#### g. Cek kasus paling tidak konsisten

In [ ]:
# CHECK MOST INCONSISTENT ANNUAL INCOME AND MONTHLY SALARY

salary_inconsistent = (
    salary_relation_data[
        [
            "ID",
            "Customer_ID",
            "Month",
            "Month_Number",
            "Annual_Income",
            "Monthly_Inhand_Salary",
            "Expected_Monthly_Salary",
            "Salary_Ratio",
            "Salary_Difference_Percentage",
            "Credit_Score"
        ]
    ]
    .sort_values(
        by="Salary_Difference_Percentage",
        ascending=False
    )
)

salary_inconsistent.head(30)

Output ini menampilkan baris-baris dengan ketidaksesuaian paling besar antara `Annual_Income` dan `Monthly_Inhand_Salary`. Data diurutkan berdasarkan `Salary_Difference_Percentage` tertinggi, sehingga baris paling atas adalah kasus yang paling tidak konsisten.

Dari hasil tersebut, terlihat bahwa banyak customer memiliki `Annual_Income` yang sangat besar, mulai dari jutaan hingga puluhan juta. Contohnya, `CUS_ANON_005` memiliki `Annual_Income` sebesar 10.318.159, sehingga `Expected_Monthly_Salary` menjadi sekitar 859.846,58. Namun, `Monthly_Inhand_Salary` aktualnya hanya 332,43. Akibatnya, `Salary_Ratio` menjadi sangat kecil, yaitu 0,0004, dan `Salary_Difference_Percentage` mencapai 99,96%.

Pola yang sama juga terlihat pada baris lain. Misalnya, `CUS_ANON_017` memiliki `Annual_Income` sebesar 20.703.498 dengan estimasi gaji bulanan sekitar 1.725.291,50, tetapi `Monthly_Inhand_Salary` aktualnya hanya 680,36. Selisih yang sangat besar ini menunjukkan bahwa nilai `Annual_Income` pada baris tersebut tidak sebanding dengan gaji bulanan aktual.

Secara logika, jika `Annual_Income` benar-benar berada pada angka jutaan atau puluhan juta, maka `Monthly_Inhand_Salary` seharusnya juga jauh lebih tinggi. Namun, pada output ini, `Monthly_Inhand_Salary` hanya berada pada kisaran ratusan sampai sekitar dua ribuan. Hal ini menunjukkan bahwa masalah utama kemungkinan besar berasal dari nilai ekstrem pada `Annual_Income`, bukan dari `Monthly_Inhand_Salary`.

Nilai `Salary_Ratio` yang sangat kecil, seperti 0,0004 sampai 0,0012, menunjukkan bahwa gaji bulanan aktual hanya sebagian sangat kecil dari estimasi gaji bulanan berdasarkan pendapatan tahunan. Selain itu, `Salary_Difference_Percentage` yang mendekati 100% memperkuat bahwa terdapat ketidaksesuaian ekstrem antara kedua kolom.

Dari output ini, dapat disimpulkan bahwa sebagian kecil nilai `Annual_Income` memiliki indikasi outlier yang sangat kuat berdasarkan hubungan antarvariabel. Nilai-nilai tersebut perlu diperiksa lebih lanjut menggunakan riwayat `Customer_ID` yang sama. Jika pada bulan lain customer tersebut memiliki `Annual_Income` yang jauh lebih rendah dan lebih konsisten dengan `Monthly_Inhand_Salary`, maka nilai ekstrem tersebut dapat dipertimbangkan untuk diperbaiki menggunakan referensi customer yang sama atau dijadikan missing value sebelum masuk ke pipeline preprocessing.

#### h. Cek riwayat customer dari kasus ekstrem

In [ ]:
# CHECK CUSTOMER HISTORY FOR MOST INCONSISTENT SALARY CASES

sample_salary_customers = (
    salary_inconsistent["Customer_ID"]
    .drop_duplicates()
    .head(5)
    .tolist()
)

for customer_id in sample_salary_customers:
    print("\nCustomer_ID:", customer_id)

    customer_history = income_salary_eda.loc[
        income_salary_eda["Customer_ID"] == customer_id,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Month_Number",
            "Annual_Income",
            "Monthly_Inhand_Salary",
            "Credit_Score"
        ]
    ].sort_values("Month_Number")

    print(customer_history)

Output ini menunjukkan riwayat bulanan beberapa `Customer_ID` yang sebelumnya terdeteksi memiliki ketidaksesuaian ekstrem antara `Annual_Income` dan `Monthly_Inhand_Salary`. Pengecekan ini dilakukan untuk memastikan apakah nilai ekstrem pada `Annual_Income` memang konsisten dengan riwayat customer yang sama atau hanya muncul pada bulan tertentu.

Dari hasil yang ditampilkan, terlihat bahwa pada beberapa customer, `Monthly_Inhand_Salary` cenderung stabil antarbulan, sedangkan `Annual_Income` tiba-tiba melonjak sangat besar pada satu bulan tertentu.

Contohnya, pada `CUS_ANON_005`, bulan March memiliki `Annual_Income` sebesar 10.318.159, tetapi pada bulan June untuk customer yang sama nilai `Annual_Income` hanya 7.577,175. Menariknya, `Monthly_Inhand_Salary` pada kedua bulan tersebut tetap sama, yaitu 332,4312. Hal ini menunjukkan bahwa nilai `Annual_Income` pada bulan March kemungkinan besar tidak konsisten dengan riwayat customer tersebut.

Pola yang sama juga terlihat pada `CUS_ANON_017`. Pada January, March, dan June, `Annual_Income` tercatat sebesar 11.176,2700, sedangkan pada July tiba-tiba melonjak menjadi 20.703.498. Namun, `Monthly_Inhand_Salary` tetap berada pada nilai 680,3558. Jika pendapatan tahunan benar-benar naik sampai puluhan juta, seharusnya gaji bulanan juga ikut meningkat secara signifikan. Karena hal tersebut tidak terjadi, nilai `Annual_Income` pada July patut dicurigai sebagai outlier.

Pada `CUS_ANON_003`, nilai `Annual_Income` pada March, June, dan July adalah 14.455,1500, tetapi pada April melonjak menjadi 24.096.975. Sementara itu, `Monthly_Inhand_Salary` tetap sama, yaitu 912,5958. Hal ini semakin memperkuat bahwa anomali lebih mungkin terjadi pada `Annual_Income`, bukan pada `Monthly_Inhand_Salary`.

Selain itu, pada `CUS_ANON_034`, beberapa bulan memiliki `Annual_Income` sebesar 8.422,3850 dengan `Monthly_Inhand_Salary` sebesar 873,8654. Namun, pada June, `Annual_Income` tiba-tiba menjadi 21.367.283, sedangkan `Monthly_Inhand_Salary` tetap sama. Pola ini menunjukkan adanya lonjakan tidak wajar pada `Annual_Income`.

Berdasarkan output ini, dapat disimpulkan bahwa nilai ekstrem pada `Annual_Income` tidak konsisten dengan riwayat bulanan customer yang sama. Sebaliknya, `Monthly_Inhand_Salary` terlihat lebih stabil dan lebih masuk akal. Oleh karena itu, masalah utama pada hubungan kedua kolom ini lebih kuat berasal dari outlier pada `Annual_Income`.

Nilai ekstrem pada `Annual_Income` sebaiknya tidak langsung dihapus, tetapi dapat ditangani dengan memanfaatkan referensi valid dari `Customer_ID` yang sama. Jika customer tersebut memiliki nilai `Annual_Income` lain yang lebih konsisten pada bulan berbeda, nilai ekstrem dapat diganti menggunakan median dari nilai valid tersebut. Jika tidak tersedia referensi valid, nilai ekstrem dapat dijadikan missing value dan ditangani pada pipeline preprocessing model.

#### i. Cek jumlah kasus selisih besar

In [ ]:
# CHECK SALARY DIFFERENCE THRESHOLDS

salary_difference_thresholds = [
    20,
    30,
    50,
    70,
    90
]

salary_difference_summary = []

for threshold in salary_difference_thresholds:
    count_above = (
        salary_relation_data["Salary_Difference_Percentage"] > threshold
    ).sum()

    percentage_above = (
        count_above / len(salary_relation_data) * 100
    )

    salary_difference_summary.append({
        "Salary_Difference_Threshold": threshold,
        "Count_Above_Threshold": count_above,
        "Percentage": percentage_above
    })

salary_difference_summary = pd.DataFrame(
    salary_difference_summary
)

salary_difference_summary

Kode ini digunakan untuk melihat berapa banyak data yang memiliki selisih besar antara `Monthly_Inhand_Salary` aktual dan `Expected_Monthly_Salary` dari `Annual_Income / 12`.

Pertama, dibuat beberapa batas atau threshold, yaitu 20%, 30%, 50%, 70%, dan 90%. Kemudian kode menghitung jumlah baris yang memiliki `Salary_Difference_Percentage` lebih besar dari masing-masing threshold tersebut. Setelah itu, jumlahnya juga diubah menjadi persentase terhadap total data valid pada `salary_relation_data`.

Hasil output menunjukkan bahwa terdapat 1.398 baris atau sekitar 6,61% data dengan selisih gaji lebih dari 20%. Untuk threshold 30%, jumlahnya turun menjadi 620 baris atau sekitar 2,93%. Sementara itu, untuk threshold 50% terdapat 188 baris atau sekitar 0,89%.

Menariknya, jumlah data pada threshold 50% dan 70% sama-sama 188 baris, lalu pada threshold 90% hanya turun sedikit menjadi 187 baris. Hal ini menunjukkan bahwa data yang sudah memiliki selisih di atas 50% umumnya memang sangat ekstrem, bahkan hampir semuanya memiliki selisih lebih dari 90%.

Dengan demikian, threshold 90% dapat digunakan sebagai batas investigasi untuk kasus yang benar-benar sangat tidak konsisten antara `Annual_Income` dan `Monthly_Inhand_Salary`. Jumlahnya hanya 187 baris atau sekitar 0,88% dari data valid, sehingga kasus ini merupakan sebagian kecil data tetapi memiliki indikasi anomali yang kuat.

#### j. Cek kasus Salary Difference > 90%

In [ ]:
# CHECK EXTREME SALARY DIFFERENCE CASES

salary_anomaly_threshold = 90

salary_anomaly = salary_relation_data[
    salary_relation_data["Salary_Difference_Percentage"] > salary_anomaly_threshold
].copy()

print("Jumlah kasus Salary Difference > 90%:", len(salary_anomaly))
print(
    "Persentase kasus Salary Difference > 90%:",
    round(len(salary_anomaly) / len(salary_relation_data) * 100, 2),
    "%"
)

salary_anomaly[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Expected_Monthly_Salary",
        "Salary_Ratio",
        "Salary_Difference_Percentage",
        "Credit_Score"
    ]
].sort_values(
    by="Salary_Difference_Percentage",
    ascending=False
).head(30)

Output ini menampilkan baris-baris yang memiliki `Salary_Difference_Percentage` paling tinggi, yaitu kasus ketika `Monthly_Inhand_Salary` sangat tidak sesuai dengan `Expected_Monthly_Salary` yang dihitung dari `Annual_Income / 12`.

Pada output ini, seluruh baris yang muncul memiliki `Salary_Difference_Percentage` sekitar 99,88% sampai 99,96%. Artinya, selisih antara gaji bulanan aktual dan estimasi gaji bulanan dari annual income hampir mencapai 100%. Kondisi ini menunjukkan ketidaksesuaian yang sangat ekstrem.

Sebagai contoh, pada `Customer_ID = CUS_ANON_005`, nilai `Annual_Income` adalah 10.318.159, sehingga `Expected_Monthly_Salary` menjadi sekitar 859.846,58. Namun, `Monthly_Inhand_Salary` aktual hanya 332,43. Akibatnya, `Salary_Ratio` hanya 0,0004 dan `Salary_Difference_Percentage` mencapai 99,96%. Secara logika, jika pendapatan tahunan benar-benar sebesar itu, maka gaji bulanan yang diterima seharusnya jauh lebih tinggi.

Pola yang sama terlihat pada banyak baris lain. Nilai `Annual_Income` berada pada angka jutaan hingga puluhan juta, sedangkan `Monthly_Inhand_Salary` hanya berada pada kisaran ratusan sampai sekitar dua ribuan. Hal ini menunjukkan bahwa nilai ekstrem lebih kuat berasal dari `Annual_Income`, bukan dari `Monthly_Inhand_Salary`.

Threshold 90% digunakan sebagai batas investigasi karena nilai tersebut menunjukkan ketidaksesuaian yang sangat besar antara dua kolom yang secara logika seharusnya saling berhubungan. Jika `Salary_Difference_Percentage` lebih dari 90%, berarti `Monthly_Inhand_Salary` aktual berbeda lebih dari 90% dari estimasi gaji bulanan berdasarkan `Annual_Income / 12`.

Pemilihan threshold 90% juga didukung oleh hasil perbandingan threshold sebelumnya. Jumlah data dengan selisih lebih dari 50% adalah 188 baris, sedangkan jumlah data dengan selisih lebih dari 90% adalah 187 baris. Artinya, hampir semua data yang sudah melewati threshold 50% ternyata memang berada pada kondisi ekstrem, yaitu selisih di atas 90%. Oleh karena itu, threshold 90% dipilih agar investigasi difokuskan pada kasus yang benar-benar paling tidak wajar, bukan pada perbedaan kecil atau sedang yang masih mungkin terjadi karena potongan gaji, pajak, atau perbedaan antara pendapatan kotor dan pendapatan bersih.

Dengan demikian, output ini memperkuat bahwa terdapat sebagian kecil data yang memiliki anomali serius pada hubungan `Annual_Income` dan `Monthly_Inhand_Salary`. Nilai-nilai ini tidak langsung dihapus, tetapi perlu diperiksa lebih lanjut menggunakan riwayat `Customer_ID` yang sama. Jika customer yang sama memiliki nilai `Annual_Income` lain yang lebih konsisten dengan `Monthly_Inhand_Salary`, maka nilai ekstrem tersebut dapat diperbaiki menggunakan referensi customer tersebut. Jika tidak ada referensi valid, nilai tersebut dapat dijadikan missing value dan ditangani pada pipeline preprocessing model.

#### k. Cek referensi valid Annual Income dari customer yang sama

In [ ]:
# CHECK ANNUAL INCOME EXTREME CASES WITH SAME CUSTOMER REFERENCE

valid_annual_reference = (
    salary_relation_data[
        salary_relation_data["Salary_Difference_Percentage"] <= 50
    ]
    .groupby("Customer_ID")["Annual_Income"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)

salary_anomaly_check = (
    salary_anomaly[
        [
            "ID",
            "Customer_ID",
            "Month",
            "Annual_Income",
            "Monthly_Inhand_Salary",
            "Expected_Monthly_Salary",
            "Salary_Ratio",
            "Salary_Difference_Percentage",
            "Credit_Score"
        ]
    ]
    .merge(
        valid_annual_reference,
        on="Customer_ID",
        how="left"
    )
)

salary_anomaly_check["Has_Valid_Annual_Reference"] = (
    salary_anomaly_check["count"].notna()
)

print("Jumlah kasus Salary Difference > 90%:", len(salary_anomaly_check))
print(
    "Memiliki referensi Annual_Income valid dari Customer_ID yang sama:",
    salary_anomaly_check["Has_Valid_Annual_Reference"].sum()
)
print(
    "Tidak memiliki referensi valid:",
    (~salary_anomaly_check["Has_Valid_Annual_Reference"]).sum()
)

salary_anomaly_check.sort_values(
    by="Salary_Difference_Percentage",
    ascending=False
).head(30)

Output ini menunjukkan hasil pengecekan kasus ekstrem pada hubungan antara `Annual_Income` dan `Monthly_Inhand_Salary`, khususnya untuk baris dengan `Salary_Difference_Percentage > 90%`.

Hasilnya menunjukkan terdapat 187 kasus dengan selisih lebih dari 90%. Artinya, terdapat 187 baris di mana `Monthly_Inhand_Salary` aktual sangat tidak sesuai dengan estimasi gaji bulanan dari `Annual_Income / 12`. Jumlah ini relatif kecil dibanding total data valid, tetapi kasusnya sangat penting karena menunjukkan ketidaksesuaian yang ekstrem.

Dari 187 kasus tersebut, sebanyak 149 kasus memiliki referensi `Annual_Income` valid dari `Customer_ID` yang sama. Artinya, pada customer yang sama terdapat catatan bulan lain dengan nilai `Annual_Income` yang lebih masuk akal dan lebih konsisten dengan `Monthly_Inhand_Salary`. Sementara itu, 38 kasus tidak memiliki referensi valid dari customer yang sama, sehingga nilai ekstrem pada kasus tersebut tidak bisa diperbaiki langsung menggunakan riwayat customer.

Output tabel juga memperkuat bahwa masalah utama berasal dari `Annual_Income`. Contohnya, `CUS_ANON_005` memiliki `Annual_Income` ekstrem sebesar 10.318.159 dengan `Monthly_Inhand_Salary` hanya 332,4312. Namun, customer yang sama memiliki referensi valid `Annual_Income` sebesar 7.577,175. Perbedaan ini menunjukkan bahwa nilai 10.318.159 kemungkinan besar merupakan outlier.

Contoh lain adalah `CUS_ANON_017`, yang memiliki `Annual_Income` ekstrem sebesar 20.703.498, tetapi referensi valid dari customer yang sama menunjukkan median `Annual_Income` hanya 11.176,2700. Pola seperti ini juga terlihat pada banyak baris lain, di mana nilai `Annual_Income` ekstrem berada pada jutaan hingga puluhan juta, sedangkan referensi valid dari customer yang sama hanya berada pada ribuan sampai puluhan ribu.

Kolom `Has_Valid_Annual_Reference` membantu membedakan kasus yang bisa diperbaiki menggunakan riwayat customer dan kasus yang tidak memiliki referensi valid. Jika nilainya `True`, maka nilai ekstrem dapat dipertimbangkan untuk diganti menggunakan median `Annual_Income` valid dari customer yang sama. Jika nilainya `False`, maka nilai ekstrem tidak sebaiknya dipaksakan dengan nilai customer lain, tetapi lebih aman dijadikan missing value dan nanti ditangani oleh pipeline preprocessing.

Berdasarkan output ini, keputusan yang paling aman adalah menandai `Annual_Income` ekstrem sebagai anomali berbasis hubungan antarvariabel. Untuk 149 kasus yang memiliki referensi valid, nilai ekstrem dapat diperbaiki menggunakan median `Annual_Income` dari `Customer_ID` yang sama. Untuk 38 kasus tanpa referensi valid, nilai ekstrem dapat diubah menjadi missing value agar tidak merusak distribusi data dan tidak memberi sinyal yang salah pada model.

#### l. Cek threshold Annual Income ekstrem

In [ ]:
# CHECK ANNUAL INCOME EXTREME THRESHOLD

annual_income_thresholds = [
    150000,
    180000,
    200000,
    500000,
    1000000,
    5000000
]

annual_income_threshold_summary = []

for threshold in annual_income_thresholds:
    count_above = (
        income_salary_eda["Annual_Income"] > threshold
    ).sum()

    percentage_above = (
        count_above / len(income_salary_eda) * 100
    )

    annual_income_threshold_summary.append({
        "Annual_Income_Threshold": threshold,
        "Count_Above_Threshold": count_above,
        "Percentage": percentage_above
    })

annual_income_threshold_summary = pd.DataFrame(
    annual_income_threshold_summary
)

annual_income_threshold_summary

Output ini digunakan untuk membandingkan beberapa batas atau threshold pada kolom `Annual_Income`. Tujuannya adalah menentukan batas investigasi yang masuk akal untuk menandai nilai `Annual_Income` yang ekstrem.

Hasilnya menunjukkan bahwa terdapat 749 baris atau sekitar 3,00% data dengan `Annual_Income > 150.000`. Namun, ketika threshold dinaikkan menjadi 180.000, jumlahnya turun cukup besar menjadi 224 baris atau sekitar 0,90%. Jumlah ini sama dengan threshold 200.000, yaitu tetap 224 baris. Artinya, tidak ada nilai `Annual_Income` yang berada di antara 180.000 sampai 200.000.

Ketika threshold dinaikkan lagi menjadi 500.000, jumlah data hanya turun sedikit menjadi 222 baris. Pada threshold 1.000.000, jumlahnya masih 218 baris, dan pada threshold 5.000.000 masih terdapat 184 baris. Hal ini menunjukkan bahwa sebagian besar nilai ekstrem pada `Annual_Income` bukan hanya sedikit di atas batas normal, tetapi melonjak sangat jauh sampai jutaan.

Berdasarkan output ini, threshold 200.000 dapat digunakan sebagai batas investigasi yang cukup aman. Alasannya, nilai persentil ke-99 sebelumnya berada di sekitar 179.078, sehingga batas 200.000 berada sedikit di atas P99 dan tidak terlalu agresif. Selain itu, jumlah data di atas 200.000 hanya 224 baris atau sekitar 0,90% dari total data, sehingga kasus ini merupakan sebagian kecil data yang memang layak diperiksa lebih lanjut.

Dengan demikian, nilai `Annual_Income > 200.000` dapat ditandai sebagai kandidat outlier. Nilai tersebut tidak langsung dihapus, tetapi perlu dicek menggunakan riwayat `Customer_ID` yang sama. Jika customer memiliki nilai `Annual_Income` lain yang valid dan lebih konsisten, maka nilai ekstrem dapat diganti menggunakan median dari customer tersebut. Jika tidak ada referensi valid, nilai ekstrem dapat dijadikan missing value dan ditangani pada pipeline preprocessing model.

#### m. Cek Annual Income > 200000 dengan referensi customer

In [ ]:
# CHECK ANNUAL INCOME ABOVE 200000 WITH SAME CUSTOMER REFERENCE

annual_income_boundary = 200000

annual_income_extreme = income_salary_eda[
    income_salary_eda["Annual_Income"] > annual_income_boundary
].copy()

annual_income_valid_reference = (
    income_salary_eda[
        income_salary_eda["Annual_Income"].le(annual_income_boundary)
        & income_salary_eda["Annual_Income"].notna()
    ]
    .groupby("Customer_ID")["Annual_Income"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)

annual_income_extreme_check = (
    annual_income_extreme[
        [
            "ID",
            "Customer_ID",
            "Month",
            "Annual_Income",
            "Monthly_Inhand_Salary",
            "Credit_Score"
        ]
    ]
    .merge(
        annual_income_valid_reference,
        on="Customer_ID",
        how="left"
    )
)

annual_income_extreme_check["Has_Valid_Annual_Reference"] = (
    annual_income_extreme_check["count"].notna()
)

print("Total Annual_Income > 200000:", len(annual_income_extreme_check))
print(
    "Memiliki referensi valid dari Customer_ID yang sama:",
    annual_income_extreme_check["Has_Valid_Annual_Reference"].sum()
)
print(
    "Tidak memiliki referensi valid:",
    (~annual_income_extreme_check["Has_Valid_Annual_Reference"]).sum()
)

annual_income_extreme_check.head(30)

Output ini menunjukkan hasil pengecekan nilai `Annual_Income` yang berada di atas batas investigasi 200.000. Batas 200.000 digunakan karena sebelumnya terlihat bahwa persentil ke-99 `Annual_Income` masih berada di sekitar 179.078, sehingga nilai di atas 200.000 sudah berada di luar pola mayoritas data.

Hasil pengecekan menunjukkan bahwa terdapat 224 baris dengan `Annual_Income > 200.000`. Dari 224 baris tersebut, sebanyak 195 baris memiliki referensi valid dari `Customer_ID` yang sama, sedangkan 29 baris tidak memiliki referensi valid.

Referensi valid berarti customer tersebut memiliki catatan `Annual_Income` lain pada bulan berbeda dengan nilai yang lebih masuk akal, yaitu tidak melebihi batas 200.000. Referensi ini digunakan untuk melihat apakah nilai ekstrem tersebut hanya muncul pada satu bulan tertentu atau memang konsisten pada customer yang sama.

Dari output tabel, terlihat bahwa banyak nilai `Annual_Income` ekstrem berada pada jutaan hingga puluhan juta. Namun, untuk customer yang sama, nilai referensi validnya jauh lebih rendah. Contohnya, `CUS_ANON_024` memiliki `Annual_Income` ekstrem sebesar 918.988, tetapi median referensi valid customer tersebut hanya 30.821,30. Contoh lain, `CUS_ANON_020` memiliki `Annual_Income` ekstrem sebesar 24.160.009, tetapi median referensi validnya hanya 47.291,79. Perbedaan yang sangat besar ini menunjukkan bahwa nilai ekstrem tersebut tidak konsisten dengan riwayat customer.

Kolom `count`, `median`, `min`, dan `max` menunjukkan ringkasan nilai `Annual_Income` valid dari customer yang sama. Jika `Has_Valid_Annual_Reference` bernilai `True`, artinya nilai ekstrem memiliki referensi dari bulan lain dan dapat dipertimbangkan untuk diperbaiki menggunakan median customer tersebut. Jika bernilai `False`, artinya tidak ada referensi valid dari customer yang sama, sehingga nilai ekstrem tersebut lebih aman dijadikan missing value.

Berdasarkan output ini, keputusan yang paling aman adalah menangani `Annual_Income > 200.000` sebagai kandidat outlier. Untuk 195 baris yang memiliki referensi valid, nilai ekstrem dapat diganti menggunakan median `Annual_Income` dari `Customer_ID` yang sama. Untuk 29 baris yang tidak memiliki referensi valid, nilai ekstrem tidak dipaksakan diganti, tetapi dapat diubah menjadi missing value agar nanti ditangani oleh pipeline preprocessing model.

### Perbaikan Kolom Annual Income dan Monthly Inhand Salaryu

In [ ]:
# PERBAIKAN ANNUAL_INCOME DAN MONTHLY_INHAND_SALARY
pd.options.display.float_format = "{:,.4f}".format

df = df.copy()

annual_income_threshold = 200000

In [ ]:
# CEK KONDISI SEBELUM PERBAIKAN
annual_extreme_mask = df["Annual_Income"] > annual_income_threshold
monthly_salary_missing_mask = df["Monthly_Inhand_Salary"].isna()

print("Sebelum perbaikan:")
print("Annual_Income > 200000:", annual_extreme_mask.sum())
print("Missing Annual_Income:", df["Annual_Income"].isna().sum())
print("Missing Monthly_Inhand_Salary:", monthly_salary_missing_mask.sum())

In [ ]:
# BUAT REFERENSI ANNUAL_INCOME VALID DARI CUSTOMER_ID YANG SAMA

valid_annual_reference = (
    df[
        (df["Annual_Income"].notna()) &
        (df["Annual_Income"] <= annual_income_threshold)
    ]
    .groupby("Customer_ID")["Annual_Income"]
    .median()
)

suggested_annual_income = df.loc[
    annual_extreme_mask, "Customer_ID"
].map(valid_annual_reference)

annual_fillable_indices = suggested_annual_income[
    suggested_annual_income.notna()
].index

annual_unresolved_indices = suggested_annual_income[
    suggested_annual_income.isna()
].index

print("Annual_Income ekstrem yang bisa diperbaiki:", len(annual_fillable_indices))
print("Annual_Income ekstrem tanpa referensi valid:", len(annual_unresolved_indices))

Cell pertama digunakan untuk mengecek kondisi awal sebelum perbaikan data. Hasilnya menunjukkan bahwa terdapat 224 baris dengan `Annual_Income > 200000`, tidak ada missing value pada `Annual_Income`, dan terdapat 3.836 missing value pada `Monthly_Inhand_Salary`.

Cell kedua digunakan untuk membuat referensi `Annual_Income` valid berdasarkan `Customer_ID` yang sama. Nilai yang dianggap valid adalah `Annual_Income` yang tidak missing dan tidak melebihi threshold 200.000. Kemudian, data ekstrem `Annual_Income > 200000` dicek apakah customer tersebut memiliki referensi valid dari bulan lain.

Hasilnya menunjukkan bahwa dari 224 nilai `Annual_Income` ekstrem, sebanyak 195 baris masih bisa diperbaiki menggunakan median `Annual_Income` valid dari customer yang sama. Sementara itu, 29 baris tidak memiliki referensi valid, sehingga nantinya lebih aman dijadikan missing value daripada dipaksakan memakai nilai dari customer lain.

In [ ]:
# SIMPAN AUDIT SEBELUM NILAI DIPERBAIKI

annual_income_repair_audit = df.loc[
    annual_extreme_mask,
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Credit_Score"
    ]
].copy()

annual_income_repair_audit["Suggested_Annual_Income"] = suggested_annual_income

annual_income_repair_audit["Repair_Action"] = np.where(
    annual_income_repair_audit["Suggested_Annual_Income"].notna(),
    "Replace with customer median",
    "Set as missing value"
)

annual_income_repair_audit.head(30)

Cell ini digunakan untuk menyimpan audit sebelum nilai `Annual_Income` ekstrem diperbaiki. Tujuannya supaya perubahan data tetap bisa ditelusuri dan tidak dilakukan secara langsung tanpa bukti.

Kode mengambil semua baris yang memiliki `Annual_Income > 200000`, lalu menyimpan beberapa kolom penting seperti `ID`, `Customer_ID`, `Month`, `Annual_Income`, `Monthly_Inhand_Salary`, dan `Credit_Score`.

Setelah itu, kode menambahkan kolom `Suggested_Annual_Income`, yaitu nilai pengganti yang diambil dari median `Annual_Income` valid milik `Customer_ID` yang sama. Jika customer tersebut memiliki referensi valid, maka kolom `Repair_Action` akan berisi `"Replace with customer median"`. Jika tidak ada referensi valid, maka `Repair_Action` akan berisi `"Set as missing value"`.

Output yang muncul adalah 30 baris pertama dari data audit. Baris-baris tersebut menunjukkan nilai `Annual_Income` ekstrem sebelum diperbaiki, beserta rekomendasi nilai pengganti dan tindakan perbaikannya. Jadi, cell ini belum memperbaiki data utama, tetapi hanya menyimpan catatan pemeriksaan agar proses cleaning lebih transparan dan bisa dijelaskan dalam laporan.

In [ ]:
# PERBAIKI ANNUAL_INCOME

df.loc[
    annual_fillable_indices,
    "Annual_Income"
] = suggested_annual_income.loc[annual_fillable_indices]

df.loc[
    annual_unresolved_indices,
    "Annual_Income"
] = np.nan

In [ ]:
# CEK KONDISI SETELAH PERBAIKAN

print("Setelah perbaikan:")
print("Annual_Income > 200000:", (df["Annual_Income"] > annual_income_threshold).sum())
print("Missing Annual_Income:", df["Annual_Income"].isna().sum())
print("Missing Monthly_Inhand_Salary:", df["Monthly_Inhand_Salary"].isna().sum())

df[["Annual_Income", "Monthly_Inhand_Salary"]].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.97, 0.98, 0.99]
).T

Output ini menunjukkan hasil setelah perbaikan `Annual_Income`.

Setelah perbaikan, jumlah `Annual_Income > 200000` menjadi 0. Artinya, seluruh 224 nilai ekstrem sudah berhasil ditangani. Sebanyak 195 nilai ekstrem diganti menggunakan median `Annual_Income` valid dari `Customer_ID` yang sama, sedangkan 29 nilai ekstrem yang tidak memiliki referensi valid dijadikan missing value.

Karena 29 data ekstrem diubah menjadi missing value, jumlah missing pada `Annual_Income` sekarang menjadi 29. Ini wajar dan lebih aman dibandingkan mempertahankan nilai annual income jutaan atau puluhan juta yang tidak konsisten dengan data customer.

Distribusi `Annual_Income` juga menjadi jauh lebih masuk akal. Sebelum perbaikan, nilai maksimum mencapai 24.160.009, sedangkan setelah perbaikan nilai maksimum turun menjadi 179.987,28. Nilai mean juga turun menjadi 50.504,71, sehingga tidak lagi terlalu tertarik oleh outlier ekstrem.

Kolom `Monthly_Inhand_Salary` belum berubah pada tahap ini. Jumlah missing masih 3.836 karena cell yang dijalankan hanya memperbaiki `Annual_Income`, belum mengisi missing value pada `Monthly_Inhand_Salary`.

Kesimpulannya, perbaikan `Annual_Income` sudah berhasil. Outlier ekstrem sudah hilang, distribusi menjadi lebih stabil, dan 29 data yang tidak punya referensi valid dibiarkan sebagai missing value untuk ditangani nanti pada preprocessing pipeline.

In [ ]:
# CEK ULANG HUBUNGAN ANNUAL_INCOME DAN MONTHLY_INHAND_SALARY SETELAH PERBAIKAN

salary_relation_after_repair = df[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Credit_Score"
    ]
].dropna(subset=["Annual_Income", "Monthly_Inhand_Salary"]).copy()

salary_relation_after_repair["Expected_Monthly_Salary"] = (
    salary_relation_after_repair["Annual_Income"] / 12
)

salary_relation_after_repair["Salary_Ratio"] = (
    salary_relation_after_repair["Monthly_Inhand_Salary"] /
    salary_relation_after_repair["Expected_Monthly_Salary"]
)

salary_relation_after_repair["Salary_Difference_Percentage"] = (
    (
        salary_relation_after_repair["Monthly_Inhand_Salary"] -
        salary_relation_after_repair["Expected_Monthly_Salary"]
    ).abs() /
    salary_relation_after_repair["Expected_Monthly_Salary"]
) * 100

print(
    "Jumlah Salary Difference > 90% setelah perbaikan:",
    (salary_relation_after_repair["Salary_Difference_Percentage"] > 90).sum()
)

salary_relation_after_repair[
    [
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Expected_Monthly_Salary",
        "Salary_Ratio",
        "Salary_Difference_Percentage"
    ]
].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

Output ini menunjukkan hasil pengecekan ulang hubungan antara `Annual_Income` dan `Monthly_Inhand_Salary` setelah perbaikan dilakukan.

Hasil paling penting adalah jumlah `Salary_Difference > 90%` setelah perbaikan menjadi 0. Artinya, setelah nilai ekstrem pada `Annual_Income` diperbaiki atau dijadikan missing value, tidak ada lagi data yang memiliki ketidaksesuaian sangat ekstrem antara pendapatan tahunan dan gaji bulanan.

Distribusi hubungan kedua kolom juga menjadi lebih wajar. Rata-rata `Annual_Income` setelah perbaikan adalah sekitar 50.395,92, sehingga `Expected_Monthly_Salary` rata-ratanya menjadi sekitar 4.199,66. Nilai ini sangat dekat dengan rata-rata `Monthly_Inhand_Salary`, yaitu sekitar 4.189,47. Hal ini menunjukkan bahwa secara umum hubungan antara pendapatan tahunan dan gaji bulanan sudah konsisten.

Median `Salary_Ratio` juga berada di sekitar 0,9982, yang berarti gaji bulanan aktual hampir sama dengan estimasi gaji bulanan dari `Annual_Income / 12`. Selain itu, median `Salary_Difference_Percentage` hanya sekitar 4,09%, sehingga mayoritas data memiliki selisih yang kecil dan masih masuk akal.

Nilai maksimum `Salary_Difference_Percentage` setelah perbaikan adalah sekitar 48,33%. Nilai ini jauh lebih baik dibandingkan sebelum perbaikan, ketika selisih maksimum mencapai hampir 100%. Dengan demikian, perbaikan `Annual_Income` berhasil menghilangkan anomali ekstrem tanpa merusak pola mayoritas data.

Kesimpulannya, proses cleaning pada `Annual_Income` berhasil. Outlier ekstrem yang sebelumnya menyebabkan hubungan tidak wajar dengan `Monthly_Inhand_Salary` sudah hilang, dan hubungan kedua kolom sekarang jauh lebih stabil serta lebih siap digunakan untuk tahap preprocessing dan modeling.

In [ ]:
# TAMPILKAN SISA KASUS PALING TIDAK KONSISTEN SETELAH PERBAIKAN

salary_relation_after_repair.sort_values(
    by="Salary_Difference_Percentage",
    ascending=False
).head(30)

Output ini menampilkan 30 kasus dengan `Salary_Difference_Percentage` tertinggi setelah perbaikan `Annual_Income`.

Setelah perbaikan, sisa kasus yang paling tidak konsisten hanya memiliki selisih sekitar 43% sampai 48%. Nilai ini jauh lebih rendah dibandingkan sebelum perbaikan, ketika selisihnya bisa mencapai hampir 100%. Hal ini menunjukkan bahwa anomali ekstrem sudah berhasil dihilangkan.

Contohnya, `CUS_ANON_029` memiliki `Annual_Income` sebesar 7.174,99, sehingga `Expected_Monthly_Salary` adalah sekitar 597,92. Namun, `Monthly_Inhand_Salary` aktualnya sebesar 886,92, sehingga selisihnya sekitar 48,33%. Perbedaan ini memang cukup besar, tetapi tidak seekstrem kasus sebelumnya yang memiliki annual income jutaan atau puluhan juta.

Pola yang muncul setelah perbaikan juga berbeda dari sebelumnya. Sebelum perbaikan, masalah utama berasal dari `Annual_Income` yang terlalu besar secara tidak wajar. Setelah perbaikan, nilai `Annual_Income` pada kasus tertinggi justru berada di angka yang rendah, sekitar 7.000 sampai 8.000, dan beberapa perbedaan bisa terjadi karena perbedaan antara pendapatan tahunan, gaji bersih bulanan, potongan, pembulatan, atau kemungkinan variasi pencatatan.

Selain itu, beberapa kasus muncul berulang pada `Customer_ID` yang sama, misalnya `CUS_ANON_029`, `CUS_ANON_025`, `CUS_ANON_008`, dan `CUS_ANON_005`. Nilai `Annual_Income` dan `Monthly_Inhand_Salary` pada customer yang sama terlihat konsisten antarbulan. Artinya, sisa perbedaan ini tidak terlihat seperti lonjakan salah input pada satu bulan tertentu, melainkan pola yang relatif stabil untuk customer tersebut.

Kasus dengan selisih sekitar 40% sampai 48% juga tidak dianalisis lebih lanjut karena tidak memiliki bukti yang cukup kuat untuk disebut sebagai data error. Pada tahap sebelumnya, anomali yang diperbaiki memiliki pola yang sangat jelas, yaitu `Annual_Income` melonjak sampai jutaan atau puluhan juta pada satu bulan, sementara bulan lain dari customer yang sama memiliki nilai yang jauh lebih rendah. Pada kasus sisa ini, pola tersebut tidak terlihat. Nilai `Annual_Income` masih berada dalam rentang valid dan tidak melebihi threshold 200.000.

Selain itu, hubungan antara `Annual_Income` dan `Monthly_Inhand_Salary` tidak harus selalu tepat sama dengan perhitungan `Annual_Income / 12`. `Monthly_Inhand_Salary` dapat merepresentasikan gaji bersih bulanan, sedangkan `Annual_Income` dapat merepresentasikan pendapatan tahunan yang mungkin mencakup perbedaan pencatatan, potongan, tunjangan, bonus, pajak, atau komponen pendapatan lain. Karena itu, selisih sedang masih mungkin terjadi dan tidak otomatis berarti nilai tersebut salah.

Jika kasus sisa dengan selisih 40%–48% tetap dipaksa diperbaiki, proses cleaning akan menjadi terlalu agresif. Hal ini berisiko mengubah data yang sebenarnya valid dan menghilangkan variasi alami pada dataset. Dalam konteks machine learning, over-cleaning dapat membuat data menjadi terlalu “rapi” secara buatan dan dapat mengurangi kemampuan model untuk belajar dari pola asli data.

Berdasarkan output ini, tidak perlu dilakukan perbaikan tambahan secara agresif pada kasus-kasus sisa ini. Setelah threshold ekstrem `Salary_Difference > 90%` menjadi 0, hubungan antara `Annual_Income` dan `Monthly_Inhand_Salary` sudah jauh lebih wajar. Jika sisa kasus dengan selisih sekitar 40%–48% tetap diperbaiki secara paksa, ada risiko mengubah data valid yang sebenarnya masih mungkin terjadi secara realistis.

Dengan demikian, proses perbaikan `Annual_Income` dapat dianggap selesai. Nilai ekstrem sudah ditangani, sedangkan sisa perbedaan yang masih ada tetap dipertahankan agar tidak melakukan over-cleaning pada data.

### Analisis isi Total_EMI_per_month sebelum aku melihat hubungannya dengan variabel" lain

In [ ]:
# CHECK TOTAL EMI BASIC INFORMATION (INI SEBENERNYA TADI UDAH DI EDA SBLMNYA, TAPI SAYA ULANG LAGII AJA)

total_emi_data = eda.dropna(
    subset=[
        "Total_EMI_per_month"
    ]
).copy()

print("Data Type:", total_emi_data["Total_EMI_per_month"].dtype)
print("Jumlah data valid:", len(total_emi_data))
print("Missing Value di df asli:", eda["Total_EMI_per_month"].isna().sum())
print("Minimum:", total_emi_data["Total_EMI_per_month"].min())
print("Maximum:", total_emi_data["Total_EMI_per_month"].max())
print("Negative Value:", (total_emi_data["Total_EMI_per_month"] < 0).sum())

In [ ]:
# CHECK TOTAL EMI DISTRIBUTION

total_emi_distribution = total_emi_data["Total_EMI_per_month"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.97,
        0.98,
        0.99,
        0.995
    ]
)

total_emi_distribution

In [ ]:
# CHECK TOTAL EMI PERCENTILE JUMP

total_emi_percentiles = (
    total_emi_data["Total_EMI_per_month"]
    .quantile([
        0.50,
        0.75,
        0.90,
        0.95,
        0.97,
        0.98,
        0.99,
        0.995,
        1.00
    ])
    .reset_index()
)

total_emi_percentiles.columns = [
    "Percentile",
    "Total_EMI_per_month"
]

total_emi_percentiles["Percentile"] = (
    total_emi_percentiles["Percentile"] * 100
)

total_emi_percentiles["Increase"] = (
    total_emi_percentiles["Total_EMI_per_month"].diff()
)

total_emi_percentiles

In [ ]:
# VISUALIZE TOTAL EMI FULL DISTRIBUTION

plt.figure(figsize=(7, 5))
plt.hist(
    total_emi_data["Total_EMI_per_month"],
    bins=50
)
plt.title("Distribution of Total EMI per Month")
plt.xlabel("Total EMI per Month")
plt.ylabel("Frequency")
plt.show()

Output ini menunjukkan distribusi kolom `Total_EMI_per_month`. Jumlah datanya lengkap, yaitu 25.000 baris, sehingga tidak ada missing value pada kolom ini.

Mayoritas nilai `Total_EMI_per_month` sebenarnya masih kecil. Median hanya sekitar 68,25, persentil ke-75 sekitar 159,99, persentil ke-90 sekitar 288,12, dan persentil ke-95 sekitar 446,26. Artinya, sekitar 95% data masih memiliki nilai EMI bulanan di bawah 446,26.

Namun, setelah persentil ke-95, terjadi lonjakan yang sangat besar. Pada persentil ke-97, nilai langsung naik menjadi 8.705,24. Lalu pada persentil ke-98 menjadi 32.164,44, persentil ke-99 menjadi 58.579,07, dan maksimum mencapai 82.331. Lonjakan ini menunjukkan adanya nilai ekstrem pada sebagian kecil data.

Histogram juga memperlihatkan pola yang sama. Hampir semua data menumpuk di area nilai rendah, sedangkan sebagian kecil data menyebar sangat jauh sampai lebih dari 80.000. Karena nilai ekstrem ini sangat besar, grafik terlihat sangat condong ke kanan atau right-skewed.

Tabel percentile increase memperjelas titik lonjakan terbesar. Kenaikan dari P95 ke P97 sangat besar, yaitu sekitar 8.258,98. Setelah itu, kenaikan dari P97 ke P98 juga sangat besar, yaitu sekitar 23.459,20. Hal ini menunjukkan bahwa nilai ekstrem mulai muncul kuat setelah sekitar persentil ke-95 sampai ke-97.

Kesimpulannya, `Total_EMI_per_month` tidak memiliki masalah missing value, tetapi memiliki outlier ekstrem yang cukup kuat. Nilai EMI kecil sampai sekitar 446 masih terlihat sebagai pola mayoritas data, sedangkan nilai di atas ribuan sampai puluhan ribu perlu dianalisis lebih lanjut. Namun, nilai tersebut tidak boleh langsung dihapus karena bisa saja berkaitan dengan jumlah pinjaman, tipe pinjaman, pendapatan, atau riwayat customer. Oleh karena itu, tahap berikutnya perlu membandingkan `Total_EMI_per_month` dengan `Monthly_Inhand_Salary`, `Annual_Income`, `Num_of_Loan`, dan `Customer_ID`.

In [ ]:
# COMPARE TOTAL EMI THRESHOLDS

emi_thresholds = [
    500,
    1000,
    1500,
    2000,
    5000,
    10000
]

emi_threshold_summary = []

for threshold in emi_thresholds:
    count_above = (
        total_emi_data["Total_EMI_per_month"] > threshold
    ).sum()

    percentage_above = (
        count_above / len(total_emi_data) * 100
    )

    emi_threshold_summary.append({
        "Threshold": threshold,
        "Count_Above_Threshold": count_above,
        "Percentage": percentage_above
    })

emi_threshold_summary = pd.DataFrame(emi_threshold_summary)

emi_threshold_summary

Output ini menunjukkan jumlah data `Total_EMI_per_month` yang berada di atas beberapa threshold tertentu.

Dari hasil tersebut, terdapat 1.138 baris atau sekitar 4,55% data dengan `Total_EMI_per_month > 500`. Angka ini masuk akal sebagai batas awal investigasi karena sebelumnya persentil ke-95 masih berada di sekitar 446,26. Artinya, nilai di atas 500 sudah berada di luar mayoritas data.

Namun, saat threshold dinaikkan menjadi 1.000, jumlahnya masih 875 baris atau sekitar 3,50%. Pada threshold 2.000, jumlahnya masih 812 baris atau sekitar 3,25%. Ini menunjukkan bahwa cukup banyak data yang memiliki EMI jauh lebih besar dari pola umum.

Hal yang menarik adalah jumlah data tidak turun terlalu banyak dari threshold 2.000 ke 10.000. Pada threshold 2.000 terdapat 812 baris, sedangkan pada threshold 10.000 masih ada 735 baris. Artinya, sebagian besar data yang sudah melewati 2.000 ternyata memang berada pada nilai yang sangat tinggi, bahkan banyak yang mencapai puluhan ribu.

Berdasarkan output ini, threshold 2.000 dapat digunakan sebagai batas investigasi awal untuk `Total_EMI_per_month`. Alasannya, nilai ini sudah jauh di atas P95 dan mencakup sekitar 3,25% data, sehingga cukup spesifik untuk melihat kasus EMI yang tidak biasa tanpa terlalu banyak mengambil data normal.

Namun, nilai `Total_EMI_per_month > 2.000` belum bisa langsung dianggap salah atau dihapus. EMI yang tinggi bisa saja masuk akal jika customer memiliki banyak pinjaman, pendapatan tinggi, atau jenis pinjaman tertentu. Oleh karena itu, nilai EMI ekstrem perlu dianalisis lebih lanjut dengan membandingkannya terhadap `Monthly_Inhand_Salary`, `Annual_Income`, `Num_of_Loan`, `Type_of_Loan`, dan riwayat `Customer_ID` yang sama.

In [ ]:
# VISUALIZE TOTAL EMI ZOOMED DISTRIBUTION AFTER THRESHOLD COMPARISON

plt.figure(figsize=(7, 5))
plt.hist(
    total_emi_data.loc[
        total_emi_data["Total_EMI_per_month"] <= 2000,
        "Total_EMI_per_month"
    ],
    bins=50
)
plt.title("Distribution of Total EMI per Month <= 2000")
plt.xlabel("Total EMI per Month")
plt.ylabel("Frequency")
plt.show()

Grafik ini menampilkan distribusi `Total_EMI_per_month` setelah dibatasi hanya pada nilai `<= 2000`. Tujuannya adalah untuk melihat pola mayoritas data dengan lebih jelas, karena pada grafik distribusi penuh sebelumnya nilai ekstrem sampai puluhan ribu membuat bagian data normal menjadi sulit dibaca.

Dari grafik ini terlihat bahwa sebagian besar customer memiliki `Total_EMI_per_month` yang rendah, terutama di bawah sekitar 300. Frekuensi paling tinggi berada pada nilai EMI kecil, lalu semakin menurun ketika nilai EMI semakin besar. Pola ini menunjukkan bahwa distribusi `Total_EMI_per_month` tetap right-skewed, tetapi setelah data ekstrem di atas 2000 tidak ditampilkan, bentuk mayoritas data menjadi lebih terlihat.

Visualisasi ini juga mendukung hasil statistik sebelumnya, yaitu median `Total_EMI_per_month` hanya sekitar 68,25, P75 sekitar 159,99, P90 sekitar 288,12, dan P95 sekitar 446,26. Artinya, mayoritas data memang berada pada nilai EMI yang relatif rendah. Nilai di atas 2000 hanya sebagian kecil data, tetapi jumlahnya tetap perlu dianalisis karena sangat jauh dari pola utama.

Batas 2000 di sini bukan berarti nilai di atas 2000 langsung dihapus. Threshold ini hanya digunakan sebagai batas investigasi awal untuk memisahkan pola mayoritas data dari nilai EMI yang sangat tinggi. Nilai EMI tinggi masih perlu dicek lebih lanjut dengan kolom lain seperti `Monthly_Inhand_Salary`, `Annual_Income`, `Num_of_Loan`, `Type_of_Loan`, dan riwayat `Customer_ID` sebelum diputuskan apakah nilai tersebut valid atau perlu diperbaiki.

### Setelah distribusi terlihat, baru cek Total_EMI_per_month >

In [ ]:
# CHECK EXTREME TOTAL EMI AFTER DISTRIBUTION ANALYSIS

extreme_emi = eda.dropna(
    subset=[
        "Total_EMI_per_month"
    ]
).copy()

extreme_emi = extreme_emi[
    extreme_emi["Total_EMI_per_month"] > 2000
].copy()

print("Jumlah Total_EMI_per_month > 2000:", len(extreme_emi))
print(
    "Persentase Total_EMI_per_month > 2000:",
    round(len(extreme_emi) / len(total_emi_data) * 100, 2),
    "%"
)

extreme_emi[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Total_EMI_per_month",
        "Credit_Score"
    ]
].sort_values(
    by="Total_EMI_per_month",
    ascending=False
).head(30)

Output ini menunjukkan hasil investigasi awal terhadap nilai `Total_EMI_per_month` yang lebih besar dari 2.000.

Terdapat 812 baris dengan `Total_EMI_per_month > 2000`, atau sekitar 3,25% dari total data. Jumlah ini relatif kecil, tetapi tetap penting karena nilai EMI pada kelompok ini sangat jauh dari pola mayoritas data. Sebelumnya, P95 `Total_EMI_per_month` hanya sekitar 446,26, sedangkan pada output ini banyak nilai EMI mencapai sekitar 78.000 sampai 82.000.

Beberapa baris terlihat sangat tidak wajar. Contohnya, `CUS_ANON_004` memiliki `Annual_Income` 15.704, `Monthly_Inhand_Salary` 1.606,52, dan `Num_of_Loan` 8, tetapi `Total_EMI_per_month` mencapai 82.331. Secara logika, EMI bulanan sebesar 82.331 jauh lebih besar daripada gaji bulanan customer tersebut, sehingga perlu dicurigai sebagai anomali.

Ada juga kasus yang lebih jelas tidak konsisten, yaitu customer dengan `Num_of_Loan = 0` tetapi memiliki `Total_EMI_per_month` sangat tinggi. Misalnya, `CUS_ANON_013` memiliki `Num_of_Loan` 0 tetapi `Total_EMI_per_month` sebesar 82.163. Jika jumlah pinjaman adalah 0, maka secara logika EMI bulanan seharusnya 0 atau sangat kecil, bukan puluhan ribu. Pola seperti ini menunjukkan kemungkinan error pada `Total_EMI_per_month` atau pada kolom terkait pinjaman.

Namun, nilai `Total_EMI_per_month > 2000` belum boleh langsung dihapus seluruhnya. Ada kemungkinan sebagian EMI tinggi masih masuk akal untuk customer dengan pendapatan tinggi, jumlah pinjaman besar, atau jenis pinjaman tertentu. Oleh karena itu, kolom ini perlu dianalisis lebih lanjut dengan membandingkan `Total_EMI_per_month` terhadap `Monthly_Inhand_Salary`, `Annual_Income`, `Num_of_Loan`, `Type_of_Loan`, dan riwayat `Customer_ID` yang sama.

Catatan penting: pada output ini masih terlihat ada `Annual_Income` yang sangat ekstrem, misalnya 22.828.106. Padahal sebelumnya `Annual_Income > 200000` sudah diperbaiki menjadi 0. Artinya, kemungkinan `total_emi_data` dibuat sebelum perbaikan `Annual_Income`, sehingga masih menyimpan versi data lama. Sebelum mengambil kesimpulan final, `total_emi_data` sebaiknya dibuat ulang dari `df` yang sudah dibersihkan agar analisis EMI menggunakan data terbaru.

Kesimpulannya, `Total_EMI_per_month` memiliki indikasi outlier yang kuat, terutama pada nilai puluhan ribu dan pada baris dengan `Num_of_Loan = 0`. Tahap berikutnya perlu melakukan relationship-based EDA untuk melihat apakah EMI tinggi tersebut konsisten dengan pendapatan, jumlah pinjaman, dan riwayat customer. Setelah itu baru bisa diputuskan apakah nilai ekstrem tersebut diperbaiki, dijadikan missing value, atau tetap dipertahankan.

#### a. Hubungan Total_EMI_per_month dengan gaji dan pendapatan

In [ ]:
# RELATIONSHIP BETWEEN TOTAL EMI AND INCOME AFTER REPAIR

emi_income_data = df.dropna(
    subset=[
        "Total_EMI_per_month",
        "Monthly_Inhand_Salary",
        "Annual_Income"
    ]
).copy()

emi_income_data["EMI_to_Monthly_Salary_Ratio"] = (
    emi_income_data["Total_EMI_per_month"] /
    emi_income_data["Monthly_Inhand_Salary"]
)

emi_income_data["EMI_to_Annual_Income_Ratio"] = (
    emi_income_data["Total_EMI_per_month"] * 12 /
    emi_income_data["Annual_Income"]
)

emi_income_data = emi_income_data.replace(
    [np.inf, -np.inf],
    np.nan
)

emi_income_clean = emi_income_data.dropna(
    subset=[
        "EMI_to_Monthly_Salary_Ratio",
        "EMI_to_Annual_Income_Ratio"
    ]
).copy()

emi_income_summary = emi_income_clean[
    [
        "Total_EMI_per_month",
        "Monthly_Inhand_Salary",
        "Annual_Income",
        "EMI_to_Monthly_Salary_Ratio",
        "EMI_to_Annual_Income_Ratio"
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.97,
        0.98,
        0.99
    ]
)

print("Jumlah data relationship EMI dan income:", len(emi_income_clean))
print("Annual_Income > 200000:", (df["Annual_Income"] > 200000).sum())
print("Missing Annual_Income:", df["Annual_Income"].isna().sum())
print("Missing Monthly_Inhand_Salary:", df["Monthly_Inhand_Salary"].isna().sum())

emi_income_summary

Output ini menunjukkan hubungan antara `Total_EMI_per_month`, `Monthly_Inhand_Salary`, dan `Annual_Income` setelah `Annual_Income` diperbaiki.

Jumlah data yang masuk ke analisis ini adalah 21.143 baris. Angka ini lebih kecil dari 25.000 karena analisis ini hanya memakai baris yang memiliki nilai lengkap pada `Total_EMI_per_month`, `Monthly_Inhand_Salary`, dan `Annual_Income`.

Hasil ini sudah lebih valid dibandingkan output sebelumnya karena `Annual_Income` sudah tidak memiliki nilai ekstrem jutaan atau puluhan juta. Nilai maksimum `Annual_Income` sekarang adalah 179.987,28, sehingga perbaikan sebelumnya berhasil diterapkan.

Mayoritas data masih terlihat wajar. Median `Total_EMI_per_month` adalah sekitar 68,26, sedangkan median `Monthly_Inhand_Salary` adalah sekitar 3.070,17. Median `EMI_to_Monthly_Salary_Ratio` hanya 0,0253, artinya pada sebagian besar data, EMI bulanan hanya sekitar 2,53% dari gaji bulanan. Sampai P95, rasio EMI terhadap gaji bulanan juga masih sekitar 0,1065 atau 10,65%, sehingga mayoritas data masih masuk akal.

Namun, mulai dari P97 terjadi lonjakan yang sangat besar. Nilai `Total_EMI_per_month` naik dari P95 sekitar 448,96 menjadi P97 sekitar 10.376,96. Pada titik ini, `EMI_to_Monthly_Salary_Ratio` juga naik menjadi 2,3224, yang berarti EMI bulanan sudah sekitar 2,32 kali lebih besar daripada gaji bulanan.

Pada P98 dan P99, kondisi semakin ekstrem. P98 `Total_EMI_per_month` mencapai 33.017,48, dan P99 mencapai 58.547,36. Rasio EMI terhadap gaji bulanan pada P99 mencapai sekitar 19,85 kali. Nilai maksimum bahkan menunjukkan rasio 227,74 kali gaji bulanan. Secara logika, ini sangat tidak wajar karena EMI bulanan seharusnya tidak jauh melebihi pendapatan bulanan customer.

Rasio terhadap `Annual_Income` juga menunjukkan pola yang sama. Median `EMI_to_Annual_Income_Ratio` hanya 0,0253, tetapi pada P97 naik menjadi 2,3576, P98 menjadi 8,4296, dan P99 menjadi 19,8487. Artinya, pada sebagian kecil data, total EMI dalam setahun bisa jauh lebih besar daripada pendapatan tahunan customer. Ini memperkuat indikasi bahwa masalah utama sekarang berada pada nilai ekstrem `Total_EMI_per_month`.

Kesimpulannya, setelah `Annual_Income` diperbaiki, mayoritas hubungan EMI dan income sudah wajar. Namun, terdapat sebagian kecil nilai `Total_EMI_per_month` yang sangat ekstrem, terutama mulai sekitar P97 ke atas. Nilai ekstrem ini belum boleh langsung dihapus, tetapi perlu dianalisis lagi dengan `Num_of_Loan`, `Type_of_Loan`, dan riwayat `Customer_ID` yang sama untuk melihat apakah EMI tinggi tersebut konsisten atau merupakan kesalahan data.

In [ ]:
# TAMPILKAN KASUS EMI PALING TIDAK KONSISTEN DENGAN INCOME

emi_income_clean.sort_values(
    by="EMI_to_Monthly_Salary_Ratio",
    ascending=False
)[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Total_EMI_per_month",
        "EMI_to_Monthly_Salary_Ratio",
        "EMI_to_Annual_Income_Ratio",
        "Credit_Score"
    ]
].head(30)

Output ini menampilkan 30 kasus `Total_EMI_per_month` yang paling tidak konsisten dengan pendapatan customer, berdasarkan nilai `EMI_to_Monthly_Salary_Ratio` tertinggi.

Kolom `EMI_to_Monthly_Salary_Ratio` menunjukkan perbandingan antara EMI bulanan dan gaji bulanan customer. Jika nilainya 1, berarti EMI sama dengan 100% gaji bulanan. Jika nilainya lebih dari 1, berarti EMI lebih besar daripada gaji bulanan. Pada output ini, rasionya sangat ekstrem, mulai dari sekitar 60 kali sampai 227 kali gaji bulanan.

Contohnya, `CUS_ANON_025` memiliki `Monthly_Inhand_Salary` sebesar 303,65, tetapi `Total_EMI_per_month` sebesar 69.152. Nilai `EMI_to_Monthly_Salary_Ratio` mencapai 227,74. Artinya, EMI bulanannya sekitar 227 kali lebih besar daripada gaji bulanannya. Secara logika, kondisi ini sangat tidak wajar.

Contoh lain adalah `CUS_ANON_040` dengan `Monthly_Inhand_Salary` sebesar 511,75 dan `Total_EMI_per_month` sebesar 79.247. Rasionya mencapai 154,86 kali gaji bulanan. Selain itu, rasio terhadap annual income juga sangat tinggi, yaitu 132,80. Artinya, total EMI dalam satu tahun jauh lebih besar daripada annual income customer tersebut.

Output ini juga menunjukkan beberapa kasus yang semakin kuat dicurigai sebagai data error karena `Num_of_Loan` bernilai 0, tetapi `Total_EMI_per_month` sangat tinggi. Misalnya, `CUS_ANON_009` memiliki `Num_of_Loan = 0`, tetapi `Total_EMI_per_month` sebesar 66.045. Jika customer tidak memiliki loan, maka EMI bulanan seharusnya 0 atau sangat kecil, bukan puluhan ribu.

Setelah `Annual_Income` diperbaiki, masalah utama pada bagian ini terlihat lebih jelas berasal dari `Total_EMI_per_month`. Nilai `Annual_Income` dan `Monthly_Inhand_Salary` pada output ini sudah berada pada rentang yang lebih masuk akal, tetapi `Total_EMI_per_month` masih sangat ekstrem dan tidak proporsional terhadap pendapatan customer.

Kesimpulannya, nilai ekstrem pada `Total_EMI_per_month` perlu dianalisis lebih lanjut. Indikasi outlier sangat kuat terutama pada kasus dengan EMI puluhan ribu, rasio EMI terhadap gaji bulanan jauh di atas 1, dan kasus dengan `Num_of_Loan = 0` tetapi EMI sangat besar. Tahap berikutnya sebaiknya mengecek riwayat `Customer_ID` yang sama untuk melihat apakah EMI ekstrem hanya muncul pada satu bulan tertentu. Jika customer memiliki referensi EMI valid pada bulan lain, nilai ekstrem dapat diperbaiki menggunakan median EMI valid dari customer tersebut. Jika tidak ada referensi valid, nilai ekstrem lebih aman dijadikan missing value agar tidak memberikan sinyal yang salah pada model.

In [ ]:
# CHECK EMI HIGHER THAN MONTHLY SALARY

emi_higher_than_salary = emi_income_clean[
    emi_income_clean["EMI_to_Monthly_Salary_Ratio"] > 1
].copy()

print("Jumlah EMI lebih besar dari gaji bulanan:", len(emi_higher_than_salary))
print(
    "Persentase EMI lebih besar dari gaji bulanan:",
    round(len(emi_higher_than_salary) / len(emi_income_clean) * 100, 2),
    "%"
)

emi_higher_than_salary[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Total_EMI_per_month",
        "EMI_to_Monthly_Salary_Ratio",
        "Credit_Score"
    ]
].sort_values(
    by="EMI_to_Monthly_Salary_Ratio",
    ascending=False
).head(30)

Output ini menunjukkan bahwa terdapat 687 baris di mana `Total_EMI_per_month` lebih besar daripada `Monthly_Inhand_Salary`. Persentasenya sekitar 3,25% dari data yang memiliki nilai lengkap untuk EMI, gaji bulanan, dan annual income.

Secara logika, kondisi ini perlu dicurigai karena EMI bulanan seharusnya tidak melebihi kemampuan pendapatan bulanan customer dalam jumlah yang sangat besar. Jika rasio `EMI_to_Monthly_Salary_Ratio` lebih dari 1, berarti EMI sudah lebih besar daripada gaji bulanan. Pada output ini, rasionya tidak hanya sedikit di atas 1, tetapi sangat ekstrem, bahkan mencapai 227,74 kali gaji bulanan.

Contohnya, `CUS_ANON_025` memiliki `Monthly_Inhand_Salary` hanya 303,65, tetapi `Total_EMI_per_month` sebesar 69.152. Rasio EMI terhadap gaji bulanannya mencapai 227,74. Artinya, EMI bulanan jauh melebihi pendapatan bulanan customer, sehingga sangat tidak realistis.

Beberapa kasus lain juga menunjukkan pola yang sama. Misalnya, `CUS_ANON_040` memiliki gaji bulanan 511,75 tetapi EMI sebesar 79.247, dengan rasio 154,86. `CUS_ANON_038` memiliki gaji bulanan 557,10 tetapi EMI sebesar 61.264, dengan rasio 109,97. Nilai-nilai ini menunjukkan bahwa masalah utama kemungkinan besar berasal dari nilai ekstrem pada `Total_EMI_per_month`.

Output ini juga memperlihatkan adanya kasus yang semakin tidak konsisten, yaitu `Num_of_Loan = 0` tetapi `Total_EMI_per_month` sangat tinggi. Contohnya, `CUS_ANON_009` memiliki `Num_of_Loan = 0`, tetapi EMI bulanannya sebesar 66.045. Jika customer tidak memiliki loan, maka EMI seharusnya 0 atau mendekati 0, bukan puluhan ribu.

Namun, semua baris dengan EMI lebih besar dari gaji bulanan tidak boleh langsung dihapus. Ada kemungkinan beberapa customer memang memiliki EMI tinggi karena banyak pinjaman atau data pendapatan yang tidak sepenuhnya merepresentasikan seluruh kemampuan finansial. Oleh karena itu, perbaikan tidak dilakukan hanya berdasarkan rasio ini saja.

Kesimpulannya, output ini memperkuat bahwa `Total_EMI_per_month` memiliki outlier ekstrem. Kasus yang paling perlu dicurigai adalah EMI puluhan ribu, rasio EMI terhadap gaji bulanan jauh di atas 1, dan baris dengan `Num_of_Loan = 0` tetapi EMI sangat besar. Tahap berikutnya perlu mengecek riwayat `Customer_ID` yang sama untuk melihat apakah nilai EMI ekstrem hanya muncul pada satu bulan tertentu atau konsisten di seluruh bulan.

In [ ]:
# VISUALIZE EMI VS MONTHLY SALARY

plt.figure(figsize=(7, 5))
plt.scatter(
    emi_income_clean["Monthly_Inhand_Salary"],
    emi_income_clean["Total_EMI_per_month"],
    alpha=0.3
)
plt.title("Total EMI vs Monthly Inhand Salary")
plt.xlabel("Monthly Inhand Salary")
plt.ylabel("Total EMI per Month")
plt.show()

### Hubungan Total_EMI_per_monthdenganNum_of_Loan

In [ ]:
# RELATIONSHIP BETWEEN TOTAL EMI AND NUMBER OF LOANS

emi_loan_data = eda.dropna(
    subset=[
        "Total_EMI_per_month",
        "Num_of_Loan"
    ]
).copy()

emi_loan_data = emi_loan_data[
    emi_loan_data["Num_of_Loan"] > 0
].copy()

emi_loan_data["EMI_per_Loan"] = (
    emi_loan_data["Total_EMI_per_month"]
    / emi_loan_data["Num_of_Loan"]
)

emi_loan_data = emi_loan_data.replace(
    [np.inf, -np.inf],
    np.nan
)

emi_loan_data = emi_loan_data.dropna(
    subset=[
        "EMI_per_Loan"
    ]
).copy()

emi_loan_summary = (
    emi_loan_data
    .groupby("Num_of_Loan")["Total_EMI_per_month"]
    .agg(["count", "mean", "median", "min", "max"])
)

emi_loan_summary

Kode ini digunakan untuk menganalisis hubungan antara `Num_of_Loan` dan `Total_EMI_per_month`.

Pertama, data difilter hanya untuk customer yang memiliki `Num_of_Loan > 0`, karena jika jumlah loan 0 maka tidak masuk akal untuk menghitung EMI per loan. Setelah itu, dibuat kolom baru `EMI_per_Loan`, yaitu hasil pembagian `Total_EMI_per_month` dengan `Num_of_Loan`. Tujuannya adalah melihat rata-rata beban EMI per pinjaman.

Output kemudian menampilkan ringkasan `Total_EMI_per_month` berdasarkan jumlah loan. Dari hasilnya, terlihat bahwa median EMI cenderung meningkat ketika `Num_of_Loan` bertambah. Misalnya, median EMI untuk 1 loan adalah sekitar 31,57, sedangkan untuk 9 loan median EMI naik menjadi sekitar 135,96. Ini cukup masuk akal karena semakin banyak pinjaman, total EMI biasanya semakin besar.

Namun, nilai mean jauh lebih besar daripada median pada semua jumlah loan. Contohnya, untuk `Num_of_Loan = 1`, mean EMI sekitar 1.431,99, tetapi median hanya 31,57. Perbedaan besar ini menunjukkan bahwa rata-rata tertarik ke atas oleh nilai EMI ekstrem.

Selain itu, nilai maksimum pada hampir semua kelompok loan masih sangat tinggi, yaitu sekitar 77.000 sampai 82.000. Ini menunjukkan bahwa outlier `Total_EMI_per_month` tidak hanya terjadi pada customer dengan banyak loan, tetapi juga muncul pada hampir semua kategori jumlah loan, termasuk customer dengan hanya 1 atau 2 loan.

Kesimpulannya, jumlah loan memang memiliki hubungan logis dengan EMI, terutama jika dilihat dari median yang meningkat. Namun, nilai maksimum yang sangat ekstrem menunjukkan bahwa masalah outlier pada `Total_EMI_per_month` masih kuat. Karena itu, kolom ini perlu dianalisis lebih lanjut menggunakan riwayat `Customer_ID` yang sama sebelum diperbaiki, bukan hanya berdasarkan `Num_of_Loan`.

In [ ]:
# HITUNG PERSENTIL TOTAL_EMI_PER_MONTH SEBELUM ANALISIS HIGH EMI LOW LOAN

emi_loan_data = df[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Total_EMI_per_month",
        "Credit_Score"
    ]
].copy()

emi_loan_data = emi_loan_data.dropna(
    subset=[
        "Num_of_Loan",
        "Total_EMI_per_month"
    ]
).copy()

emi_percentiles = [
    0.50,
    0.75,
    0.90,
    0.95,
    0.97,
    0.98,
    0.99,
    0.995,
    1.00
]

total_emi_percentile = (
    emi_loan_data["Total_EMI_per_month"]
    .quantile(emi_percentiles)
    .reset_index()
)

total_emi_percentile.columns = [
    "Percentile",
    "Total_EMI_per_month"
]

total_emi_percentile["Percentile"] = (
    total_emi_percentile["Percentile"] * 100
)

total_emi_percentile["Increase"] = (
    total_emi_percentile["Total_EMI_per_month"].diff()
)

total_emi_percentile

In [ ]:
# CEK JUMLAH DATA DI ATAS BEBERAPA THRESHOLD TOTAL EMI

emi_thresholds = [
    500,
    1000,
    1500,
    2000,
    5000,
    10000
]

emi_threshold_summary = []

for threshold in emi_thresholds:
    count_above = (
        emi_loan_data["Total_EMI_per_month"] > threshold
    ).sum()

    percentage_above = (
        count_above / len(emi_loan_data) * 100
    )

    emi_threshold_summary.append({
        "Threshold": threshold,
        "Count_Above_Threshold": count_above,
        "Percentage": percentage_above
    })

emi_threshold_summary = pd.DataFrame(emi_threshold_summary)

emi_threshold_summary

In [ ]:
# CEK DISTRIBUSI NUM_OF_LOAN

num_of_loan_distribution = (
    emi_loan_data["Num_of_Loan"]
    .value_counts(dropna=False)
    .sort_index()
    .reset_index()
)

num_of_loan_distribution.columns = [
    "Num_of_Loan",
    "Count"
]

num_of_loan_distribution["Percentage"] = (
    num_of_loan_distribution["Count"] /
    len(emi_loan_data) * 100
)

num_of_loan_distribution

In [ ]:
# CHECK HIGH EMI WITH LOW NUMBER OF LOANS

emi_loan_data = emi_loan_data[
    emi_loan_data["Num_of_Loan"] > 0
].copy()

emi_loan_data["EMI_per_Loan"] = (
    emi_loan_data["Total_EMI_per_month"] /
    emi_loan_data["Num_of_Loan"]
)

emi_loan_data = emi_loan_data.replace(
    [np.inf, -np.inf],
    np.nan
)

emi_loan_data = emi_loan_data.dropna(
    subset=[
        "EMI_per_Loan"
    ]
).copy()

high_emi_low_loan = emi_loan_data[
    (emi_loan_data["Total_EMI_per_month"] > 2000)
    & (emi_loan_data["Num_of_Loan"] <= 2)
].copy()

print("Jumlah EMI > 2000 dengan Num_of_Loan <= 2:", len(high_emi_low_loan))

high_emi_low_loan[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Total_EMI_per_month",
        "EMI_per_Loan",
        "Credit_Score"
    ]
].sort_values(
    by="Total_EMI_per_month",
    ascending=False
).head(30)

Output ini digunakan untuk mengecek kasus `Total_EMI_per_month` yang tinggi, tetapi jumlah pinjamannya rendah. Data difilter dengan dua kondisi, yaitu `Total_EMI_per_month > 2000` dan `Num_of_Loan <= 2`.

Threshold `Total_EMI_per_month > 2000` dipakai sebagai batas investigasi karena dari distribusi sebelumnya, mayoritas nilai EMI jauh lebih kecil. Median `Total_EMI_per_month` hanya sekitar 68, P75 sekitar 160, P90 sekitar 288, dan P95 sekitar 446. Jadi, nilai di atas 2000 sudah jauh di atas pola mayoritas data. Selain itu, jumlah data di atas 2000 adalah 812 baris atau sekitar 3,25%, sehingga jumlahnya tidak terlalu besar dan cocok digunakan sebagai kelompok yang perlu dianalisis lebih lanjut.

Batas `Num_of_Loan <= 2` dipakai untuk melihat kasus EMI tinggi pada customer yang hanya memiliki sedikit pinjaman. Secara logika, customer dengan 1 atau 2 pinjaman masih mungkin memiliki EMI, tetapi jika EMI bulanannya mencapai puluhan ribu, maka nilainya menjadi mencurigakan. Jika jumlah pinjaman sedikit, seharusnya total EMI tidak terlalu ekstrem, kecuali pinjamannya sangat besar. Oleh karena itu, kombinasi `EMI > 2000` dan `Num_of_Loan <= 2` digunakan untuk mencari kasus yang lebih tidak wajar.

Hasil output menunjukkan terdapat 200 baris dengan `Total_EMI_per_month > 2000` dan `Num_of_Loan <= 2`. Ini berarti dari seluruh kasus EMI tinggi, ada cukup banyak customer yang hanya memiliki 1 atau 2 pinjaman tetapi EMI bulanannya sangat besar.

Pada tabel, terlihat beberapa kasus yang sangat ekstrem. Misalnya, `CUS_ANON_014` hanya memiliki `Num_of_Loan = 1`, tetapi `Total_EMI_per_month` mencapai 81.935. Karena hanya ada satu loan, nilai `EMI_per_Loan` juga menjadi 81.935. Nilai ini sangat tinggi dan perlu dicurigai sebagai outlier.

Contoh lain, `CUS_ANON_028` memiliki `Num_of_Loan = 2` dengan `Total_EMI_per_month` sebesar 81.257, sehingga `EMI_per_Loan` sekitar 40.628,5. Secara logika, EMI per loan sebesar puluhan ribu sangat tidak wajar jika dibandingkan dengan pola umum data, terutama jika pendapatan bulanan customer tidak sebanding dengan nilai tersebut.

Beberapa baris juga memiliki `Monthly_Inhand_Salary` yang missing, misalnya `CUS_ANON_014`, `CUS_ANON_018`, dan `CUS_ANON_007`. Untuk kasus seperti ini, rasio EMI terhadap gaji bulanan belum bisa dihitung secara langsung. Namun, nilai EMI yang sangat besar tetap perlu diperiksa menggunakan riwayat `Customer_ID` dan referensi bulan lain.

Kesimpulannya, output ini memperkuat bahwa outlier `Total_EMI_per_month` tidak hanya terjadi pada customer dengan banyak pinjaman. Bahkan customer dengan 1 atau 2 pinjaman pun memiliki EMI puluhan ribu. Ini menjadi indikasi kuat bahwa sebagian nilai `Total_EMI_per_month` kemungkinan tidak konsisten dan perlu dicek lebih lanjut berdasarkan riwayat customer yang sama.

Namun, hasil ini belum menjadi dasar untuk langsung menghapus data. Threshold 2000 dan batas 2 loan hanya digunakan sebagai alat investigasi. Keputusan perbaikan sebaiknya tetap dilakukan setelah mengecek apakah nilai EMI ekstrem tersebut hanya muncul pada satu bulan tertentu, apakah customer yang sama memiliki referensi EMI valid pada bulan lain, dan apakah nilai tersebut konsisten dengan pendapatan serta jumlah pinjamannya.

In [ ]:
# CEK RIWAYAT CUSTOMER DENGAN HIGH EMI LOW LOAN

high_emi_low_loan_customer_ids = (
    high_emi_low_loan["Customer_ID"]
    .drop_duplicates()
    .head(10)
    .tolist()
)

emi_history_columns = [
    "ID",
    "Customer_ID",
    "Month",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Num_of_Loan",
    "Total_EMI_per_month",
    "Credit_Score"
]

if "Month_Number" in df.columns:
    emi_history_columns.insert(3, "Month_Number")
    sort_columns = ["Customer_ID", "Month_Number"]
else:
    sort_columns = ["Customer_ID", "Month"]

emi_customer_history = df[
    df["Customer_ID"].isin(high_emi_low_loan_customer_ids)
][emi_history_columns].copy()

emi_customer_history.sort_values(
    by=sort_columns
)

Output ini menampilkan riwayat beberapa `Customer_ID` yang memiliki kasus `Total_EMI_per_month` tinggi dengan jumlah loan rendah. Tujuannya adalah mengecek apakah nilai EMI ekstrem tersebut konsisten di beberapa bulan, atau hanya muncul sebagai lonjakan pada satu bulan tertentu.

Dari output ini, terlihat bahwa banyak customer memiliki `Annual_Income`, `Monthly_Inhand_Salary`, dan `Num_of_Loan` yang stabil, tetapi `Total_EMI_per_month` tiba-tiba melonjak sangat besar pada satu bulan tertentu.

Contohnya, `CUS_ANON_001` memiliki `Annual_Income` 118.636,88, `Monthly_Inhand_Salary` 9.865,41, dan `Num_of_Loan = 1`. Pada bulan May dan July, `Total_EMI_per_month` hanya 86,54. Namun, pada bulan March, nilai EMI tiba-tiba menjadi 27.780. Karena kondisi income dan jumlah loan tetap sama, lonjakan EMI ini sangat mencurigakan sebagai outlier.

Pola yang sama juga terlihat pada `CUS_ANON_019`. Customer ini memiliki `Annual_Income` 20.272,23, `Monthly_Inhand_Salary` 1.776,35, dan `Num_of_Loan = 1`. Pada beberapa bulan, EMI hanya 11,48, tetapi pada bulan March melonjak menjadi 32.656. Karena nilai EMI normal customer tersebut muncul berulang di bulan lain, nilai 32.656 kemungkinan besar merupakan data error.

Contoh lain adalah `CUS_ANON_023`. Pada bulan July, EMI hanya 104,35, tetapi pada bulan June EMI menjadi 79.861, padahal `Annual_Income`, `Monthly_Inhand_Salary`, dan `Num_of_Loan` tetap sama. Ini semakin memperkuat bahwa masalah utama berada pada `Total_EMI_per_month`, bukan pada income atau jumlah loan.

Output ini juga menunjukkan bahwa beberapa customer memiliki `Monthly_Inhand_Salary` yang missing pada bulan tertentu, tetapi masih memiliki referensi EMI valid dari bulan lain. Misalnya, `CUS_ANON_032` memiliki EMI 6.110 pada April, tetapi pada June EMI hanya 32,82. Untuk kasus seperti ini, nilai EMI ekstrem tetap bisa diperiksa menggunakan riwayat customer yang sama.

Kesimpulannya, output ini memberikan bukti kuat bahwa sebagian nilai ekstrem `Total_EMI_per_month` merupakan lonjakan tidak wajar pada bulan tertentu. Karena customer yang sama memiliki nilai EMI yang jauh lebih kecil dan konsisten pada bulan lain, maka nilai EMI ekstrem dapat diperbaiki menggunakan median `Total_EMI_per_month` valid dari `Customer_ID` yang sama. Jika tidak ada referensi valid dari customer yang sama, nilai ekstrem lebih aman dijadikan missing value dan ditangani pada pipeline preprocessing.

In [ ]:
# CEK REFERENSI EMI VALID DARI CUSTOMER YANG SAMA

valid_emi_reference = (
    df[
        (df["Total_EMI_per_month"].notna()) &
        (df["Total_EMI_per_month"] <= 2000)
    ]
    .groupby("Customer_ID")["Total_EMI_per_month"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)

high_emi_reference_check = (
    high_emi_low_loan[
        [
            "ID",
            "Customer_ID",
            "Month",
            "Annual_Income",
            "Monthly_Inhand_Salary",
            "Num_of_Loan",
            "Total_EMI_per_month",
            "EMI_per_Loan",
            "Credit_Score"
        ]
    ]
    .merge(
        valid_emi_reference,
        on="Customer_ID",
        how="left"
    )
)

high_emi_reference_check["Has_Valid_EMI_Reference"] = (
    high_emi_reference_check["count"].notna()
)

print("Jumlah high EMI low loan:", len(high_emi_reference_check))
print(
    "Memiliki referensi EMI valid dari Customer_ID yang sama:",
    high_emi_reference_check["Has_Valid_EMI_Reference"].sum()
)
print(
    "Tidak memiliki referensi EMI valid:",
    (~high_emi_reference_check["Has_Valid_EMI_Reference"]).sum()
)

high_emi_reference_check.sort_values(
    by="Total_EMI_per_month",
    ascending=False
).head(30)

Output ini menunjukkan hasil pengecekan lanjutan untuk kasus `Total_EMI_per_month` yang tinggi, tetapi jumlah pinjaman rendah (`Num_of_Loan <= 2`).

Terdapat 200 baris yang masuk kategori `high EMI low loan`. Artinya, ada 200 baris dengan `Total_EMI_per_month > 2000` meskipun customer hanya memiliki 1 atau 2 loan. Kondisi ini perlu dicurigai karena EMI yang sangat tinggi seharusnya lebih mungkin terjadi pada customer dengan jumlah pinjaman besar atau pendapatan yang sangat tinggi.

Dari 200 kasus tersebut, sebanyak 177 kasus memiliki referensi EMI valid dari `Customer_ID` yang sama. Artinya, pada bulan lain customer tersebut memiliki nilai `Total_EMI_per_month` yang lebih rendah dan lebih masuk akal. Sementara itu, 23 kasus tidak memiliki referensi valid, sehingga nilai ekstremnya tidak bisa diperbaiki langsung menggunakan riwayat customer yang sama.

Tabel juga menunjukkan bahwa banyak nilai ekstrem memiliki perbedaan yang sangat jauh dari referensi valid customer. Contohnya, `CUS_ANON_014` memiliki `Total_EMI_per_month` sebesar 81.935 dengan `Num_of_Loan = 1`, tetapi referensi valid dari customer yang sama hanya sekitar 78,42. Hal ini menunjukkan bahwa nilai 81.935 kemungkinan besar merupakan outlier.

Contoh lain adalah `CUS_ANON_023`, yang memiliki `Total_EMI_per_month` sebesar 79.861 dengan `Num_of_Loan = 1`, sedangkan referensi valid customer yang sama hanya sekitar 104,35. Perbedaan yang sangat besar ini menunjukkan bahwa nilai EMI ekstrem tidak konsisten dengan riwayat customer.

Kolom `count`, `median`, `min`, dan `max` menunjukkan ringkasan nilai EMI valid dari customer yang sama. Jika `Has_Valid_EMI_Reference` bernilai `True`, maka nilai ekstrem dapat diperbaiki menggunakan median EMI valid customer tersebut. Jika bernilai `False`, maka nilai ekstrem lebih aman dijadikan missing value daripada dipaksa menggunakan nilai dari customer lain.

Kesimpulannya, output ini memberikan bukti kuat bahwa sebagian besar kasus `Total_EMI_per_month` ekstrem pada customer dengan loan rendah merupakan anomali. Karena 177 dari 200 kasus memiliki referensi valid, perbaikan dapat dilakukan dengan mengganti nilai ekstrem menggunakan median `Total_EMI_per_month` dari `Customer_ID` yang sama. Untuk 23 kasus tanpa referensi valid, nilai ekstrem dapat dijadikan missing value dan nanti ditangani pada pipeline preprocessing model.

In [ ]:
# CEK KASUS NUM_OF_LOAN = 0 TETAPI TOTAL EMI TINGGI

zero_loan_high_emi = df[
    (df["Num_of_Loan"] == 0) &
    (df["Total_EMI_per_month"] > 2000)
].copy()

print("Jumlah Num_of_Loan = 0 tetapi Total_EMI_per_month > 2000:", len(zero_loan_high_emi))

zero_loan_high_emi[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Total_EMI_per_month",
        "Credit_Score"
    ]
].sort_values(
    by="Total_EMI_per_month",
    ascending=False
).head(30)

Output ini menampilkan kasus ketika `Num_of_Loan = 0`, tetapi `Total_EMI_per_month > 2000`.

Secara logika, ini adalah ketidaksesuaian yang sangat kuat. Jika customer tidak memiliki loan (`Num_of_Loan = 0`), maka seharusnya `Total_EMI_per_month` bernilai 0 atau setidaknya tidak muncul sebagai nilai puluhan ribu. EMI adalah cicilan bulanan dari pinjaman, sehingga jika jumlah pinjaman nol, beban EMI bulanan tidak seharusnya sangat besar.

Pada output ini, banyak customer memiliki `Num_of_Loan = 0`, tetapi `Total_EMI_per_month` berada di kisaran 64.000 sampai 82.000. Contohnya, `CUS_ANON_013` memiliki `Num_of_Loan = 0`, tetapi `Total_EMI_per_month` sebesar 82.163. Contoh lain, `CUS_ANON_010` juga memiliki `Num_of_Loan = 0`, tetapi EMI bulanannya sebesar 81.171. Kondisi seperti ini tidak konsisten secara bisnis.

Beberapa customer juga memiliki `Monthly_Inhand_Salary` yang jauh lebih kecil daripada EMI. Misalnya, `CUS_ANON_010` memiliki gaji bulanan 2.011,50, tetapi EMI sebesar 81.171. Ini semakin memperkuat bahwa nilai `Total_EMI_per_month` pada baris tersebut tidak realistis.

Kasus ini lebih kuat dibandingkan hanya membandingkan EMI dengan income. Alasannya, meskipun customer memiliki pendapatan tinggi, jika `Num_of_Loan = 0`, maka EMI tetap tidak seharusnya besar. Jadi, masalah utama pada output ini kemungkinan besar berasal dari nilai ekstrem pada `Total_EMI_per_month`.

Berdasarkan output ini, baris dengan `Num_of_Loan = 0` dan `Total_EMI_per_month > 2000` dapat ditandai sebagai anomali kuat. Pada tahap perbaikan, nilai EMI seperti ini dapat diperbaiki dengan aturan logis: jika `Num_of_Loan = 0`, maka `Total_EMI_per_month` sebaiknya dibuat menjadi 0. Namun sebelum final cleaning, tetap baik untuk menyimpan audit agar perubahan data dapat dijelaskan dalam laporan.

-----

Berdasarkan seluruh output yang sudah diperiksa, kolom `Total_EMI_per_month` memiliki indikasi outlier yang sangat kuat. Secara distribusi, mayoritas nilai EMI sebenarnya kecil. Median hanya sekitar 68, P75 sekitar 160, P90 sekitar 288, dan P95 sekitar 446. Namun, setelah itu terjadi lonjakan besar, terutama mulai P97 sampai maksimum. Nilai maksimum bahkan mencapai 82.331. Artinya, sebagian kecil data memiliki EMI bulanan yang sangat jauh dari pola mayoritas data.

Dari threshold analysis, terdapat 812 baris atau sekitar 3,25% data dengan `Total_EMI_per_month > 2000`. Threshold 2000 dipakai sebagai batas investigasi karena nilai ini sudah jauh di atas P95 dan hanya mencakup sebagian kecil data. Jadi, nilai di atas 2000 tidak langsung dihapus, tetapi dianggap sebagai kandidat anomali yang perlu dicek lebih lanjut.

Ketika dibandingkan dengan income, ditemukan 687 baris di mana `Total_EMI_per_month` lebih besar daripada `Monthly_Inhand_Salary`. Beberapa kasus bahkan memiliki rasio EMI terhadap gaji bulanan puluhan hingga ratusan kali lipat. Ini sangat tidak wajar karena secara logika EMI bulanan tidak seharusnya jauh lebih besar daripada gaji bulanan customer.

Analisis dengan `Num_of_Loan` juga memperkuat indikasi masalah. Terdapat 200 baris dengan `Total_EMI_per_month > 2000` padahal `Num_of_Loan <= 2`. Ini mencurigakan karena customer dengan hanya 1 atau 2 loan seharusnya tidak memiliki total EMI puluhan ribu, kecuali ada alasan finansial yang sangat kuat. Namun, dari riwayat customer, banyak kasus menunjukkan bahwa EMI ekstrem hanya muncul pada satu bulan tertentu, sedangkan bulan lain memiliki EMI yang jauh lebih kecil dan konsisten.

Hasil pengecekan referensi juga penting. Dari 200 kasus high EMI low loan, sebanyak 177 kasus memiliki referensi EMI valid dari `Customer_ID` yang sama. Artinya, sebagian besar nilai EMI ekstrem bisa diperbaiki menggunakan median EMI valid dari customer tersebut. Sementara itu, 23 kasus tidak memiliki referensi valid, sehingga lebih aman dijadikan missing value daripada dipaksakan menggunakan nilai dari customer lain.

Selain itu, ditemukan kasus yang lebih kuat lagi, yaitu `Num_of_Loan = 0` tetapi `Total_EMI_per_month > 2000`. Secara bisnis, ini tidak konsisten karena jika customer tidak memiliki loan, maka EMI bulanan seharusnya 0 atau sangat kecil. Pada output, banyak baris dengan `Num_of_Loan = 0` justru memiliki EMI puluhan ribu. Kasus seperti ini dapat dianggap sebagai anomali kuat pada `Total_EMI_per_month`.

Kesimpulannya, `Total_EMI_per_month` perlu diperbaiki. Rule yang paling aman adalah: jika `Num_of_Loan = 0` dan `Total_EMI_per_month > 2000`, nilai EMI dapat diubah menjadi 0. Untuk kasus EMI ekstrem lain yang memiliki referensi valid dari customer yang sama, nilai EMI dapat diganti menggunakan median EMI valid customer tersebut. Jika tidak ada referensi valid, nilai EMI ekstrem dijadikan missing value dan nanti ditangani oleh pipeline preprocessing model.

In [ ]:
# CHECK NUM_OF_LOAN = 0 BUT TOTAL EMI IS POSITIVE

zero_loan_positive_emi = df[
    (df["Num_of_Loan"] == 0) &
    (df["Total_EMI_per_month"] > 0)
].copy()

print("Jumlah Num_of_Loan = 0 tetapi Total_EMI_per_month > 0:", len(zero_loan_positive_emi))
print(
    "Persentase:",
    round(len(zero_loan_positive_emi) / len(df) * 100, 2),
    "%"
)

zero_loan_positive_emi[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Type_of_Loan",
        "Total_EMI_per_month",
        "Credit_Score"
    ]
].sort_values(
    by="Total_EMI_per_month",
    ascending=False
).head(30)

In [ ]:
# SUMMARY TOTAL EMI WHEN NUM_OF_LOAN = 0

zero_loan_emi_summary = df[
    df["Num_of_Loan"] == 0
]["Total_EMI_per_month"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.97,
        0.98,
        0.99
    ]
)

zero_loan_emi_summary

In [ ]:
# CHECK ZERO LOAN EMI BY THRESHOLD

zero_loan_thresholds = [
    0,
    1,
    10,
    100,
    500,
    1000,
    2000
]

zero_loan_threshold_summary = []

zero_loan_data = df[
    df["Num_of_Loan"] == 0
].copy()

for threshold in zero_loan_thresholds:
    count_above = (
        zero_loan_data["Total_EMI_per_month"] > threshold
    ).sum()
    
    percentage_above = (
        count_above / len(zero_loan_data) * 100
    )
    
    zero_loan_threshold_summary.append({
        "Threshold": threshold,
        "Count_Above_Threshold": count_above,
        "Percentage": percentage_above
    })

zero_loan_threshold_summary = pd.DataFrame(zero_loan_threshold_summary)

zero_loan_threshold_summary

Output ini mengecek kasus `Num_of_Loan = 0` tetapi `Total_EMI_per_month` masih memiliki nilai positif.

Dari tabel pertama, terlihat bahwa banyak customer memiliki `Num_of_Loan = 0`, `Type_of_Loan = NaN`, tetapi `Total_EMI_per_month` sangat tinggi, bahkan mencapai 64.000 sampai 82.000. Secara bisnis, ini tidak konsisten karena jika customer tidak memiliki loan dan tipe loan juga kosong, maka EMI bulanan seharusnya 0.

Contohnya, `CUS_ANON_013` memiliki `Num_of_Loan = 0`, `Type_of_Loan = NaN`, tetapi `Total_EMI_per_month` sebesar 82.163. Contoh lain, `CUS_ANON_010` juga memiliki `Num_of_Loan = 0`, tetapi EMI sebesar 81.171. Kondisi ini memperkuat bahwa nilai ekstrem pada `Total_EMI_per_month` kemungkinan besar merupakan data error.

Ringkasan statistik juga mendukung hal tersebut. Untuk data dengan `Num_of_Loan = 0`, jumlahnya ada 2.826 baris. Median, P75, dan P90 semuanya bernilai 0, artinya mayoritas customer tanpa loan memang memiliki EMI 0. Ini pola yang benar dan logis.

Namun, setelah P90, mulai muncul nilai positif. P95 sudah naik menjadi 802,91, P97 melonjak menjadi 21.903,25, P98 menjadi 42.438, P99 menjadi 65.023,25, dan maksimum mencapai 82.163. Lonjakan ini menunjukkan bahwa hanya sebagian kecil data `Num_of_Loan = 0` yang memiliki EMI sangat tinggi, sedangkan mayoritasnya 0.

Karena mayoritas data tanpa loan memiliki EMI 0, maka kasus `Num_of_Loan = 0` tetapi `Total_EMI_per_month` tinggi dapat dianggap sebagai anomali kuat. Hal ini juga diperkuat oleh `Type_of_Loan` yang bernilai missing/NaN, sehingga tidak ada informasi loan yang mendukung adanya EMI.

Berdasarkan hasil ini, aturan cleaning yang paling masuk akal adalah: jika `Num_of_Loan = 0`, maka `Total_EMI_per_month` sebaiknya dibuat menjadi 0. Rule ini tidak terlalu agresif karena sesuai dengan logika bisnis dan didukung oleh pola mayoritas data, di mana median sampai P90 EMI untuk customer tanpa loan memang 0.

Kesimpulannya, output ini memperkuat bahwa sebagian nilai ekstrem `Total_EMI_per_month` berasal dari baris yang sebenarnya tidak memiliki loan. Jadi, pada tahap perbaikan nanti, kasus `Num_of_Loan = 0` dapat diperbaiki dengan mengubah `Total_EMI_per_month` menjadi 0, lalu kasus EMI ekstrem lainnya dicek berdasarkan referensi valid dari `Customer_ID` yang sama.

In [ ]:
# CEK APAKAH NUM_OF_LOAN = 0 TAPI EMI POSITIF BENAR-BENAR NO LOAN ATAU ADA KONFLIK RIWAYAT

zero_loan_positive_emi = df[
    (df["Num_of_Loan"] == 0) &
    (df["Total_EMI_per_month"] > 0)
].copy()

customer_loan_history = (
    df.groupby("Customer_ID")
    .agg(
        Max_Num_of_Loan=("Num_of_Loan", "max"),
        Min_Num_of_Loan=("Num_of_Loan", "min"),
        Non_Missing_Type_of_Loan=("Type_of_Loan", lambda x: x.notna().sum()),
        Valid_EMI_Count=("Total_EMI_per_month", lambda x: ((x.notna()) & (x <= 2000)).sum()),
        Valid_EMI_Median=("Total_EMI_per_month", lambda x: x[(x.notna()) & (x <= 2000)].median())
    )
    .reset_index()
)

zero_loan_positive_emi_check = zero_loan_positive_emi.merge(
    customer_loan_history,
    on="Customer_ID",
    how="left"
)

zero_loan_positive_emi_check["Type_of_Loan_Is_Missing"] = (
    zero_loan_positive_emi_check["Type_of_Loan"].isna()
)

zero_loan_positive_emi_check["Consistent_No_Loan_History"] = (
    (zero_loan_positive_emi_check["Max_Num_of_Loan"] == 0) &
    (zero_loan_positive_emi_check["Non_Missing_Type_of_Loan"] == 0)
)

zero_loan_positive_emi_check["Conflict_With_Customer_History"] = (
    (zero_loan_positive_emi_check["Max_Num_of_Loan"] > 0) |
    (zero_loan_positive_emi_check["Non_Missing_Type_of_Loan"] > 0)
)

zero_loan_positive_emi_check["Strong_Set_EMI_Zero_Candidate"] = (
    (zero_loan_positive_emi_check["Num_of_Loan"] == 0) &
    (zero_loan_positive_emi_check["Type_of_Loan_Is_Missing"]) &
    (zero_loan_positive_emi_check["Consistent_No_Loan_History"])
)

print("Jumlah Num_of_Loan = 0 tetapi Total_EMI_per_month > 0:", len(zero_loan_positive_emi_check))
print(
    "Kandidat kuat EMI diubah menjadi 0:",
    zero_loan_positive_emi_check["Strong_Set_EMI_Zero_Candidate"].sum()
)
print(
    "Ada konflik dengan riwayat customer:",
    zero_loan_positive_emi_check["Conflict_With_Customer_History"].sum()
)

zero_loan_positive_emi_check[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Type_of_Loan",
        "Total_EMI_per_month",
        "Max_Num_of_Loan",
        "Non_Missing_Type_of_Loan",
        "Valid_EMI_Count",
        "Valid_EMI_Median",
        "Consistent_No_Loan_History",
        "Conflict_With_Customer_History",
        "Strong_Set_EMI_Zero_Candidate",
        "Credit_Score"
    ]
].sort_values(
    by="Total_EMI_per_month",
    ascending=False
).head(30)

Output ini mengecek kasus `Num_of_Loan = 0` tetapi `Total_EMI_per_month` masih memiliki nilai positif.

Dari tabel pertama, terlihat bahwa banyak customer memiliki `Num_of_Loan = 0`, `Type_of_Loan = NaN`, tetapi `Total_EMI_per_month` sangat tinggi, bahkan mencapai 64.000 sampai 82.000. Secara bisnis, ini terlihat tidak konsisten karena jika customer tidak memiliki loan dan tipe loan juga kosong, maka EMI bulanan seharusnya 0 atau tidak muncul sebagai nilai yang sangat besar.

Contohnya, `CUS_ANON_013` memiliki `Num_of_Loan = 0`, `Type_of_Loan = NaN`, tetapi `Total_EMI_per_month` sebesar 82.163. Contoh lain, `CUS_ANON_010` juga memiliki `Num_of_Loan = 0`, tetapi EMI sebesar 81.171. Kondisi ini menjadi indikasi awal bahwa nilai ekstrem pada `Total_EMI_per_month` kemungkinan merupakan data yang tidak konsisten.

Ringkasan statistik juga mendukung indikasi tersebut. Untuk data dengan `Num_of_Loan = 0`, jumlahnya ada 2.826 baris. Median, P75, dan P90 semuanya bernilai 0, artinya mayoritas customer tanpa loan memang memiliki EMI 0. Ini merupakan pola yang logis dan sesuai dengan konsep EMI.

Namun, setelah P90 mulai muncul nilai positif yang cukup besar. P95 naik menjadi 802,91, P97 melonjak menjadi 21.903,25, P98 menjadi 42.438, P99 menjadi 65.023,25, dan maksimum mencapai 82.163. Lonjakan ini menunjukkan bahwa hanya sebagian kecil data `Num_of_Loan = 0` yang memiliki EMI sangat tinggi, sedangkan mayoritasnya tetap 0.

Hasil ini belum langsung digunakan sebagai keputusan final untuk mengubah semua EMI menjadi 0. Output ini baru menjadi bukti awal bahwa kasus `Num_of_Loan = 0` tetapi `Total_EMI_per_month` positif perlu dianalisis lebih lanjut. Alasannya, masih ada kemungkinan bahwa yang salah bukan EMI, tetapi `Num_of_Loan` atau `Type_of_Loan` yang tercatat tidak lengkap.

Oleh karena itu, tahap berikutnya perlu mengecek riwayat `Customer_ID` yang sama. Jika customer tersebut sepanjang riwayatnya selalu memiliki `Num_of_Loan = 0`, `Type_of_Loan` selalu kosong, dan referensi EMI validnya bernilai 0, maka EMI positif atau ekstrem lebih kuat dianggap sebagai kesalahan pada `Total_EMI_per_month`. Namun, jika pada bulan lain customer pernah memiliki loan atau `Type_of_Loan` tidak kosong, maka kasus tersebut tidak boleh langsung diubah menjadi 0 karena bisa saja yang bermasalah adalah pencatatan jumlah loan.

Kesimpulannya, output ini menunjukkan adanya indikasi kuat anomali pada `Total_EMI_per_month`, terutama untuk baris dengan `Num_of_Loan = 0`, `Type_of_Loan = NaN`, dan EMI puluhan ribu. Namun, keputusan cleaning final belum diambil pada tahap ini. Analisis perlu dilanjutkan dengan pengecekan riwayat customer untuk membedakan apakah nilai yang salah adalah `Total_EMI_per_month` atau justru informasi loan-nya.

In [ ]:
# RINGKASAN KATEGORI NUM_OF_LOAN = 0 TAPI EMI POSITIF

zero_loan_positive_emi_summary = pd.DataFrame({
    "Category": [
        "Total Num_of_Loan = 0 but EMI > 0",
        "Strong candidate set EMI to 0",
        "Conflict with customer loan history",
        "Has valid EMI reference from same customer"
    ],
    "Count": [
        len(zero_loan_positive_emi_check),
        zero_loan_positive_emi_check["Strong_Set_EMI_Zero_Candidate"].sum(),
        zero_loan_positive_emi_check["Conflict_With_Customer_History"].sum(),
        zero_loan_positive_emi_check["Valid_EMI_Count"].gt(0).sum()
    ]
})

zero_loan_positive_emi_summary["Percentage"] = (
    zero_loan_positive_emi_summary["Count"] /
    len(zero_loan_positive_emi_check) * 100
)

zero_loan_positive_emi_summary

In [ ]:
# CEK HIPOTESIS: APAKAH NUM_OF_LOAN YANG SALAH ATAU TOTAL EMI YANG SALAH

zero_loan_positive_emi = df[
    (df["Num_of_Loan"] == 0) &
    (df["Total_EMI_per_month"] > 0)
].copy()

customer_loan_emi_history = (
    df.groupby("Customer_ID")
    .agg(
        Max_Num_of_Loan=("Num_of_Loan", "max"),
        Min_Num_of_Loan=("Num_of_Loan", "min"),
        Non_Missing_Type_of_Loan=("Type_of_Loan", lambda x: x.notna().sum()),
        Valid_EMI_Count=("Total_EMI_per_month", lambda x: ((x.notna()) & (x <= 2000)).sum()),
        Valid_EMI_Median=("Total_EMI_per_month", lambda x: x[(x.notna()) & (x <= 2000)].median()),
        Zero_EMI_Count=("Total_EMI_per_month", lambda x: (x == 0).sum())
    )
    .reset_index()
)

zero_loan_emi_diagnosis = zero_loan_positive_emi.merge(
    customer_loan_emi_history,
    on="Customer_ID",
    how="left"
)

zero_loan_emi_diagnosis["Likely_EMI_Error"] = (
    (zero_loan_emi_diagnosis["Max_Num_of_Loan"] == 0) &
    (zero_loan_emi_diagnosis["Non_Missing_Type_of_Loan"] == 0) &
    (
        (zero_loan_emi_diagnosis["Valid_EMI_Median"] == 0) |
        (zero_loan_emi_diagnosis["Valid_EMI_Median"].isna())
    )
)

zero_loan_emi_diagnosis["Possible_Num_of_Loan_Error"] = (
    (zero_loan_emi_diagnosis["Max_Num_of_Loan"] > 0) |
    (zero_loan_emi_diagnosis["Non_Missing_Type_of_Loan"] > 0)
)

zero_loan_emi_diagnosis["Diagnosis"] = np.select(
    [
        zero_loan_emi_diagnosis["Likely_EMI_Error"],
        zero_loan_emi_diagnosis["Possible_Num_of_Loan_Error"]
    ],
    [
        "Likely Total_EMI_per_month error",
        "Possible Num_of_Loan / Type_of_Loan error"
    ],
    default="Needs further checking"
)

print("Jumlah Num_of_Loan = 0 tetapi EMI > 0:", len(zero_loan_emi_diagnosis))
print(
    "Kemungkinan EMI yang salah:",
    zero_loan_emi_diagnosis["Likely_EMI_Error"].sum()
)
print(
    "Kemungkinan Num_of_Loan / Type_of_Loan yang salah:",
    zero_loan_emi_diagnosis["Possible_Num_of_Loan_Error"].sum()
)

zero_loan_emi_diagnosis[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Type_of_Loan",
        "Total_EMI_per_month",
        "Max_Num_of_Loan",
        "Non_Missing_Type_of_Loan",
        "Valid_EMI_Count",
        "Valid_EMI_Median",
        "Zero_EMI_Count",
        "Likely_EMI_Error",
        "Possible_Num_of_Loan_Error",
        "Diagnosis",
        "Credit_Score"
    ]
].sort_values(
    by="Total_EMI_per_month",
    ascending=False
).head(30)

Output ini mengecek kasus `Num_of_Loan = 0` tetapi `Total_EMI_per_month` masih memiliki nilai positif.

Dari tabel pertama, terlihat bahwa banyak customer memiliki `Num_of_Loan = 0`, `Type_of_Loan = NaN`, tetapi `Total_EMI_per_month` sangat tinggi, bahkan mencapai 64.000 sampai 82.000. Secara bisnis, ini tidak konsisten karena jika customer tidak memiliki loan dan tipe loan juga kosong, maka EMI bulanan seharusnya 0.

Contohnya, `CUS_ANON_013` memiliki `Num_of_Loan = 0`, `Type_of_Loan = NaN`, tetapi `Total_EMI_per_month` sebesar 82.163. Contoh lain, `CUS_ANON_010` juga memiliki `Num_of_Loan = 0`, tetapi EMI sebesar 81.171. Kondisi ini memperkuat bahwa nilai ekstrem pada `Total_EMI_per_month` kemungkinan besar merupakan data error.

Ringkasan statistik juga mendukung hal tersebut. Untuk data dengan `Num_of_Loan = 0`, jumlahnya ada 2.826 baris. Median, P75, dan P90 semuanya bernilai 0, artinya mayoritas customer tanpa loan memang memiliki EMI 0. Ini pola yang benar dan logis.

Namun, setelah P90, mulai muncul nilai positif. P95 sudah naik menjadi 802,91, P97 melonjak menjadi 21.903,25, P98 menjadi 42.438, P99 menjadi 65.023,25, dan maksimum mencapai 82.163. Lonjakan ini menunjukkan bahwa hanya sebagian kecil data `Num_of_Loan = 0` yang memiliki EMI sangat tinggi, sedangkan mayoritasnya 0.

Karena mayoritas data tanpa loan memiliki EMI 0, maka kasus `Num_of_Loan = 0` tetapi `Total_EMI_per_month` tinggi dapat dianggap sebagai anomali kuat. Hal ini juga diperkuat oleh `Type_of_Loan` yang bernilai missing/NaN, sehingga tidak ada informasi loan yang mendukung adanya EMI.

Berdasarkan hasil ini, aturan cleaning yang paling masuk akal adalah: jika `Num_of_Loan = 0`, maka `Total_EMI_per_month` sebaiknya dibuat menjadi 0. Rule ini tidak terlalu agresif karena sesuai dengan logika bisnis dan didukung oleh pola mayoritas data, di mana median sampai P90 EMI untuk customer tanpa loan memang 0.

Kesimpulannya, output ini memperkuat bahwa sebagian nilai ekstrem `Total_EMI_per_month` berasal dari baris yang sebenarnya tidak memiliki loan. Jadi, pada tahap perbaikan nanti, kasus `Num_of_Loan = 0` dapat diperbaiki dengan mengubah `Total_EMI_per_month` menjadi 0, lalu kasus EMI ekstrem lainnya dicek berdasarkan referensi valid dari `Customer_ID` yang sama.

### Hubungan Total_EMI_per_monthdengan riwayatCustomer_ID

In [ ]:
# CHECK TOTAL EMI USING SAME CUSTOMER HISTORY

customer_emi_data = eda.dropna(
    subset=[
        "Customer_ID",
        "Total_EMI_per_month"
    ]
).copy()

customer_emi_reference = (
    customer_emi_data
    .groupby("Customer_ID")["Total_EMI_per_month"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

customer_emi_reference["EMI_Range"] = (
    customer_emi_reference["max"]
    - customer_emi_reference["min"]
)

extreme_emi_history = (
    extreme_emi[
        [
            "ID",
            "Customer_ID",
            "Month",
            "Annual_Income",
            "Monthly_Inhand_Salary",
            "Num_of_Loan",
            "Total_EMI_per_month",
            "Credit_Score"
        ]
    ]
    .merge(
        customer_emi_reference,
        on="Customer_ID",
        how="left"
    )
)

extreme_emi_history["EMI_to_Customer_Median"] = (
    extreme_emi_history["Total_EMI_per_month"]
    / extreme_emi_history["median"]
)

extreme_emi_history = extreme_emi_history.replace(
    [np.inf, -np.inf],
    np.nan
)

extreme_emi_history.sort_values(
    by="EMI_to_Customer_Median",
    ascending=False
).head(30)

Kode ini digunakan untuk mengecek apakah nilai `Total_EMI_per_month` yang ekstrem konsisten dengan riwayat EMI customer yang sama.

Pertama, kode membuat `customer_emi_data` dengan mengambil data yang memiliki `Customer_ID` dan `Total_EMI_per_month`. Setelah itu, data dikelompokkan berdasarkan `Customer_ID` untuk menghitung ringkasan EMI setiap customer, yaitu `count`, `mean`, `median`, `min`, dan `max`.

Kolom `EMI_Range` dibuat dari selisih antara EMI maksimum dan EMI minimum customer yang sama. Jika `EMI_Range` sangat besar, berarti EMI customer tersebut sangat tidak stabil antarbulan. Ini bisa menjadi tanda bahwa ada lonjakan EMI yang tidak wajar pada salah satu bulan.

Kemudian, data `extreme_emi` digabungkan dengan ringkasan EMI customer tersebut. Tujuannya adalah membandingkan nilai EMI ekstrem dengan pola EMI normal dari customer yang sama.

Kolom paling penting adalah `EMI_to_Customer_Median`, yaitu:

`Total_EMI_per_month / median EMI customer yang sama`

Jika nilainya sangat besar, berarti EMI pada baris tersebut jauh lebih tinggi dibandingkan median EMI customer tersebut.

Pada output, terlihat banyak kasus dengan `EMI_to_Customer_Median` ribuan kali lipat. Contohnya, `CUS_ANON_012` memiliki `Total_EMI_per_month` sebesar 59.414, tetapi median EMI customer tersebut hanya sekitar 9,08. Nilai `EMI_to_Customer_Median` menjadi 6.546 kali. Ini menunjukkan bahwa EMI 59.414 sangat tidak konsisten dengan riwayat customer tersebut.

Contoh lain, `CUS_ANON_037` memiliki EMI ekstrem sebesar 77.033, sedangkan median EMI customer hanya 13,61. Rasionya mencapai 5.661 kali. Artinya, nilai EMI ekstrem ini jauh berbeda dari pola EMI customer yang sama.

Kolom `EMI_Range` juga memperkuat bukti ini. Banyak customer memiliki selisih antara EMI maksimum dan minimum yang sangat besar, misalnya puluhan ribu. Hal ini menunjukkan bahwa nilai ekstrem muncul sebagai lonjakan besar, bukan sebagai pola EMI yang stabil.

Jadi, output ini menjadi evidence tambahan bahwa sebagian nilai `Total_EMI_per_month` ekstrem kemungkinan besar merupakan anomali. Alasannya, nilai tersebut bukan hanya tinggi dibandingkan dataset secara umum, tetapi juga sangat jauh dibandingkan riwayat customer yang sama.

Namun, untuk keputusan cleaning final, sebaiknya referensi EMI valid dibuat dari dataframe `df` yang sudah dibersihkan dan menggunakan batas EMI valid, misalnya `Total_EMI_per_month <= 2000`. Kode ini masih lebih cocok sebagai tahap EDA pembuktian, karena ringkasan median di sini masih bisa ikut terpengaruh oleh nilai ekstrem jika jumlah riwayat customer sedikit.

Kesimpulannya, output ini memperkuat bahwa banyak nilai `Total_EMI_per_month` ekstrem muncul sebagai lonjakan tidak wajar pada bulan tertentu. Karena median EMI customer yang sama jauh lebih rendah, nilai ekstrem tersebut layak diperbaiki menggunakan median EMI valid customer yang sama jika tersedia. Jika tidak tersedia referensi valid, nilai ekstrem lebih aman dijadikan missing value.

In [ ]:
# SHOW MONTHLY HISTORY OF SAMPLE CUSTOMERS WITH EXTREME EMI

sample_extreme_customers = (
    extreme_emi_history["Customer_ID"]
    .dropna()
    .drop_duplicates()
    .head(5)
    .tolist()
)

for customer_id in sample_extreme_customers:
    print("\nCustomer_ID:", customer_id)

    customer_history = eda.loc[
        eda["Customer_ID"] == customer_id,
        [
            "ID",
            "Customer_ID",
            "Month",
            "Month_Number",
            "Annual_Income",
            "Monthly_Inhand_Salary",
            "Num_of_Loan",
            "Total_EMI_per_month",
            "Credit_Score"
        ]
    ].sort_values("Month_Number")

    print(customer_history)

Output ini menunjukkan riwayat `Customer_ID` untuk mengecek apakah `Total_EMI_per_month` yang ekstrem konsisten di bulan lain atau hanya muncul sebagai lonjakan pada satu bulan tertentu.

Pada `CUS_ANON_036`, hanya ada satu baris yang terlihat. Customer ini memiliki `Num_of_Loan = 0`, tetapi `Total_EMI_per_month` sebesar 9.505. Karena hanya ada satu riwayat yang tampil, kasus ini belum bisa dipastikan dari pola bulan lain. Namun, karena `Num_of_Loan = 0`, nilai EMI positif tetap perlu dicurigai dan perlu dicek apakah customer ini benar-benar tidak memiliki loan sepanjang data.

Pada `CUS_ANON_039`, pola anomalinya lebih jelas. Customer ini memiliki `Annual_Income` yang sama, yaitu 122.322, dan `Num_of_Loan` yang konsisten, yaitu 2. Pada June, July, dan August, `Total_EMI_per_month` berada di sekitar 1.120,38. Namun, pada February, nilai EMI tiba-tiba melonjak menjadi 38.848. Karena jumlah loan dan annual income tetap sama, lonjakan EMI pada February terlihat tidak konsisten dengan riwayat customer yang sama. Jadi, nilai 38.848 lebih kuat dianggap sebagai outlier pada `Total_EMI_per_month`.

Pada `CUS_ANON_015`, walaupun output terpotong, terlihat pola yang mirip. Customer memiliki nilai pendapatan dan jumlah loan yang sama pada beberapa bulan, tetapi terdapat satu nilai EMI yang sangat tinggi, sedangkan bulan lain memiliki EMI yang jauh lebih rendah. Ini menunjukkan bahwa sebagian EMI ekstrem kemungkinan hanya muncul sebagai lonjakan satu bulan, bukan pola normal customer.

Dengan demikian, output ini memperkuat bukti bahwa banyak nilai ekstrem `Total_EMI_per_month` bukan berasal dari perubahan kondisi customer yang konsisten, melainkan dari lonjakan tidak wajar pada bulan tertentu. Jika customer yang sama memiliki referensi EMI valid pada bulan lain, maka nilai ekstrem dapat diperbaiki menggunakan median EMI valid customer tersebut. Jika hanya ada satu riwayat atau tidak ada referensi valid, nilai ekstrem lebih aman dijadikan missing value daripada dipaksakan diganti.

### Perbaikan Kolom Total_Emi_per_monthh

In [ ]:

pd.options.display.float_format = "{:,.4f}".format

df = df.copy()

high_emi_threshold = 2000

df["Total_EMI_per_month"] = pd.to_numeric(
    df["Total_EMI_per_month"],
    errors="coerce"
)

df["Num_of_Loan"] = pd.to_numeric(
    df["Num_of_Loan"],
    errors="coerce"
)

In [ ]:
# BUILD CUSTOMER HISTORY REFERENCE FOR TOTAL_EMI_PER_MONTH REPAIR

customer_loan_emi_history = (
    df.groupby("Customer_ID")
    .agg(
        Max_Num_of_Loan=("Num_of_Loan", "max"),
        Non_Missing_Type_of_Loan=("Type_of_Loan", lambda x: x.notna().sum()),
        Valid_EMI_Count=("Total_EMI_per_month", lambda x: ((x.notna()) & (x <= high_emi_threshold)).sum()),
        Valid_EMI_Median=("Total_EMI_per_month", lambda x: x[(x.notna()) & (x <= high_emi_threshold)].median()),
        Zero_EMI_Count=("Total_EMI_per_month", lambda x: (x == 0).sum())
    )
    .reset_index()
)

valid_emi_reference = (
    df[
        (df["Total_EMI_per_month"].notna()) &
        (df["Total_EMI_per_month"] <= high_emi_threshold)
    ]
    .groupby("Customer_ID")["Total_EMI_per_month"]
    .median()
)

In [ ]:
# IDENTIFY STRONG ZERO-LOAN EMI ERROR CANDIDATES

zero_loan_positive_emi = df[
    (df["Num_of_Loan"] == 0) &
    (df["Total_EMI_per_month"] > 0)
].copy()

zero_loan_positive_emi["Original_Index"] = zero_loan_positive_emi.index

zero_loan_emi_repair_check = zero_loan_positive_emi.merge(
    customer_loan_emi_history,
    on="Customer_ID",
    how="left"
)

zero_loan_emi_repair_check["Likely_EMI_Error"] = (
    (zero_loan_emi_repair_check["Max_Num_of_Loan"] == 0) &
    (zero_loan_emi_repair_check["Non_Missing_Type_of_Loan"] == 0) &
    (
        (zero_loan_emi_repair_check["Valid_EMI_Median"] == 0) |
        (zero_loan_emi_repair_check["Valid_EMI_Median"].isna())
    )
)

zero_loan_repair_indices = zero_loan_emi_repair_check.loc[
    zero_loan_emi_repair_check["Likely_EMI_Error"],
    "Original_Index"
].tolist()

print("Strong zero-loan EMI error candidates:", len(zero_loan_repair_indices))

In [ ]:
# SAVE AUDIT BEFORE TOTAL_EMI_PER_MONTH REPAIR

total_emi_repair_audit = df[
    (
        df.index.isin(zero_loan_repair_indices)
    ) |
    (
        df["Total_EMI_per_month"] > high_emi_threshold
    )
][
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Type_of_Loan",
        "Total_EMI_per_month",
        "Credit_Score"
    ]
].copy()

total_emi_repair_audit["Suggested_Total_EMI"] = (
    total_emi_repair_audit["Customer_ID"]
    .map(valid_emi_reference)
)

total_emi_repair_audit["Repair_Action"] = np.select(
    [
        total_emi_repair_audit.index.isin(zero_loan_repair_indices),
        (
            (total_emi_repair_audit["Total_EMI_per_month"] > high_emi_threshold) &
            (total_emi_repair_audit["Suggested_Total_EMI"].notna())
        ),
        (
            (total_emi_repair_audit["Total_EMI_per_month"] > high_emi_threshold) &
            (total_emi_repair_audit["Suggested_Total_EMI"].isna())
        )
    ],
    [
        "Set EMI to 0",
        "Replace with customer median valid EMI",
        "Set as missing value"
    ],
    default="No repair"
)

total_emi_repair_audit.sort_values(
    by="Total_EMI_per_month",
    ascending=False
).head(30)

In [ ]:
# REPAIR TOTAL_EMI_PER_MONTH

before_high_emi_count = (
    df["Total_EMI_per_month"] > high_emi_threshold
).sum()

before_missing_emi_count = df["Total_EMI_per_month"].isna().sum()

before_zero_loan_positive_emi_count = (
    (df["Num_of_Loan"] == 0) &
    (df["Total_EMI_per_month"] > 0)
).sum()

# Rule 1:
# If customer consistently has no loan history, positive EMI is likely an EMI error.
df.loc[
    zero_loan_repair_indices,
    "Total_EMI_per_month"
] = 0

# Rule 2:
# Remaining high EMI values are replaced using valid customer median if available.
high_emi_mask_after_zero_rule = (
    df["Total_EMI_per_month"] > high_emi_threshold
)

suggested_emi_after_zero_rule = df.loc[
    high_emi_mask_after_zero_rule,
    "Customer_ID"
].map(valid_emi_reference)

emi_fillable_indices = suggested_emi_after_zero_rule[
    suggested_emi_after_zero_rule.notna()
].index

emi_unresolved_indices = suggested_emi_after_zero_rule[
    suggested_emi_after_zero_rule.isna()
].index

df.loc[
    emi_fillable_indices,
    "Total_EMI_per_month"
] = suggested_emi_after_zero_rule.loc[emi_fillable_indices]

df.loc[
    emi_unresolved_indices,
    "Total_EMI_per_month"
] = np.nan

print("Sebelum repair - Total_EMI_per_month > 2000:", before_high_emi_count)
print("Sebelum repair - Missing Total_EMI_per_month:", before_missing_emi_count)
print("Sebelum repair - Num_of_Loan = 0 tetapi EMI > 0:", before_zero_loan_positive_emi_count)
print()
print("Set EMI to 0:", len(zero_loan_repair_indices))
print("Replace high EMI dengan median customer:", len(emi_fillable_indices))
print("High EMI tanpa referensi valid menjadi missing:", len(emi_unresolved_indices))

Kode ini merupakan proses perbaikan untuk kolom `Total_EMI_per_month` berdasarkan hasil relationship-based EDA yang sudah dilakukan sebelumnya. Perbaikan dilakukan dengan dua aturan utama, yaitu memperbaiki EMI pada customer yang konsisten tidak memiliki loan, dan memperbaiki nilai EMI ekstrem menggunakan median EMI valid dari customer yang sama.

Bagian pertama kode menyimpan kondisi sebelum perbaikan. Variabel `before_high_emi_count` menghitung jumlah data dengan `Total_EMI_per_month > 2000`. Variabel `before_missing_emi_count` menghitung jumlah missing value pada `Total_EMI_per_month`. Variabel `before_zero_loan_positive_emi_count` menghitung jumlah baris yang memiliki `Num_of_Loan = 0` tetapi `Total_EMI_per_month > 0`.

Hasil sebelum repair menunjukkan bahwa terdapat 812 baris dengan `Total_EMI_per_month > 2000`. Ini sesuai dengan hasil EDA sebelumnya, bahwa sekitar 3,25% data memiliki EMI yang sangat tinggi dan jauh dari pola mayoritas data. Missing value pada `Total_EMI_per_month` sebelum perbaikan adalah 0, artinya kolom ini awalnya lengkap. Selain itu, terdapat 211 baris dengan `Num_of_Loan = 0` tetapi EMI masih positif. Kondisi ini mencurigakan karena jika customer tidak memiliki loan, maka EMI seharusnya 0.

Rule pertama digunakan untuk kasus yang paling kuat buktinya, yaitu customer yang konsisten tidak memiliki loan. Pada bagian ini, kode menggunakan `zero_loan_repair_indices`, yaitu indeks baris yang sebelumnya sudah terbukti memiliki pola: `Num_of_Loan = 0`, `Type_of_Loan` kosong, riwayat customer tidak menunjukkan adanya loan, dan EMI valid customer bernilai 0 atau tidak tersedia. Untuk kasus seperti ini, nilai `Total_EMI_per_month` diperbaiki menjadi 0.

Output menunjukkan bahwa terdapat 120 baris yang diperbaiki dengan cara `Set EMI to 0`. Artinya, ada 120 baris yang bukti historisnya cukup kuat untuk disimpulkan bahwa customer memang tidak memiliki loan, sehingga EMI positif pada baris tersebut lebih mungkin merupakan kesalahan pencatatan.

Rule kedua digunakan untuk sisa nilai EMI yang masih ekstrem setelah rule pertama dijalankan. Kode membuat `high_emi_mask_after_zero_rule`, yaitu data yang masih memiliki `Total_EMI_per_month > 2000` setelah sebagian kasus zero-loan diperbaiki. Untuk baris-baris ini, kode mencari referensi EMI valid dari customer yang sama menggunakan `valid_emi_reference`.

Jika customer memiliki referensi EMI valid dari bulan lain, maka nilai ekstrem diganti dengan median EMI valid customer tersebut. Output menunjukkan bahwa terdapat 609 baris yang diperbaiki dengan cara `Replace high EMI dengan median customer`. Ini berarti sebagian besar EMI ekstrem masih bisa diperbaiki tanpa menebak dari customer lain, karena tersedia referensi historis dari customer yang sama.

Jika customer tidak memiliki referensi EMI valid, maka nilai EMI ekstrem tidak dipaksakan untuk diganti. Kode mengubahnya menjadi missing value (`NaN`). Output menunjukkan bahwa terdapat 89 baris yang menjadi missing karena tidak memiliki referensi valid. Ini lebih aman daripada mengisi dengan median global atau nilai customer lain, karena hal tersebut bisa memasukkan informasi yang kurang sesuai.

Secara total, proses repair ini cukup konservatif. Kode tidak menghapus baris dan tidak menghapus kolom `Total_EMI_per_month`. Nilai hanya diperbaiki ketika ada bukti yang cukup kuat dari riwayat customer yang sama. Untuk kasus yang sangat jelas tidak memiliki loan, EMI diubah menjadi 0. Untuk EMI ekstrem yang memiliki referensi valid, nilai diganti dengan median customer. Untuk kasus tanpa referensi, nilai dijadikan missing value agar nanti ditangani oleh preprocessing pipeline.


In [ ]:
# CHECK AFTER TOTAL_EMI_PER_MONTH REPAIR

print("Setelah repair:")
print("Total_EMI_per_month > 2000:", (df["Total_EMI_per_month"] > high_emi_threshold).sum())
print("Missing Total_EMI_per_month:", df["Total_EMI_per_month"].isna().sum())
print(
    "Num_of_Loan = 0 tetapi Total_EMI_per_month > 0:",
    (
        (df["Num_of_Loan"] == 0) &
        (df["Total_EMI_per_month"] > 0)
    ).sum()
)

df["Total_EMI_per_month"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.97,
        0.98,
        0.99
    ]
)

Output ini menunjukkan hasil setelah proses repair `Total_EMI_per_month`.

Berdasarkan statistik setelah perbaikan, jumlah data valid pada `Total_EMI_per_month` menjadi 24.911. Karena total data awal adalah 25.000, berarti terdapat 89 missing value. Ini sesuai dengan hasil repair sebelumnya, yaitu 89 nilai EMI ekstrem tidak memiliki referensi valid dari customer yang sama, sehingga lebih aman dijadikan missing value daripada dipaksakan diganti.

Nilai maksimum `Total_EMI_per_month` setelah repair adalah 1.950. Ini berarti sudah tidak ada lagi nilai `Total_EMI_per_month > 2000`. Sebelumnya terdapat 812 baris dengan EMI di atas 2000, bahkan maksimum mencapai lebih dari 82.000. Setelah repair, semua nilai ekstrem tersebut berhasil ditangani.

Distribusi `Total_EMI_per_month` juga menjadi jauh lebih stabil. Median sekarang sebesar 65,44, P75 sebesar 145,49, P90 sebesar 249,10, P95 sebesar 324,47, dan P99 sebesar 577,28. Nilai-nilai ini jauh lebih masuk akal dibandingkan sebelum repair, ketika P97 ke atas melonjak sampai puluhan ribu.

Mean setelah repair adalah 106,34 dan standar deviasi 129,73. Ini menunjukkan bahwa rata-rata tidak lagi tertarik oleh outlier ekstrem. Sebelumnya, mean dan standar deviasi sangat besar karena adanya EMI puluhan ribu.

Jadi, proses repair `Total_EMI_per_month` berhasil. Nilai ekstrem di atas 2000 sudah hilang, distribusi menjadi lebih wajar, dan hanya 89 nilai yang dijadikan missing karena tidak memiliki referensi valid dari customer yang sama. Missing value ini nanti akan ditangani pada tahap preprocessing pipeline model.

In [ ]:
# RECHECK EMI RELATIONSHIP AFTER REPAIR

emi_income_after_repair = df.dropna(
    subset=[
        "Total_EMI_per_month",
        "Monthly_Inhand_Salary",
        "Annual_Income"
    ]
).copy()

emi_income_after_repair["EMI_to_Monthly_Salary_Ratio"] = (
    emi_income_after_repair["Total_EMI_per_month"] /
    emi_income_after_repair["Monthly_Inhand_Salary"]
)

emi_income_after_repair["EMI_to_Annual_Income_Ratio"] = (
    emi_income_after_repair["Total_EMI_per_month"] * 12 /
    emi_income_after_repair["Annual_Income"]
)

emi_income_after_repair = emi_income_after_repair.replace(
    [np.inf, -np.inf],
    np.nan
)

print(
    "Jumlah EMI lebih besar dari gaji bulanan setelah repair:",
    (
        emi_income_after_repair["Total_EMI_per_month"] >
        emi_income_after_repair["Monthly_Inhand_Salary"]
    ).sum()
)

emi_income_after_repair[
    [
        "Total_EMI_per_month",
        "Monthly_Inhand_Salary",
        "Annual_Income",
        "EMI_to_Monthly_Salary_Ratio",
        "EMI_to_Annual_Income_Ratio"
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.97,
        0.98,
        0.99
    ]
)

Output ini menunjukkan hasil pengecekan ulang hubungan antara `Total_EMI_per_month`, `Monthly_Inhand_Salary`, dan `Annual_Income` setelah proses repair EMI dilakukan.

Hasil paling penting adalah jumlah EMI yang lebih besar dari gaji bulanan setelah repair hanya tersisa 5 kasus. Sebelumnya, jumlahnya mencapai 687 kasus. Artinya, proses repair berhasil mengurangi ketidaksesuaian ekstrem antara EMI dan gaji bulanan secara signifikan.

Distribusi `Total_EMI_per_month` juga sudah jauh lebih stabil. Median EMI sekarang sekitar 65,64, P75 sekitar 144,78, P90 sekitar 248,13, P95 sekitar 323,27, dan P99 sekitar 576,68. Nilai maksimum juga hanya 1.950, sehingga sudah tidak ada lagi EMI puluhan ribu seperti sebelum perbaikan.

Rasio `EMI_to_Monthly_Salary_Ratio` juga menjadi jauh lebih masuk akal. Median rasionya sekitar 0,0244, artinya pada mayoritas data, EMI hanya sekitar 2,44% dari gaji bulanan. Pada P95, rasionya sekitar 0,0733 atau 7,33%, dan pada P99 sekitar 0,1302 atau 13,02%. Ini menunjukkan bahwa sebagian besar EMI sudah berada dalam proporsi yang wajar terhadap pendapatan bulanan.

Nilai maksimum `EMI_to_Monthly_Salary_Ratio` masih mencapai sekitar 1,9792. Artinya, masih ada sedikit kasus di mana EMI hampir dua kali lebih besar dari gaji bulanan. Namun jumlahnya hanya 5 kasus, sehingga tidak lagi menunjukkan masalah sistematis seperti sebelum repair.

Secara keseluruhan, output ini menunjukkan bahwa proses repair `Total_EMI_per_month` berhasil. Outlier ekstrem sudah hilang, hubungan EMI dengan income menjadi lebih masuk akal, dan hanya tersisa sedikit kasus yang dapat diperiksa secara manual atau dibiarkan karena jumlahnya sangat kecil.

5 kasus ini jangan langsung diperbaiki lagi. Karena setelah repair, rasionya sudah maksimal sekitar 1,98, bukan lagi puluhan/ratusan kali gaji seperti sebelumnya. Ini bisa saja memang customer yang sangat terbebani cicilan, bukan pasti error.

Jadi langkah terbaik: tampilkan 5 kasusnya dulu, lalu cek riwayat customer-nya. Kalau ternyata EMI itu konsisten dengan bulan lain, biarkan. Kalau ternyata hanya lonjakan aneh satu bulan, baru repair pakai median customer.

In [ ]:
# DISPLAY REMAINING CASES WHERE EMI IS GREATER THAN MONTHLY SALARY AFTER REPAIR

remaining_emi_greater_salary = emi_income_after_repair[
    emi_income_after_repair["Total_EMI_per_month"] >
    emi_income_after_repair["Monthly_Inhand_Salary"]
].copy()

remaining_emi_greater_salary[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Type_of_Loan",
        "Total_EMI_per_month",
        "EMI_to_Monthly_Salary_Ratio",
        "EMI_to_Annual_Income_Ratio",
        "Credit_Score"
    ]
].sort_values(
    by="EMI_to_Monthly_Salary_Ratio",
    ascending=False
)

Output ini menampilkan 5 kasus yang masih memiliki `Total_EMI_per_month` lebih besar daripada `Monthly_Inhand_Salary` setelah proses repair dilakukan.

Berbeda dengan kasus sebelum repair, 5 kasus ini tidak terlihat sebagai outlier ekstrem. Nilai `Total_EMI_per_month` berada pada kisaran 1.325 sampai 1.745, sehingga masih berada di bawah threshold 2.000 yang sebelumnya digunakan sebagai batas investigasi EMI ekstrem. Jadi, secara distribusi, nilai ini tidak lagi termasuk nilai EMI yang sangat ekstrem.

Selain itu, kelima customer ini memang memiliki loan. Hal ini terlihat dari `Num_of_Loan` yang lebih dari 0 dan `Type_of_Loan` yang juga terisi. Misalnya, `CUS_ANON_016` memiliki 3 loan, `CUS_ANON_006` memiliki 5 loan, `CUS_ANON_027` memiliki 9 loan, `CUS_ANON_002` memiliki 4 loan, dan `CUS_0x4be` memiliki 7 loan. Karena jumlah loan dan tipe loan tersedia, EMI yang lebih besar dari gaji bulanan masih mungkin terjadi pada customer dengan beban cicilan tinggi.

Rasio `EMI_to_Monthly_Salary_Ratio` juga sudah jauh lebih masuk akal dibandingkan sebelum repair. Rasio tertinggi hanya sekitar 1,98, artinya EMI hampir dua kali gaji bulanan. Nilai ini memang menunjukkan beban cicilan berat, tetapi tidak seekstrem kondisi sebelumnya yang mencapai puluhan sampai ratusan kali gaji bulanan.

Karena kelima kasus ini memiliki informasi loan yang jelas, nilai EMI masih di bawah 2.000, dan jumlah kasusnya sangat kecil, maka tidak perlu dilakukan repair tambahan. Jika nilai ini dipaksa diperbaiki lagi, proses cleaning bisa menjadi terlalu agresif dan berisiko menghapus variasi data yang sebenarnya valid.

Dengan demikian, 5 kasus sisa ini sebaiknya dipertahankan. Kasus ini dapat dianggap sebagai customer dengan beban EMI yang sangat tinggi dibandingkan gaji bulanan, bukan sebagai data error yang kuat. Proses repair `Total_EMI_per_month` dapat dianggap selesai.

In [ ]:
# CHECK CUSTOMER HISTORY FOR REMAINING EMI GREATER THAN SALARY CASES

remaining_customer_ids = (
    remaining_emi_greater_salary["Customer_ID"]
    .drop_duplicates()
    .tolist()
)

remaining_history_columns = [
    "ID",
    "Customer_ID",
    "Month",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Num_of_Loan",
    "Type_of_Loan",
    "Total_EMI_per_month",
    "Credit_Score"
]

if "Month_Number" in df.columns:
    remaining_history_columns.insert(3, "Month_Number")
    sort_columns = ["Customer_ID", "Month_Number"]
else:
    sort_columns = ["Customer_ID", "Month"]

remaining_emi_customer_history = df[
    df["Customer_ID"].isin(remaining_customer_ids)
][remaining_history_columns].copy()

remaining_emi_customer_history.sort_values(
    by=sort_columns
)

Setelah proses repair utama pada kolom `Total_EMI_per_month`, dilakukan pengecekan ulang terhadap hubungan antara EMI dan gaji bulanan. Hasilnya menunjukkan bahwa masih terdapat 5 kasus di mana `Total_EMI_per_month` lebih besar daripada `Monthly_Inhand_Salary`.

Pada tahap ini, 5 kasus tersebut tidak langsung diperbaiki karena nilainya sudah tidak lagi termasuk outlier ekstrem seperti sebelum repair. Sebelum perbaikan, terdapat banyak nilai EMI yang mencapai puluhan ribu hingga lebih dari 80.000, bahkan rasio EMI terhadap gaji bulanan bisa mencapai puluhan sampai ratusan kali. Setelah repair utama, nilai EMI maksimum sudah turun menjadi 1.950 dan jumlah kasus EMI lebih besar dari gaji bulanan hanya tersisa 5 kasus.

Kelima kasus tersebut kemudian dicek menggunakan riwayat `Customer_ID` yang sama. Tujuannya adalah memastikan apakah nilai EMI yang masih lebih besar dari gaji bulanan merupakan pola yang konsisten pada customer tersebut, atau hanya lonjakan tidak wajar pada satu bulan tertentu.

Dari hasil riwayat customer, terlihat bahwa beberapa kasus masih menunjukkan pola lonjakan EMI yang tidak konsisten. Contohnya, `CUS_ANON_002` memiliki `Num_of_Loan = 4`, `Annual_Income = 10.542,0450`, dan `Monthly_Inhand_Salary = 1.082,5038`. Pada bulan June, EMI customer ini hanya sebesar 25,9487. Namun, pada bulan March, EMI tiba-tiba menjadi 1.627. Karena income, gaji bulanan, dan jumlah loan sama, nilai EMI 1.627 terlihat sebagai lonjakan yang tidak konsisten dengan riwayat customer tersebut.

Pola yang sama terlihat pada `CUS_ANON_006`. Customer ini memiliki `Annual_Income = 10.277,1900`, `Monthly_Inhand_Salary = 920,4325`, dan `Num_of_Loan = 5`. Pada beberapa bulan seperti January, February, May, dan June, nilai EMI konsisten sekitar 25,0298. Namun, pada August, EMI menjadi 1.689. Karena customer yang sama memiliki nilai EMI valid yang sangat konsisten di bulan lain, nilai 1.689 lebih kuat dianggap sebagai lonjakan tidak wajar.

Kasus `CUS_ANON_027` juga menunjukkan indikasi serupa. Customer ini memiliki `Num_of_Loan = 9` dan `Type_of_Loan` terisi. Pada beberapa bulan, EMI berada di sekitar 100,8532 sampai 172,6673. Namun, pada June, EMI naik menjadi 1.614. Walaupun customer ini memiliki banyak loan, nilai 1.614 tetap tampak jauh lebih tinggi dibandingkan pola EMI customer yang sama pada bulan lain.

Sementara itu, beberapa kasus lain seperti `CUS_0x4be` dan `CUS_ANON_016` tidak memiliki riwayat pembanding yang cukup pada output yang ditampilkan. Karena kedua customer tersebut memiliki `Num_of_Loan > 0`, `Type_of_Loan` terisi, dan nilai EMI masih berada di bawah threshold 2.000, maka nilai tersebut tidak langsung dianggap sebagai error. Mengubah nilai tersebut tanpa referensi historis yang kuat dapat menyebabkan over-cleaning.

Dengan demikian, keputusan sebelum final micro-repair adalah tidak memperbaiki semua 5 kasus secara otomatis. Perbaikan hanya dilakukan pada kasus yang memiliki bukti historis kuat dari `Customer_ID` yang sama, yaitu ketika nilai EMI pada satu bulan jauh lebih tinggi dibandingkan median atau pola EMI valid pada bulan lain. Jika tidak tersedia referensi valid dari customer yang sama, nilai EMI akan dibiarkan agar tidak mengubah data yang masih mungkin valid.

Pendekatan ini tetap konsisten dengan prinsip conservative cleaning. Data hanya diperbaiki jika terdapat bukti kuat dari riwayat customer, bukan hanya karena EMI lebih besar daripada gaji bulanan. Hal ini penting karena dalam data kredit, EMI yang lebih besar dari gaji bulanan masih mungkin terjadi pada sebagian kecil customer dengan beban cicilan sangat tinggi. Namun, jika nilai EMI tersebut terbukti sebagai lonjakan tidak wajar dibandingkan riwayat customer yang sama, maka nilai tersebut dapat diperbaiki menggunakan median EMI valid dari customer tersebut.

In [ ]:
# DISPLAY REMAINING MOST INCONSISTENT EMI CASES AFTER REPAIR

emi_income_after_repair.sort_values(
    by="EMI_to_Monthly_Salary_Ratio",
    ascending=False
)[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Num_of_Loan",
        "Type_of_Loan",
        "Total_EMI_per_month",
        "EMI_to_Monthly_Salary_Ratio",
        "EMI_to_Annual_Income_Ratio",
        "Credit_Score"
    ]
].head(30)

Output ini menampilkan 30 kasus dengan rasio `EMI_to_Monthly_Salary_Ratio` tertinggi setelah proses repair utama pada `Total_EMI_per_month`.

Setelah repair, hanya terdapat 5 kasus di mana `Total_EMI_per_month` masih lebih besar daripada `Monthly_Inhand_Salary`, yaitu rasio di atas 1. Kasus tersebut adalah `CUS_ANON_016`, `CUS_ANON_006`, `CUS_ANON_027`, `CUS_ANON_002`, dan `CUS_0x4be`. Rasio tertinggi adalah sekitar 1,9792, artinya EMI hampir 1,98 kali lebih besar dari gaji bulanan.

Meskipun EMI pada 5 kasus tersebut lebih besar daripada gaji bulanan, nilainya sudah tidak termasuk outlier ekstrem seperti sebelum repair. Nilai `Total_EMI_per_month` pada 5 kasus ini berada pada rentang 1.325 sampai 1.745, masih di bawah threshold 2.000 yang digunakan sebagai batas investigasi EMI ekstrem. Sebelumnya, terdapat EMI puluhan ribu hingga lebih dari 80.000, sehingga kondisi setelah repair sudah jauh lebih stabil.

Selain itu, kelima customer tersebut memiliki `Num_of_Loan > 0` dan `Type_of_Loan` terisi. Artinya, secara informasi fitur, mereka memang memiliki pinjaman. Hal ini berbeda dengan kasus sebelumnya yang memiliki `Num_of_Loan = 0`, `Type_of_Loan = NaN`, tetapi EMI puluhan ribu. Pada 5 kasus sisa ini, keberadaan EMI masih dapat dijelaskan oleh adanya loan.

Mulai dari baris keenam ke bawah, rasio `EMI_to_Monthly_Salary_Ratio` sudah berada di bawah 1. Artinya, EMI tidak lagi melebihi gaji bulanan. Contohnya, `CUS_ANON_021` memiliki rasio 0,9953, sehingga EMI hampir sama dengan gaji bulanan tetapi masih sedikit lebih rendah. Baris-baris berikutnya memiliki rasio yang semakin kecil dan lebih wajar.

Berdasarkan output ini, tidak dilakukan repair tambahan atau micro-repair. Alasannya, sisa kasus sudah tidak ekstrem, jumlahnya sangat kecil, memiliki informasi loan yang jelas, dan masih berada di bawah threshold EMI ekstrem. Jika nilai-nilai ini tetap dipaksa diperbaiki, proses cleaning menjadi terlalu agresif dan berisiko mengubah data yang masih mungkin valid.

Dengan demikian, proses repair `Total_EMI_per_month` dianggap selesai pada tahap repair utama. Sisa 5 kasus dicatat sebagai customer dengan beban EMI tinggi dibandingkan gaji bulanan, bukan sebagai data error yang kuat.

### CHECK SSN CONSISTENCY WITH CUSTOMER ID

In [ ]:
# CHECK SSN CONSISTENCY WITH CUSTOMER ID

valid_ssn = df[
    df["SSN"].notna() &
    (df["SSN"] != "#F%$D@*&8")
].copy()

ssn_customer_count = (
    valid_ssn
    .groupby("SSN")["Customer_ID"]
    .nunique()
)

invalid_ssn_mapping = ssn_customer_count[
    ssn_customer_count > 1
]

print(
    "Jumlah SSN yang digunakan oleh lebih dari satu Customer_ID:",
    len(invalid_ssn_mapping)
)

display(invalid_ssn_mapping)

amaann

### Hubungan Type_of_LoandenganNum_of_Loan

In [ ]:
# PARSE TYPE OF LOAN COUNT

def count_loan_type(value):
    if pd.isna(value):
        return np.nan

    value = str(value)
    value = value.replace(", and ", ", ")
    value = value.replace(" and ", ", ")

    loan_list = [
        loan.strip()
        for loan in value.split(",")
        if loan.strip() != ""
    ]

    return len(loan_list)


loan_relation_data = eda.copy()

loan_relation_data["Parsed_Loan_Count"] = (
    loan_relation_data["Type_of_Loan"]
    .apply(count_loan_type)
)

loan_relation_data["Loan_Count_Difference"] = (
    loan_relation_data["Parsed_Loan_Count"]
    - loan_relation_data["Num_of_Loan"]
)

loan_count_relation = loan_relation_data[
    [
        "Num_of_Loan",
        "Parsed_Loan_Count",
        "Loan_Count_Difference"
    ]
].dropna().describe()

loan_count_relation

Output ini menunjukkan hasil validasi antara `Num_of_Loan` dan jumlah loan yang berhasil dihitung dari kolom `Type_of_Loan`.

`Parsed_Loan_Count` adalah jumlah loan yang dihitung dari teks pada `Type_of_Loan`. Misalnya, jika `Type_of_Loan` berisi beberapa jenis loan yang dipisahkan koma, maka jumlah item loan dihitung dan dibandingkan dengan nilai pada `Num_of_Loan`.

Kolom `Loan_Count_Difference` menunjukkan selisih antara `Num_of_Loan` dan `Parsed_Loan_Count`. Pada output ini, seluruh nilai `Loan_Count_Difference` adalah 0. Hal ini terlihat dari mean, min, 25%, median, 75%, dan max yang semuanya bernilai 0.

Artinya, untuk 21.992 baris yang memiliki data loan valid, jumlah loan pada `Num_of_Loan` sudah sesuai dengan jumlah loan yang tertulis di `Type_of_Loan`. Tidak ada perbedaan antara jumlah loan numerik dan jumlah loan hasil parsing dari teks.

Hasil ini penting karena menunjukkan bahwa kolom `Num_of_Loan` dan `Type_of_Loan` sudah konsisten satu sama lain. Jadi, tidak diperlukan perbaikan tambahan pada hubungan antara kedua kolom ini. Kolom `Type_of_Loan` nantinya tidak perlu digunakan dalam bentuk teks panjang secara langsung, tetapi bisa diubah menjadi fitur turunan seperti jumlah atau jenis loan tertentu jika dibutuhkan untuk modeling.

In [ ]:
# CHECK LOAN COUNT MISMATCH

loan_mismatch_data = loan_relation_data.dropna(
    subset=[
        "Num_of_Loan",
        "Parsed_Loan_Count",
        "Loan_Count_Difference"
    ]
).copy()

loan_mismatch = loan_mismatch_data[
    loan_mismatch_data["Loan_Count_Difference"] != 0
].copy()

print("Jumlah data valid untuk cek mismatch:", len(loan_mismatch_data))
print("Jumlah mismatch Num_of_Loan dan Type_of_Loan:", len(loan_mismatch))
print(
    "Persentase mismatch:",
    round(len(loan_mismatch) / len(loan_mismatch_data) * 100, 2),
    "%"
)

loan_mismatch[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Num_of_Loan",
        "Type_of_Loan",
        "Parsed_Loan_Count",
        "Loan_Count_Difference",
        "Credit_Score"
    ]
].head(30)

Keduanya berhubungan baik

In [ ]:
# CHECK TYPE OF LOAN MISSING BY NUMBER OF LOANS

loan_missing_data = loan_relation_data.copy()

loan_missing_data["Num_of_Loan_Label"] = (
    loan_missing_data["Num_of_Loan"]
    .astype("string")
    .fillna("Missing")
)

loan_missing_by_count = pd.crosstab(
    loan_missing_data["Type_of_Loan"].isna(),
    loan_missing_data["Num_of_Loan_Label"]
)

loan_missing_by_count

Output ini digunakan untuk mengecek hubungan antara `Num_of_Loan` dan missing value pada `Type_of_Loan`.

Kode membuat kolom sementara `Num_of_Loan_Label` dari `Num_of_Loan`, lalu membuat crosstab antara status missing pada `Type_of_Loan` dan nilai `Num_of_Loan`. Nilai `False` berarti `Type_of_Loan` tidak missing, sedangkan `True` berarti `Type_of_Loan` missing.

Hasilnya menunjukkan pola yang sangat jelas. Untuk `Num_of_Loan` dari 1 sampai 9, seluruh data memiliki `Type_of_Loan` yang tidak missing. Contohnya, untuk `Num_of_Loan = 1` terdapat 2.796 baris dan semuanya berada pada baris `False`, artinya `Type_of_Loan` tersedia. Pola yang sama juga terjadi untuk `Num_of_Loan = 2` sampai `Num_of_Loan = 9`.

Sebaliknya, untuk `Num_of_Loan = 0`, terdapat 2.826 baris dengan `Type_of_Loan` missing. Ini masuk akal karena jika customer tidak memiliki loan, maka tidak ada jenis loan yang perlu dicatat. Jadi, missing pada `Type_of_Loan` untuk `Num_of_Loan = 0` bukan error, melainkan missing yang logis secara bisnis.

Namun, terdapat 26 baris dengan `Num_of_Loan` missing dan `Type_of_Loan` juga missing. Ini berarti baris tersebut belum bisa dipastikan apakah customer benar-benar tidak memiliki loan atau datanya memang belum lengkap. Kasus ini sebaiknya tetap dibiarkan sebagai missing dan nanti ditangani pada preprocessing pipeline.

Kesimpulannya, hubungan antara `Num_of_Loan` dan `Type_of_Loan` sudah konsisten. Jika `Num_of_Loan > 0`, maka `Type_of_Loan` tersedia. Jika `Num_of_Loan = 0`, maka `Type_of_Loan` missing, dan ini wajar. Oleh karena itu, tidak perlu memperbaiki `Type_of_Loan` yang missing pada customer dengan `Num_of_Loan = 0`.

### Hubungan Outstanding_Debt dengan pendapatan

In [ ]:
# RELATIONSHIP BETWEEN OUTSTANDING DEBT AND INCOME

debt_income_data = eda.dropna(
    subset=[
        "Outstanding_Debt",
        "Annual_Income",
        "Monthly_Inhand_Salary"
    ]
).copy()

debt_income_data["Debt_to_Annual_Income_Ratio"] = (
    debt_income_data["Outstanding_Debt"]
    / debt_income_data["Annual_Income"]
)

debt_income_data["Debt_to_Monthly_Salary_Ratio"] = (
    debt_income_data["Outstanding_Debt"]
    / debt_income_data["Monthly_Inhand_Salary"]
)

debt_income_data = debt_income_data.replace(
    [np.inf, -np.inf],
    np.nan
)

debt_income_clean = debt_income_data.dropna(
    subset=[
        "Debt_to_Annual_Income_Ratio",
        "Debt_to_Monthly_Salary_Ratio"
    ]
).copy()

debt_income_summary = debt_income_clean[
    [
        "Outstanding_Debt",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Debt_to_Annual_Income_Ratio",
        "Debt_to_Monthly_Salary_Ratio"
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

debt_income_summary

Output ini menunjukkan hubungan antara `Outstanding_Debt`, `Annual_Income`, dan `Monthly_Inhand_Salary`.

Secara umum, nilai `Outstanding_Debt` terlihat masih berada dalam rentang yang cukup stabil. Nilai minimum adalah 0,34, median sekitar 1.164,47, P75 sekitar 1.961,40, P95 sekitar 4.103,79, P99 sekitar 4.811,78, dan maksimum 4.997,10. Tidak terlihat lonjakan ekstrem seperti pada `Annual_Income` atau `Total_EMI_per_month` sebelumnya.

Kolom `Debt_to_Annual_Income_Ratio` menunjukkan perbandingan antara total debt dan annual income. Median rasionya sekitar 0,0278 atau 2,78% dari annual income. P95 sekitar 24,72%, P99 sekitar 46,19%, dan maksimum sekitar 68,33%. Secara umum, rasio ini masih cukup masuk akal karena outstanding debt memang bisa menjadi sebagian dari pendapatan tahunan customer.

Kolom `Debt_to_Monthly_Salary_Ratio` menunjukkan perbandingan antara outstanding debt dan gaji bulanan. Median rasionya sekitar 0,3415, artinya pada mayoritas data, outstanding debt sekitar 34% dari gaji bulanan. Namun, pada P90 rasio naik menjadi sekitar 2,0171, P95 menjadi 3,0317, P99 menjadi 5,6378, dan maksimum 12,7624. Ini berarti pada sebagian kecil customer, outstanding debt bisa beberapa kali lebih besar daripada gaji bulanan. Kondisi ini tidak otomatis error, karena outstanding debt adalah total sisa utang, bukan cicilan bulanan.

Namun, ada catatan penting: output ini masih menunjukkan `Annual_Income` maksimum sebesar 24.160.009 dan mean sebesar 161.988,26. Padahal sebelumnya `Annual_Income` sudah diperbaiki sehingga maksimum seharusnya sekitar 179.987. Artinya, kemungkinan analisis ini masih memakai dataframe lama sebelum repair `Annual_Income`, misalnya `eda`, bukan `df` yang sudah dibersihkan.

Jadi, kesimpulan sementara dari output ini adalah `Outstanding_Debt` tidak menunjukkan outlier ekstrem yang perlu langsung diperbaiki. Rasio terhadap monthly salary memang memiliki beberapa nilai tinggi, tetapi masih bisa masuk akal karena debt adalah total utang, bukan EMI bulanan. Namun, sebelum mengambil kesimpulan final, analisis ini perlu dijalankan ulang menggunakan `df` yang sudah melewati repair `Annual_Income`.

In [ ]:
# CHECK HIGH DEBT TO INCOME

high_debt_income = debt_income_clean[
    debt_income_clean["Debt_to_Annual_Income_Ratio"] > 1
].copy()

print("Jumlah data valid debt-income:", len(debt_income_clean))
print("Jumlah Outstanding_Debt lebih besar dari Annual_Income:", len(high_debt_income))
print(
    "Persentase Outstanding_Debt lebih besar dari Annual_Income:",
    round(len(high_debt_income) / len(debt_income_clean) * 100, 2),
    "%"
)

high_debt_income[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Outstanding_Debt",
        "Debt_to_Annual_Income_Ratio",
        "Credit_Mix",
        "Credit_Score"
    ]
].sort_values(
    by="Debt_to_Annual_Income_Ratio",
    ascending=False
).head(30)

Aman

In [ ]:
# VISUALIZE OUTSTANDING DEBT VS ANNUAL INCOME

plt.figure(figsize=(7, 5))
plt.scatter(
    debt_income_clean["Annual_Income"],
    debt_income_clean["Outstanding_Debt"],
    alpha=0.3
)
plt.title("Outstanding Debt vs Annual Income")
plt.xlabel("Annual Income")
plt.ylabel("Outstanding Debt")
plt.show()

Grafik ini menunjukkan hubungan antara `Outstanding_Debt` dan `Annual_Income`.

Dari grafik, sebagian besar titik menumpuk di bagian kiri. Ini berarti mayoritas customer memiliki `Annual_Income` pada rentang rendah sampai normal, sedangkan `Outstanding_Debt` tersebar dari nilai kecil sampai sekitar 5.000. Pola ini menunjukkan bahwa `Outstanding_Debt` tidak naik secara jelas mengikuti `Annual_Income`.

Namun, grafik ini juga memperlihatkan adanya beberapa titik `Annual_Income` yang sangat jauh di kanan, bahkan sampai sekitar 20 juta lebih. Ini tidak sesuai dengan hasil cleaning sebelumnya, karena `Annual_Income` seharusnya sudah diperbaiki dan maksimum setelah repair sekitar 179.987. Jadi, kemungkinan besar grafik ini masih dibuat dari dataframe lama, misalnya `eda`, bukan dari `df` yang sudah dibersihkan.

Karena itu, grafik ini belum bisa dipakai sebagai kesimpulan final hubungan `Outstanding_Debt` dan `Annual_Income`. Grafik ini justru menunjukkan bahwa data yang digunakan masih mengandung outlier lama pada `Annual_Income`.

Kesimpulannya, `Outstanding_Debt` sendiri tidak terlihat memiliki outlier ekstrem karena nilainya masih berada di bawah sekitar 5.000. Tetapi visualisasi ini perlu dibuat ulang menggunakan `df` yang sudah diperbaiki agar hubungan dengan `Annual_Income` lebih akurat.

In [ ]:
# VISUALIZE DEBT RATIO BY CREDIT SCORE

credit_order = [
    "Poor",
    "Standard",
    "Good"
]

debt_ratio_plot_data = debt_income_clean.dropna(
    subset=[
        "Credit_Score",
        "Debt_to_Annual_Income_Ratio"
    ]
).copy()

debt_ratio_data = [
    debt_ratio_plot_data.loc[
        debt_ratio_plot_data["Credit_Score"] == score,
        "Debt_to_Annual_Income_Ratio"
    ]
    for score in credit_order
]

plt.figure(figsize=(7, 5))
plt.boxplot(
    debt_ratio_data,
    labels=credit_order,
    showfliers=False
)
plt.title("Debt to Annual Income Ratio by Credit Score")
plt.xlabel("Credit Score")
plt.ylabel("Debt to Annual Income Ratio")
plt.show()

Grafik boxplot ini menunjukkan perbandingan `Debt_to_Annual_Income_Ratio` berdasarkan kelas `Credit_Score`, yaitu Poor, Standard, dan Good.

`Debt_to_Annual_Income_Ratio` menggambarkan seberapa besar `Outstanding_Debt` dibandingkan dengan `Annual_Income`. Semakin besar rasionya, semakin besar beban utang customer dibandingkan pendapatan tahunannya.

Dari grafik terlihat bahwa kelompok `Poor` memiliki rasio debt terhadap annual income paling tinggi. Median pada kelas Poor berada lebih tinggi dibandingkan Standard dan Good. Selain itu, rentang datanya juga paling lebar, dengan nilai maksimum mencapai sekitar 0,30. Ini menunjukkan bahwa customer dengan credit score Poor cenderung memiliki beban utang yang lebih besar relatif terhadap pendapatannya.

Kelompok `Standard` berada di tengah. Median rasionya lebih rendah daripada Poor, tetapi lebih tinggi daripada Good. Rentangnya juga tidak selebar Poor. Ini menunjukkan bahwa customer Standard memiliki beban utang yang sedang, tidak seberat kelompok Poor tetapi juga tidak serendah kelompok Good.

Kelompok `Good` memiliki rasio paling rendah. Median dan sebaran datanya kecil, serta nilai maksimumnya lebih rendah dibandingkan Poor dan Standard. Hal ini menunjukkan bahwa customer dengan credit score Good cenderung memiliki outstanding debt yang lebih rendah dibandingkan annual income mereka.

Secara keseluruhan, grafik ini menunjukkan pola yang logis: semakin tinggi rasio utang terhadap pendapatan tahunan, semakin buruk kecenderungan credit score. Dengan kata lain, `Debt_to_Annual_Income_Ratio` terlihat sebagai fitur yang relevan untuk membedakan kelas `Credit_Score`, karena customer dengan rasio utang lebih tinggi lebih sering berada pada kelas Poor, sedangkan customer dengan rasio utang lebih rendah lebih sering berada pada kelas Good.

### Hubungan Credit_Utilization_Ratiodengan Outstanding_DebtdanCredit_Score

In [ ]:
# RELATIONSHIP BETWEEN CREDIT UTILIZATION AND OUTSTANDING DEBT

credit_util_data = eda.dropna(
    subset=[
        "Credit_Utilization_Ratio",
        "Outstanding_Debt"
    ]
).copy()

credit_util_debt_summary = credit_util_data[
    [
        "Credit_Utilization_Ratio",
        "Outstanding_Debt"
    ]
].corr(method="spearman")

credit_util_debt_summary

Kode ini digunakan untuk mengecek hubungan antara `Credit_Utilization_Ratio` dan `Outstanding_Debt`.

Pertama, kode membuat dataframe `credit_util_data` dengan mengambil baris yang tidak missing pada kolom `Credit_Utilization_Ratio` dan `Outstanding_Debt`. Setelah itu, kode menghitung korelasi Spearman antara kedua kolom tersebut.

Spearman correlation digunakan karena lebih cocok untuk melihat hubungan monotonic, yaitu apakah ketika satu variabel naik, variabel lain cenderung ikut naik atau turun, tanpa harus mengasumsikan hubungan linear sempurna.

Hasil korelasi antara `Credit_Utilization_Ratio` dan `Outstanding_Debt` adalah -0,0684. Nilai ini sangat dekat dengan 0, sehingga hubungan antara kedua variabel tersebut sangat lemah. Tanda negatif menunjukkan arah hubungan yang sedikit berlawanan, tetapi karena nilainya sangat kecil, hubungan ini tidak kuat secara praktis.

Artinya, pada dataset ini, customer dengan `Credit_Utilization_Ratio` tinggi belum tentu memiliki `Outstanding_Debt` yang tinggi juga. Begitu pula sebaliknya, customer dengan outstanding debt tinggi belum tentu memiliki credit utilization ratio tinggi.

Secara bisnis, ini masih masuk akal karena `Credit_Utilization_Ratio` biasanya menggambarkan persentase penggunaan limit kredit, sedangkan `Outstanding_Debt` menunjukkan jumlah utang yang masih belum dibayar. Seseorang bisa memiliki debt besar tetapi limit kreditnya juga besar, sehingga utilization ratio tidak selalu tinggi. Sebaliknya, seseorang dengan debt lebih kecil bisa memiliki utilization ratio tinggi jika limit kreditnya rendah.

Kesimpulannya, `Credit_Utilization_Ratio` dan `Outstanding_Debt` tidak memiliki hubungan yang kuat secara langsung. Keduanya tetap dapat dipertahankan sebagai fitur karena merepresentasikan aspek kredit yang berbeda: `Outstanding_Debt` menunjukkan besar utang, sedangkan `Credit_Utilization_Ratio` menunjukkan tingkat pemakaian kredit relatif terhadap limit.

In [ ]:
# CREDIT UTILIZATION BY CREDIT SCORE

credit_util_score_data = eda.dropna(
    subset=[
        "Credit_Utilization_Ratio",
        "Outstanding_Debt",
        "Credit_Score"
    ]
).copy()

if "Debt_to_Annual_Income_Ratio" in debt_income_clean.columns:
    credit_util_score_data = credit_util_score_data.merge(
        debt_income_clean[
            [
                "ID",
                "Debt_to_Annual_Income_Ratio"
            ]
        ],
        on="ID",
        how="left"
    )

credit_util_by_score = (
    credit_util_score_data
    .groupby("Credit_Score")[
        [
            "Credit_Utilization_Ratio",
            "Outstanding_Debt",
            "Debt_to_Annual_Income_Ratio"
        ]
    ]
    .median()
    .reindex(credit_order)
)

credit_util_by_score

Kode ini digunakan untuk melihat apakah `Credit_Utilization_Ratio`, `Outstanding_Debt`, dan `Debt_to_Annual_Income_Ratio` memiliki pola yang berbeda pada setiap kelas `Credit_Score`.

Pertama, kode membuat dataframe `credit_util_score_data` dengan mengambil data yang tidak missing pada kolom `Credit_Utilization_Ratio`, `Outstanding_Debt`, dan `Credit_Score`. Setelah itu, jika kolom `Debt_to_Annual_Income_Ratio` sudah tersedia di `debt_income_clean`, kode menggabungkan rasio tersebut ke `credit_util_score_data` berdasarkan kolom `ID`.

`Debt_to_Annual_Income_Ratio` bukan kolom asli dari dataset, melainkan kolom turunan yang dibuat dari perbandingan antara `Outstanding_Debt` dan `Annual_Income`. Rasio ini digunakan untuk melihat seberapa besar beban utang customer dibandingkan dengan pendapatan tahunannya.

Kemudian, data dikelompokkan berdasarkan `Credit_Score`, lalu dihitung nilai median untuk tiga kolom: `Credit_Utilization_Ratio`, `Outstanding_Debt`, dan `Debt_to_Annual_Income_Ratio`. Median digunakan karena lebih tahan terhadap outlier dibandingkan mean.

Dari output, terlihat bahwa median `Credit_Utilization_Ratio` antar kelas tidak berbeda jauh. Kelas Poor memiliki median sekitar 31,97, Standard sekitar 32,33, dan Good sekitar 32,79. Perbedaan ini sangat kecil, sehingga `Credit_Utilization_Ratio` tidak terlihat sebagai pembeda yang kuat antara kelas Poor, Standard, dan Good.

Sebaliknya, `Outstanding_Debt` menunjukkan pola yang jauh lebih jelas. Median `Outstanding_Debt` pada kelas Poor adalah sekitar 1.960,88, lebih tinggi dibandingkan Standard sebesar 994,73 dan Good sebesar 737,50. Artinya, customer dengan credit score Poor cenderung memiliki outstanding debt yang lebih besar.

Pola yang sama juga terlihat pada `Debt_to_Annual_Income_Ratio`. Kelas Poor memiliki median rasio sekitar 0,0638, Standard sekitar 0,0219, dan Good sekitar 0,0119. Ini menunjukkan bahwa customer dengan credit score Poor memiliki beban utang yang lebih besar relatif terhadap pendapatan tahunannya.

Berdasarkan pola tersebut, `Debt_to_Annual_Income_Ratio` dapat dipertimbangkan sebagai kandidat fitur baru pada tahap feature engineering. Fitur ini berpotensi membantu model karena tidak hanya melihat jumlah utang secara absolut, tetapi juga menyesuaikannya dengan kemampuan pendapatan customer. Dua customer dapat memiliki `Outstanding_Debt` yang sama, tetapi risikonya berbeda jika pendapatan tahunannya berbeda.

Namun, fitur ini tidak dibuat secara permanen pada tahap EDA sebagai keputusan final modeling. Pada tahap modeling nanti, `Debt_to_Annual_Income_Ratio` sebaiknya dibuat di dalam proses feature engineering atau preprocessing pipeline agar perhitungannya konsisten pada data train, validation, test, dan data inference. Dengan begitu, fitur turunan ini dapat digunakan tanpa mengganggu alur deployment.

Kesimpulannya, `Outstanding_Debt` dan `Debt_to_Annual_Income_Ratio` terlihat lebih relevan dalam membedakan kelas `Credit_Score` dibandingkan `Credit_Utilization_Ratio`. Customer dengan kelas Poor cenderung memiliki debt dan rasio debt terhadap income yang lebih tinggi, sedangkan customer dengan kelas Good cenderung memiliki debt dan rasio debt yang lebih rendah. Oleh karena itu, `Debt_to_Annual_Income_Ratio` layak dicatat sebagai ide feature engineering untuk tahap modeling berikutnya.

In [ ]:
# VISUALIZE CREDIT UTILIZATION BY CREDIT SCORE

credit_util_plot_data = credit_util_score_data.dropna(
    subset=[
        "Credit_Score",
        "Credit_Utilization_Ratio"
    ]
).copy()

credit_util_box_data = [
    credit_util_plot_data.loc[
        credit_util_plot_data["Credit_Score"] == score,
        "Credit_Utilization_Ratio"
    ]
    for score in credit_order
]

plt.figure(figsize=(7, 5))
plt.boxplot(
    credit_util_box_data,
    labels=credit_order,
    showfliers=False
)
plt.title("Credit Utilization Ratio by Credit Score")
plt.xlabel("Credit Score")
plt.ylabel("Credit Utilization Ratio")
plt.show()

Grafik boxplot ini menunjukkan perbandingan `Credit_Utilization_Ratio` berdasarkan kelas `Credit_Score`, yaitu Poor, Standard, dan Good.

Dari grafik terlihat bahwa ketiga kelas memiliki pola distribusi yang sangat mirip. Median `Credit_Utilization_Ratio` untuk Poor, Standard, dan Good berada pada rentang sekitar 32–33. Perbedaan median antar kelas sangat kecil, sehingga variabel ini tidak menunjukkan pemisahan yang kuat terhadap kelas `Credit_Score`.

Sebaran datanya juga terlihat cukup mirip. Ketiga kelas memiliki rentang nilai yang hampir sama, yaitu dari sekitar 20 sampai 48. Box atau area tengah distribusi pada masing-masing kelas juga saling berdekatan. Artinya, customer dengan credit score Poor, Standard, maupun Good dapat memiliki tingkat credit utilization yang relatif sama.

Pola ini berbeda dengan `Outstanding_Debt` dan `Debt_to_Annual_Income_Ratio` yang sebelumnya menunjukkan perbedaan lebih jelas antar kelas. Pada variabel debt, kelas Poor cenderung memiliki nilai yang lebih tinggi dibandingkan Standard dan Good. Namun, pada `Credit_Utilization_Ratio`, perbedaannya tidak terlalu terlihat.

Kesimpulannya, `Credit_Utilization_Ratio` tidak terlihat sebagai fitur yang kuat untuk membedakan kelas `Credit_Score` secara langsung berdasarkan grafik ini. Walaupun begitu, kolom ini tetap dapat dipertahankan dalam modeling karena masih merepresentasikan tingkat penggunaan kredit customer. Hanya saja, berdasarkan EDA, kontribusinya kemungkinan tidak sekuat `Outstanding_Debt` atau rasio turunan seperti `Debt_to_Annual_Income_Ratio`.

### Hubungan Credit_History_Age dengan Age

Pengecekan hubungan antara `Age` dan `Credit_History_Age` dilakukan untuk memastikan konsistensi logis antarfitur. `Age` menunjukkan umur nasabah, sedangkan `Credit_History_Age` menunjukkan lama riwayat kredit nasabah. Secara logika, lama riwayat kredit tidak boleh lebih besar daripada umur nasabah, karena seseorang tidak mungkin memiliki riwayat kredit sebelum lahir.

Oleh karena itu, dibuat fitur sementara `Credit_History_Years` dan `Estimated_Age_When_Credit_Started`. Fitur ini bukan kolom asli dataset, tetapi hanya digunakan untuk validasi data. Jika `Estimated_Age_When_Credit_Started` bernilai negatif, maka artinya `Credit_History_Age` lebih panjang daripada umur nasabah, sehingga data tersebut tidak konsisten secara logis.

Dengan demikian, pengecekan ini bertujuan untuk mendeteksi kemungkinan nilai tidak valid pada `Age` atau `Credit_History_Age`, bukan karena kedua kolom tersebut harus memiliki nilai yang sama.

In [ ]:
# RELATIONSHIP BETWEEN CREDIT HISTORY AGE AND AGE

credit_history_data = eda.dropna(
    subset=[
        "Age",
        "Credit_History_Age"
    ]
).copy()

credit_history_data["Credit_History_Years"] = (
    credit_history_data["Credit_History_Age"] / 12
)

credit_history_data["Estimated_Age_When_Credit_Started"] = (
    credit_history_data["Age"]
    - credit_history_data["Credit_History_Years"]
)

credit_history_relation = credit_history_data[
    [
        "Age",
        "Credit_History_Age",
        "Credit_History_Years",
        "Estimated_Age_When_Credit_Started"
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

credit_history_relation

Output ini menunjukkan hubungan antara `Age` dan `Credit_History_Age`. Pada analisis ini, `Credit_History_Age` sudah dikonversi ke bulan, lalu dibuat kolom tambahan `Credit_History_Years` dengan membagi `Credit_History_Age` dengan 12. Setelah itu dibuat `Estimated_Age_When_Credit_Started`, yaitu perkiraan umur customer ketika mulai memiliki riwayat kredit.

Rumusnya adalah:

`Estimated_Age_When_Credit_Started = Age - Credit_History_Years`

Jumlah data yang masuk ke analisis ini adalah 22.659 baris. Jumlah ini lebih kecil dari total data karena hanya baris yang memiliki nilai lengkap pada `Age` dan `Credit_History_Age` yang digunakan.

Secara umum, `Age` terlihat berada pada rentang 14 sampai 56 tahun, dengan median 33 tahun. `Credit_History_Age` memiliki median 219 bulan atau sekitar 18,25 tahun. Artinya, secara umum customer memiliki riwayat kredit yang cukup panjang.

Namun, kolom `Estimated_Age_When_Credit_Started` menunjukkan adanya beberapa nilai yang tidak masuk akal. Nilai minimumnya adalah -17,83 tahun. Artinya, berdasarkan perhitungan, ada customer yang seolah-olah sudah memiliki riwayat kredit sebelum lahir. Ini jelas tidak logis dan menunjukkan adanya ketidaksesuaian antara `Age` dan `Credit_History_Age`.

Selain nilai negatif, rata-rata `Estimated_Age_When_Credit_Started` adalah sekitar 14,78 tahun. Ini juga cukup rendah, karena secara umum seseorang biasanya mulai memiliki riwayat kredit pada usia yang lebih dewasa. Jadi, sebagian data mungkin memiliki `Credit_History_Age` yang terlalu panjang dibandingkan usia customer.

Namun, tidak semua data bermasalah. Median `Estimated_Age_When_Credit_Started` adalah sekitar 14,83 tahun, P75 sekitar 24,08 tahun, P90 sekitar 30,42 tahun, dan maksimum sekitar 45,08 tahun. Artinya, sebagian data masih masuk akal, tetapi ada kelompok data yang perlu dicek lebih lanjut karena estimasi umur mulai kredit terlalu rendah atau bahkan negatif.

Kesimpulannya, hubungan antara `Age` dan `Credit_History_Age` perlu dianalisis lebih lanjut. Nilai `Credit_History_Age` tidak boleh langsung digunakan begitu saja jika lebih besar daripada usia customer. Tahap berikutnya sebaiknya mengecek jumlah kasus di mana `Credit_History_Years > Age`, karena kondisi tersebut berarti riwayat kredit lebih panjang daripada umur customer dan secara logika tidak valid.

In [ ]:
# CHECK IMPOSSIBLE CREDIT HISTORY

credit_history_too_long = credit_history_data[
    credit_history_data["Credit_History_Years"]
    > credit_history_data["Age"]
].copy()

credit_started_too_early = credit_history_data[
    credit_history_data["Estimated_Age_When_Credit_Started"] < 10
].copy()

print("Jumlah data valid credit history-age:", len(credit_history_data))
print("Credit history lebih panjang dari umur:", len(credit_history_too_long))
print("Estimasi mulai kredit sebelum umur 10:", len(credit_started_too_early))

credit_started_too_early[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Age",
        "Credit_History_Age",
        "Credit_History_Years",
        "Estimated_Age_When_Credit_Started",
        "Credit_Score"
    ]
].sort_values(
    by="Estimated_Age_When_Credit_Started"
).head(30)

Output ini menunjukkan hasil pengecekan konsistensi antara `Age` dan `Credit_History_Age`.

Dari 22.659 data valid, terdapat 2.745 baris di mana `Credit_History_Age` lebih panjang daripada umur customer. Ini berarti lama riwayat kredit lebih besar daripada usia customer, sehingga secara logika tidak mungkin terjadi. Persentasenya sekitar 12,11% dari data valid, sehingga masalah ini cukup besar dan perlu diperhatikan.

Selain itu, terdapat 8.102 baris dengan estimasi umur mulai kredit sebelum umur 10 tahun. Kondisi ini tidak sekuat kasus negatif, tetapi tetap mencurigakan karena secara umum seseorang sangat kecil kemungkinan sudah memiliki riwayat kredit sejak usia di bawah 10 tahun. Namun, kategori ini sebaiknya dianggap sebagai suspicious case, bukan langsung dianggap error pasti.

Tabel yang ditampilkan memperlihatkan kasus paling ekstrem. Misalnya, customer dengan `Age = 14` tetapi `Credit_History_Age = 382 bulan`, atau sekitar 31,83 tahun. Jika dihitung, customer tersebut seolah-olah mulai memiliki riwayat kredit pada usia -17,83 tahun. Ini jelas tidak masuk akal karena berarti riwayat kredit dimulai sebelum customer lahir.

Pola yang sama terlihat pada beberapa baris lain, seperti customer berusia 14–18 tahun tetapi memiliki riwayat kredit sekitar 30–33 tahun. Hal ini menunjukkan adanya ketidaksesuaian yang kuat antara `Age` dan `Credit_History_Age`.

Namun, dari output ini kita belum bisa langsung menentukan apakah yang salah adalah `Age` atau `Credit_History_Age`. Bisa saja umur customer yang tercatat terlalu kecil, atau bisa juga lama riwayat kredit yang terlalu panjang. Oleh karena itu, data ini perlu dicek lagi menggunakan riwayat `Customer_ID` yang sama sebelum melakukan repair.

Kesimpulannya, hubungan antara `Age` dan `Credit_History_Age` menunjukkan adanya masalah konsistensi logis. Kasus `Credit_History_Age > Age` adalah anomali kuat karena secara logika tidak mungkin. Sementara itu, kasus estimasi mulai kredit sebelum umur 10 tahun perlu dicatat sebagai indikasi mencurigakan, tetapi tidak langsung diperbaiki secara agresif. Tahap berikutnya sebaiknya mengecek riwayat customer untuk menentukan apakah nilai yang lebih mungkin bermasalah adalah `Age` atau `Credit_History_Age`.

### Perbaikan Credit_History_Age berdasarkan hasil hubungan dengan Age

In [ ]:
# REPAIR CREDIT_HISTORY_AGE BASED ON AGE LOGIC

df = df.copy()

df["Age"] = pd.to_numeric(
    df["Age"],
    errors="coerce"
)

df["Credit_History_Age"] = pd.to_numeric(
    df["Credit_History_Age"],
    errors="coerce"
)

invalid_credit_history_mask = (
    df["Age"].notna() &
    df["Credit_History_Age"].notna() &
    (df["Credit_History_Age"] > df["Age"] * 12)
)

print("Sebelum repair:")
print("Credit_History_Age lebih panjang dari Age:", invalid_credit_history_mask.sum())
print("Missing Credit_History_Age:", df["Credit_History_Age"].isna().sum())

In [ ]:
# BUILD VALID CREDIT HISTORY REFERENCE FROM SAME CUSTOMER

valid_credit_history_reference = (
    df[
        df["Age"].notna() &
        df["Credit_History_Age"].notna() &
        (df["Credit_History_Age"] <= df["Age"] * 12)
    ]
    .groupby("Customer_ID")["Credit_History_Age"]
    .median()
)

suggested_credit_history = df.loc[
    invalid_credit_history_mask,
    "Customer_ID"
].map(valid_credit_history_reference)

credit_history_fillable_indices = suggested_credit_history[
    suggested_credit_history.notna()
].index

credit_history_unresolved_indices = suggested_credit_history[
    suggested_credit_history.isna()
].index

print("Invalid Credit_History_Age yang bisa diperbaiki:", len(credit_history_fillable_indices))
print("Invalid Credit_History_Age tanpa referensi valid:", len(credit_history_unresolved_indices))

Kode ini digunakan untuk membuat referensi `Credit_History_Age` yang valid berdasarkan riwayat customer yang sama. Data yang dianggap valid adalah data dengan `Age` dan `Credit_History_Age` tidak missing, serta `Credit_History_Age` tidak lebih panjang daripada umur customer dalam bulan.

Setelah itu, kode mengambil median `Credit_History_Age` valid untuk setiap `Customer_ID`. Median ini digunakan sebagai nilai referensi untuk memperbaiki data yang tidak logis.

Dari output, terdapat 62 baris `Credit_History_Age` tidak logis yang masih bisa diperbaiki menggunakan median valid dari customer yang sama. Sementara itu, terdapat 2.683 baris yang tidak memiliki referensi valid dari customer yang sama.

Artinya, 62 baris dapat diganti dengan median valid customer tersebut, sedangkan 2.683 baris lainnya lebih aman dijadikan missing value. Keputusan ini dilakukan agar cleaning tidak terlalu memaksakan asumsi baru pada data.

In [ ]:
# SAVE AUDIT BEFORE CREDIT_HISTORY_AGE REPAIR

credit_history_repair_audit = df.loc[
    invalid_credit_history_mask,
    [
        "ID",
        "Customer_ID",
        "Month",
        "Age",
        "Credit_History_Age",
        "Credit_Score"
    ]
].copy()

credit_history_repair_audit["Credit_History_Years"] = (
    credit_history_repair_audit["Credit_History_Age"] / 12
)

credit_history_repair_audit["Estimated_Age_When_Credit_Started"] = (
    credit_history_repair_audit["Age"] -
    credit_history_repair_audit["Credit_History_Years"]
)

credit_history_repair_audit["Suggested_Credit_History_Age"] = (
    suggested_credit_history
)

credit_history_repair_audit["Repair_Action"] = np.where(
    credit_history_repair_audit["Suggested_Credit_History_Age"].notna(),
    "Replace with customer median valid Credit_History_Age",
    "Set as missing value"
)

credit_history_repair_audit.sort_values(
    by="Estimated_Age_When_Credit_Started",
    ascending=True
).head(30)

In [ ]:
# APPLY CREDIT_HISTORY_AGE REPAIR

suggested_credit_history_rounded = (
    suggested_credit_history
    .round()
    .astype("Int64")
)

df["Credit_History_Age"] = df["Credit_History_Age"].round().astype("Int64")

df.loc[
    credit_history_fillable_indices,
    "Credit_History_Age"
] = suggested_credit_history_rounded.loc[credit_history_fillable_indices]

df.loc[
    credit_history_unresolved_indices,
    "Credit_History_Age"
] = pd.NA

df["Credit_History_Age"] = df["Credit_History_Age"].astype("Int64")

print("Setelah repair:")
print(
    "Credit_History_Age lebih panjang dari Age:",
    (
        df["Age"].notna() &
        df["Credit_History_Age"].notna() &
        (df["Credit_History_Age"] > df["Age"] * 12)
    ).sum()
)
print("Missing Credit_History_Age:", df["Credit_History_Age"].isna().sum())

Setelah proses repair utama, masih terdapat 33 baris dengan `Credit_History_Age` yang lebih panjang daripada umur customer. Hal ini dapat terjadi karena nilai median valid dari customer yang sama masih terlalu besar untuk beberapa baris tertentu, terutama jika umur pada baris tersebut lebih kecil.

Karena nilai tersebut tetap tidak logis, 33 baris tersebut dijadikan missing value. Keputusan ini lebih aman dibandingkan memaksakan nilai pengganti yang masih melanggar aturan logika data.

Setelah final fix, tidak ada lagi kasus `Credit_History_Age` yang lebih panjang daripada `Age`. Missing value pada `Credit_History_Age` nantinya akan ditangani pada tahap preprocessing pipeline.

In [ ]:
# FIX REMAINING INVALID CREDIT_HISTORY_AGE CASES

remaining_invalid_credit_history_mask = (
    df["Age"].notna() &
    df["Credit_History_Age"].notna() &
    (df["Credit_History_Age"] > df["Age"] * 12)
)

print("Remaining invalid before final fix:", remaining_invalid_credit_history_mask.sum())

df.loc[
    remaining_invalid_credit_history_mask,
    "Credit_History_Age"
] = pd.NA

df["Credit_History_Age"] = df["Credit_History_Age"].astype("Int64")

print("After final fix:")
print(
    "Credit_History_Age lebih panjang dari Age:",
    (
        df["Age"].notna() &
        df["Credit_History_Age"].notna() &
        (df["Credit_History_Age"] > df["Age"] * 12)
    ).sum()
)
print("Missing Credit_History_Age:", df["Credit_History_Age"].isna().sum())

In [ ]:
# CHECK CREDIT HISTORY AFTER REPAIR

credit_history_after_repair = df.dropna(
    subset=[
        "Age",
        "Credit_History_Age"
    ]
).copy()

credit_history_after_repair["Credit_History_Years"] = (
    credit_history_after_repair["Credit_History_Age"] / 12
)

credit_history_after_repair["Estimated_Age_When_Credit_Started"] = (
    credit_history_after_repair["Age"] -
    credit_history_after_repair["Credit_History_Years"]
)

credit_history_after_repair[
    [
        "Age",
        "Credit_History_Age",
        "Credit_History_Years",
        "Estimated_Age_When_Credit_Started"
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

In [ ]:
# CHECK REMAINING INVALID CREDIT HISTORY CASES

remaining_invalid_credit_history = credit_history_after_repair[
    credit_history_after_repair["Credit_History_Age"] >
    credit_history_after_repair["Age"] * 12
]

print("Remaining invalid Credit_History_Age cases:", len(remaining_invalid_credit_history))

remaining_invalid_credit_history[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Age",
        "Credit_History_Age",
        "Credit_History_Years",
        "Estimated_Age_When_Credit_Started",
        "Credit_Score"
    ]
].head(20)

Tabel ini menampilkan sisa kasus `Credit_History_Age` yang masih lebih panjang daripada umur customer. Selisihnya tidak seekstrem sebelumnya, tetapi secara logika tetap tidak valid karena lama riwayat kredit tidak boleh melebihi umur customer.

Karena nilai hasil repair sebelumnya masih menghasilkan beberapa kasus yang tidak logis, sisa data tersebut dijadikan missing value. Keputusan ini lebih aman dibandingkan memaksakan pengisian nilai, karena tidak ada referensi yang cukup kuat untuk memastikan nilai `Credit_History_Age` yang benar.

Setelah final fix, jumlah kasus `Credit_History_Age > Age` harus menjadi 0. Missing value pada `Credit_History_Age` nantinya akan ditangani pada tahap preprocessing pipeline.

-----------------------

Umur 16 tahun masih mungkin muncul sebagai umur customer dalam dataset, tetapi secara logika kurang wajar jika customer tersebut sudah memiliki credit history yang sangat panjang. Jika `Age = 16`, maka `Credit_History_Age` maksimal secara logika tidak boleh lebih dari 16 tahun atau 192 bulan. Jika lebih dari itu, berarti customer seolah-olah sudah memiliki riwayat kredit sebelum lahir, sehingga data tersebut tidak valid.

Namun, karena dataset tidak menjelaskan secara spesifik jenis kredit yang dimaksud, analisis cleaning tidak menggunakan batas usia minimum pengajuan kredit seperti 17, 18, atau 21 tahun. Cleaning hanya menggunakan aturan logika paling aman, yaitu `Credit_History_Age` tidak boleh lebih panjang daripada `Age`.

In [ ]:
# CHECK AGE LESS THAN 21

age_under_21 = df[
    df["Age"].notna() &
    (df["Age"] < 21)
].copy()

print("Jumlah baris dengan Age < 21:", len(age_under_21))
print("Jumlah unique Customer_ID dengan Age < 21:", age_under_21["Customer_ID"].nunique())

age_under_21["Age"].value_counts().sort_index()

Setelah proses repair, nilai `Age` di bawah 21 tahun masih tetap ada di dataset. Hal ini karena proses cleaning tidak menggunakan batas usia minimum pengajuan kredit sebagai aturan utama. Cleaning hanya memperbaiki kondisi yang secara logika pasti tidak mungkin, yaitu ketika `Credit_History_Age` lebih panjang daripada umur customer.

Dengan demikian, customer berusia 14 tahun masih dapat muncul selama `Credit_History_Age` tidak melebihi 14 tahun atau 168 bulan. Jika `Credit_History_Age` melebihi batas tersebut, maka nilainya dianggap tidak logis dan dijadikan missing value.

Keputusan ini dilakukan agar cleaning tetap konservatif. Umur di bawah 21 tahun dapat dianggap sebagai data yang perlu dicatat atau dianalisis lebih lanjut, tetapi tidak langsung dihapus karena dataset tidak memberikan informasi spesifik mengenai jenis produk kredit, status pengguna tambahan, atau aturan usia yang digunakan.

In [ ]:
# CHECK AGE 14 CREDIT HISTORY CONDITION WITH INCOME AND CREDIT FEATURES

age_14_check = df[
    df["Age"].notna() &
    (df["Age"] == 14)
].copy()

age_14_check["Credit_History_Years"] = (
    age_14_check["Credit_History_Age"] / 12
)

age_14_check["Estimated_Age_When_Credit_Started"] = (
    age_14_check["Age"] -
    age_14_check["Credit_History_Years"]
)

print("Jumlah baris Age = 14:", len(age_14_check))
print("Jumlah unique Customer_ID Age = 14:", age_14_check["Customer_ID"].nunique())
print("Missing Credit_History_Age pada Age = 14:", age_14_check["Credit_History_Age"].isna().sum())

print(
    "Credit_History_Age > Age pada Age = 14:",
    (
        age_14_check["Credit_History_Age"].notna() &
        (age_14_check["Credit_History_Age"] > age_14_check["Age"] * 12)
    ).sum()
)

age_14_columns = [
    "ID",
    "Customer_ID",
    "Month",
    "Age",
    "Occupation",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Credit_History_Age",
    "Credit_History_Years",
    "Estimated_Age_When_Credit_Started",
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Type_of_Loan",
    "Delay_from_due_date",
    "Num_of_Delayed_Payment",
    "Outstanding_Debt",
    "Total_EMI_per_month",
    "Credit_Utilization_Ratio",
    "Payment_of_Min_Amount",
    "Payment_Behaviour",
    "Credit_Mix",
    "Credit_Score"
]

age_14_columns = [
    col for col in age_14_columns
    if col in age_14_check.columns
]

age_14_check[
    age_14_columns
].sort_values(
    by=[
        "Estimated_Age_When_Credit_Started",
        "Annual_Income",
        "Monthly_Inhand_Salary"
    ],
    ascending=[
        True,
        False,
        False
    ]
).head(30)

In [ ]:
# AGE 14 INCOME AND CREDIT SUMMARY

age_14_check[
    [
        "Annual_Income",
        "Monthly_Inhand_Salary",
        "Outstanding_Debt",
        "Total_EMI_per_month",
        "Credit_Utilization_Ratio"
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

Output ini menunjukkan ringkasan kondisi finansial customer dengan `Age = 14`.

Terdapat 326 baris dengan umur 14 tahun. Pada kelompok ini, `Annual_Income` tersedia untuk seluruh 326 baris, sedangkan `Monthly_Inhand_Salary` tersedia pada 266 baris. Median `Annual_Income` adalah sekitar 32.198,23 dan median `Monthly_Inhand_Salary` sekitar 2.787,81. Selain itu, kelompok ini juga memiliki `Outstanding_Debt`, `Total_EMI_per_month`, dan `Credit_Utilization_Ratio`.

Secara konteks, kondisi ini cukup mencurigakan karena customer berusia 14 tahun terlihat memiliki data finansial yang lengkap seperti pendapatan, utang, EMI, dan penggunaan kredit. Namun, nilai-nilai finansialnya sendiri tidak terlihat ekstrem setelah proses cleaning. Misalnya, median `Outstanding_Debt` sekitar 1.963,24, median `Total_EMI_per_month` sekitar 79,11, dan median `Credit_Utilization_Ratio` sekitar 31,67.

Dengan demikian, masalah utama bukan pada nilai pendapatan atau debt yang ekstrem, melainkan pada kewajaran konteks umur 14 tahun sebagai customer kredit. Namun, karena dataset tidak memberikan informasi mengenai regulasi usia, jenis produk kredit, atau status customer sebagai pemegang utama/tambahan, nilai `Age = 14` tidak langsung dihapus atau diubah.

Keputusan cleaning tetap dibuat secara konservatif. Data `Age < 21` hanya dicatat sebagai kelompok usia muda yang perlu diperhatikan, tetapi tidak dijadikan missing value. Cleaning hanya dilakukan pada kondisi yang secara logika pasti tidak mungkin, yaitu ketika `Credit_History_Age` lebih panjang daripada umur customer.

Untuk tahap feature engineering nanti, kelompok usia muda seperti `Age < 21` dapat dipertimbangkan sebagai fitur tambahan, misalnya `Is_Under_21` atau age group. Namun, pada tahap cleaning, nilai tersebut tetap dipertahankan agar tidak membuat asumsi berlebihan terhadap dataset.

### Hubungan Delay_from_due_date, Num_of_Delayed_Payment, danPayment_of_Min_Amount

In [ ]:
# RELATIONSHIP BETWEEN PAYMENT DELAY FEATURES

payment_delay_data = df.dropna(
    subset=[
        "Delay_from_due_date",
        "Num_of_Delayed_Payment"
    ]
).copy()

payment_delay_summary = payment_delay_data[
    [
        "Delay_from_due_date",
        "Num_of_Delayed_Payment"
    ]
].corr(method="spearman")

payment_delay_summary

Kode ini digunakan untuk melihat hubungan antara `Delay_from_due_date` dan `Num_of_Delayed_Payment`. `Delay_from_due_date` menunjukkan keterlambatan pembayaran dalam jumlah hari, sedangkan `Num_of_Delayed_Payment` menunjukkan jumlah pembayaran yang pernah terlambat.

Korelasi yang digunakan adalah Spearman karena lebih cocok untuk melihat hubungan monotonic dan lebih tahan terhadap outlier dibandingkan Pearson.

Dari output, nilai korelasi antara `Delay_from_due_date` dan `Num_of_Delayed_Payment` adalah 0,5536. Nilai ini menunjukkan hubungan positif sedang. Artinya, semakin besar keterlambatan dari tanggal jatuh tempo, maka jumlah delayed payment juga cenderung lebih tinggi.

Namun, korelasinya tidak mendekati 1, sehingga kedua fitur ini tidak sepenuhnya sama. `Delay_from_due_date` lebih menggambarkan tingkat keterlambatan dalam hari, sedangkan `Num_of_Delayed_Payment` menggambarkan frekuensi keterlambatan pembayaran. Oleh karena itu, keduanya masih dapat dipertahankan sebagai fitur yang berbeda dalam modeling.



In [ ]:
# PAYMENT DELAY SUMMARY BY PAYMENT OF MINIMUM AMOUNT

delay_min_payment_data = df.dropna(
    subset=[
        "Delay_from_due_date",
        "Num_of_Delayed_Payment",
        "Payment_of_Min_Amount"
    ]
).copy()

delay_by_min_payment = (
    delay_min_payment_data
    .groupby("Payment_of_Min_Amount")[
        [
            "Delay_from_due_date",
            "Num_of_Delayed_Payment"
        ]
    ]
    .median()
)

delay_by_min_payment

Kode ini digunakan untuk melihat pola keterlambatan pembayaran berdasarkan kategori `Payment_of_Min_Amount`.

Pertama, kode mengambil data yang tidak missing pada kolom `Delay_from_due_date`, `Num_of_Delayed_Payment`, dan `Payment_of_Min_Amount`. Setelah itu, data dikelompokkan berdasarkan `Payment_of_Min_Amount`, lalu dihitung median dari `Delay_from_due_date` dan `Num_of_Delayed_Payment`.

Dari output, terlihat bahwa kategori `Yes` memiliki median keterlambatan paling tinggi, yaitu `Delay_from_due_date` sebesar 24 hari dan `Num_of_Delayed_Payment` sebesar 17. Kategori `NM` berada di tengah dengan median delay 18 hari dan jumlah delayed payment 14. Sementara itu, kategori `No` memiliki median paling rendah, yaitu delay 11 hari dan jumlah delayed payment 9.

Pola ini menunjukkan bahwa customer dengan `Payment_of_Min_Amount = Yes` cenderung memiliki riwayat pembayaran yang lebih bermasalah dibandingkan kategori `No`. Hal ini terlihat dari keterlambatan pembayaran yang lebih lama dan jumlah delayed payment yang lebih tinggi.

Kesimpulannya, `Payment_of_Min_Amount` terlihat relevan terhadap perilaku pembayaran customer. Kolom ini sebaiknya tetap dipertahankan sebagai fitur kategorikal dalam modeling. Nilai `NM` juga tetap diperlakukan sebagai kategori tersendiri, bukan diubah menjadi `No`, karena pola median-nya berbeda dari kategori `No`.

In [ ]:
# PAYMENT DELAY SUMMARY BY CREDIT SCORE

delay_score_data = df.dropna(
    subset=[
        "Delay_from_due_date",
        "Num_of_Delayed_Payment",
        "Credit_Score"
    ]
).copy()

delay_by_score = (
    delay_score_data
    .groupby("Credit_Score")[
        [
            "Delay_from_due_date",
            "Num_of_Delayed_Payment"
        ]
    ]
    .median()
    .reindex(credit_order)
)

delay_by_score

Kode ini digunakan untuk melihat pola `Delay_from_due_date` dan `Num_of_Delayed_Payment` berdasarkan kelas `Credit_Score`.

Pertama, kode mengambil data yang tidak missing pada kolom `Delay_from_due_date`, `Num_of_Delayed_Payment`, dan `Credit_Score`. Setelah itu, data dikelompokkan berdasarkan `Credit_Score`, lalu dihitung nilai median untuk kedua fitur keterlambatan pembayaran tersebut.

Dari output, terlihat pola yang sangat jelas. Kelas Poor memiliki median `Delay_from_due_date` paling tinggi, yaitu 27 hari, dan median `Num_of_Delayed_Payment` sebesar 17. Kelas Standard berada di tengah dengan median delay 18 hari dan jumlah delayed payment 14. Sementara itu, kelas Good memiliki nilai paling rendah, yaitu median delay 10 hari dan jumlah delayed payment 8.

Pola ini menunjukkan bahwa semakin buruk `Credit_Score`, semakin tinggi keterlambatan pembayaran dan jumlah pembayaran yang terlambat. Dengan kata lain, customer dengan kelas Poor cenderung memiliki riwayat pembayaran yang lebih buruk dibandingkan Standard dan Good.

Kesimpulannya, `Delay_from_due_date` dan `Num_of_Delayed_Payment` merupakan fitur yang relevan untuk memprediksi `Credit_Score`, karena keduanya menunjukkan perbedaan yang jelas antar kelas target.

In [ ]:
# VISUALIZE DELAY FROM DUE DATE BY CREDIT SCORE

delay_plot_data = delay_score_data.dropna(
    subset=[
        "Credit_Score",
        "Delay_from_due_date"
    ]
).copy()

delay_box_data = [
    delay_plot_data.loc[
        delay_plot_data["Credit_Score"] == score,
        "Delay_from_due_date"
    ]
    for score in credit_order
]

plt.figure(figsize=(7, 5))
plt.boxplot(
    delay_box_data,
    labels=credit_order,
    showfliers=False
)
plt.title("Delay from Due Date by Credit Score")
plt.xlabel("Credit Score")
plt.ylabel("Delay from Due Date")
plt.show()

Grafik boxplot ini menunjukkan distribusi `Delay_from_due_date` berdasarkan kelas `Credit_Score`.

Dari grafik terlihat bahwa kelas Poor memiliki nilai keterlambatan pembayaran paling tinggi dibandingkan kelas lainnya. Median `Delay_from_due_date` pada kelas Poor berada sekitar 27 hari, dengan sebaran data yang juga lebih lebar. Hal ini menunjukkan bahwa customer dengan credit score Poor cenderung mengalami keterlambatan pembayaran yang lebih lama.

Kelas Standard berada di posisi tengah, dengan median keterlambatan sekitar 18 hari. Sementara itu, kelas Good memiliki median paling rendah, yaitu sekitar 10 hari. Sebaran pada kelas Good juga lebih rendah dibandingkan Poor dan Standard.

Pola ini menunjukkan hubungan yang jelas antara keterlambatan pembayaran dan kualitas credit score. Semakin tinggi keterlambatan dari tanggal jatuh tempo, semakin buruk kecenderungan kelas `Credit_Score`.

Kesimpulannya, `Delay_from_due_date` merupakan fitur yang relevan untuk modeling karena mampu menunjukkan perbedaan pola yang jelas antara kelas Poor, Standard, dan Good.

### Hubungan Amount_invested_monthly, Monthly_Balance, gaji, dan EMI

In [ ]:
# RELATIONSHIP BETWEEN INVESTMENT, BALANCE, SALARY, AND EMI

cashflow_data = eda.dropna(
    subset=[
        "Monthly_Inhand_Salary",
        "Total_EMI_per_month",
        "Amount_invested_monthly",
        "Monthly_Balance"
    ]
).copy()

cashflow_data["Investment_to_Salary_Ratio"] = (
    cashflow_data["Amount_invested_monthly"]
    / cashflow_data["Monthly_Inhand_Salary"]
)

cashflow_data["Balance_to_Salary_Ratio"] = (
    cashflow_data["Monthly_Balance"]
    / cashflow_data["Monthly_Inhand_Salary"]
)

cashflow_data["Estimated_Remaining_Balance"] = (
    cashflow_data["Monthly_Inhand_Salary"]
    - cashflow_data["Total_EMI_per_month"]
    - cashflow_data["Amount_invested_monthly"]
)

cashflow_data["Balance_Difference"] = (
    cashflow_data["Monthly_Balance"]
    - cashflow_data["Estimated_Remaining_Balance"]
)

cashflow_data = cashflow_data.replace(
    [np.inf, -np.inf],
    np.nan
)

cashflow_clean = cashflow_data.dropna(
    subset=[
        "Investment_to_Salary_Ratio",
        "Balance_to_Salary_Ratio",
        "Estimated_Remaining_Balance",
        "Balance_Difference"
    ]
).copy()

cashflow_summary = cashflow_clean[
    [
        "Monthly_Inhand_Salary",
        "Total_EMI_per_month",
        "Amount_invested_monthly",
        "Monthly_Balance",
        "Investment_to_Salary_Ratio",
        "Balance_to_Salary_Ratio",
        "Estimated_Remaining_Balance",
        "Balance_Difference"
    ]
].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

cashflow_summary

Kode ini digunakan untuk menganalisis hubungan antara `Monthly_Inhand_Salary`, `Total_EMI_per_month`, `Amount_invested_monthly`, dan `Monthly_Balance`.

Pertama, kode mengambil data yang tidak missing pada empat kolom utama, yaitu `Monthly_Inhand_Salary`, `Total_EMI_per_month`, `Amount_invested_monthly`, dan `Monthly_Balance`. Setelah itu, kode membuat beberapa kolom turunan untuk melihat kewajaran hubungan antarvariabel keuangan customer.

Kolom `Investment_to_Salary_Ratio` dibuat dengan membagi `Amount_invested_monthly` terhadap `Monthly_Inhand_Salary`. Rasio ini menunjukkan seberapa besar bagian gaji bulanan yang digunakan untuk investasi setiap bulan.

Kolom `Balance_to_Salary_Ratio` dibuat dengan membagi `Monthly_Balance` terhadap `Monthly_Inhand_Salary`. Rasio ini menunjukkan seberapa besar saldo bulanan customer dibandingkan dengan gaji bulanannya.

Selanjutnya, kode membuat `Estimated_Remaining_Balance`, yaitu estimasi sisa uang setelah gaji bulanan dikurangi EMI dan investasi bulanan. Rumusnya adalah:

`Estimated_Remaining_Balance = Monthly_Inhand_Salary - Total_EMI_per_month - Amount_invested_monthly`

Kemudian dibuat kolom `Balance_Difference`, yaitu selisih antara `Monthly_Balance` asli dengan estimasi sisa saldo. Rumusnya adalah:

`Balance_Difference = Monthly_Balance - Estimated_Remaining_Balance`

Tujuan dari kolom ini adalah untuk melihat apakah `Monthly_Balance` memiliki hubungan yang konsisten dengan salary, EMI, dan monthly investment.

Dari output, terdapat 19.822 baris data lengkap yang digunakan dalam analisis. Median `Monthly_Inhand_Salary` adalah sekitar 3.032,51, sedangkan median `Amount_invested_monthly` adalah sekitar 126,21. Median `Investment_to_Salary_Ratio` sebesar 0,0438 menunjukkan bahwa secara umum customer menginvestasikan sekitar 4,38% dari gaji bulanannya.

Median `Monthly_Balance` adalah sekitar 335,46, sedangkan median `Balance_to_Salary_Ratio` sebesar 0,1172. Artinya, secara umum monthly balance customer berada di sekitar 11,72% dari gaji bulanannya.

Namun, terdapat hal yang perlu diperhatikan pada `Total_EMI_per_month`. Nilai mean `Total_EMI_per_month` sangat tinggi, yaitu sekitar 1.516,80, sedangkan median hanya sekitar 67,41. Selain itu, nilai P99 mencapai 59.118,54 dan maksimum mencapai 82.331. Perbedaan yang sangat besar antara mean, median, P99, dan maksimum menunjukkan adanya nilai ekstrem pada `Total_EMI_per_month`.

Nilai ekstrem tersebut juga memengaruhi `Estimated_Remaining_Balance` dan `Balance_Difference`. Minimum `Estimated_Remaining_Balance` mencapai -80.812,77, sedangkan maksimum `Balance_Difference` mencapai 81.097,54. Nilai seperti ini menunjukkan bahwa sebagian data memiliki EMI yang sangat besar sehingga estimasi sisa saldo menjadi tidak masuk akal.

Kesimpulannya, kode ini membantu menunjukkan bahwa hubungan antara salary, EMI, investasi, dan balance perlu dicek secara hati-hati. `Amount_invested_monthly` dan `Monthly_Balance` terlihat masih berada pada rentang yang relatif wajar, tetapi `Total_EMI_per_month` pada output ini masih menunjukkan outlier ekstrem.



In [ ]:
# CHECK INVESTMENT HIGHER THAN SALARY

investment_higher_than_salary = cashflow_clean[
    cashflow_clean["Investment_to_Salary_Ratio"] > 1
].copy()

print("Jumlah data valid cashflow:", len(cashflow_clean))
print("Jumlah investment lebih besar dari salary:", len(investment_higher_than_salary))

investment_higher_than_salary[
    [
        "ID",
        "Customer_ID",
        "Month",
        "Monthly_Inhand_Salary",
        "Total_EMI_per_month",
        "Amount_invested_monthly",
        "Monthly_Balance",
        "Investment_to_Salary_Ratio",
        "Credit_Score"
    ]
].sort_values(
    by="Investment_to_Salary_Ratio",
    ascending=False
).head(30)

tidak ada

In [ ]:
# VISUALIZE INVESTMENT RATIO BY CREDIT SCORE

investment_plot_data = cashflow_clean.dropna(
    subset=[
        "Credit_Score",
        "Investment_to_Salary_Ratio"
    ]
).copy()

investment_ratio_data = [
    investment_plot_data.loc[
        investment_plot_data["Credit_Score"] == score,
        "Investment_to_Salary_Ratio"
    ]
    for score in credit_order
]

plt.figure(figsize=(7, 5))
plt.boxplot(
    investment_ratio_data,
    labels=credit_order,
    showfliers=False
)
plt.title("Investment to Salary Ratio by Credit Score")
plt.xlabel("Credit Score")
plt.ylabel("Investment to Salary Ratio")
plt.show()

Grafik boxplot ini menunjukkan distribusi `Investment_to_Salary_Ratio` berdasarkan kelas `Credit_Score`.

`Investment_to_Salary_Ratio` merupakan rasio antara `Amount_invested_monthly` dan `Monthly_Inhand_Salary`. Rasio ini digunakan untuk melihat seberapa besar bagian gaji bulanan customer yang digunakan untuk investasi.

Dari grafik, terlihat bahwa ketiga kelas `Credit_Score` memiliki pola yang cukup mirip. Median rasio investasi terhadap gaji pada kelas Poor sedikit lebih tinggi dibandingkan Standard dan Good, tetapi perbedaannya tidak terlalu besar. Kelas Poor memiliki median sekitar 0,05, Standard sekitar 0,04, dan Good sekitar 0,04.

Sebaran data pada ketiga kelas juga relatif mirip, dengan nilai rasio yang umumnya berada di bawah 0,15. Artinya, sebagian besar customer menginvestasikan kurang dari 15% dari gaji bulanannya.

Berdasarkan grafik ini, `Investment_to_Salary_Ratio` tidak menunjukkan pemisahan kelas yang sekuat fitur seperti `Delay_from_due_date`, `Num_of_Delayed_Payment`, atau `Outstanding_Debt`. Namun, fitur ini tetap dapat dipertimbangkan sebagai fitur turunan pada tahap feature engineering karena dapat merepresentasikan pola penggunaan pendapatan customer.

Catatan: jika kode grafik ini masih menggunakan dataframe `eda`, sebaiknya dijalankan ulang menggunakan `df` agar hasilnya sesuai dengan data yang sudah melalui proses cleaning.

### Hubungan Payment_Behaviourdengan investasi dan keseimbangan

In [ ]:
# RELATIONSHIP BETWEEN PAYMENT BEHAVIOUR, INVESTMENT, AND BALANCE

payment_behaviour_data = cashflow_clean.dropna(
    subset=[
        "Payment_Behaviour",
        "Amount_invested_monthly",
        "Monthly_Balance",
        "Investment_to_Salary_Ratio",
        "Balance_to_Salary_Ratio"
    ]
).copy()

payment_behaviour_summary = (
    payment_behaviour_data
    .groupby("Payment_Behaviour")[
        [
            "Amount_invested_monthly",
            "Monthly_Balance",
            "Investment_to_Salary_Ratio",
            "Balance_to_Salary_Ratio"
        ]
    ]
    .median()
)

payment_behaviour_summary

Kode ini digunakan untuk melihat distribusi `Credit_Score` pada setiap kategori `Payment_Behaviour`.

Fungsi `pd.crosstab()` digunakan untuk membuat tabel silang antara `Payment_Behaviour` dan `Credit_Score`. Parameter `normalize="index"` berarti persentase dihitung per baris, sehingga setiap kategori `Payment_Behaviour` memiliki total sekitar 100%. Nilai missing pada `Payment_Behaviour` juga diisi sementara sebagai `"Missing"` agar tetap terlihat dalam analisis.

Dari output, terlihat bahwa kategori `Standard` menjadi kelas terbesar pada semua jenis `Payment_Behaviour`. Namun, terdapat perbedaan pola antara kategori pembayaran.

Kategori `High_spent_Large_value_payments` memiliki proporsi `Good` paling tinggi, yaitu sekitar 23,21%, dan proporsi `Poor` relatif lebih rendah, yaitu sekitar 21,22%. Ini menunjukkan bahwa customer dengan perilaku pembayaran ini cenderung memiliki kondisi credit score yang lebih baik dibanding beberapa kategori lain.

Sebaliknya, kategori `Low_spent_Small_value_payments` memiliki proporsi `Poor` paling tinggi, yaitu sekitar 35,91%, dan proporsi `Good` paling rendah, yaitu sekitar 12,97%. Hal ini menunjukkan bahwa customer pada kategori ini cenderung lebih banyak berada pada kelas credit score Poor.

Kategori `Missing` juga memiliki proporsi `Poor` yang cukup tinggi, yaitu sekitar 32,06%. Ini menunjukkan bahwa missing value pada `Payment_Behaviour` tidak boleh diabaikan begitu saja, karena distribusinya berbeda dari beberapa kategori lain.

Kesimpulannya, `Payment_Behaviour` terlihat relevan terhadap `Credit_Score` karena setiap kategori memiliki komposisi kelas yang berbeda. Kolom ini sebaiknya tetap digunakan sebagai fitur kategorikal dalam modeling. Nilai missing juga sebaiknya ditangani secara eksplisit, misalnya sebagai kategori `"Missing"` atau melalui preprocessing pipeline, agar informasi polanya tidak hilang.

Catatan: kode ini masih menggunakan `eda`. Untuk analisis final setelah cleaning, sebaiknya gunakan `df` agar hasilnya sesuai dengan data yang sudah dibersihkan.

In [ ]:
# CREDIT SCORE DISTRIBUTION BY PAYMENT BEHAVIOUR

payment_behaviour_credit_score = pd.crosstab(
    eda["Payment_Behaviour"].fillna("Missing"),
    eda["Credit_Score"],
    normalize="index"
) * 100

payment_behaviour_credit_score

Tabel ini menunjukkan distribusi `Credit_Score` pada setiap kategori `Payment_Behaviour`. Persentase dihitung per baris, sehingga setiap kategori `Payment_Behaviour` memiliki total sekitar 100%.

Dari output, terlihat bahwa kelas `Standard` menjadi kelas terbesar pada seluruh kategori `Payment_Behaviour`. Namun, proporsi kelas `Good` dan `Poor` berbeda pada setiap kategori, sehingga `Payment_Behaviour` tetap menunjukkan pola yang relevan terhadap `Credit_Score`.

Kategori `High_spent_Large_value_payments` memiliki proporsi `Good` paling tinggi, yaitu sekitar 23,21%, dan proporsi `Poor` relatif lebih rendah, yaitu sekitar 21,22%. Hal ini menunjukkan bahwa customer dengan pola pembayaran ini cenderung memiliki credit score yang lebih baik dibandingkan beberapa kategori lainnya.

Sebaliknya, kategori `Low_spent_Small_value_payments` memiliki proporsi `Poor` paling tinggi, yaitu sekitar 35,91%, dan proporsi `Good` paling rendah, yaitu sekitar 12,97%. Ini menunjukkan bahwa customer dengan pola pembayaran tersebut lebih banyak berada pada kelas credit score Poor.

Kategori `Missing` juga memiliki proporsi `Poor` yang cukup tinggi, yaitu sekitar 32,06%, dan proporsi `Standard` sekitar 48,78%. Hal ini menunjukkan bahwa missing value pada `Payment_Behaviour` tidak sepenuhnya acak dan tetap memiliki pola terhadap target.

Kesimpulannya, `Payment_Behaviour` merupakan fitur kategorikal yang relevan untuk modeling karena distribusi `Credit_Score` berbeda pada setiap kategorinya. Nilai missing pada kolom ini juga sebaiknya tetap ditangani secara eksplisit, misalnya sebagai kategori `"Missing"` atau melalui preprocessing pipeline, agar informasi dari pola missing tersebut tidak hilang.

### Hubungan Credit_Mixdengan debt, interest, dan credit score

In [ ]:
# RELATIONSHIP BETWEEN CREDIT MIX AND FINANCIAL FEATURES

credit_mix_data = df.dropna(
    subset=[
        "Credit_Mix",
        "Outstanding_Debt",
        "Interest_Rate",
        "Credit_Utilization_Ratio",
        "Num_Credit_Inquiries"
    ]
).copy()

credit_mix_data = credit_mix_data.merge(
    debt_income_clean[
        [
            "ID",
            "Debt_to_Annual_Income_Ratio"
        ]
    ],
    on="ID",
    how="left"
)

credit_mix_data = credit_mix_data.dropna(
    subset=[
        "Debt_to_Annual_Income_Ratio"
    ]
).copy()

credit_mix_summary = (
    credit_mix_data
    .groupby("Credit_Mix")[
        [
            "Outstanding_Debt",
            "Debt_to_Annual_Income_Ratio",
            "Interest_Rate",
            "Credit_Utilization_Ratio",
            "Num_Credit_Inquiries"
        ]
    ]
    .median()
)

credit_mix_summary

Tabel ini menunjukkan median beberapa fitur kredit berdasarkan kategori `Credit_Mix`, yaitu Bad, Good, dan Standard.

Dari output, terlihat bahwa kategori `Credit_Mix = Bad` memiliki kondisi kredit yang paling berisiko. Median `Outstanding_Debt` pada kategori Bad adalah sekitar 2.896,95, jauh lebih tinggi dibandingkan Good sebesar 736,68 dan Standard sebesar 1.021,88. Hal ini menunjukkan bahwa customer dengan credit mix buruk cenderung memiliki jumlah utang yang lebih besar.

Pola yang sama terlihat pada `Debt_to_Annual_Income_Ratio`. Kategori Bad memiliki median rasio sebesar 0,1096, lebih tinggi dibandingkan Standard sebesar 0,0238 dan Good sebesar 0,0113. Artinya, customer dengan credit mix Bad tidak hanya memiliki utang yang lebih besar secara nominal, tetapi juga memiliki beban utang yang lebih tinggi relatif terhadap pendapatan tahunannya.

Selain itu, `Interest_Rate` juga menunjukkan pola yang jelas. Kategori Bad memiliki median interest rate sebesar 24, Standard sebesar 15, dan Good sebesar 6. Ini menunjukkan bahwa customer dengan credit mix lebih buruk cenderung mendapatkan tingkat bunga yang lebih tinggi.

`Num_Credit_Inquiries` juga meningkat sesuai buruknya credit mix. Kategori Good memiliki median 3 inquiries, Standard 6 inquiries, dan Bad 10 inquiries. Hal ini dapat menunjukkan bahwa customer dengan credit mix buruk cenderung lebih sering mengajukan atau dicek kreditnya.

Sementara itu, `Credit_Utilization_Ratio` tidak menunjukkan perbedaan besar antar kategori. Median pada Bad, Good, dan Standard berada di sekitar 31–33. Artinya, fitur ini tidak terlalu membedakan kategori `Credit_Mix` dibandingkan fitur seperti `Outstanding_Debt`, `Debt_to_Annual_Income_Ratio`, `Interest_Rate`, dan `Num_Credit_Inquiries`.

Kesimpulannya, `Credit_Mix` memiliki hubungan yang kuat dengan beberapa indikator risiko kredit. Kategori Bad cenderung memiliki outstanding debt lebih tinggi, debt-to-income ratio lebih besar, interest rate lebih tinggi, dan jumlah credit inquiries lebih banyak. Oleh karena itu, `Credit_Mix` merupakan fitur kategorikal yang penting untuk dipertahankan dalam modeling.

In [ ]:
# CREDIT SCORE DISTRIBUTION BY CREDIT MIX

credit_mix_score = pd.crosstab(
    eda["Credit_Mix"].fillna("Missing"),
    eda["Credit_Score"],
    normalize="index"
) * 100

credit_mix_score

Tabel ini menunjukkan distribusi `Credit_Score` pada setiap kategori `Credit_Mix`. Persentase dihitung per baris, sehingga setiap kategori `Credit_Mix` memiliki total sekitar 100%.

Dari output, terlihat bahwa `Credit_Mix` memiliki hubungan yang sangat jelas dengan `Credit_Score`.

Pada kategori `Credit_Mix = Bad`, mayoritas customer berada pada kelas `Poor`, yaitu sekitar 60,32%. Proporsi kelas `Good` sangat kecil, hanya sekitar 1,40%. Hal ini menunjukkan bahwa customer dengan credit mix buruk cenderung memiliki credit score yang buruk juga.

Sebaliknya, pada kategori `Credit_Mix = Good`, proporsi kelas `Good` paling tinggi, yaitu sekitar 49,24%. Proporsi kelas `Poor` jauh lebih rendah, yaitu sekitar 15,48%. Ini menunjukkan bahwa credit mix yang baik cenderung berkaitan dengan credit score yang lebih baik.

Kategori `Credit_Mix = Standard` didominasi oleh kelas `Standard`, yaitu sekitar 73,00%. Proporsi `Poor` pada kategori ini sekitar 21,91%, sedangkan `Good` hanya sekitar 5,09%. Artinya, credit mix standar paling banyak berhubungan dengan credit score standar.

Sementara itu, kategori `Missing` juga memiliki pola tersendiri. Pada data yang missing, mayoritas customer berada pada kelas `Standard` sekitar 50,57%, diikuti `Poor` sekitar 29,37%, dan `Good` sekitar 20,06%. Karena distribusinya berbeda dari kategori lain, missing value pada `Credit_Mix` sebaiknya tidak diabaikan begitu saja.

Kesimpulannya, `Credit_Mix` merupakan fitur kategorikal yang sangat relevan untuk modeling karena distribusi `Credit_Score` berbeda jelas pada setiap kategorinya. Kategori Bad cenderung berhubungan dengan kelas Poor, kategori Good cenderung berhubungan dengan kelas Good, dan kategori Standard paling banyak berhubungan dengan kelas Standard. Nilai missing juga sebaiknya ditangani secara eksplisit dalam preprocessing pipeline.

### Hubungan Num_Bank_Accounts, Num_Credit_Card, Interest_Rate, danCredit_Score

In [ ]:
# RELATIONSHIP BETWEEN ACCOUNT FEATURES AND CREDIT SCORE

account_data = eda.dropna(
    subset=[
        "Num_Bank_Accounts",
        "Num_Credit_Card",
        "Interest_Rate",
        "Num_Credit_Inquiries",
        "Credit_Score"
    ]
).copy()

account_feature_summary = (
    account_data
    .groupby("Credit_Score")[
        [
            "Num_Bank_Accounts",
            "Num_Credit_Card",
            "Interest_Rate",
            "Num_Credit_Inquiries"
        ]
    ]
    .median()
    .reindex(credit_order)
)

account_feature_summary


Tabel ini menunjukkan median beberapa fitur kredit berdasarkan kelas `Credit_Score`, yaitu Poor, Standard, dan Good.

Dari output, terlihat bahwa kelas Poor memiliki nilai median paling tinggi pada seluruh fitur yang ditampilkan. Customer dengan `Credit_Score = Poor` memiliki median `Num_Bank_Accounts` sebesar 7, `Num_Credit_Card` sebesar 7, `Interest_Rate` sebesar 21, dan `Num_Credit_Inquiries` sebesar 8. Hal ini menunjukkan bahwa customer pada kelas Poor cenderung memiliki lebih banyak akun bank, lebih banyak kartu kredit, tingkat bunga lebih tinggi, dan jumlah credit inquiries lebih banyak.

Kelas Standard berada di tengah. Median `Num_Bank_Accounts` dan `Num_Credit_Card` masing-masing sebesar 5, median `Interest_Rate` sebesar 13, dan median `Num_Credit_Inquiries` sebesar 5. Nilai ini lebih rendah dibandingkan kelas Poor, tetapi masih lebih tinggi dibandingkan kelas Good.

Kelas Good memiliki nilai median paling rendah. Median `Num_Bank_Accounts` sebesar 3, `Num_Credit_Card` sebesar 4, `Interest_Rate` sebesar 7, dan `Num_Credit_Inquiries` sebesar 3. Ini menunjukkan bahwa customer dengan credit score baik cenderung memiliki profil kredit yang lebih stabil dan risiko yang lebih rendah.

Pola ini menunjukkan hubungan yang jelas antara fitur-fitur kredit tersebut dengan `Credit_Score`. Semakin buruk kelas credit score, semakin tinggi jumlah akun bank, jumlah kartu kredit, interest rate, dan credit inquiries.

Kesimpulannya, `Num_Bank_Accounts`, `Num_Credit_Card`, `Interest_Rate`, dan `Num_Credit_Inquiries` merupakan fitur yang relevan untuk modeling karena menunjukkan perbedaan pola yang konsisten antar kelas `Credit_Score`.

In [ ]:
# CORRELATION BETWEEN ACCOUNT FEATURES

account_corr_data = account_data.dropna(
    subset=[
        "Num_Bank_Accounts",
        "Num_Credit_Card",
        "Interest_Rate",
        "Num_Credit_Inquiries"
    ]
).copy()

account_corr = account_corr_data[
    [
        "Num_Bank_Accounts",
        "Num_Credit_Card",
        "Interest_Rate",
        "Num_Credit_Inquiries"
    ]
].corr(method="spearman")

account_corr

Tabel ini menunjukkan korelasi antar fitur `Num_Bank_Accounts`, `Num_Credit_Card`, `Interest_Rate`, dan `Num_Credit_Inquiries`.

Dari output, terlihat bahwa seluruh fitur memiliki hubungan positif satu sama lain. Artinya, ketika salah satu fitur meningkat, fitur lainnya juga cenderung ikut meningkat.

Korelasi paling tinggi terlihat antara `Interest_Rate` dan `Num_Credit_Inquiries`, yaitu sebesar 0,6326. Ini menunjukkan hubungan positif sedang hingga cukup kuat. Customer yang memiliki jumlah credit inquiries lebih banyak cenderung memiliki interest rate yang lebih tinggi.

Korelasi antara `Num_Bank_Accounts` dan `Interest_Rate` juga cukup tinggi, yaitu 0,5947. Selain itu, `Num_Bank_Accounts` dengan `Num_Credit_Inquiries` memiliki korelasi 0,5277. Hal ini menunjukkan bahwa jumlah akun bank juga berhubungan dengan indikator risiko kredit lainnya.

Sementara itu, korelasi antara `Num_Bank_Accounts` dan `Num_Credit_Card` sebesar 0,4218, serta korelasi antara `Num_Credit_Card` dan `Num_Credit_Inquiries` sebesar 0,4383. Nilai ini menunjukkan hubungan positif sedang, tetapi tidak terlalu kuat.

Kesimpulannya, fitur-fitur ini saling berhubungan karena sama-sama menggambarkan aktivitas dan risiko kredit customer. Namun, korelasinya belum mendekati 1, sehingga fitur-fitur tersebut tidak sepenuhnya duplikatif. Oleh karena itu, `Num_Bank_Accounts`, `Num_Credit_Card`, `Interest_Rate`, dan `Num_Credit_Inquiries` masih dapat dipertahankan dalam modeling.

In [ ]:
# VISUALIZE INTEREST RATE BY CREDIT SCORE

interest_rate_plot_data = account_data.dropna(
    subset=[
        "Credit_Score",
        "Interest_Rate"
    ]
).copy()

interest_rate_data = [
    interest_rate_plot_data.loc[
        interest_rate_plot_data["Credit_Score"] == score,
        "Interest_Rate"
    ]
    for score in credit_order
]

plt.figure(figsize=(7, 5))
plt.boxplot(
    interest_rate_data,
    labels=credit_order,
    showfliers=False
)
plt.title("Interest Rate by Credit Score")
plt.xlabel("Credit Score")
plt.ylabel("Interest Rate")
plt.show()

Grafik boxplot ini menunjukkan distribusi `Interest_Rate` berdasarkan kelas `Credit_Score`.

Dari grafik terlihat bahwa kelas Poor memiliki tingkat bunga paling tinggi. Median `Interest_Rate` pada kelas Poor berada sekitar 21, dengan sebaran yang cukup lebar hingga mendekati 34. Hal ini menunjukkan bahwa customer dengan credit score Poor cenderung memiliki interest rate yang lebih besar.

Kelas Standard berada di tengah, dengan median `Interest_Rate` sekitar 13. Nilainya lebih rendah dibandingkan Poor, tetapi masih lebih tinggi dibandingkan Good.

Kelas Good memiliki median `Interest_Rate` paling rendah, yaitu sekitar 7. Sebaran datanya juga cenderung lebih rendah dibandingkan kelas Poor dan Standard. Ini menunjukkan bahwa customer dengan credit score yang lebih baik cenderung mendapatkan tingkat bunga yang lebih rendah.

Kesimpulannya, `Interest_Rate` menunjukkan pemisahan yang jelas antar kelas `Credit_Score`. Semakin buruk credit score customer, semakin tinggi interest rate yang dimiliki. Oleh karena itu, `Interest_Rate` merupakan fitur yang sangat relevan untuk digunakan dalam modeling.

### Hetmap Correlation

In [ ]:
# CORRELATION HEATMAP

corr_data = df.copy()

drop_cols = [
    "Unnamed: 0",
    "ID",
    "Customer_ID",
    "Name",
    "SSN"
]

corr_data = corr_data.drop(
    columns=drop_cols,
    errors="ignore"
)

credit_score_map = {
    "Poor": 0,
    "Standard": 1,
    "Good": 2
}

corr_data["Credit_Score_Ordinal"] = (
    corr_data["Credit_Score"]
    .map(credit_score_map)
)

numeric_corr_data = corr_data.select_dtypes(
    include=["number"]
)

print("Jumlah kolom numerik:", numeric_corr_data.shape[1])
print("Shape data korelasi:", numeric_corr_data.shape)

corr_matrix = numeric_corr_data.corr(
    method="spearman"
)

plt.figure(figsize=(18, 14))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Spearman Correlation Heatmap")
plt.tight_layout()
plt.show()

Heatmap ini menunjukkan korelasi Spearman antar 18 kolom numerik pada dataset. Shape data korelasi adalah `(25000, 18)`, artinya analisis dilakukan pada 25.000 baris dan 18 fitur numerik.

Korelasi Spearman digunakan karena lebih cocok untuk melihat hubungan monotonic dan lebih tahan terhadap outlier dibandingkan Pearson. Nilai korelasi berada pada rentang -1 sampai 1. Nilai positif berarti kedua fitur cenderung naik bersama, nilai negatif berarti ketika satu fitur naik fitur lain cenderung turun, sedangkan nilai mendekati 0 menunjukkan hubungan yang lemah.

Dari heatmap, terlihat bahwa `Annual_Income` dan `Monthly_Inhand_Salary` memiliki korelasi sangat tinggi, yaitu sekitar 0,99. Ini wajar karena monthly salary secara logika berasal dari annual income. Keduanya membawa informasi yang sangat mirip, sehingga pada tahap modeling nanti perlu diperhatikan karena dapat menyebabkan fitur yang redundan, terutama untuk model linear.

Beberapa fitur risiko kredit juga saling berkorelasi positif cukup kuat. Misalnya, `Interest_Rate`, `Num_Credit_Inquiries`, `Outstanding_Debt`, `Delay_from_due_date`, dan `Num_of_Delayed_Payment` memiliki korelasi positif sedang hingga cukup kuat satu sama lain. Ini menunjukkan bahwa customer dengan risiko kredit lebih tinggi cenderung memiliki interest rate lebih tinggi, lebih banyak inquiry, lebih besar outstanding debt, serta lebih banyak keterlambatan pembayaran.

`Credit_History_Age` memiliki korelasi negatif dengan beberapa fitur risiko, seperti `Outstanding_Debt`, `Num_Credit_Inquiries`, `Interest_Rate`, dan `Num_of_Loan`. Artinya, customer dengan riwayat kredit yang lebih panjang cenderung memiliki indikator risiko yang lebih rendah. Hal ini juga terlihat dari korelasinya yang positif dengan `Credit_Score_Ordinal`, yaitu sekitar 0,40.

Terhadap `Credit_Score_Ordinal`, fitur yang memiliki hubungan cukup kuat antara lain `Interest_Rate` sekitar -0,49, `Delay_from_due_date` sekitar -0,44, `Num_Credit_Inquiries` sekitar -0,44, `Outstanding_Debt` sekitar -0,44, `Num_Bank_Accounts` sekitar -0,39, dan `Num_Credit_Card` sekitar -0,38. Jika `Credit_Score_Ordinal` dikodekan semakin besar berarti credit score semakin baik, maka korelasi negatif ini berarti semakin tinggi fitur-fitur risiko tersebut, semakin buruk kecenderungan credit score customer.

Sebaliknya, `Credit_History_Age`, `Monthly_Balance`, `Annual_Income`, dan `Monthly_Inhand_Salary` memiliki korelasi positif terhadap `Credit_Score_Ordinal`. Artinya, customer dengan riwayat kredit lebih panjang, balance lebih tinggi, dan pendapatan lebih tinggi cenderung memiliki credit score yang lebih baik.

Sementara itu, `Credit_Utilization_Ratio` memiliki korelasi yang sangat lemah dengan sebagian besar fitur lain dan juga dengan `Credit_Score_Ordinal`. Ini sesuai dengan analisis sebelumnya, di mana distribusi `Credit_Utilization_Ratio` antar kelas Poor, Standard, dan Good tidak terlalu berbeda.

Kesimpulannya, heatmap ini menunjukkan bahwa fitur-fitur seperti `Interest_Rate`, `Delay_from_due_date`, `Num_of_Delayed_Payment`, `Outstanding_Debt`, `Num_Credit_Inquiries`, dan `Credit_History_Age` sangat relevan untuk modeling karena memiliki hubungan yang cukup jelas dengan `Credit_Score_Ordinal`. Namun, beberapa fitur juga memiliki korelasi tinggi satu sama lain, seperti `Annual_Income` dan `Monthly_Inhand_Salary`, sehingga pada tahap modeling perlu diperhatikan kemungkinan redundansi fitur, terutama jika menggunakan model linear.

In [ ]:
# NUMERIC CORRELATION WITH CREDIT SCORE

target_corr = (
    corr_matrix["Credit_Score_Ordinal"]
    .drop("Credit_Score_Ordinal")
    .sort_values(
        key=lambda x: x.abs(),
        ascending=False
    )
)

target_corr

Kode ini digunakan untuk melihat korelasi antara setiap fitur numerik dengan `Credit_Score_Ordinal`.

`Credit_Score_Ordinal` adalah versi numerik dari target `Credit_Score`, misalnya Poor, Standard, dan Good yang diubah menjadi urutan angka. Korelasi dihitung menggunakan nilai dari `corr_matrix`, lalu kolom `Credit_Score_Ordinal` diambil, target itu sendiri dihapus dengan `.drop("Credit_Score_Ordinal")`, dan hasilnya diurutkan berdasarkan nilai absolut korelasi terbesar.

Dari output, fitur dengan hubungan paling kuat terhadap `Credit_Score_Ordinal` adalah `Interest_Rate` dengan korelasi -0,4870. Nilai negatif menunjukkan bahwa semakin tinggi `Interest_Rate`, maka kecenderungan credit score semakin buruk.

Fitur lain yang juga memiliki korelasi negatif cukup kuat adalah `Num_Credit_Inquiries` sebesar -0,4430, `Delay_from_due_date` sebesar -0,4421, dan `Outstanding_Debt` sebesar -0,4352. Artinya, semakin banyak credit inquiries, semakin lama keterlambatan pembayaran, dan semakin besar outstanding debt, maka credit score cenderung semakin rendah.

`Credit_History_Age` memiliki korelasi positif sebesar 0,3978. Ini berarti semakin panjang riwayat kredit customer, credit score cenderung semakin baik. Hal ini masuk akal karena riwayat kredit yang lebih panjang dapat memberikan informasi stabilitas kredit customer.

Selain itu, `Num_Bank_Accounts`, `Num_Credit_Card`, `Num_of_Delayed_Payment`, dan `Num_of_Loan` juga memiliki korelasi negatif dengan target. Semakin tinggi nilai fitur-fitur tersebut, credit score cenderung lebih buruk.

Fitur seperti `Monthly_Balance`, `Annual_Income`, dan `Monthly_Inhand_Salary` memiliki korelasi positif, tetapi kekuatannya lebih rendah. Artinya, customer dengan balance dan pendapatan lebih tinggi cenderung memiliki credit score yang lebih baik, tetapi hubungannya tidak sekuat fitur risiko kredit seperti interest rate, inquiries, delay, dan outstanding debt.

Sementara itu, `Credit_Utilization_Ratio` memiliki korelasi paling lemah, yaitu sekitar 0,0494. Ini sesuai dengan analisis sebelumnya bahwa distribusi credit utilization antar kelas Poor, Standard, dan Good tidak terlalu berbeda.

Kesimpulannya, fitur numerik yang paling relevan terhadap `Credit_Score` adalah `Interest_Rate`, `Num_Credit_Inquiries`, `Delay_from_due_date`, `Outstanding_Debt`, `Credit_History_Age`, `Num_Bank_Accounts`, dan `Num_Credit_Card`. Fitur-fitur ini perlu dipertahankan dalam modeling karena memiliki hubungan yang cukup jelas dengan target.

Catatan penting, `Credit_Score_Ordinal` hanya digunakan untuk kebutuhan EDA korelasi. Kolom ini tidak boleh dijadikan fitur input model, karena merupakan representasi numerik dari target.

## SUMMARY Hubungan Antar Variabel dan Proses Perbaikan

## Summary Detail Analisis Hubungan Antar Variabel

Analisis hubungan antar variabel dilakukan mulai dari bagian Exploratory Data Analysis lanjutan, khususnya bagian melihat hubungan antar kolom. Tujuan utama dari tahap ini adalah untuk memahami hubungan logis antar fitur, menemukan nilai yang tidak konsisten, menentukan kolom mana yang perlu diperbaiki, serta mendapatkan ide feature engineering untuk tahap modeling.

Data utama yang digunakan pada tahap ini adalah `df`, yaitu dataframe yang sudah melalui proses cleaning awal. Pada awal bagian ini dibuat salinan `eda = df.copy()` untuk membantu proses analisis hubungan antar variabel. Namun, setelah beberapa proses repair dilakukan, analisis final sebaiknya menggunakan `df` karena `df` merupakan dataframe utama yang sudah diperbaiki.

### 1. Setup EDA Hubungan Antar Kolom

- Pada awal analisis, dibuat dataframe `eda` dari `df`.
- Shape data pada tahap ini adalah `(25000, 30)`.
- Jumlah `Customer_ID` unik adalah 11.254.
- Total missing value pada data adalah 17.569.
- Tujuan setup ini adalah membuat salinan data untuk melakukan EDA hubungan antar variabel tanpa langsung mengubah dataframe utama pada setiap pengecekan awal.
- Karena satu `Customer_ID` dapat muncul pada beberapa bulan berbeda, riwayat customer yang sama dapat digunakan sebagai referensi untuk mengecek dan memperbaiki nilai yang tidak konsisten.

Kesimpulan:

- Dataset memiliki 25.000 baris dan 30 kolom pada tahap EDA lanjutan.
- Struktur data yang memiliki customer berulang membuat analisis berbasis `Customer_ID` penting untuk proses validasi dan repair data.

### 2. Hubungan `Annual_Income` dengan `Monthly_Inhand_Salary`

#### 2.1 Tujuan Analisis Income dan Salary

- Kolom yang dianalisis adalah `Annual_Income` dan `Monthly_Inhand_Salary`.
- `Annual_Income` menunjukkan pendapatan tahunan customer.
- `Monthly_Inhand_Salary` menunjukkan pendapatan bulanan yang diterima customer.
- Secara logika, `Monthly_Inhand_Salary` seharusnya memiliki hubungan dengan `Annual_Income / 12`.
- Analisis ini dilakukan untuk mengecek apakah ada nilai `Annual_Income` yang terlalu besar dan tidak sesuai dengan salary bulanan customer.

#### 2.2 Kondisi Missing Value Awal

- Missing value pada `Annual_Income` sebelum repair adalah 0.
- Missing value pada `Monthly_Inhand_Salary` adalah 3.836.
- Artinya, `Annual_Income` lengkap, tetapi `Monthly_Inhand_Salary` memiliki missing value cukup banyak.

Kesimpulan:

- Karena `Annual_Income` tidak missing tetapi memiliki kemungkinan nilai ekstrem, kolom ini perlu dicek melalui hubungan dengan `Monthly_Inhand_Salary`.
- `Monthly_Inhand_Salary` dapat digunakan sebagai pembanding karena distribusinya lebih stabil.

#### 2.3 Distribusi `Annual_Income`

- Jumlah data valid `Annual_Income` adalah 25.000.
- Median `Annual_Income` sekitar 37.313,32.
- P75 sekitar 72.712,20.
- P90 sekitar 114.015,36.
- P95 sekitar 134.928,36.
- P99 sekitar 179.078,28.
- P99.5 melonjak sangat tinggi menjadi sekitar 11.469.299,38.
- Nilai maksimum mencapai 24.160.009.

Kesimpulan:

- Mayoritas data `Annual_Income` masih berada pada rentang wajar.
- Namun, terjadi lonjakan ekstrem setelah P99.
- Perbedaan sangat besar antara P99 dan P99.5 menunjukkan adanya outlier ekstrem pada `Annual_Income`.

#### 2.4 Percentile Jump `Annual_Income`

- Percentile jump digunakan untuk melihat perubahan nilai antar percentile.
- Kenaikan dari P98 ke P99 masih relatif wajar.
- Kenaikan dari P99 ke P99.5 sangat besar.
- Kenaikan dari P99.5 ke maksimum juga masih sangat besar.

Kesimpulan:

- Titik masalah utama berada setelah P99.
- Nilai ekstrem tidak muncul secara bertahap, tetapi melonjak tiba-tiba.
- Ini menjadi dasar awal bahwa nilai `Annual_Income` di atas sekitar 180.000 sampai 200.000 perlu dicek sebagai outlier ekstrem.

#### 2.5 Distribusi `Monthly_Inhand_Salary`

- Jumlah data valid `Monthly_Inhand_Salary` adalah 21.164.
- Median `Monthly_Inhand_Salary` sekitar 3.071,42.
- P75 sekitar 5.941,09.
- P90 sekitar 9.137,98.
- P95 sekitar 10.872,25.
- P99 sekitar 13.943,25.
- P99.5 sekitar 14.490,78.
- Nilai maksimum sekitar 15.204,63.

Kesimpulan:

- Distribusi `Monthly_Inhand_Salary` terlihat jauh lebih stabil dibandingkan `Annual_Income`.
- Tidak ada lonjakan ekstrem seperti pada `Annual_Income`.
- Ini menunjukkan bahwa masalah kemungkinan besar ada pada `Annual_Income`, bukan pada `Monthly_Inhand_Salary`.

#### 2.6 Membuat Kolom Hubungan Income-Salary

Dibuat beberapa kolom turunan untuk mengecek konsistensi:

- `Expected_Monthly_Salary`, yaitu `Annual_Income / 12`.
- `Salary_Ratio`, yaitu perbandingan `Monthly_Inhand_Salary` terhadap `Expected_Monthly_Salary`.
- `Salary_Difference_Percentage`, yaitu persentase selisih antara salary aktual dan expected salary.

Tujuan:

- Mengecek apakah annual income dan monthly salary saling konsisten.
- Jika `Salary_Ratio` sangat kecil, berarti annual income terlalu besar dibanding monthly salary.
- Jika `Salary_Difference_Percentage` sangat tinggi, berarti hubungan income-salary tidak konsisten.

#### 2.7 Ringkasan Hubungan Income-Salary

- Jumlah data valid hubungan income-salary adalah 21.164 baris.
- Median `Salary_Ratio` sekitar 0,9976.
- Median `Salary_Difference_Percentage` sekitar 4,14%.
- Nilai minimum `Salary_Ratio` sekitar 0,0004.

Kesimpulan:

- Mayoritas data income-salary masih konsisten.
- Namun, nilai minimum `Salary_Ratio` sangat kecil dan tidak masuk akal.
- Ini menunjukkan adanya sebagian kecil data annual income yang terlalu ekstrem.

#### 2.8 Cek Kasus Paling Tidak Konsisten

- Ditampilkan baris dengan `Salary_Difference_Percentage` paling tinggi.
- Beberapa customer memiliki `Annual_Income` jutaan sampai puluhan juta.
- Namun, `Monthly_Inhand_Salary` hanya ratusan sampai ribuan.
- Contoh: customer dengan `Annual_Income` lebih dari 10 juta tetapi monthly salary hanya sekitar 300 sampai 700.

Kesimpulan:

- Kasus ekstrem menunjukkan `Annual_Income` terlalu besar dan tidak konsisten dengan salary.
- Nilai ekstrem ini lebih kuat dianggap sebagai error pada `Annual_Income`.

#### 2.9 Cek Riwayat Customer pada Kasus Ekstrem

- Riwayat `Customer_ID` yang sama dicek untuk melihat apakah nilai ekstrem konsisten atau hanya muncul pada satu bulan tertentu.
- Pada beberapa customer, `Annual_Income` normal pada bulan lain, tetapi tiba-tiba sangat besar pada satu bulan.
- `Monthly_Inhand_Salary` cenderung tetap stabil.

Kesimpulan:

- Nilai ekstrem `Annual_Income` muncul sebagai spike pada bulan tertentu.
- Riwayat customer menunjukkan bahwa nilai normal customer dapat digunakan sebagai referensi repair.
- Ini memperkuat keputusan bahwa nilai ekstrem sebaiknya diperbaiki menggunakan median valid dari customer yang sama.

#### 2.10 Cek Jumlah Kasus Selisih Besar

Jumlah kasus berdasarkan threshold `Salary_Difference_Percentage`:

- Selisih > 20%: 1.398 baris.
- Selisih > 30%: 620 baris.
- Selisih > 50%: 188 baris.
- Selisih > 70%: 188 baris.
- Selisih > 90%: 187 baris.

Kesimpulan:

- Kasus yang benar-benar ekstrem adalah kelompok dengan selisih > 90%.
- Jumlahnya kecil dibanding total data valid.
- Repair sebaiknya dilakukan secara selektif pada nilai ekstrem, bukan seluruh kolom.

#### 2.11 Cek Threshold Annual Income Ekstrem

Jumlah data di atas beberapa threshold:

- `Annual_Income > 150000`: 749 baris.
- `Annual_Income > 180000`: 224 baris.
- `Annual_Income > 200000`: 224 baris.
- `Annual_Income > 500000`: 222 baris.
- `Annual_Income > 1000000`: 218 baris.
- `Annual_Income > 5000000`: 184 baris.

Kesimpulan:

- Jumlah data di atas 180.000 dan 200.000 sama-sama 224.
- Ini menunjukkan tidak ada nilai antara 180.000 dan 200.000.
- Threshold 200.000 aman digunakan sebagai batas outlier ekstrem.

### 3. Repair `Annual_Income`

#### 3.1 Kondisi Sebelum Repair

- `Annual_Income > 200000`: 224 baris.
- Missing `Annual_Income`: 0.
- Missing `Monthly_Inhand_Salary`: 3.836.

Keputusan repair:

- Nilai `Annual_Income > 200000` dianggap outlier ekstrem.
- Jika ada referensi valid dari customer yang sama, nilai diganti dengan median `Annual_Income` valid customer tersebut.
- Jika tidak ada referensi valid, nilai dijadikan missing value.

#### 3.2 Referensi Valid dari Customer yang Sama

- Annual income ekstrem yang bisa diperbaiki: 195 baris.
- Annual income ekstrem tanpa referensi valid: 29 baris.

Kesimpulan:

- Sebanyak 195 nilai ekstrem diganti dengan median valid customer.
- Sebanyak 29 nilai ekstrem dijadikan missing value.

#### 3.3 Audit Sebelum Repair

- Audit dibuat sebelum repair dilakukan.
- Audit menyimpan nilai lama, suggested value, dan action repair.
- Kolom audit berisi `ID`, `Customer_ID`, `Month`, `Annual_Income`, `Monthly_Inhand_Salary`, `Credit_Score`, `Suggested_Annual_Income`, dan `Repair_Action`.

Kesimpulan:

- Audit penting agar proses cleaning dapat ditelusuri kembali.
- Repair tidak dilakukan secara sembarangan karena setiap perubahan memiliki action yang jelas.

#### 3.4 Hasil Setelah Repair Annual Income

Setelah repair:

- `Annual_Income > 200000`: 0.
- Missing `Annual_Income`: 29.
- Count valid `Annual_Income`: 24.971.
- Mean `Annual_Income` sekitar 50.504,71.
- Median `Annual_Income` sekitar 36.857,22.
- P99 `Annual_Income` sekitar 167.787,16.
- Maksimum `Annual_Income` sekitar 179.987,28.

Kesimpulan:

- Nilai ekstrem annual income berhasil dihapus.
- Distribusi annual income menjadi jauh lebih masuk akal.
- Missing value bertambah 29 karena kasus tanpa referensi valid dijadikan missing.

#### 3.5 Cek Ulang Hubungan Income-Salary Setelah Repair

Setelah repair:

- Jumlah `Salary_Difference > 90%`: 0.
- Median `Salary_Ratio` sekitar 0,9982.
- Median `Salary_Difference_Percentage` sekitar 4,09%.
- P99 salary difference turun menjadi sekitar 35,23%.

Kesimpulan:

- Hubungan `Annual_Income` dan `Monthly_Inhand_Salary` menjadi jauh lebih konsisten.
- Kasus ekstrem dengan salary difference > 90% sudah hilang.
- Repair annual income berhasil.

### 4. Analisis Awal `Total_EMI_per_month`

#### 4.1 Distribusi Awal `Total_EMI_per_month`

- Count: 25.000.
- Mean sekitar 1.465,71.
- Median sekitar 68,25.
- P95 sekitar 446,26.
- P97 sekitar 8.705,24.
- P98 sekitar 32.164,44.
- P99 sekitar 58.579,07.
- Maksimum 82.331.

Kesimpulan:

- Mean jauh lebih besar daripada median.
- Nilai setelah P95 melonjak sangat tinggi.
- `Total_EMI_per_month` memiliki outlier ekstrem.

#### 4.2 Threshold EMI

Jumlah data di atas threshold:

- `Total_EMI_per_month > 500`: 1.138 baris.
- `Total_EMI_per_month > 1000`: 875 baris.
- `Total_EMI_per_month > 1500`: 829 baris.
- `Total_EMI_per_month > 2000`: 812 baris.
- `Total_EMI_per_month > 5000`: 784 baris.
- `Total_EMI_per_month > 10000`: 735 baris.

Kesimpulan:

- Threshold 2.000 digunakan sebagai batas investigasi outlier EMI.
- Batas ini tidak langsung berarti semua nilai di atas 2.000 pasti salah, tetapi digunakan untuk mendeteksi kelompok ekstrem.

#### 4.3 Visualisasi Distribusi EMI

- Distribusi penuh menunjukkan nilai ekstrem yang sangat jauh.
- Visualisasi zoom untuk `Total_EMI_per_month <= 2000` menunjukkan mayoritas data terkonsentrasi pada nilai rendah.

Kesimpulan:

- Mayoritas EMI berada pada rentang kecil.
- Nilai puluhan ribu tidak mengikuti pola mayoritas data.

### 5. Hubungan `Total_EMI_per_month` dengan Income

#### 5.1 Hubungan EMI dengan Salary dan Annual Income

- Setelah annual income diperbaiki, EMI dibandingkan dengan income.
- Sebelum repair EMI, masih terdapat nilai EMI ekstrem.
- Beberapa customer memiliki EMI puluhan ribu, padahal salary hanya ratusan atau ribuan.
- Rasio EMI terhadap salary pada beberapa kasus sangat tinggi.

Kesimpulan:

- EMI ekstrem tidak konsisten dengan kemampuan pembayaran customer.
- Perlu dicek lebih lanjut dengan jumlah loan dan riwayat customer.

#### 5.2 EMI Lebih Besar dari Gaji Bulanan

- Terdapat 687 baris di mana `Total_EMI_per_month > Monthly_Inhand_Salary`.

Kesimpulan:

- Kasus EMI lebih besar dari gaji bulanan menjadi indikator kuat untuk investigasi outlier EMI.
- Namun, repair tidak hanya berdasarkan EMI > salary, tetapi juga mempertimbangkan loan count dan riwayat customer.

### 6. Hubungan `Total_EMI_per_month` dengan `Num_of_Loan`

#### 6.1 Hubungan EMI dan Jumlah Loan

- Secara umum, semakin banyak loan, EMI cenderung meningkat.
- Namun, ditemukan EMI ekstrem pada customer dengan jumlah loan rendah.
- EMI tinggi dengan jumlah loan rendah mencurigakan dan perlu dicek melalui riwayat customer.

Kesimpulan:

- `Num_of_Loan` digunakan sebagai variabel pembanding untuk memvalidasi kewajaran EMI.
- EMI tinggi dengan jumlah loan rendah lebih kuat dicurigai sebagai error.

#### 6.2 High EMI dengan Low Number of Loans

- Data difilter untuk `Total_EMI_per_month > 2000` dan `Num_of_Loan <= 2`.
- Jumlah high EMI low loan: 200 baris.
- Dari 200 baris tersebut, 177 memiliki referensi EMI valid dari customer yang sama.
- Sebanyak 23 tidak memiliki referensi valid.

Kesimpulan:

- Mayoritas high EMI low loan dapat diperbaiki menggunakan median EMI valid dari customer yang sama.
- Jika tidak ada referensi valid, nilai lebih aman dijadikan missing.

#### 6.3 Riwayat Customer High EMI Low Loan

- Riwayat customer menunjukkan EMI ekstrem sering muncul hanya pada satu bulan.
- Pada bulan lain, EMI customer jauh lebih kecil dan konsisten.
- Income dan loan count relatif konsisten.

Kesimpulan:

- EMI ekstrem cenderung merupakan spike error.
- Riwayat customer mendukung penggunaan median valid sebagai referensi repair.

#### 6.4 Kasus `Num_of_Loan = 0` tetapi EMI Positif

- Terdapat 211 baris dengan `Num_of_Loan = 0` tetapi `Total_EMI_per_month > 0`.
- Pada kelompok `Num_of_Loan = 0`, median EMI adalah 0.
- P75 dan P90 juga masih 0.
- Namun, P97 dan P99 melonjak sangat tinggi.

Kesimpulan:

- Mayoritas customer tanpa loan memang memiliki EMI 0.
- EMI positif pada customer tanpa loan adalah anomali.
- EMI tinggi pada customer tanpa loan merupakan indikasi error kuat.

#### 6.5 Cek Apakah yang Salah `Num_of_Loan` atau `Total_EMI_per_month`

- Dicek apakah customer dengan no loan tetapi EMI positif memiliki bukti loan dari `Type_of_Loan` atau riwayat customer.
- Tidak ada bukti kuat bahwa `Num_of_Loan` yang salah.
- Masalah lebih kuat berada pada `Total_EMI_per_month`.

Kesimpulan:

- Untuk kasus no-loan yang kuat, EMI dapat diubah menjadi 0.
- Untuk high EMI yang punya referensi customer, EMI diganti median valid.
- Untuk high EMI tanpa referensi, EMI dijadikan missing.

### 7. Repair `Total_EMI_per_month`

#### 7.1 Kondisi Sebelum Repair

- `Total_EMI_per_month > 2000`: 812 baris.
- Missing `Total_EMI_per_month`: 0.
- `Num_of_Loan = 0` tetapi EMI > 0: 211 baris.

#### 7.2 Action Repair EMI

- Set EMI to 0: 120 baris.
- Replace high EMI dengan median customer: 609 baris.
- High EMI tanpa referensi valid menjadi missing: 89 baris.

Kesimpulan:

- Repair dilakukan dengan tiga pendekatan:
  - no-loan strong error diubah menjadi 0;
  - high EMI dengan referensi customer diganti median valid;
  - high EMI tanpa referensi dijadikan missing.

#### 7.3 Hasil Setelah Repair EMI

Setelah repair:

- `Total_EMI_per_month > 2000`: 0.
- Missing `Total_EMI_per_month`: 89.
- Count valid: 24.911.
- Mean: 106,34.
- Median: 65,44.
- P75: 145,49.
- P90: 249,10.
- P95: 324,47.
- P99: 577,28.
- Maksimum: 1.950.

Kesimpulan:

- Outlier ekstrem EMI berhasil dihapus.
- Distribusi EMI menjadi jauh lebih masuk akal.
- Nilai maksimum setelah repair berada di bawah threshold 2.000.

#### 7.4 Cek Ulang EMI Setelah Repair

- Setelah repair, jumlah EMI lebih besar dari gaji bulanan tersisa 5 kasus.
- Median `EMI_to_Monthly_Salary_Ratio` sekitar 0,0244.
- P95 rasio sekitar 0,0733.
- P99 rasio sekitar 0,1302.
- Maksimum rasio sekitar 1,9792.

Kesimpulan:

- Setelah repair, mayoritas EMI sudah wajar dibandingkan salary.
- Sisa 5 kasus tidak lagi ekstrem.
- Karena sisa kasus memiliki loan dan type of loan, tidak dilakukan repair agresif tambahan.

### 8. Hubungan `Type_of_Loan` dengan `Num_of_Loan`

#### 8.1 Parse Jumlah Loan dari `Type_of_Loan`

- Kolom `Type_of_Loan` diparse untuk menghitung jumlah jenis loan.
- Hasil parsed loan count dibandingkan dengan `Num_of_Loan`.
- Jumlah data valid untuk perbandingan: 21.992.
- Rata-rata `Num_of_Loan` dan parsed loan count sama, yaitu sekitar 3,9761.
- `Loan_Count_Difference` semuanya 0.

Kesimpulan:

- `Num_of_Loan` dan jumlah loan yang diparse dari `Type_of_Loan` konsisten.
- Tidak ada mismatch pada data valid.

#### 8.2 Missing `Type_of_Loan` Berdasarkan `Num_of_Loan`

- Jika `Num_of_Loan = 0`, maka `Type_of_Loan` missing pada 2.826 baris.
- Jika `Num_of_Loan` 1 sampai 9, `Type_of_Loan` terisi.
- Terdapat 26 baris di mana `Num_of_Loan` missing dan `Type_of_Loan` juga missing.

Kesimpulan:

- Missing pada `Type_of_Loan` sebagian besar logis karena customer tidak memiliki loan.
- `Type_of_Loan` missing tidak selalu berarti data rusak.
- Untuk modeling, `Type_of_Loan` perlu diproses hati-hati karena bentuknya multi-label text.

### 9. Hubungan `Outstanding_Debt` dengan Pendapatan

#### 9.1 Analisis Debt-Income

- Dibuat rasio `Debt_to_Annual_Income_Ratio`.
- Rasio ini menunjukkan beban utang relatif terhadap pendapatan tahunan.
- `Outstanding_Debt` tidak menunjukkan outlier ekstrem seperti annual income atau EMI.
- Maksimum `Outstanding_Debt` masih sekitar 4.997,10.
- Tidak ada kasus `Outstanding_Debt > Annual_Income`.

Kesimpulan:

- `Outstanding_Debt` relatif aman dan tidak perlu repair berdasarkan hubungan dengan income.
- Rasio debt terhadap income berguna untuk melihat beban utang secara relatif.

#### 9.2 Debt Ratio Berdasarkan `Credit_Score`

- Kelas Poor memiliki `Debt_to_Annual_Income_Ratio` paling tinggi.
- Kelas Standard berada di tengah.
- Kelas Good memiliki rasio paling rendah.

Kesimpulan:

- `Debt_to_Annual_Income_Ratio` berpotensi menjadi fitur engineered yang kuat.
- Fitur ini menangkap beban utang relatif, bukan hanya nominal debt.

### 10. Hubungan `Credit_Utilization_Ratio` dengan `Outstanding_Debt` dan `Credit_Score`

#### 10.1 Korelasi Credit Utilization dan Outstanding Debt

- Korelasi Spearman antara `Credit_Utilization_Ratio` dan `Outstanding_Debt` sekitar -0,0684.
- Nilai ini sangat lemah.

Kesimpulan:

- `Credit_Utilization_Ratio` dan `Outstanding_Debt` tidak membawa informasi yang sama.
- Keduanya tetap dapat dipertahankan karena tidak duplikatif.

#### 10.2 Credit Utilization Berdasarkan Credit Score

Median berdasarkan `Credit_Score`:

- Poor:
  - `Credit_Utilization_Ratio`: sekitar 31,97.
  - `Outstanding_Debt`: sekitar 1.960,88.
  - `Debt_to_Annual_Income_Ratio`: sekitar 0,0638.
- Standard:
  - `Credit_Utilization_Ratio`: sekitar 32,33.
  - `Outstanding_Debt`: sekitar 994,73.
  - `Debt_to_Annual_Income_Ratio`: sekitar 0,0219.
- Good:
  - `Credit_Utilization_Ratio`: sekitar 32,79.
  - `Outstanding_Debt`: sekitar 737,50.
  - `Debt_to_Annual_Income_Ratio`: sekitar 0,0119.

Kesimpulan:

- `Credit_Utilization_Ratio` tidak berbeda jauh antar kelas.
- `Outstanding_Debt` dan `Debt_to_Annual_Income_Ratio` jauh lebih membedakan kelas.
- Customer Poor memiliki debt dan debt-to-income ratio lebih tinggi.

### 11. Hubungan `Credit_History_Age` dengan `Age`

#### 11.1 Tujuan Validasi

- `Age` menunjukkan umur customer dalam tahun.
- `Credit_History_Age` menunjukkan lama riwayat kredit dalam bulan.
- Secara logika, credit history tidak boleh lebih panjang daripada umur customer.
- Dibuat `Credit_History_Years` dan `Estimated_Age_When_Credit_Started`.

Rumus:

- `Credit_History_Years = Credit_History_Age / 12`
- `Estimated_Age_When_Credit_Started = Age - Credit_History_Years`

#### 11.2 Output Awal Hubungan Credit History dan Age

- Data valid: 22.659 baris.
- Median `Age`: 33.
- Median `Credit_History_Age`: 219 bulan atau 18,25 tahun.
- Minimum `Estimated_Age_When_Credit_Started`: -17,83.

Kesimpulan:

- Ada kasus tidak logis pada hubungan age dan credit history.
- Nilai negatif berarti customer seolah-olah memiliki credit history sebelum lahir.

#### 11.3 Cek Impossible Credit History

- Jumlah data valid credit history-age: 22.659.
- Credit history lebih panjang dari umur: 2.745.
- Estimasi mulai kredit sebelum umur 10: 8.102.

Kesimpulan:

- Fokus utama cleaning adalah kasus yang pasti tidak mungkin, yaitu `Credit_History_Age > Age * 12`.
- Kasus sebelum umur 10 hanya dicatat sebagai suspicious, tetapi tidak digunakan sebagai aturan repair utama.

### 12. Repair `Credit_History_Age`

#### 12.1 Kondisi Sebelum Repair

- `Credit_History_Age` lebih panjang dari `Age`: 2.745 baris.
- Missing `Credit_History_Age`: 2.240.

Keputusan repair:

- Jika customer memiliki referensi `Credit_History_Age` valid dari bulan lain, gunakan median valid customer tersebut.
- Jika tidak ada referensi valid, jadikan missing value.

#### 12.2 Referensi Valid dari Customer yang Sama

- Invalid credit history yang bisa diperbaiki: 62 baris.
- Invalid credit history tanpa referensi valid: 2.683 baris.

Kesimpulan:

- Hanya sedikit kasus yang memiliki referensi valid.
- Mayoritas kasus tidak dapat diperbaiki secara aman, sehingga dijadikan missing value.

#### 12.3 Apply Repair dan Final Fix

Setelah repair utama:

- `Credit_History_Age` lebih panjang dari `Age`: 33 baris.
- Missing `Credit_History_Age`: 4.923.

Setelah final fix:

- `Credit_History_Age` lebih panjang dari `Age`: 0.
- Missing `Credit_History_Age`: 4.956.

Kesimpulan:

- Semua kasus tidak logis berhasil dihilangkan.
- Missing value bertambah karena kasus tanpa referensi valid dijadikan missing.

#### 12.4 Cek Umur di Bawah 21 Tahun

- Jumlah baris dengan `Age < 21`: 3.566.
- Jumlah unique `Customer_ID` dengan `Age < 21`: 1.603.
- Distribusi umur:
  - Age 14: 326 baris.
  - Age 15: 407 baris.
  - Age 16: 362 baris.
  - Age 17: 403 baris.
  - Age 18: 620 baris.
  - Age 19: 710 baris.
  - Age 20: 738 baris.

Kesimpulan:

- Data usia muda cukup banyak.
- `Age < 21` tidak langsung dihapus karena dataset tidak menjelaskan jenis kredit, negara, regulasi, atau status pemegang kredit.
- Cleaning hanya dilakukan pada kondisi yang pasti tidak mungkin.

#### 12.5 Cek Khusus `Age = 14`

- Jumlah baris `Age = 14`: 326.
- Jumlah unique `Customer_ID Age = 14`: 155.
- Missing `Credit_History_Age` pada Age 14: 152.
- `Credit_History_Age > Age` pada Age 14: 0.
- Median `Annual_Income` Age 14 sekitar 32.198,23.
- Median `Monthly_Inhand_Salary` sekitar 2.787,81.
- Median `Outstanding_Debt` sekitar 1.963,24.
- Median `Total_EMI_per_month` sekitar 79,11.
- Median `Credit_Utilization_Ratio` sekitar 31,67.

Kesimpulan:

- Age 14 secara konteks mencurigakan karena memiliki data finansial lengkap.
- Namun, setelah repair tidak ada lagi kasus credit history lebih panjang dari umur.
- Age 14 tetap dipertahankan, tetapi dapat menjadi ide fitur age group atau `Is_Under_21`.

### 13. Hubungan `Delay_from_due_date`, `Num_of_Delayed_Payment`, dan `Payment_of_Min_Amount`

#### 13.1 Korelasi Delay Features

- Korelasi Spearman antara `Delay_from_due_date` dan `Num_of_Delayed_Payment`: 0,5536.
- Ini menunjukkan hubungan positif sedang.
- Semakin besar delay dari due date, jumlah delayed payment juga cenderung meningkat.

Kesimpulan:

- Kedua fitur saling berhubungan tetapi tidak duplikatif.
- `Delay_from_due_date` mengukur durasi keterlambatan.
- `Num_of_Delayed_Payment` mengukur frekuensi keterlambatan.
- Keduanya layak dipertahankan.

#### 13.2 Payment Delay Berdasarkan `Payment_of_Min_Amount`

Median berdasarkan kategori:

- `NM`:
  - `Delay_from_due_date`: 18.
  - `Num_of_Delayed_Payment`: 14.
- `No`:
  - `Delay_from_due_date`: 11.
  - `Num_of_Delayed_Payment`: 9.
- `Yes`:
  - `Delay_from_due_date`: 24.
  - `Num_of_Delayed_Payment`: 17.

Kesimpulan:

- Customer dengan `Payment_of_Min_Amount = Yes` cenderung memiliki perilaku pembayaran lebih bermasalah.
- Kategori `NM` memiliki pola sendiri, bukan sama dengan `No`.
- `Payment_of_Min_Amount` relevan sebagai fitur kategorikal.

#### 13.3 Payment Delay Berdasarkan `Credit_Score`

Median berdasarkan `Credit_Score`:

- Poor:
  - `Delay_from_due_date`: 27.
  - `Num_of_Delayed_Payment`: 17.
- Standard:
  - `Delay_from_due_date`: 18.
  - `Num_of_Delayed_Payment`: 14.
- Good:
  - `Delay_from_due_date`: 10.
  - `Num_of_Delayed_Payment`: 8.

Kesimpulan:

- Poor memiliki delay paling tinggi.
- Standard berada di tengah.
- Good memiliki delay paling rendah.
- Fitur delay sangat relevan untuk modeling.

### 14. Hubungan `Amount_invested_monthly`, `Monthly_Balance`, Salary, dan EMI

#### 14.1 Fitur Turunan Cashflow

Dibuat fitur turunan:

- `Investment_to_Salary_Ratio`
- `Balance_to_Salary_Ratio`
- `Estimated_Remaining_Balance`
- `Balance_Difference`

Tujuan:

- Melihat proporsi investasi terhadap salary.
- Melihat proporsi balance terhadap salary.
- Mengecek apakah balance konsisten dengan salary, EMI, dan investment.

#### 14.2 Ringkasan Cashflow

- Count valid: 19.822.
- Median `Monthly_Inhand_Salary`: 3.032,51.
- Median `Total_EMI_per_month`: 67,41.
- Median `Amount_invested_monthly`: 126,21.
- Median `Monthly_Balance`: 335,46.
- Median `Investment_to_Salary_Ratio`: 0,0438.
- Median `Balance_to_Salary_Ratio`: 0,1172.

Kesimpulan:

- Sebagian besar customer menginvestasikan sekitar 4,38% dari gaji bulanan.
- Monthly balance sekitar 11,72% dari gaji.
- Rasio lebih informatif dibanding nilai nominal.
- Fitur cashflow harus dibuat ulang di pipeline menggunakan `df` final.

#### 14.3 Investment Lebih Besar dari Salary

- Jumlah data valid cashflow: 19.822.
- Jumlah `Amount_invested_monthly > Monthly_Inhand_Salary`: 0.

Kesimpulan:

- Tidak ada kasus investment lebih besar dari salary.
- `Amount_invested_monthly` terlihat aman berdasarkan hubungan dengan salary.

#### 14.4 Visualisasi Investment Ratio by Credit Score

- Boxplot menunjukkan `Investment_to_Salary_Ratio` antar Poor, Standard, dan Good cukup mirip.
- Poor sedikit lebih tinggi, tetapi perbedaannya tidak besar.
- Rasio umumnya berada di bawah 0,15.

Kesimpulan:

- `Investment_to_Salary_Ratio` bukan fitur pembeda utama.
- Namun, fitur ini tetap dapat dicoba sebagai fitur tambahan.

### 15. Hubungan `Payment_Behaviour` dengan Investment, Balance, dan Credit Score

#### 15.1 Payment Behaviour dengan Investment dan Balance

- `High_spent_Large_value_payments` memiliki `Monthly_Balance` tertinggi dan `Investment_to_Salary_Ratio` terendah.
- `Low_spent_Small_value_payments` memiliki `Monthly_Balance` terendah dan `Investment_to_Salary_Ratio` tertinggi.
- Kategori lain berada di antara dua pola tersebut.

Kesimpulan:

- `Payment_Behaviour` berhubungan dengan pola investasi dan balance.
- Fitur ini relevan sebagai fitur kategorikal.

#### 15.2 Distribusi `Credit_Score` Berdasarkan `Payment_Behaviour`

- `High_spent_Large_value_payments` memiliki proporsi Good sekitar 23,21% dan Poor sekitar 21,22%.
- `Low_spent_Small_value_payments` memiliki proporsi Poor paling tinggi, sekitar 35,91%.
- Missing pada `Payment_Behaviour` memiliki proporsi Poor sekitar 32,06%.

Kesimpulan:

- `Payment_Behaviour` memiliki pola terhadap target.
- Missing value juga memiliki pola target.
- Missing value sebaiknya diperlakukan sebagai kategori khusus.

### 16. Hubungan `Credit_Mix` dengan Debt, Interest, dan Credit Score

#### 16.1 `Credit_Mix` dengan Fitur Risiko Kredit

- `Credit_Mix = Bad` memiliki median `Outstanding_Debt`, `Debt_to_Annual_Income_Ratio`, `Interest_Rate`, dan `Num_Credit_Inquiries` paling tinggi.
- `Credit_Mix = Good` memiliki nilai risiko paling rendah.
- `Credit_Mix = Standard` berada di tengah.
- `Credit_Utilization_Ratio` tetap mirip antar kategori.

Kesimpulan:

- `Credit_Mix` sangat kuat hubungannya dengan indikator risiko kredit.
- Fitur ini penting untuk dipertahankan.

#### 16.2 Distribusi `Credit_Score` Berdasarkan `Credit_Mix`

- Bad:
  - Good: 1,40%.
  - Poor: 60,32%.
  - Standard: 38,28%.
- Good:
  - Good: 49,24%.
  - Poor: 15,48%.
  - Standard: 35,28%.
- Missing:
  - Good: 20,06%.
  - Poor: 29,37%.
  - Standard: 50,57%.
- Standard:
  - Good: 5,09%.
  - Poor: 21,91%.
  - Standard: 73,00%.

Kesimpulan:

- `Credit_Mix` sangat kuat hubungannya dengan `Credit_Score`.
- `Credit_Mix = Bad` didominasi Poor.
- `Credit_Mix = Good` memiliki proporsi Good paling tinggi.
- Missing value pada `Credit_Mix` sebaiknya ditangani eksplisit.

### 17. Hubungan `Num_Bank_Accounts`, `Num_Credit_Card`, `Interest_Rate`, dan `Num_Credit_Inquiries`

#### 17.1 Median Berdasarkan `Credit_Score`

- Poor:
  - `Num_Bank_Accounts`: 7.
  - `Num_Credit_Card`: 7.
  - `Interest_Rate`: 21.
  - `Num_Credit_Inquiries`: 8.
- Standard:
  - `Num_Bank_Accounts`: 5.
  - `Num_Credit_Card`: 5.
  - `Interest_Rate`: 13.
  - `Num_Credit_Inquiries`: 5.
- Good:
  - `Num_Bank_Accounts`: 3.
  - `Num_Credit_Card`: 4.
  - `Interest_Rate`: 7.
  - `Num_Credit_Inquiries`: 3.

Kesimpulan:

- Semakin buruk credit score, semakin tinggi jumlah akun, kartu kredit, interest rate, dan inquiries.
- `Interest_Rate` terlihat sangat kuat sebagai fitur prediktif.

#### 17.2 Korelasi Antar Account Features

- `Num_Bank_Accounts` dengan `Num_Credit_Card`: 0,4218.
- `Num_Bank_Accounts` dengan `Interest_Rate`: 0,5947.
- `Num_Bank_Accounts` dengan `Num_Credit_Inquiries`: 0,5277.
- `Num_Credit_Card` dengan `Interest_Rate`: 0,4642.
- `Num_Credit_Card` dengan `Num_Credit_Inquiries`: 0,4383.
- `Interest_Rate` dengan `Num_Credit_Inquiries`: 0,6326.

Kesimpulan:

- Semua fitur saling berkorelasi positif.
- Korelasi tertinggi adalah `Interest_Rate` dengan `Num_Credit_Inquiries`.
- Korelasi belum mendekati 1, sehingga fitur tidak sepenuhnya duplikatif.

#### 17.3 Visualisasi Interest Rate by Credit Score

- Poor memiliki interest rate tertinggi.
- Standard berada di tengah.
- Good memiliki interest rate terendah.
- Median Poor sekitar 21.
- Median Standard sekitar 13.
- Median Good sekitar 7.

Kesimpulan:

- `Interest_Rate` sangat relevan terhadap `Credit_Score`.
- Fitur ini wajib dipertahankan dalam modeling.

### 18. Heatmap Korelasi Numerik

#### 18.1 Setup Heatmap

- Jumlah kolom numerik: 18.
- Shape data korelasi: `(25000, 18)`.
- Korelasi yang digunakan adalah Spearman.
- `Credit_Score_Ordinal` dibuat hanya untuk kebutuhan EDA korelasi.

Catatan:

- `Credit_Score_Ordinal` tidak boleh digunakan sebagai fitur model karena berasal dari target.

#### 18.2 Insight Heatmap

- `Annual_Income` dan `Monthly_Inhand_Salary` memiliki korelasi sangat tinggi, sekitar 0,99.
- Fitur risiko seperti `Interest_Rate`, `Num_Credit_Inquiries`, `Outstanding_Debt`, `Delay_from_due_date`, dan `Num_of_Delayed_Payment` saling berkorelasi positif.
- `Credit_History_Age` berkorelasi negatif dengan beberapa fitur risiko.
- Customer dengan riwayat kredit lebih panjang cenderung memiliki risiko lebih rendah.

Kesimpulan:

- Beberapa fitur numerik saling berhubungan tetapi tetap membawa informasi penting.
- Untuk model linear, korelasi tinggi perlu diperhatikan.
- Untuk tree-based model, fitur-fitur ini masih dapat diuji.

#### 18.3 Korelasi Numerik dengan `Credit_Score_Ordinal`

Urutan korelasi terbesar terhadap `Credit_Score_Ordinal`:

- `Interest_Rate`: -0,4870.
- `Num_Credit_Inquiries`: -0,4430.
- `Delay_from_due_date`: -0,4421.
- `Outstanding_Debt`: -0,4352.
- `Credit_History_Age`: 0,3978.
- `Num_Bank_Accounts`: -0,3939.
- `Num_Credit_Card`: -0,3839.
- `Num_of_Delayed_Payment`: -0,3678.
- `Num_of_Loan`: -0,3561.
- `Monthly_Balance`: 0,2301.
- `Annual_Income`: 0,2142.
- `Monthly_Inhand_Salary`: 0,2074.
- `Changed_Credit_Limit`: -0,1676.
- `Age`: 0,1457.
- `Amount_invested_monthly`: 0,1288.
- `Total_EMI_per_month`: -0,0820.
- `Credit_Utilization_Ratio`: 0,0494.

Kesimpulan:

- Fitur risiko kredit memiliki hubungan paling kuat dengan target.
- `Interest_Rate` adalah fitur numerik dengan korelasi paling kuat.
- `Credit_Utilization_Ratio` memiliki hubungan paling lemah.
- `Credit_Score_Ordinal` hanya digunakan untuk EDA, bukan untuk training model.

## Ide yang Didapat untuk Tahap Modeling

### 1. Fitur Risiko Kredit Utama Perlu Dipertahankan

Fitur yang paling kuat hubungannya dengan `Credit_Score`:

- `Interest_Rate`
- `Num_Credit_Inquiries`
- `Delay_from_due_date`
- `Outstanding_Debt`
- `Credit_History_Age`
- `Num_Bank_Accounts`
- `Num_Credit_Card`
- `Num_of_Delayed_Payment`
- `Num_of_Loan`

Alasan:

- Fitur-fitur ini memiliki korelasi cukup kuat dengan `Credit_Score_Ordinal`.
- Median fitur-fitur ini berbeda jelas pada kelas Poor, Standard, dan Good.
- Fitur ini berhubungan langsung dengan risiko kredit dan perilaku pembayaran customer.

### 2. Fitur Kategorikal Penting Perlu Dipertahankan

Fitur kategorikal yang penting:

- `Credit_Mix`
- `Payment_Behaviour`
- `Payment_of_Min_Amount`
- `Occupation`

Alasan:

- `Credit_Mix` memiliki hubungan sangat kuat dengan `Credit_Score`.
- `Payment_Behaviour` menunjukkan distribusi target yang berbeda pada setiap kategori.
- `Payment_of_Min_Amount` berhubungan dengan delay payment.
- `Occupation` tetap dapat digunakan sebagai fitur kategorikal umum.

### 3. Missing Value Kategorikal Sebaiknya Dibuat Kategori Khusus

Fitur yang missing-nya memiliki pola:

- `Credit_Mix`
- `Payment_Behaviour`

Ide preprocessing:

- Missing value pada fitur kategorikal dapat diisi dengan kategori `Missing`.
- Jangan langsung menghapus baris missing karena distribusi target pada missing juga memiliki pola.

Alasan:

- Missing value tidak sepenuhnya acak.
- Missing pada `Credit_Mix` dan `Payment_Behaviour` tetap membawa informasi terhadap `Credit_Score`.

### 4. Fitur Turunan yang Layak Dibuat

Fitur engineered yang dapat dicoba:

- `Debt_to_Annual_Income_Ratio`
- `Debt_to_Monthly_Salary_Ratio`
- `Investment_to_Salary_Ratio`
- `Balance_to_Salary_Ratio`
- `Income_to_EMI_Ratio`
- `EMI_to_Monthly_Salary_Ratio`
- `Is_Under_21`
- `Age_Group`

Alasan:

- Rasio lebih informatif daripada nilai nominal.
- Dua customer dengan debt yang sama bisa memiliki risiko berbeda jika income berbeda.
- Age group dapat membantu model menangkap pola usia muda tanpa menghapus data.

### 5. Feature Engineering Harus Dilakukan di Pipeline

Fitur turunan sebaiknya dibuat dalam preprocessing pipeline, bukan hanya manual di EDA.

Alasan:

- Agar konsisten antara train, validation, test, dan inference.
- Menghindari data leakage.
- Memudahkan deployment ke Streamlit.
- Sesuai kebutuhan final project Model Deployment.

### 6. Perlu Hati-Hati dengan Fitur yang Sangat Berkorelasi

Fitur yang sangat berkorelasi:

- `Annual_Income`
- `Monthly_Inhand_Salary`

Korelasi keduanya sekitar 0,99.

Ide modeling:

- Untuk Logistic Regression, perlu memperhatikan redundansi fitur.
- Untuk model tree-based, keduanya masih bisa diuji.
- Bisa membandingkan performa model dengan menggunakan keduanya, menggunakan salah satu, atau menambahkan fitur rasio.

### 7. Model yang Cocok Dicoba

Model yang disarankan:

- Logistic Regression sebagai baseline.
- Random Forest sebagai model tree-based dasar.
- XGBoost sebagai model boosting utama.
- LightGBM jika diperbolehkan dan environment mendukung.

Alasan:

- Banyak fitur memiliki hubungan non-linear.
- Banyak fitur saling berinteraksi.
- Tree-based model lebih cocok untuk menangkap interaksi antara fitur risiko kredit, payment behaviour, credit mix, income, dan debt.

### 8. Strategi Preprocessing yang Disarankan

Untuk fitur numerik:

- Gunakan imputer untuk missing value.
- Gunakan scaling untuk Logistic Regression.
- Scaling tidak wajib untuk tree-based model, tetapi boleh tetap dibuat dalam pipeline terpisah.

Untuk fitur kategorikal:

- Missing value diisi dengan `Missing`.
- Gunakan One Hot Encoding.
- Pastikan kategori baru saat inference dapat ditangani dengan `handle_unknown="ignore"`.

Untuk fitur ID:

- `ID`, `Customer_ID`, `Name`, `SSN`, dan `Unnamed: 0` tidak digunakan sebagai fitur model.
- `Customer_ID` hanya digunakan untuk cleaning, validasi, dan split berbasis customer.

### 9. Strategi Split Data

Karena terdapat banyak `Customer_ID` yang berulang dalam beberapa bulan:

- Split sebaiknya dilakukan berdasarkan `Customer_ID`, bukan random row biasa.
- Tujuannya agar data customer yang sama tidak muncul di train dan test sekaligus.
- Ini penting untuk menghindari data leakage.

### 10. Kolom yang Tidak Boleh Masuk ke Model

Kolom yang tidak boleh menjadi fitur input:

- `Credit_Score_Ordinal`
- `ID`
- `Customer_ID`
- `Name`
- `SSN`
- `Unnamed: 0`

Alasan:

- `Credit_Score_Ordinal` berasal dari target.
- Kolom identitas tidak merepresentasikan pola kredit yang general.
- Memasukkan ID dapat menyebabkan model menghafal data, bukan belajar pola.

## Kesimpulan Akhir

Dari seluruh analisis hubungan antar variabel, fitur yang paling kuat untuk modeling adalah fitur yang berkaitan dengan risiko kredit, perilaku pembayaran, beban utang, riwayat kredit, interest rate, credit inquiries, dan credit mix. Beberapa fitur numerik seperti `Interest_Rate`, `Delay_from_due_date`, `Outstanding_Debt`, `Num_Credit_Inquiries`, dan `Credit_History_Age` menunjukkan hubungan yang jelas terhadap `Credit_Score`.

Selain fitur asli, beberapa fitur turunan seperti `Debt_to_Annual_Income_Ratio`, `Investment_to_Salary_Ratio`, `Balance_to_Salary_Ratio`, dan `EMI_to_Monthly_Salary_Ratio` dapat dicoba pada tahap feature engineering. Missing value pada fitur kategorikal juga sebaiknya dipertahankan sebagai kategori khusus karena memiliki pola terhadap target.

Insight ini menjadi dasar untuk tahap modeling, khususnya dalam menentukan fitur yang dipakai, fitur yang direkayasa, strategi preprocessing, jenis model yang dicoba, dan cara menghindari data leakage saat training dan deployment.


# E. Data Prepocessing(Penangganan Mising Value daan Ide untuk Pipeline nanti)

In [ ]:
df.info()

In [ ]:
missing_summary = missing_value_summary(df)

display(missing_summary)


### 1. Name

In [ ]:
show_missing_rows(
    df,
    "Name"
)

In [ ]:
# CHECK WHETHER MISSING NAME CAN BE FILLED FROM SAME CUSTOMER_ID

valid_name_reference = (
    df[df["Name"].notna()]
    .groupby("Customer_ID")["Name"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

missing_name_customer = df.loc[
    df["Name"].isna(),
    "Customer_ID"
]

suggested_name = missing_name_customer.map(valid_name_reference)

print("Missing Name total:", df["Name"].isna().sum())
print("Missing Name yang punya referensi dari Customer_ID sama:", suggested_name.notna().sum())
print("Missing Name tanpa referensi:", suggested_name.isna().sum())

Dari hasil pengecekan, terdapat 2.430 baris dengan nilai Name yang missing. Sebanyak 1.984 baris masih memiliki referensi nama dari Customer_ID yang sama, sedangkan 446 baris tidak memiliki referensi nama yang valid.

Namun, 446 baris tersebut tidak dihapus karena Name merupakan kolom identitas dan tidak digunakan sebagai fitur modeling. Menghapus baris hanya karena nama missing dapat membuang informasi penting dari fitur lain, seperti income, debt, credit history, payment behavior, dan target Credit_Score.

Oleh karena itu, missing value pada Name hanya dicatat sebagai bagian dari data quality check. Kolom Name akan dihapus sebelum modeling bersama kolom identitas lain seperti ID, Customer_ID, SSN, dan Unnamed: 0.

### 2. Age

In [ ]:
show_missing_rows(
    df,
    "Age"
)

Terdapaty 110 Baris yang memiliki missing value kolom age

In [ ]:
# CHECK AGE MISSING REFERENCE FROM SAME CUSTOMER_ID

valid_age_reference = (
    df[df["Age"].notna()]
    .groupby("Customer_ID")["Age"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

missing_age_indices = df[df["Age"].isna()].index

suggested_age = df.loc[
    missing_age_indices,
    "Customer_ID"
].map(valid_age_reference)

age_fillable_indices = suggested_age[
    suggested_age.notna()
].index

age_unresolved_indices = suggested_age[
    suggested_age.isna()
].index

print("Missing Age total:", df["Age"].isna().sum())
print("Missing Age yang punya referensi dari Customer_ID sama:", len(age_fillable_indices))
print("Missing Age tanpa referensi:", len(age_unresolved_indices))

Dari hasil pengecekan, terdapat 110 baris dengan nilai Age yang missing.

Sebelum menentukan cara penanganannya, dilakukan pengecekan apakah missing value pada Age dapat diisi menggunakan referensi dari Customer_ID yang sama. Tujuannya adalah untuk melihat apakah customer yang sama memiliki nilai umur valid pada bulan lain. Pendekatan ini lebih aman karena masih menggunakan riwayat customer yang sama, bukan menebak dari customer lain.

Namun, dari hasil pengecekan, seluruh 110 baris missing Age tidak memiliki referensi valid dari Customer_ID yang sama. Artinya, nilai Age yang missing tidak dapat diperbaiki secara aman menggunakan riwayat customer tersebut.

Karena tidak tersedia referensi valid dari customer yang sama, nilai missing pada Age tidak dipaksa untuk diisi secara manual pada tahap cleaning. Baris data juga tidak dihapus karena fitur lain pada baris tersebut masih memiliki informasi penting, seperti Annual_Income, Occupation, Interest_Rate, Outstanding_Debt, Credit_History_Age, dan target Credit_Score.

Keputusan cleaning untuk kolom Age adalah membiarkan 110 nilai tersebut sebagai missing value. Missing value ini nantinya akan ditangani pada tahap preprocessing pipeline menggunakan median imputer. Median dipilih karena lebih tahan terhadap outlier dibandingkan mean, sehingga lebih aman untuk mengisi missing value pada fitur numerik.

Pada tahap modeling, pengisian mediannya dilakukan di dalam preprocessing pipeline dan median hanya dihitung dari data train. Hal ini penting agar tidak terjadi data leakage dari data validation atau test ke proses training model.

### 3. Occupation

In [ ]:
show_missing_rows(
    df,
    "Occupation"
)

In [ ]:
# CHECK OCCUPATION REFERENCE FROM SAME CUSTOMER_ID

valid_occupation_reference = (
    df[df["Occupation"].notna()]
    .groupby("Customer_ID")["Occupation"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

missing_occupation_indices = df[df["Occupation"].isna()].index

suggested_occupation = df.loc[
    missing_occupation_indices,
    "Customer_ID"
].map(valid_occupation_reference)

occupation_fillable_indices = suggested_occupation[
    suggested_occupation.notna()
].index

occupation_unresolved_indices = suggested_occupation[
    suggested_occupation.isna()
].index

print("Missing Occupation total:", df["Occupation"].isna().sum())
print("Missing Occupation yang punya referensi dari Customer_ID sama:", len(occupation_fillable_indices))
print("Missing Occupation tanpa referensi:", len(occupation_unresolved_indices))

Dari hasil pengecekan, terdapat 259 baris dengan nilai Occupation yang missing.

Kolom ini masih relevan untuk modeling karena jenis pekerjaan dapat berkaitan dengan profil pendapatan, kemampuan pembayaran, dan risiko kredit customer. Oleh karena itu, baris dengan Occupation missing tidak dihapus, karena fitur lain pada baris tersebut masih memiliki informasi penting seperti Age, Annual_Income, Monthly_Inhand_Salary, Interest_Rate, Outstanding_Debt, Payment_Behaviour, dan target Credit_Score.

Untuk menangani missing value pada Occupation, pendekatan yang paling aman adalah menggunakan referensi dari Customer_ID yang sama jika tersedia. Jika customer yang sama memiliki nilai occupation valid pada bulan lain, maka nilai tersebut dapat digunakan untuk mengisi missing value karena masih berasal dari riwayat customer yang sama.

Namun, jika setelah pengecekan masih terdapat Occupation yang tidak memiliki referensi valid dari Customer_ID yang sama, nilai tersebut tidak dipaksa untuk diisi secara manual. Missing value tersebut lebih aman dibiarkan sebagai missing dan nantinya ditangani pada tahap preprocessing pipeline.

Pada tahap modeling, missing value pada Occupation sebaiknya diisi dengan kategori khusus seperti Missing. Hal ini dilakukan karena Occupation adalah fitur kategorikal, sehingga missing value dapat tetap dianggap sebagai salah satu informasi. Dengan pendekatan ini, data tidak dibuang secara berlebihan dan model tetap dapat mempelajari pola dari customer yang occupation-nya tidak tersedia.

Keputusan cleaning untuk kolom Occupation adalah tidak menghapus baris missing. Nilai yang bisa diperbaiki dari Customer_ID yang sama dapat diisi terlebih dahulu, sedangkan sisa missing value akan ditangani sebagai kategori Missing di preprocessing pipeline.

### 4. Annual_Income

In [ ]:
show_missing_rows(
    df,
    "Annual_Income"
)


In [ ]:
# CHECK ANNUAL_INCOME REPAIR USING MONTHLY_INHAND_SALARY

missing_annual_indices = df[df["Annual_Income"].isna()].index

annual_from_salary = df.loc[
    missing_annual_indices,
    "Monthly_Inhand_Salary"
] * 12

annual_fillable_from_salary = annual_from_salary[
    annual_from_salary.notna()
].index

annual_unresolved_indices = annual_from_salary[
    annual_from_salary.isna()
].index

print("Missing Annual_Income total:", df["Annual_Income"].isna().sum())
print("Annual_Income yang bisa diisi dari Monthly_Inhand_Salary x 12:", len(annual_fillable_from_salary))
print("Annual_Income yang tetap missing karena Monthly_Inhand_Salary juga missing:", len(annual_unresolved_indices))

In [ ]:
# REPAIR ANNUAL_INCOME USING MONTHLY_INHAND_SALARY x 12

df.loc[
    annual_fillable_from_salary,
    "Annual_Income"
] = annual_from_salary.loc[annual_fillable_from_salary]

print("Missing Annual_Income setelah repair:", df["Annual_Income"].isna().sum())
print("Maximum Annual_Income setelah repair:", df["Annual_Income"].max())

df["Annual_Income"].describe()

In [ ]:
# CHECK REMAINING ANNUAL_INCOME REFERENCE FROM SAME CUSTOMER_ID

remaining_annual_missing_indices = df[df["Annual_Income"].isna()].index

valid_annual_reference = (
    df[df["Annual_Income"].notna()]
    .groupby("Customer_ID")["Annual_Income"]
    .median()
)

suggested_annual_from_customer = df.loc[
    remaining_annual_missing_indices,
    "Customer_ID"
].map(valid_annual_reference)

annual_fillable_from_customer_indices = suggested_annual_from_customer[
    suggested_annual_from_customer.notna()
].index

annual_unresolved_indices = suggested_annual_from_customer[
    suggested_annual_from_customer.isna()
].index

print("Sisa missing Annual_Income total:", df["Annual_Income"].isna().sum())
print("Annual_Income yang punya referensi dari Customer_ID sama:", len(annual_fillable_from_customer_indices))
print("Annual_Income yang tetap tidak punya referensi:", len(annual_unresolved_indices))

In [ ]:
# DROP ROWS WITH MISSING ANNUAL_INCOME

before_rows = df.shape[0]

df = df.dropna(
    subset=["Annual_Income"]
).reset_index(drop=True)

after_rows = df.shape[0]

print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Annual_Income setelah drop:", df["Annual_Income"].isna().sum())

Setelah proses perbaikan nilai ekstrem dan estimasi menggunakan Monthly_Inhand_Salary × 12, masih terdapat 8 baris dengan nilai Annual_Income yang missing. Sisa missing value ini juga tidak memiliki referensi valid dari Customer_ID yang sama.

Karena jumlahnya sangat kecil dibandingkan total dataset, yaitu hanya 8 baris dari sekitar 25.000 baris, baris tersebut dihapus pada tahap data cleaning. Penghapusan dilakukan sebelum EDA final, feature engineering, train-test split, dan modeling agar dataset yang digunakan pada tahap analisis dan modeling sudah konsisten.

Keputusan ini tidak dilakukan di tahap model pipeline biasa, tetapi termasuk bagian dari cleaning rule. Oleh karena itu, aturan drop ini juga perlu diterapkan kembali pada pipeline training .py agar proses retraining menghasilkan data yang konsisten dengan notebook.

### 5. Monthly_Inhand_Salary

In [ ]:
show_missing_rows(
    df,
    "Monthly_Inhand_Salary"
)


In [ ]:
# CHECK MONTHLY_INHAND_SALARY REFERENCE AND ANNUAL_INCOME AVAILABILITY

missing_salary_indices = df[df["Monthly_Inhand_Salary"].isna()].index

valid_salary_reference = (
    df[df["Monthly_Inhand_Salary"].notna()]
    .groupby("Customer_ID")["Monthly_Inhand_Salary"]
    .median()
)

suggested_salary_from_customer = df.loc[
    missing_salary_indices,
    "Customer_ID"
].map(valid_salary_reference)

salary_fillable_from_customer_indices = suggested_salary_from_customer[
    suggested_salary_from_customer.notna()
].index

salary_unresolved_after_customer_indices = suggested_salary_from_customer[
    suggested_salary_from_customer.isna()
].index

annual_income_available_indices = df.loc[
    salary_unresolved_after_customer_indices
][
    df.loc[salary_unresolved_after_customer_indices, "Annual_Income"].notna()
].index

annual_income_missing_indices = df.loc[
    salary_unresolved_after_customer_indices
][
    df.loc[salary_unresolved_after_customer_indices, "Annual_Income"].isna()
].index

print("Missing Monthly_Inhand_Salary total:", df["Monthly_Inhand_Salary"].isna().sum())
print("Monthly_Inhand_Salary yang punya referensi dari Customer_ID sama:", len(salary_fillable_from_customer_indices))
print("Monthly_Inhand_Salary tanpa referensi dari Customer_ID sama:", len(salary_unresolved_after_customer_indices))
print("Dari yang tidak punya referensi Customer_ID, Annual_Income tersedia:", len(annual_income_available_indices))
print("Dari yang tidak punya referensi Customer_ID, Annual_Income juga missing:", len(annual_income_missing_indices))

In [ ]:
# REPAIR MONTHLY_INHAND_SALARY USING CUSTOMER_ID AND ANNUAL_INCOME

before_missing_salary = df["Monthly_Inhand_Salary"].isna().sum()

valid_salary_reference = (
    df[df["Monthly_Inhand_Salary"].notna()]
    .groupby("Customer_ID")["Monthly_Inhand_Salary"]
    .median()
)

missing_salary_indices = df[df["Monthly_Inhand_Salary"].isna()].index

suggested_salary_from_customer = df.loc[
    missing_salary_indices,
    "Customer_ID"
].map(valid_salary_reference)

salary_fillable_from_customer_indices = suggested_salary_from_customer[
    suggested_salary_from_customer.notna()
].index

df.loc[
    salary_fillable_from_customer_indices,
    "Monthly_Inhand_Salary"
] = suggested_salary_from_customer.loc[
    salary_fillable_from_customer_indices
]

missing_after_customer = df["Monthly_Inhand_Salary"].isna().sum()

remaining_salary_missing_indices = df[
    df["Monthly_Inhand_Salary"].isna() &
    df["Annual_Income"].notna()
].index

df.loc[
    remaining_salary_missing_indices,
    "Monthly_Inhand_Salary"
] = df.loc[
    remaining_salary_missing_indices,
    "Annual_Income"
] / 12

after_missing_salary = df["Monthly_Inhand_Salary"].isna().sum()

print("Missing Monthly_Inhand_Salary sebelum repair:", before_missing_salary)
print("Diisi dari Customer_ID yang sama:", len(salary_fillable_from_customer_indices))
print("Missing setelah repair Customer_ID:", missing_after_customer)
print("Diisi dari Annual_Income / 12:", len(remaining_salary_missing_indices))
print("Missing Monthly_Inhand_Salary setelah semua repair:", after_missing_salary)

In [ ]:
# DROP MONTHLY_INHAND_SALARY ROWS WITHOUT CUSTOMER_ID REFERENCE

before_rows = df.shape[0]

rows_to_drop_salary = salary_unresolved_after_customer_indices

df = df.drop(
    index=rows_to_drop_salary
).reset_index(drop=True)

after_rows = df.shape[0]

print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Monthly_Inhand_Salary setelah drop:", df["Monthly_Inhand_Salary"].isna().sum())

Pada tahap pengecekan missing value, ditemukan bahwa terdapat 3.828 baris dengan nilai Monthly_Inhand_Salary yang missing. Karena jumlah missing value cukup besar, baris tersebut tidak langsung dihapus tanpa analisis lebih lanjut. Jika seluruh missing value langsung dihapus, maka jumlah data yang hilang akan cukup banyak dan dapat mengurangi informasi penting dalam dataset.

Langkah pertama yang dilakukan adalah mengecek apakah missing value pada Monthly_Inhand_Salary dapat diperbaiki menggunakan referensi dari Customer_ID yang sama. Hal ini dilakukan karena dataset memiliki data customer dalam beberapa bulan yang berbeda. Jika customer yang sama memiliki nilai Monthly_Inhand_Salary valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk mengisi missing value.

Dari hasil pengecekan, terdapat 3.101 missing value yang memiliki referensi valid dari Customer_ID yang sama. Nilai tersebut kemudian diperbaiki menggunakan median Monthly_Inhand_Salary dari customer yang sama. Median dipilih karena lebih stabil dan lebih tahan terhadap nilai ekstrem dibandingkan mean.

Setelah proses perbaikan menggunakan referensi Customer_ID yang sama, masih terdapat 727 baris yang tidak memiliki referensi Monthly_Inhand_Salary valid dari customer yang sama. Selanjutnya, dilakukan pengecekan tambahan untuk melihat apakah 727 baris tersebut masih memiliki nilai Annual_Income. Hasil pengecekan menunjukkan bahwa seluruh 727 baris tersebut memiliki nilai Annual_Income.

Namun, pada tahap cleaning ini, nilai Monthly_Inhand_Salary untuk 727 baris tersebut tidak diisi menggunakan estimasi Annual_Income / 12. Hal ini dilakukan agar proses cleaning tetap konservatif dan tidak menambahkan nilai estimasi baru yang terlalu bergantung pada asumsi. Meskipun Annual_Income tersedia, Monthly_Inhand_Salary dan Annual_Income tidak selalu identik secara proporsional karena pendapatan bulanan yang diterima dapat dipengaruhi oleh potongan, komponen gaji, atau perbedaan pencatatan.

Oleh karena itu, keputusan akhir untuk kolom Monthly_Inhand_Salary adalah mengisi 3.101 missing value menggunakan median dari Customer_ID yang sama, lalu menghapus 727 baris yang masih tidak memiliki referensi salary valid dari customer yang sama. Keputusan drop ini dipilih karena 727 baris tersebut tidak dapat diperbaiki menggunakan referensi langsung dari riwayat customer, sehingga lebih aman untuk menghapusnya daripada memasukkan nilai estimasi yang terlalu asumtif.

### 6. Num_Bank_Accounts

In [ ]:
show_missing_rows(
    df,
    "Num_Bank_Accounts"
)




In [ ]:
# CHECK NUM_BANK_ACCOUNTS REFERENCE FROM SAME CUSTOMER_ID

missing_bank_account_indices = df[
    df["Num_Bank_Accounts"].isna()
].index

valid_bank_account_reference = (
    df[df["Num_Bank_Accounts"].notna()]
    .groupby("Customer_ID")["Num_Bank_Accounts"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

suggested_bank_account_from_customer = df.loc[
    missing_bank_account_indices,
    "Customer_ID"
].map(valid_bank_account_reference)

bank_account_fillable_from_customer_indices = suggested_bank_account_from_customer[
    suggested_bank_account_from_customer.notna()
].index

bank_account_unresolved_indices = suggested_bank_account_from_customer[
    suggested_bank_account_from_customer.isna()
].index

print("Missing Num_Bank_Accounts total:", df["Num_Bank_Accounts"].isna().sum())
print("Num_Bank_Accounts yang punya referensi dari Customer_ID sama:", len(bank_account_fillable_from_customer_indices))
print("Num_Bank_Accounts tanpa referensi dari Customer_ID sama:", len(bank_account_unresolved_indices))

In [ ]:
# DROP ROWS WITH MISSING NUM_BANK_ACCOUNTS

before_rows = df.shape[0]

df = df.dropna(
    subset=["Num_Bank_Accounts"]
).reset_index(drop=True)

after_rows = df.shape[0]

print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Num_Bank_Accounts setelah drop:", df["Num_Bank_Accounts"].isna().sum())

Pada tahap pengecekan missing value, ditemukan bahwa terdapat 40 baris dengan nilai Num_Bank_Accounts yang missing. Sebelum menentukan apakah baris tersebut akan dihapus atau diperbaiki, dilakukan pengecekan terlebih dahulu menggunakan referensi dari Customer_ID yang sama. Hal ini dilakukan karena dataset memiliki data customer dalam beberapa bulan, sehingga apabila customer yang sama memiliki nilai Num_Bank_Accounts valid pada bulan lain, nilai tersebut dapat digunakan sebagai referensi perbaikan.

Hasil pengecekan menunjukkan bahwa dari 40 missing value pada kolom Num_Bank_Accounts, tidak ada satupun yang memiliki referensi valid dari Customer_ID yang sama. Dengan kata lain, seluruh 40 baris missing tersebut tidak dapat diperbaiki secara aman menggunakan riwayat customer yang sama.

Karena tidak tersedia referensi valid dari customer yang sama, nilai missing pada kolom ini tidak dipaksa untuk diisi secara manual. Mengisi nilai secara sembarangan dapat menambahkan asumsi baru ke dalam dataset dan berisiko membuat hasil cleaning menjadi kurang akurat.

Setelah mempertimbangkan jumlah missing value yang sangat kecil dibandingkan total dataset, yaitu hanya 40 baris, maka keputusan cleaning untuk kolom Num_Bank_Accounts adalah menghapus baris yang masih memiliki nilai missing. Penghapusan ini dilakukan karena jumlahnya kecil dan tidak tersedia dasar yang cukup kuat untuk melakukan imputasi manual.

Setelah proses drop dilakukan, kolom Num_Bank_Accounts sudah tidak memiliki missing value.

### 7. Num_Credit_Card

In [ ]:
show_missing_rows(
    df,
    "Num_Credit_Card"
)

In [ ]:
# CHECK NUM_CREDIT_CARD REFERENCE FROM SAME CUSTOMER_ID

missing_credit_card_indices = df[
    df["Num_Credit_Card"].isna()
].index

valid_credit_card_reference = (
    df[df["Num_Credit_Card"].notna()]
    .groupby("Customer_ID")["Num_Credit_Card"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

suggested_credit_card_from_customer = df.loc[
    missing_credit_card_indices,
    "Customer_ID"
].map(valid_credit_card_reference)

credit_card_fillable_from_customer_indices = suggested_credit_card_from_customer[
    suggested_credit_card_from_customer.notna()
].index

credit_card_unresolved_indices = suggested_credit_card_from_customer[
    suggested_credit_card_from_customer.isna()
].index

print("Missing Num_Credit_Card total:", df["Num_Credit_Card"].isna().sum())
print("Num_Credit_Card yang punya referensi dari Customer_ID sama:", len(credit_card_fillable_from_customer_indices))
print("Num_Credit_Card tanpa referensi dari Customer_ID sama:", len(credit_card_unresolved_indices))

In [ ]:
# DROP ROWS WITH MISSING NUM_CREDIT_CARD

before_rows = df.shape[0]

df = df.dropna(
    subset=["Num_Credit_Card"]
).reset_index(drop=True)

after_rows = df.shape[0]

print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Num_Credit_Card setelah drop:", df["Num_Credit_Card"].isna().sum())

Pada tahap pengecekan missing value, ditemukan bahwa terdapat 73 baris dengan nilai Num_Credit_Card yang missing. Sebelum menentukan keputusan cleaning, missing value ini tidak langsung dihapus. Terlebih dahulu dilakukan pengecekan apakah nilai tersebut dapat diperbaiki menggunakan referensi dari Customer_ID yang sama.

Pengecekan berdasarkan Customer_ID dilakukan karena dataset memiliki beberapa data bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai Num_Credit_Card valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk mengisi missing value secara lebih aman.

Namun, hasil pengecekan menunjukkan bahwa dari 73 missing value pada kolom Num_Credit_Card, tidak ada satupun yang memiliki referensi valid dari Customer_ID yang sama. Artinya, seluruh 73 baris tersebut tidak dapat diperbaiki secara aman menggunakan riwayat customer yang sama.

Karena tidak tersedia referensi valid, nilai missing pada kolom ini tidak dipaksa untuk diisi secara manual. Mengisi jumlah kartu kredit tanpa dasar yang kuat dapat menambahkan asumsi baru ke dalam dataset dan berisiko membuat data menjadi kurang akurat.

Setelah mempertimbangkan jumlah missing value yang relatif kecil dibandingkan total dataset, keputusan cleaning untuk kolom Num_Credit_Card adalah menghapus 73 baris yang masih memiliki nilai missing. Penghapusan ini dilakukan karena jumlahnya kecil dan tidak tersedia referensi yang cukup kuat untuk melakukan imputasi manual.

Setelah proses drop dilakukan, kolom Num_Credit_Card sudah tidak memiliki missing value. 

### 8. Interest Rate

In [ ]:
show_missing_rows(
    df,
    "Monthly_Inhand_Salary"
)


In [ ]:
# CHECK INTEREST_RATE REFERENCE FROM SAME CUSTOMER_ID

missing_interest_rate_indices = df[
    df["Interest_Rate"].isna()
].index

valid_interest_rate_reference = (
    df[df["Interest_Rate"].notna()]
    .groupby("Customer_ID")["Interest_Rate"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

suggested_interest_rate_from_customer = df.loc[
    missing_interest_rate_indices,
    "Customer_ID"
].map(valid_interest_rate_reference)

interest_rate_fillable_from_customer_indices = suggested_interest_rate_from_customer[
    suggested_interest_rate_from_customer.notna()
].index

interest_rate_unresolved_indices = suggested_interest_rate_from_customer[
    suggested_interest_rate_from_customer.isna()
].index

print("Missing Interest_Rate total:", df["Interest_Rate"].isna().sum())
print("Interest_Rate yang punya referensi dari Customer_ID sama:", len(interest_rate_fillable_from_customer_indices))
print("Interest_Rate tanpa referensi dari Customer_ID sama:", len(interest_rate_unresolved_indices))

In [ ]:
# REPAIR AND DROP MISSING INTEREST_RATE

before_rows = df.shape[0]
before_missing_interest = df["Interest_Rate"].isna().sum()

# Get valid Interest_Rate reference from the same Customer_ID
valid_interest_rate_reference = (
    df[df["Interest_Rate"].notna()]
    .groupby("Customer_ID")["Interest_Rate"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

# Find missing Interest_Rate rows
missing_interest_rate_indices = df[
    df["Interest_Rate"].isna()
].index

# Map missing rows to same Customer_ID reference
suggested_interest_rate_from_customer = df.loc[
    missing_interest_rate_indices,
    "Customer_ID"
].map(valid_interest_rate_reference)

# Rows that can be repaired
interest_rate_fillable_from_customer_indices = suggested_interest_rate_from_customer[
    suggested_interest_rate_from_customer.notna()
].index

# Repair fillable rows
df.loc[
    interest_rate_fillable_from_customer_indices,
    "Interest_Rate"
] = suggested_interest_rate_from_customer.loc[
    interest_rate_fillable_from_customer_indices
]

missing_after_repair = df["Interest_Rate"].isna().sum()

# Drop remaining missing Interest_Rate rows
df = df.dropna(
    subset=["Interest_Rate"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_interest = df["Interest_Rate"].isna().sum()

print("Missing Interest_Rate sebelum repair:", before_missing_interest)
print("Interest_Rate yang diisi dari Customer_ID sama:", len(interest_rate_fillable_from_customer_indices))
print("Missing Interest_Rate setelah repair Customer_ID:", missing_after_repair)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Interest_Rate setelah drop:", after_missing_interest)

ilakukan pengecekan missing value pada kolom Interest_Rate. Hasil pengecekan menunjukkan bahwa terdapat 68 baris dengan nilai Interest_Rate yang masih missing.

Sebelum memutuskan untuk menghapus baris, dilakukan pengecekan apakah nilai missing tersebut dapat diperbaiki menggunakan referensi dari Customer_ID yang sama. Pendekatan ini digunakan karena dataset memiliki data customer dalam beberapa bulan, sehingga apabila customer yang sama memiliki nilai Interest_Rate valid pada bulan lain, nilai tersebut dapat digunakan sebagai referensi perbaikan.

Dari hasil pengecekan, terdapat 6 missing value yang memiliki referensi valid dari Customer_ID yang sama. Enam nilai tersebut kemudian diperbaiki menggunakan nilai Interest_Rate dari customer yang sama. Cara ini lebih aman dibandingkan mengisi dengan rata-rata seluruh dataset karena masih mempertahankan konsistensi karakteristik customer tersebut.

Setelah 6 nilai berhasil diperbaiki, masih terdapat 62 baris yang tidak memiliki referensi Interest_Rate valid dari Customer_ID yang sama. Karena tidak tersedia dasar yang cukup kuat untuk mengisi nilai tersebut, maka nilai missing ini tidak dipaksa untuk diimputasi secara manual.

Keputusan akhir untuk kolom Interest_Rate adalah memperbaiki 6 missing value menggunakan referensi dari Customer_ID yang sama, kemudian menghapus 62 baris yang masih missing. Keputusan drop dilakukan karena jumlahnya relatif kecil dibandingkan total dataset dan tidak tersedia referensi yang aman untuk melakukan perbaikan.

Dengan proses ini, kolom Interest_Rate menjadi lebih bersih, tidak memiliki missing value, dan tidak mengandung nilai ekstrem di atas batas wajar. Kolom ini kemudian dapat digunakan sebagai fitur numerik dalam proses modeling.

### 9. Num_of_Loan 

In [ ]:
show_missing_rows(
    df,
    "Num_of_Loan"
)



In [ ]:
# CHECK NUM_OF_LOAN REFERENCE FROM SAME CUSTOMER_ID AND TYPE_OF_LOAN

missing_num_loan_indices = df[
    df["Num_of_Loan"].isna()
].index

valid_num_loan_reference = (
    df[df["Num_of_Loan"].notna()]
    .groupby("Customer_ID")["Num_of_Loan"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

suggested_num_loan_from_customer = df.loc[
    missing_num_loan_indices,
    "Customer_ID"
].map(valid_num_loan_reference)

num_loan_fillable_from_customer_indices = suggested_num_loan_from_customer[
    suggested_num_loan_from_customer.notna()
].index

num_loan_unresolved_indices = suggested_num_loan_from_customer[
    suggested_num_loan_from_customer.isna()
].index

type_of_loan_available_indices = df.loc[
    missing_num_loan_indices
][
    df.loc[missing_num_loan_indices, "Type_of_Loan"].notna()
].index

type_of_loan_missing_indices = df.loc[
    missing_num_loan_indices
][
    df.loc[missing_num_loan_indices, "Type_of_Loan"].isna()
].index

print("Missing Num_of_Loan total:", df["Num_of_Loan"].isna().sum())
print("Num_of_Loan yang punya referensi dari Customer_ID sama:", len(num_loan_fillable_from_customer_indices))
print("Num_of_Loan tanpa referensi dari Customer_ID sama:", len(num_loan_unresolved_indices))
print("Dari missing Num_of_Loan, Type_of_Loan terisi:", len(type_of_loan_available_indices))
print("Dari missing Num_of_Loan, Type_of_Loan juga missing:", len(type_of_loan_missing_indices))

In [ ]:
# CHECK TYPE_OF_LOAN ITEM COUNT VS UNIQUE TYPE COUNT

def parse_loan_list(value):
    if pd.isna(value):
        return []
    
    cleaned_value = (
        str(value)
        .replace(", and ", ", ")
        .replace(" and ", ", ")
    )
    
    loan_list = [
        loan.strip()
        for loan in cleaned_value.split(",")
        if loan.strip() != ""
    ]
    
    return loan_list


df["Loan_List_From_Type"] = df["Type_of_Loan"].apply(parse_loan_list)

df["Loan_Item_Count"] = df["Loan_List_From_Type"].apply(
    lambda x: len(x) if len(x) > 0 else pd.NA
)

df["Unique_Loan_Type_Count"] = df["Loan_List_From_Type"].apply(
    lambda x: len(set(x)) if len(x) > 0 else pd.NA
)

loan_count_validation = df[
    df["Num_of_Loan"].notna() &
    df["Loan_Item_Count"].notna()
].copy()

loan_count_validation["Difference_With_Item_Count"] = (
    loan_count_validation["Num_of_Loan"] -
    loan_count_validation["Loan_Item_Count"]
)

loan_count_validation["Difference_With_Unique_Count"] = (
    loan_count_validation["Num_of_Loan"] -
    loan_count_validation["Unique_Loan_Type_Count"]
)

print("Jumlah data yang bisa dibandingkan:", loan_count_validation.shape[0])
print(
    "Jumlah beda jika pakai jumlah item loan:",
    (loan_count_validation["Difference_With_Item_Count"] != 0).sum()
)
print(
    "Jumlah beda jika pakai unique jenis loan:",
    (loan_count_validation["Difference_With_Unique_Count"] != 0).sum()
)

display(
    loan_count_validation[
        [
            "Num_of_Loan",
            "Type_of_Loan",
            "Loan_Item_Count",
            "Unique_Loan_Type_Count",
            "Difference_With_Item_Count",
            "Difference_With_Unique_Count"
        ]
    ].head(20)
)

In [ ]:
# CHECK MISSING NUM_OF_LOAN THAT CAN BE FILLED FROM TYPE_OF_LOAN

missing_num_loan_indices = df[
    df["Num_of_Loan"].isna()
].index

num_loan_fillable_from_type_indices = df[
    df["Num_of_Loan"].isna() &
    df["Loan_Item_Count"].notna()
].index

num_loan_unresolved_indices = df[
    df["Num_of_Loan"].isna() &
    df["Loan_Item_Count"].isna()
].index

print("Missing Num_of_Loan total:", len(missing_num_loan_indices))
print("Num_of_Loan yang bisa diisi dari Type_of_Loan item count:", len(num_loan_fillable_from_type_indices))
print("Num_of_Loan yang masih tidak punya referensi Type_of_Loan:", len(num_loan_unresolved_indices))

In [ ]:
# REPAIR MISSING NUM_OF_LOAN USING TYPE_OF_LOAN ITEM COUNT

before_missing_num_loan = df["Num_of_Loan"].isna().sum()

df.loc[
    num_loan_fillable_from_type_indices,
    "Num_of_Loan"
] = df.loc[
    num_loan_fillable_from_type_indices,
    "Loan_Item_Count"
]

df["Num_of_Loan"] = df["Num_of_Loan"].astype("Int64")

after_missing_num_loan = df["Num_of_Loan"].isna().sum()

print("Missing Num_of_Loan sebelum repair:", before_missing_num_loan)
print("Num_of_Loan yang diisi dari Type_of_Loan:", len(num_loan_fillable_from_type_indices))
print("Missing Num_of_Loan setelah repair:", after_missing_num_loan)

In [ ]:
# DROP REMAINING MISSING NUM_OF_LOAN

before_rows = df.shape[0]
before_missing_num_loan = df["Num_of_Loan"].isna().sum()

df = df.dropna(
    subset=["Num_of_Loan"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_num_loan = df["Num_of_Loan"].isna().sum()

df["Num_of_Loan"] = df["Num_of_Loan"].astype("Int64")

print("Missing Num_of_Loan sebelum drop:", before_missing_num_loan)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Num_of_Loan setelah drop:", after_missing_num_loan)

dilakukan pengecekan missing value pada kolom Num_of_Loan. Missing value pada kolom ini tidak langsung dihapus karena masih ada kemungkinan nilai tersebut dapat diperbaiki menggunakan informasi dari kolom lain, khususnya Type_of_Loan.

Sebelum menggunakan Type_of_Loan sebagai referensi, dilakukan validasi terlebih dahulu untuk memastikan apakah jumlah loan yang tertulis di Type_of_Loan benar-benar sesuai dengan nilai pada Num_of_Loan. Pada tahap ini, Type_of_Loan diproses dengan cara menghitung jumlah item loan yang tertulis, bukan menghitung jumlah jenis loan yang unik.

Hal ini penting karena dalam beberapa baris, jenis loan yang sama dapat muncul lebih dari satu kali. Contohnya, jika Type_of_Loan berisi Student Loan, Mortgage Loan, and Student Loan, maka jumlah item loan adalah 3, meskipun jenis loan uniknya hanya 2. Oleh karena itu, untuk mencocokkan dengan Num_of_Loan, yang digunakan adalah jumlah item loan, bukan jumlah unique loan type.

Hasil validasi menunjukkan bahwa terdapat 21.227 data yang dapat dibandingkan antara Num_of_Loan dan hasil parsing dari Type_of_Loan. Ketika menggunakan jumlah item loan, jumlah data yang berbeda adalah 0. Artinya, seluruh data yang dapat dibandingkan memiliki nilai Num_of_Loan yang sama dengan jumlah item loan pada Type_of_Loan.

Sebaliknya, ketika menggunakan jumlah unique loan type, terdapat 10.089 data yang berbeda. Hasil ini membuktikan bahwa Num_of_Loan pada dataset ini merepresentasikan jumlah item loan yang tertulis di Type_of_Loan, bukan jumlah jenis loan unik.

Berdasarkan hasil validasi tersebut, missing value pada Num_of_Loan yang masih memiliki Type_of_Loan terisi dapat diperbaiki menggunakan hasil parsing jumlah item loan dari Type_of_Loan. Pendekatan ini aman karena sudah terbukti konsisten dengan data valid yang tersedia.

Setelah missing value yang masih memiliki referensi Type_of_Loan diperbaiki, masih terdapat 23 baris dengan Num_of_Loan yang tetap missing. Sisa 23 baris ini tidak dapat diperbaiki karena tidak memiliki referensi yang cukup kuat dari Type_of_Loan.

Karena jumlah sisa missing value tersebut sangat kecil dibandingkan total dataset, maka keputusan akhir adalah menghapus 23 baris yang masih memiliki missing value pada Num_of_Loan. Penghapusan ini dilakukan agar kolom Num_of_Loan menjadi bersih dan tidak memiliki missing value sebelum digunakan dalam proses modeling.

Dengan proses ini, kolom Num_of_Loan sudah diperbaiki berdasarkan informasi yang valid dari Type_of_Loan, dan sisa data yang tidak dapat diperbaiki telah dihapus secara konservatif. Kolom ini kemudian dapat digunakan sebagai fitur numerik dalam model prediksi Credit_Score.

### 10. Type_of_Loan

In [ ]:
show_missing_rows(
    df,
    "Type_of_Loan"
)

In [ ]:
# CHECK TYPE_OF_LOAN REFERENCE FROM SAME CUSTOMER_ID AND NUM_OF_LOAN STATUS

missing_type_loan_indices = df[
    df["Type_of_Loan"].isna()
].index

valid_type_loan_reference = (
    df[df["Type_of_Loan"].notna()]
    .groupby("Customer_ID")["Type_of_Loan"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

suggested_type_loan_from_customer = df.loc[
    missing_type_loan_indices,
    "Customer_ID"
].map(valid_type_loan_reference)

type_loan_fillable_from_customer_indices = suggested_type_loan_from_customer[
    suggested_type_loan_from_customer.notna()
].index

type_loan_unresolved_indices = suggested_type_loan_from_customer[
    suggested_type_loan_from_customer.isna()
].index

num_loan_available_indices = df.loc[
    type_loan_unresolved_indices
][
    df.loc[type_loan_unresolved_indices, "Num_of_Loan"].notna()
].index

num_loan_missing_indices = df.loc[
    type_loan_unresolved_indices
][
    df.loc[type_loan_unresolved_indices, "Num_of_Loan"].isna()
].index

num_loan_zero_indices = df.loc[
    type_loan_unresolved_indices
][
    df.loc[type_loan_unresolved_indices, "Num_of_Loan"] == 0
].index

num_loan_more_than_zero_indices = df.loc[
    type_loan_unresolved_indices
][
    df.loc[type_loan_unresolved_indices, "Num_of_Loan"] > 0
].index

print("Missing Type_of_Loan total:", df["Type_of_Loan"].isna().sum())
print("Type_of_Loan yang punya referensi dari Customer_ID sama:", len(type_loan_fillable_from_customer_indices))
print("Type_of_Loan tanpa referensi dari Customer_ID sama:", len(type_loan_unresolved_indices))
print("Dari yang tidak punya referensi Customer_ID, Num_of_Loan terisi:", len(num_loan_available_indices))
print("Dari yang tidak punya referensi Customer_ID, Num_of_Loan juga missing:", len(num_loan_missing_indices))
print("Dari yang tidak punya referensi Customer_ID, Num_of_Loan = 0:", len(num_loan_zero_indices))
print("Dari yang tidak punya referensi Customer_ID, Num_of_Loan > 0:", len(num_loan_more_than_zero_indices))

In [ ]:
# FILL TYPE_OF_LOAN MISSING WHEN NUM_OF_LOAN IS 0

type_loan_no_loan_indices = df[
    df["Type_of_Loan"].isna() &
    (df["Num_of_Loan"] == 0)
].index

df.loc[
    type_loan_no_loan_indices,
    "Type_of_Loan"
] = "No Loan"

print("Jumlah Type_of_Loan yang diisi sebagai No Loan:", len(type_loan_no_loan_indices))
print("Missing Type_of_Loan setelah repair:", df["Type_of_Loan"].isna().sum())

Pada tahap pengecekan missing value, ditemukan bahwa terdapat 2.717 baris dengan nilai Type_of_Loan yang missing. Nilai missing ini tidak langsung dianggap sebagai error, karena perlu dicek terlebih dahulu apakah customer tersebut memang memiliki pinjaman atau tidak.

Langkah pertama yang dilakukan adalah mengecek apakah missing value pada Type_of_Loan dapat diperbaiki menggunakan referensi dari Customer_ID yang sama. Hal ini dilakukan karena dataset memiliki beberapa observasi bulanan untuk customer yang sama. Jika customer yang sama memiliki Type_of_Loan valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi perbaikan.

Namun, hasil pengecekan menunjukkan bahwa dari 2.717 missing value pada Type_of_Loan, tidak ada satupun yang memiliki referensi valid dari Customer_ID yang sama. Artinya, kolom ini tidak dapat diperbaiki menggunakan riwayat customer pada bulan lain.

Setelah itu, dilakukan pengecekan terhadap kolom Num_of_Loan untuk baris-baris yang memiliki missing value pada Type_of_Loan. Hasil pengecekan menunjukkan bahwa seluruh 2.717 baris tersebut memiliki Num_of_Loan yang terisi, dan semuanya bernilai 0. Tidak ada baris dengan Type_of_Loan missing yang memiliki Num_of_Loan > 0.

Hasil ini menunjukkan bahwa missing value pada Type_of_Loan bukan merupakan missing value yang bermasalah. Secara logika, jika Num_of_Loan = 0, maka customer memang tidak memiliki pinjaman. Oleh karena itu, wajar jika Type_of_Loan kosong, karena tidak ada jenis pinjaman yang dapat dicatat.

Berdasarkan hasil tersebut, nilai missing pada Type_of_Loan tidak dihapus dan tidak ditangani sebagai missing biasa di pipeline. Sebaliknya, nilai tersebut diisi dengan kategori khusus yaitu No Loan. Keputusan ini dilakukan pada tahap cleaning karena memiliki dasar logis yang jelas, bukan sekadar imputasi statistik.

Dengan mengisi Type_of_Loan menjadi No Loan, informasi bahwa customer tidak memiliki pinjaman tetap dipertahankan secara eksplisit. Pendekatan ini lebih tepat dibandingkan mengisi dengan kategori umum seperti Missing, karena dalam kasus ini nilai kosong sebenarnya memiliki makna bahwa customer tidak memiliki loan.

### 11. Num_of_Delayed_Payment 

In [ ]:
show_missing_rows(
    df,
    "Num_of_Delayed_Payment"
)


In [ ]:
# CHECK, REPAIR, AND DROP MISSING NUM_OF_DELAYED_PAYMENT

before_rows = df.shape[0]
before_missing_delayed_payment = df["Num_of_Delayed_Payment"].isna().sum()

# Get valid Num_of_Delayed_Payment reference from the same Customer_ID
valid_delayed_payment_reference = (
    df[df["Num_of_Delayed_Payment"].notna()]
    .groupby("Customer_ID")["Num_of_Delayed_Payment"]
    .median()
)

# Get rows with missing Num_of_Delayed_Payment
missing_delayed_payment_indices = df[
    df["Num_of_Delayed_Payment"].isna()
].index

# Map missing rows to same Customer_ID reference
suggested_delayed_payment_from_customer = df.loc[
    missing_delayed_payment_indices,
    "Customer_ID"
].map(valid_delayed_payment_reference)

# Rows that can be repaired from same Customer_ID
delayed_payment_fillable_from_customer_indices = suggested_delayed_payment_from_customer[
    suggested_delayed_payment_from_customer.notna()
].index

# Repair fillable rows
df.loc[
    delayed_payment_fillable_from_customer_indices,
    "Num_of_Delayed_Payment"
] = suggested_delayed_payment_from_customer.loc[
    delayed_payment_fillable_from_customer_indices
]

missing_after_repair = df["Num_of_Delayed_Payment"].isna().sum()

# Drop remaining rows without valid Customer_ID reference
df = df.dropna(
    subset=["Num_of_Delayed_Payment"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_delayed_payment = df["Num_of_Delayed_Payment"].isna().sum()

print("Missing Num_of_Delayed_Payment sebelum repair:", before_missing_delayed_payment)
print("Num_of_Delayed_Payment yang diisi dari Customer_ID sama:", len(delayed_payment_fillable_from_customer_indices))
print("Missing Num_of_Delayed_Payment setelah repair Customer_ID:", missing_after_repair)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Num_of_Delayed_Payment setelah drop:", after_missing_delayed_payment)

Pada tahap pengecekan missing value, ditemukan adanya nilai missing pada kolom Num_of_Delayed_Payment. Missing value ini tidak langsung dihapus karena masih ada kemungkinan nilai tersebut dapat diperbaiki menggunakan informasi dari Customer_ID yang sama.

Pengecekan berdasarkan Customer_ID dilakukan karena dataset memiliki beberapa observasi bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai Num_of_Delayed_Payment valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk memperbaiki missing value. Pendekatan ini lebih aman karena masih menggunakan riwayat customer yang sama, bukan menebak dari customer lain.

Dalam proses perbaikan, missing value yang memiliki referensi valid dari Customer_ID yang sama diisi menggunakan median Num_of_Delayed_Payment dari customer tersebut. Median dipilih karena lebih stabil terhadap nilai ekstrem dibandingkan mean, terutama karena jumlah keterlambatan pembayaran dapat memiliki variasi yang cukup besar antar customer.

Setelah proses repair menggunakan referensi dari Customer_ID yang sama, masih terdapat beberapa baris yang tidak memiliki referensi valid. Artinya, customer tersebut tidak memiliki nilai Num_of_Delayed_Payment valid pada bulan lain yang dapat digunakan sebagai dasar perbaikan.

Karena tidak tersedia referensi yang cukup kuat, sisa missing value tersebut tidak dipaksa untuk diisi secara manual. Mengisi jumlah keterlambatan pembayaran tanpa dasar yang jelas dapat menambahkan asumsi baru ke dalam dataset dan berisiko membuat hasil cleaning menjadi kurang akurat.

Keputusan akhir untuk kolom Num_of_Delayed_Payment adalah memperbaiki missing value yang memiliki referensi dari Customer_ID yang sama, kemudian menghapus baris yang masih memiliki missing value setelah proses repair. Penghapusan ini dilakukan karena sisa missing value tidak memiliki referensi yang aman untuk diperbaiki.

Dengan proses ini, kolom Num_of_Delayed_Payment menjadi lebih bersih, tidak memiliki missing value, dan dapat digunakan sebagai fitur numerik dalam proses modeling. 

### 12. Changed_Credit_Limit

In [ ]:
show_missing_rows(
    df,
    "Changed_Credit_Limit"
)


In [ ]:
# CHECK CHANGED_CREDIT_LIMIT REFERENCE FROM SAME CUSTOMER_ID

missing_changed_limit_indices = df[
    df["Changed_Credit_Limit"].isna()
].index

valid_changed_limit_reference = (
    df[df["Changed_Credit_Limit"].notna()]
    .groupby("Customer_ID")["Changed_Credit_Limit"]
    .median()
)

suggested_changed_limit_from_customer = df.loc[
    missing_changed_limit_indices,
    "Customer_ID"
].map(valid_changed_limit_reference)

changed_limit_fillable_from_customer_indices = suggested_changed_limit_from_customer[
    suggested_changed_limit_from_customer.notna()
].index

changed_limit_unresolved_indices = suggested_changed_limit_from_customer[
    suggested_changed_limit_from_customer.isna()
].index

print("Missing Changed_Credit_Limit total:", df["Changed_Credit_Limit"].isna().sum())
print("Changed_Credit_Limit yang punya referensi dari Customer_ID sama:", len(changed_limit_fillable_from_customer_indices))
print("Changed_Credit_Limit tanpa referensi dari Customer_ID sama:", len(changed_limit_unresolved_indices))

In [ ]:
# DROP ROWS WITH MISSING CHANGED_CREDIT_LIMIT

before_rows = df.shape[0]
before_missing_changed_limit = df["Changed_Credit_Limit"].isna().sum()

df = df.dropna(
    subset=["Changed_Credit_Limit"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_changed_limit = df["Changed_Credit_Limit"].isna().sum()

print("Missing Changed_Credit_Limit sebelum drop:", before_missing_changed_limit)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Changed_Credit_Limit setelah drop:", after_missing_changed_limit)

Pada tahap pengecekan missing value, ditemukan bahwa terdapat 61 baris dengan nilai Changed_Credit_Limit yang missing. Sebelum menentukan keputusan cleaning, missing value ini tidak langsung dihapus. Terlebih dahulu dilakukan pengecekan apakah nilai tersebut dapat diperbaiki menggunakan referensi dari Customer_ID yang sama.

Pengecekan berdasarkan Customer_ID dilakukan karena dataset memiliki beberapa data bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai Changed_Credit_Limit valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk mengisi missing value secara lebih aman.

Namun, hasil pengecekan menunjukkan bahwa dari 61 missing value pada kolom Changed_Credit_Limit, tidak ada satupun yang memiliki referensi valid dari Customer_ID yang sama. Artinya, seluruh 61 baris tersebut tidak dapat diperbaiki secara aman menggunakan riwayat customer yang sama.

Karena tidak tersedia referensi valid, nilai missing pada kolom ini tidak dipaksa untuk diisi secara manual. Mengisi nilai Changed_Credit_Limit tanpa dasar yang kuat dapat menambahkan asumsi baru ke dalam dataset dan berisiko membuat data menjadi kurang akurat.

Selain itu, nilai negatif pada Changed_Credit_Limit tidak langsung dianggap sebagai error, karena perubahan limit kredit dapat bernilai positif atau negatif. Nilai negatif dapat menunjukkan adanya penurunan limit kredit, sedangkan nilai positif dapat menunjukkan adanya kenaikan limit kredit.

Setelah mempertimbangkan jumlah missing value yang relatif kecil dibandingkan total dataset, keputusan cleaning untuk kolom Changed_Credit_Limit adalah menghapus 61 baris yang masih memiliki nilai missing. Penghapusan ini dilakukan karena jumlahnya kecil dan tidak tersedia referensi yang cukup kuat untuk melakukan imputasi manual.

Setelah proses drop dilakukan, kolom Changed_Credit_Limit sudah tidak memiliki missing value. Dengan demikian, kolom ini dapat digunakan sebagai fitur numerik dalam proses modeling.

### 13. Num_Credit_Inquiries

In [ ]:
show_missing_rows(
    df,
    "Num_Credit_Inquiries"
)

In [ ]:
# CHECK, REPAIR, AND DROP MISSING NUM_CREDIT_INQUIRIES

before_rows = df.shape[0]
before_missing_inquiries = df["Num_Credit_Inquiries"].isna().sum()

# Get valid Num_Credit_Inquiries reference from the same Customer_ID
valid_inquiries_reference = (
    df[df["Num_Credit_Inquiries"].notna()]
    .groupby("Customer_ID")["Num_Credit_Inquiries"]
    .median()
)

# Get rows with missing Num_Credit_Inquiries
missing_inquiries_indices = df[
    df["Num_Credit_Inquiries"].isna()
].index

# Map missing rows to same Customer_ID reference
suggested_inquiries_from_customer = df.loc[
    missing_inquiries_indices,
    "Customer_ID"
].map(valid_inquiries_reference)

# Rows that can be repaired from same Customer_ID
inquiries_fillable_from_customer_indices = suggested_inquiries_from_customer[
    suggested_inquiries_from_customer.notna()
].index

# Repair fillable rows
df.loc[
    inquiries_fillable_from_customer_indices,
    "Num_Credit_Inquiries"
] = suggested_inquiries_from_customer.loc[
    inquiries_fillable_from_customer_indices
]

missing_after_repair = df["Num_Credit_Inquiries"].isna().sum()

# Drop remaining rows without valid Customer_ID reference
df = df.dropna(
    subset=["Num_Credit_Inquiries"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_inquiries = df["Num_Credit_Inquiries"].isna().sum()

print("Missing Num_Credit_Inquiries sebelum repair:", before_missing_inquiries)
print("Num_Credit_Inquiries yang diisi dari Customer_ID sama:", len(inquiries_fillable_from_customer_indices))
print("Missing Num_Credit_Inquiries setelah repair Customer_ID:", missing_after_repair)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Num_Credit_Inquiries setelah drop:", after_missing_inquiries)

Pada tahap pengecekan missing value, ditemukan adanya nilai missing pada kolom Num_Credit_Inquiries. Missing value ini tidak langsung dihapus, karena terlebih dahulu dilakukan pengecekan apakah nilai tersebut dapat diperbaiki menggunakan referensi dari Customer_ID yang sama.

Pengecekan berdasarkan Customer_ID dilakukan karena dataset memiliki beberapa observasi bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai Num_Credit_Inquiries valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk memperbaiki missing value. Pendekatan ini lebih aman dibandingkan langsung mengisi dengan rata-rata seluruh dataset, karena masih mempertahankan karakteristik customer yang sama.

Dalam proses perbaikan, nilai missing yang memiliki referensi valid dari Customer_ID yang sama diisi menggunakan nilai median Num_Credit_Inquiries dari customer tersebut. Median dipilih karena lebih stabil terhadap nilai ekstrem dibandingkan mean, terutama untuk fitur numerik yang berhubungan dengan jumlah aktivitas kredit.

Setelah proses repair menggunakan referensi dari Customer_ID yang sama, masih terdapat beberapa baris yang tidak memiliki referensi valid. Artinya, customer tersebut tidak memiliki nilai Num_Credit_Inquiries valid pada bulan lain yang dapat digunakan sebagai dasar perbaikan.

Karena tidak tersedia referensi yang cukup kuat, sisa missing value tersebut tidak dipaksa untuk diisi secara manual. Mengisi nilai Num_Credit_Inquiries tanpa dasar yang jelas dapat menambahkan asumsi baru ke dalam dataset dan berisiko membuat hasil cleaning menjadi kurang akurat.

Keputusan akhir untuk kolom Num_Credit_Inquiries adalah memperbaiki missing value yang memiliki referensi dari Customer_ID yang sama, kemudian menghapus baris yang masih memiliki missing value setelah proses repair. Penghapusan ini dilakukan karena sisa missing value tidak memiliki referensi yang aman untuk diperbaiki.

Dengan proses ini, kolom Num_Credit_Inquiries menjadi lebih bersih, tidak memiliki missing value, dan dapat digunakan sebagai fitur numerik dalam proses modeling. 

### 14. Credit Mix

In [ ]:
show_missing_rows(
    df,
    "Credit_Mix"
)

In [ ]:
# CHECK, REPAIR, AND DROP MISSING CREDIT_MIX

before_rows = df.shape[0]
before_missing_credit_mix = df["Credit_Mix"].isna().sum()

# Get valid Credit_Mix reference from the same Customer_ID
valid_credit_mix_reference = (
    df[df["Credit_Mix"].notna()]
    .groupby("Customer_ID")["Credit_Mix"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

# Get rows with missing Credit_Mix
missing_credit_mix_indices = df[
    df["Credit_Mix"].isna()
].index

# Map missing rows to same Customer_ID reference
suggested_credit_mix_from_customer = df.loc[
    missing_credit_mix_indices,
    "Customer_ID"
].map(valid_credit_mix_reference)

# Rows that can be repaired from same Customer_ID
credit_mix_fillable_from_customer_indices = suggested_credit_mix_from_customer[
    suggested_credit_mix_from_customer.notna()
].index

# Repair fillable rows
df.loc[
    credit_mix_fillable_from_customer_indices,
    "Credit_Mix"
] = suggested_credit_mix_from_customer.loc[
    credit_mix_fillable_from_customer_indices
]

missing_after_repair = df["Credit_Mix"].isna().sum()

# Drop remaining rows without valid Customer_ID reference
df = df.dropna(
    subset=["Credit_Mix"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_credit_mix = df["Credit_Mix"].isna().sum()

print("Missing Credit_Mix sebelum repair:", before_missing_credit_mix)
print("Credit_Mix yang diisi dari Customer_ID sama:", len(credit_mix_fillable_from_customer_indices))
print("Missing Credit_Mix setelah repair Customer_ID:", missing_after_repair)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Credit_Mix setelah drop:", after_missing_credit_mix)

ada tahap pengecekan missing value, ditemukan adanya nilai missing pada kolom Credit_Mix. Missing value ini tidak langsung dihapus karena masih ada kemungkinan nilai tersebut dapat diperbaiki menggunakan referensi dari Customer_ID yang sama.

Pengecekan berdasarkan Customer_ID dilakukan karena dataset memiliki beberapa observasi bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai Credit_Mix valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk memperbaiki missing value. Pendekatan ini lebih aman karena masih menggunakan data dari customer yang sama, bukan menebak berdasarkan customer lain.

Dalam proses perbaikan, missing value yang memiliki referensi valid dari Customer_ID yang sama diisi menggunakan nilai modus Credit_Mix dari customer tersebut. Modus digunakan karena Credit_Mix merupakan fitur kategorikal, sehingga nilai yang paling sering muncul pada customer yang sama dianggap sebagai referensi yang paling representatif.

Setelah proses repair menggunakan referensi dari Customer_ID yang sama, masih terdapat beberapa baris yang tidak memiliki referensi valid. Artinya, customer tersebut tidak memiliki nilai Credit_Mix valid pada bulan lain yang dapat digunakan sebagai dasar perbaikan.

Karena tidak tersedia referensi yang cukup kuat, sisa missing value tersebut tidak dipaksa untuk diisi secara manual. Mengisi kategori Credit_Mix tanpa dasar yang jelas dapat menambahkan asumsi baru ke dalam dataset dan berisiko membuat hasil cleaning menjadi kurang akurat.

Keputusan akhir untuk kolom Credit_Mix adalah memperbaiki missing value yang memiliki referensi dari Customer_ID yang sama, kemudian menghapus baris yang masih memiliki missing value setelah proses repair. Penghapusan ini dilakukan karena sisa missing value tidak memiliki referensi yang aman untuk diperbaiki.

Dengan proses ini, kolom Credit_Mix menjadi lebih bersih, tidak memiliki missing value, dan tetap dapat digunakan sebagai fitur kategorikal dalam proses modeling

## SUMMARY Hasil Cek dan Penangganan Missing Value



Pada tahap data cleaning, setiap kolom dicek terlebih dahulu untuk melihat jumlah missing value dan apakah missing value tersebut masih dapat diperbaiki menggunakan informasi lain yang valid. Karena dataset memiliki data customer dalam beberapa bulan, pengecekan utama dilakukan menggunakan referensi dari `Customer_ID` yang sama pada bulan lain. Jika tersedia referensi valid dari customer yang sama, nilai missing diperbaiki menggunakan referensi tersebut. Namun, jika tidak tersedia referensi valid dan jumlah datanya kecil, baris tersebut dihapus agar dataset menjadi lebih bersih. Untuk missing value yang memiliki makna khusus, nilai tersebut tidak dihapus, tetapi diubah menjadi kategori yang lebih tepat.

- **1. Name**
  - Kolom `Name` merupakan kolom identitas customer.
  - Dari hasil pengecekan, terdapat 2.430 baris dengan nilai `Name` yang missing.
  - Missing `Name` kemudian dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Hasil pengecekan menunjukkan bahwa 1.984 missing `Name` memiliki referensi dari `Customer_ID` yang sama, sedangkan 446 tidak memiliki referensi.
  - Meskipun sebagian nilai dapat diperbaiki, kolom `Name` tetap tidak digunakan untuk modeling karena termasuk kolom identitas.
  - Jika kolom `Name` digunakan sebagai fitur, model berisiko menghafal identitas customer, bukan mempelajari pola risiko kredit.
  - Keputusan akhirnya, baris dengan missing `Name` tidak dihapus, tetapi kolom `Name` akan dihapus sebelum modeling bersama kolom identitas lain seperti `ID`, `Customer_ID`, `SSN`, dan `Unnamed: 0`.

- **2. Age**
  - Kolom `Age` merupakan fitur numerik yang menunjukkan umur customer.
  - Dari hasil pengecekan, terdapat 110 baris dengan nilai `Age` yang missing.
  - Missing `Age` dicek menggunakan referensi dari `Customer_ID` yang sama pada bulan lain.
  - Hasil pengecekan menunjukkan bahwa tidak ada missing `Age` yang memiliki referensi valid dari `Customer_ID` yang sama.
  - Karena tidak tersedia referensi valid, nilai missing pada `Age` tidak dipaksa untuk diisi secara manual.
  - Baris dengan missing `Age` juga tidak dihapus karena fitur lain pada baris tersebut masih memiliki informasi penting.
  - Keputusan akhirnya, missing `Age` akan ditangani menggunakan median imputer pada tahap preprocessing pipeline.
  - Median dipilih karena lebih tahan terhadap outlier dibandingkan mean, dan proses imputasi dilakukan di pipeline agar median hanya dihitung dari data train.

- **3. Occupation**
  - Kolom `Occupation` merupakan fitur kategorikal yang menunjukkan jenis pekerjaan customer.
  - Dari hasil pengecekan, terdapat 259 baris dengan nilai `Occupation` yang missing.
  - Kolom ini masih relevan untuk modeling karena pekerjaan dapat berhubungan dengan profil pendapatan, kemampuan pembayaran, dan risiko kredit.
  - Missing `Occupation` dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Jika customer yang sama memiliki occupation valid pada bulan lain, nilai tersebut dapat digunakan sebagai referensi perbaikan.
  - Karena `Occupation` adalah fitur kategorikal, nilai referensi yang digunakan adalah modus dari occupation customer yang sama.
  - Jika masih terdapat sisa missing tanpa referensi valid, nilainya tidak dipaksa untuk diisi manual.
  - Keputusan akhirnya, missing `Occupation` yang memiliki referensi dari `Customer_ID` yang sama dapat diperbaiki, sedangkan sisa missing akan ditangani sebagai kategori khusus seperti `Missing` pada preprocessing pipeline.

- **4. Annual_Income**
  - Kolom `Annual_Income` merupakan fitur numerik yang menunjukkan pendapatan tahunan customer.
  - Sebelumnya, kolom ini sudah melalui proses cleaning untuk menangani nilai ekstrem.
  - Nilai `Annual_Income` yang terlalu besar dan tidak wajar diperbaiki menggunakan referensi dari `Customer_ID` yang sama jika tersedia.
  - Jika tidak tersedia referensi valid, nilai ekstrem tersebut dijadikan missing value.
  - Setelah proses tersebut, terdapat 29 baris dengan nilai `Annual_Income` yang missing.
  - Missing `Annual_Income` kemudian diperbaiki menggunakan `Monthly_Inhand_Salary × 12` jika `Monthly_Inhand_Salary` tersedia.
  - Setelah proses repair menggunakan salary bulanan, masih tersisa 8 missing value pada `Annual_Income`.
  - Sisa 8 missing tersebut dicek kembali menggunakan referensi dari `Customer_ID` yang sama.
  - Hasil pengecekan menunjukkan bahwa 8 missing tersebut tidak memiliki referensi valid dari `Customer_ID` yang sama.
  - Karena jumlahnya sangat kecil dibandingkan total dataset dan tidak tersedia referensi valid, 8 baris tersebut dihapus.
  - Keputusan akhirnya, missing `Annual_Income` yang bisa diperbaiki dari `Monthly_Inhand_Salary × 12` diperbaiki, sedangkan 8 baris yang tetap missing dihapus.
  - Setelah proses ini, kolom `Annual_Income` tidak memiliki missing value dan nilai maksimumnya tetap berada pada batas wajar.

- **5. Monthly_Inhand_Salary**
  - Kolom `Monthly_Inhand_Salary` merupakan fitur numerik yang menunjukkan estimasi pendapatan bulanan customer.
  - Kolom ini penting karena berhubungan langsung dengan kemampuan bayar customer.
  - Dari hasil pengecekan, terdapat 3.828 baris dengan nilai `Monthly_Inhand_Salary` yang missing.
  - Missing value tidak langsung dihapus karena jumlahnya cukup besar.
  - Missing `Monthly_Inhand_Salary` dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Hasil pengecekan menunjukkan bahwa 3.101 missing value memiliki referensi dari `Customer_ID` yang sama.
  - Nilai tersebut diperbaiki menggunakan median `Monthly_Inhand_Salary` dari customer yang sama.
  - Setelah itu, masih terdapat 727 baris yang tidak memiliki referensi salary dari `Customer_ID` yang sama.
  - Meskipun seluruh 727 baris tersebut memiliki `Annual_Income`, nilai tersebut tidak digunakan untuk mengisi `Monthly_Inhand_Salary` karena dianggap terlalu asumtif.
  - `Monthly_Inhand_Salary` dan `Annual_Income` tidak selalu identik secara proporsional karena pendapatan bulanan dapat dipengaruhi oleh potongan, komponen gaji, atau perbedaan pencatatan.
  - Keputusan akhirnya, 3.101 missing value diperbaiki menggunakan median dari `Customer_ID` yang sama, sedangkan 727 baris tanpa referensi customer dihapus.

- **6. Num_Bank_Accounts**
  - Kolom `Num_Bank_Accounts` merupakan fitur numerik yang menunjukkan jumlah rekening bank yang dimiliki customer.
  - Dari hasil pengecekan, terdapat 40 baris dengan nilai `Num_Bank_Accounts` yang missing.
  - Missing value dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Hasil pengecekan menunjukkan bahwa tidak ada satupun missing `Num_Bank_Accounts` yang memiliki referensi valid dari `Customer_ID` yang sama.
  - Karena tidak ada referensi valid, nilai missing tidak dipaksa untuk diisi manual.
  - Jumlah missing relatif kecil dibandingkan total dataset.
  - Keputusan akhirnya, 40 baris dengan missing `Num_Bank_Accounts` dihapus.
  - Setelah drop, kolom `Num_Bank_Accounts` tidak memiliki missing value dan dapat digunakan sebagai fitur numerik.

- **7. Num_Credit_Card**
  - Kolom `Num_Credit_Card` merupakan fitur numerik yang menunjukkan jumlah kartu kredit yang dimiliki customer.
  - Dari hasil pengecekan, terdapat 73 baris dengan nilai `Num_Credit_Card` yang missing.
  - Missing value dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Hasil pengecekan menunjukkan bahwa tidak ada satupun missing `Num_Credit_Card` yang memiliki referensi valid dari `Customer_ID` yang sama.
  - Karena tidak ada referensi valid, nilai missing tidak dipaksa untuk diisi manual.
  - Jumlah missing relatif kecil dibandingkan total dataset.
  - Keputusan akhirnya, 73 baris dengan missing `Num_Credit_Card` dihapus.
  - Setelah drop, kolom `Num_Credit_Card` tidak memiliki missing value dan tetap digunakan sebagai fitur numerik.

- **8. Interest_Rate**
  - Kolom `Interest_Rate` merupakan fitur numerik yang menunjukkan tingkat bunga kredit customer.
  - Kolom ini penting karena tingkat bunga biasanya berkaitan dengan profil risiko customer.
  - Sebelumnya, nilai `Interest_Rate` yang terlalu ekstrem, yaitu lebih dari 34, sudah dianggap tidak valid dan diperbaiki jika ada referensi dari `Customer_ID` yang sama.
  - Setelah proses cleaning nilai ekstrem, terdapat 68 baris dengan nilai `Interest_Rate` yang missing.
  - Missing value dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Hasil pengecekan menunjukkan bahwa 6 missing value memiliki referensi valid dari `Customer_ID` yang sama, sedangkan 62 tidak memiliki referensi.
  - Enam missing value yang memiliki referensi diperbaiki menggunakan nilai `Interest_Rate` dari customer yang sama.
  - Sisa 62 baris tidak memiliki referensi valid, sehingga tidak dipaksa untuk diisi manual.
  - Keputusan akhirnya, 6 missing value diperbaiki menggunakan referensi dari `Customer_ID` yang sama, sedangkan 62 baris yang masih missing dihapus.
  - Setelah proses ini, kolom `Interest_Rate` tidak memiliki missing value.

- **9. Num_of_Loan**
  - Kolom `Num_of_Loan` merupakan fitur numerik yang menunjukkan jumlah pinjaman customer.
  - Kolom ini berkaitan erat dengan `Type_of_Loan`.
  - Missing value pada `Num_of_Loan` tidak langsung dihapus karena masih bisa divalidasi dan diperbaiki menggunakan informasi dari `Type_of_Loan`.
  - Sebelum digunakan untuk repair, dilakukan validasi apakah jumlah item loan pada `Type_of_Loan` sesuai dengan `Num_of_Loan`.
  - Pada tahap validasi, `Type_of_Loan` diproses dengan menghitung jumlah item loan yang tertulis, bukan jumlah jenis loan unik.
  - Contohnya, `Student Loan, Mortgage Loan, and Student Loan` memiliki 3 item loan, meskipun unique jenis loan-nya hanya 2.
  - Hasil validasi menunjukkan bahwa terdapat 21.227 data yang dapat dibandingkan.
  - Jika menggunakan jumlah item loan, jumlah data yang berbeda adalah 0.
  - Jika menggunakan unique jenis loan, jumlah data yang berbeda adalah 10.089.
  - Hasil ini membuktikan bahwa `Num_of_Loan` merepresentasikan jumlah item loan, bukan jumlah jenis loan unik.
  - Oleh karena itu, missing `Num_of_Loan` yang memiliki `Type_of_Loan` terisi diperbaiki menggunakan jumlah item loan dari `Type_of_Loan`.
  - Setelah repair menggunakan `Type_of_Loan`, masih tersisa 23 baris dengan `Num_of_Loan` missing.
  - Karena 23 baris tersebut tidak memiliki referensi yang cukup kuat, baris tersebut dihapus.
  - Keputusan akhirnya, missing `Num_of_Loan` yang bisa dihitung dari `Type_of_Loan` diperbaiki, sedangkan 23 baris yang masih missing dihapus.

- **10. Type_of_Loan**
  - Kolom `Type_of_Loan` merupakan fitur kategorikal yang menunjukkan jenis pinjaman customer.
  - Kolom ini berkaitan langsung dengan `Num_of_Loan`.
  - Dari hasil pengecekan, terdapat 2.717 baris dengan nilai `Type_of_Loan` yang missing.
  - Missing `Type_of_Loan` dicek terlebih dahulu menggunakan referensi dari `Customer_ID` yang sama.
  - Hasil pengecekan menunjukkan bahwa tidak ada missing `Type_of_Loan` yang memiliki referensi valid dari `Customer_ID` yang sama.
  - Setelah itu, dicek kondisi `Num_of_Loan` pada baris yang `Type_of_Loan`-nya missing.
  - Hasil pengecekan menunjukkan bahwa seluruh 2.717 baris tersebut memiliki `Num_of_Loan` terisi dan semuanya bernilai 0.
  - Tidak ada baris dengan `Type_of_Loan` missing yang memiliki `Num_of_Loan > 0`.
  - Artinya, missing `Type_of_Loan` terjadi karena customer memang tidak memiliki loan.
  - Secara logika, jika `Num_of_Loan = 0`, maka `Type_of_Loan` kosong adalah wajar karena tidak ada jenis pinjaman yang perlu dicatat.
  - Keputusan akhirnya, 2.717 missing value pada `Type_of_Loan` tidak dihapus, tetapi diisi dengan kategori `No Loan`.
  - Pengisian dilakukan pada tahap cleaning karena ini bukan missing biasa, melainkan missing yang memiliki makna jelas.
  - Jika nanti dilakukan parsing jumlah loan dari `Type_of_Loan`, kategori `No Loan` harus dihitung sebagai 0 loan.

- **11. Changed_Credit_Limit**
  - Kolom `Changed_Credit_Limit` merupakan fitur numerik yang menunjukkan perubahan limit kredit customer.
  - Nilai negatif pada kolom ini tidak langsung dianggap error karena perubahan limit kredit bisa naik atau turun.
  - Nilai negatif dapat menunjukkan penurunan limit kredit, sedangkan nilai positif dapat menunjukkan kenaikan limit kredit.
  - Dari hasil pengecekan, terdapat 61 baris dengan nilai `Changed_Credit_Limit` yang missing.
  - Missing value dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Hasil pengecekan menunjukkan bahwa tidak ada missing `Changed_Credit_Limit` yang memiliki referensi valid dari `Customer_ID` yang sama.
  - Karena tidak ada referensi valid, nilai missing tidak dipaksa untuk diisi manual.
  - Jumlah missing relatif kecil dibandingkan total dataset.
  - Keputusan akhirnya, 61 baris dengan missing `Changed_Credit_Limit` dihapus.
  - Setelah drop, kolom `Changed_Credit_Limit` tidak memiliki missing value dan tetap digunakan sebagai fitur numerik.

- **12. Num_Credit_Inquiries**
  - Kolom `Num_Credit_Inquiries` merupakan fitur numerik yang menunjukkan jumlah inquiry atau pengecekan kredit terhadap customer.
  - Kolom ini penting karena jumlah inquiry dapat menggambarkan seberapa sering customer mengajukan atau dicek untuk fasilitas kredit baru.
  - Pada tahap pengecekan missing value, nilai missing tidak langsung dihapus.
  - Missing value dicek terlebih dahulu menggunakan referensi dari `Customer_ID` yang sama.
  - Jika customer yang sama memiliki nilai `Num_Credit_Inquiries` valid pada bulan lain, nilai tersebut digunakan sebagai referensi.
  - Karena `Num_Credit_Inquiries` merupakan fitur numerik, nilai missing yang memiliki referensi diperbaiki menggunakan median dari customer yang sama.
  - Sisa missing value yang tidak memiliki referensi valid tidak dipaksa untuk diisi manual.
  - Keputusan akhirnya, missing `Num_Credit_Inquiries` yang memiliki referensi dari `Customer_ID` yang sama diperbaiki menggunakan median customer tersebut.
  - Sisa missing value yang tidak memiliki referensi valid dihapus.
  - Setelah proses ini, kolom `Num_Credit_Inquiries` tidak memiliki missing value dan dapat digunakan sebagai fitur numerik.

- **13. Num_of_Delayed_Payment**
  - Kolom `Num_of_Delayed_Payment` merupakan fitur numerik yang menunjukkan jumlah keterlambatan pembayaran customer.
  - Kolom ini sangat penting dalam analisis risiko kredit karena keterlambatan pembayaran merupakan salah satu indikator utama kualitas kredit.
  - Pada tahap pengecekan missing value, nilai missing tidak langsung dihapus.
  - Missing value dicek terlebih dahulu menggunakan referensi dari `Customer_ID` yang sama.
  - Jika customer yang sama memiliki nilai `Num_of_Delayed_Payment` valid pada bulan lain, nilai tersebut digunakan sebagai referensi.
  - Karena fitur ini numerik, nilai missing yang memiliki referensi diperbaiki menggunakan median `Num_of_Delayed_Payment` dari customer yang sama.
  - Median dipilih karena lebih stabil terhadap nilai ekstrem dibandingkan mean.
  - Sisa missing value yang tidak memiliki referensi valid tidak dipaksa untuk diisi manual.
  - Keputusan akhirnya, missing `Num_of_Delayed_Payment` yang memiliki referensi dari `Customer_ID` yang sama diperbaiki menggunakan median customer tersebut.
  - Sisa missing value tanpa referensi valid dihapus.
  - Setelah proses ini, kolom `Num_of_Delayed_Payment` tidak memiliki missing value dan tetap digunakan sebagai fitur numerik.

- **14. Credit_Mix**
  - Kolom `Credit_Mix` merupakan fitur kategorikal yang menunjukkan kualitas campuran kredit customer.
  - Kategori pada kolom ini dapat menggambarkan apakah komposisi kredit customer tergolong `Good`, `Standard`, atau `Bad`.
  - Pada tahap pengecekan missing value, nilai missing tidak langsung dihapus.
  - Missing value dicek terlebih dahulu menggunakan referensi dari `Customer_ID` yang sama.
  - Jika customer yang sama memiliki nilai `Credit_Mix` valid pada bulan lain, nilai tersebut digunakan sebagai referensi.
  - Karena `Credit_Mix` merupakan fitur kategorikal, nilai missing yang memiliki referensi diperbaiki menggunakan modus `Credit_Mix` dari customer yang sama.
  - Modus dipilih karena merepresentasikan kategori yang paling sering muncul pada customer tersebut.
  - Sisa missing value yang tidak memiliki referensi valid dari `Customer_ID` yang sama tidak dipaksa untuk diisi manual.
  - Keputusan akhirnya, missing `Credit_Mix` yang memiliki referensi dari `Customer_ID` yang sama diperbaiki menggunakan modus customer tersebut.
  - Sisa missing value tanpa referensi valid dihapus.
  - Setelah proses ini, kolom `Credit_Mix` tidak memiliki missing value dan tetap digunakan sebagai fitur kategorikal dalam modeling.

- **Ringkasan keputusan akhir**
  - Kolom identitas seperti `Name`, `ID`, `Customer_ID`, `SSN`, dan `Unnamed: 0` tidak digunakan sebagai fitur modeling.
  - `Age` tidak dihapus meskipun masih memiliki missing value, karena akan ditangani menggunakan median imputer pada preprocessing pipeline.
  - `Occupation` ditangani sebagai fitur kategorikal; jika masih ada missing tanpa referensi valid, akan ditangani sebagai kategori khusus seperti `Missing`.
  - `Annual_Income` diperbaiki menggunakan `Monthly_Inhand_Salary × 12` jika memungkinkan, lalu sisa 8 baris yang tetap missing dihapus.
  - `Monthly_Inhand_Salary` diperbaiki menggunakan median dari `Customer_ID` yang sama, lalu 727 baris tanpa referensi customer dihapus.
  - `Num_Bank_Accounts`, `Num_Credit_Card`, dan `Changed_Credit_Limit` dihapus jika tetap missing dan tidak memiliki referensi customer yang valid.
  - `Interest_Rate`, `Num_Credit_Inquiries`, `Num_of_Delayed_Payment`, dan `Credit_Mix` diperbaiki terlebih dahulu menggunakan referensi dari `Customer_ID` yang sama, lalu sisa missing tanpa referensi dihapus.
  - `Num_of_Loan` diperbaiki menggunakan jumlah item loan dari `Type_of_Loan`, lalu sisa 23 baris yang tetap missing dihapus.
  - `Type_of_Loan` yang missing ketika `Num_of_Loan = 0` diisi sebagai `No Loan`, karena missing tersebut memiliki makna bahwa customer tidak memiliki pinjaman.
  - Semua aturan cleaning ini perlu diterapkan kembali pada file pipeline `.py` agar proses retraining dan deployment konsisten dengan notebook.


## Tambahan Pengcekan Missing Value karena saya sebelumnya masih belum kecek semua

In [ ]:
df.info()

### 15. Credit_History_Age 

In [ ]:
show_missing_rows(
    df,
    "Credit_History_Age"
)

In [ ]:
# CHECK AND REPAIR MISSING CREDIT_HISTORY_AGE USING CUSTOMER_ID

before_missing_credit_history = df["Credit_History_Age"].isna().sum()

# Convert temporarily to nullable float to avoid assignment error
df["Credit_History_Age"] = pd.to_numeric(
    df["Credit_History_Age"],
    errors="coerce"
).astype("Float64")

valid_credit_history_reference = (
    df[df["Credit_History_Age"].notna()]
    .groupby("Customer_ID")["Credit_History_Age"]
    .median()
    .round()
)

missing_credit_history_indices = df[
    df["Credit_History_Age"].isna()
].index

suggested_credit_history_from_customer = df.loc[
    missing_credit_history_indices,
    "Customer_ID"
].map(valid_credit_history_reference)

credit_history_fillable_from_customer_indices = suggested_credit_history_from_customer[
    suggested_credit_history_from_customer.notna()
].index

credit_history_unresolved_indices = suggested_credit_history_from_customer[
    suggested_credit_history_from_customer.isna()
].index

df.loc[
    credit_history_fillable_from_customer_indices,
    "Credit_History_Age"
] = suggested_credit_history_from_customer.loc[
    credit_history_fillable_from_customer_indices
].round()

df["Credit_History_Age"] = df["Credit_History_Age"].round().astype("Int64")

after_missing_credit_history = df["Credit_History_Age"].isna().sum()

print("Missing Credit_History_Age sebelum repair:", before_missing_credit_history)
print("Credit_History_Age yang diisi dari Customer_ID sama:", len(credit_history_fillable_from_customer_indices))
print("Credit_History_Age tanpa referensi dari Customer_ID sama:", len(credit_history_unresolved_indices))
print("Missing Credit_History_Age setelah repair Customer_ID:", after_missing_credit_history)

In [ ]:
# CREATE MISSING INDICATOR FOR CREDIT_HISTORY_AGE

df["Credit_History_Age_Missing"] = df["Credit_History_Age"].isna().astype(int)

print("Missing Credit_History_Age:", df["Credit_History_Age"].isna().sum())
print("Credit_History_Age_Missing value counts:")
display(df["Credit_History_Age_Missing"].value_counts())

In [ ]:
# FINAL CHECK CREDIT_HISTORY_AGE

print("Missing Credit_History_Age:", df["Credit_History_Age"].isna().sum())
print("Credit_History_Age < 0:", (df["Credit_History_Age"] < 0).sum())

invalid_credit_history = (
    df["Credit_History_Age"].notna() &
    df["Age"].notna() &
    (df["Credit_History_Age"] > df["Age"] * 12)
)

print("Credit_History_Age > Age * 12:", invalid_credit_history.sum())

display(df["Credit_History_Age"].describe())

Pada tahap pengecekan missing value, ditemukan 4.524 baris dengan nilai Credit_History_Age yang missing. Missing value ini tidak langsung dihapus karena jumlahnya cukup besar dan kolom ini masih memiliki nilai penting untuk modeling.

Langkah pertama yang dilakukan adalah mengecek apakah missing value dapat diperbaiki menggunakan referensi dari Customer_ID yang sama. Karena dataset memiliki data bulanan untuk customer yang sama, jika customer tersebut memiliki nilai Credit_History_Age valid pada bulan lain, nilai tersebut dapat digunakan sebagai referensi perbaikan.

Dari hasil pengecekan, sebanyak 1.615 missing value berhasil diperbaiki menggunakan median Credit_History_Age dari Customer_ID yang sama. Median digunakan karena lebih stabil terhadap nilai ekstrem dibandingkan mean.

Setelah proses repair, masih terdapat 2.909 missing value yang tidak memiliki referensi valid dari customer yang sama. Nilai ini tidak dipaksa untuk diisi secara manual karena tidak tersedia dasar yang cukup kuat. Namun, baris tersebut juga tidak dihapus karena jumlahnya cukup besar dan penghapusan dapat menyebabkan kehilangan banyak informasi penting dari dataset.

Keputusan akhir untuk kolom Credit_History_Age adalah memperbaiki missing value yang memiliki referensi dari Customer_ID yang sama, kemudian membiarkan sisa missing value untuk ditangani pada tahap preprocessing pipeline menggunakan median imputer. Selain itu, dibuat fitur tambahan Credit_History_Age_Missing sebagai indikator apakah nilai Credit_History_Age sebelumnya missing atau tidak. Fitur ini dapat membantu model menangkap informasi bahwa ketiadaan data riwayat kredit juga dapat memiliki pola tertentu terhadap Credit_Score.

### 16. Total_EMI_per_month

In [ ]:
show_missing_rows(
    df,
    "Total_EMI_per_month"
)

In [ ]:
# CHECK, REPAIR, AND DROP MISSING TOTAL_EMI_PER_MONTH

before_rows = df.shape[0]
before_missing_emi = df["Total_EMI_per_month"].isna().sum()

# 1. If Num_of_Loan = 0 and EMI is missing, set EMI to 0
no_loan_missing_emi_indices = df[
    (df["Total_EMI_per_month"].isna()) &
    (df["Num_of_Loan"] == 0)
].index

df.loc[
    no_loan_missing_emi_indices,
    "Total_EMI_per_month"
] = 0

print("Missing EMI dengan Num_of_Loan = 0 yang diisi 0:", len(no_loan_missing_emi_indices))


# 2. Fill remaining missing EMI using same Customer_ID median
valid_emi_reference = (
    df[df["Total_EMI_per_month"].notna()]
    .groupby("Customer_ID")["Total_EMI_per_month"]
    .median()
)

missing_emi_indices = df[
    df["Total_EMI_per_month"].isna()
].index

suggested_emi_from_customer = df.loc[
    missing_emi_indices,
    "Customer_ID"
].map(valid_emi_reference)

emi_fillable_from_customer_indices = suggested_emi_from_customer[
    suggested_emi_from_customer.notna()
].index

emi_unresolved_indices = suggested_emi_from_customer[
    suggested_emi_from_customer.isna()
].index

df.loc[
    emi_fillable_from_customer_indices,
    "Total_EMI_per_month"
] = suggested_emi_from_customer.loc[
    emi_fillable_from_customer_indices
]

missing_after_customer_repair = df["Total_EMI_per_month"].isna().sum()

print("Missing Total_EMI_per_month sebelum repair:", before_missing_emi)
print("Total_EMI_per_month yang diisi dari Customer_ID sama:", len(emi_fillable_from_customer_indices))
print("Total_EMI_per_month tanpa referensi Customer_ID:", len(emi_unresolved_indices))
print("Missing Total_EMI_per_month setelah repair Customer_ID:", missing_after_customer_repair)


# 3. Drop remaining missing EMI
df = df.dropna(
    subset=["Total_EMI_per_month"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_emi = df["Total_EMI_per_month"].isna().sum()

print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Total_EMI_per_month setelah drop:", after_missing_emi)

Pada tahap pengecekan outlier menggunakan metode IQR, kolom Total_EMI_per_month memiliki upper bound sebesar 318,1575 dan terdapat 1.206 baris yang terdeteksi sebagai outlier. Namun, outlier ini tidak langsung dihapus karena nilai EMI yang tinggi masih dapat masuk akal secara konteks, terutama untuk customer yang memiliki jumlah loan lebih banyak atau pendapatan bulanan yang lebih tinggi.

Oleh karena itu, outlier pada Total_EMI_per_month tidak ditangani hanya berdasarkan batas IQR. Validasi dilakukan dengan melihat hubungan antara Total_EMI_per_month, Monthly_Inhand_Salary, dan Num_of_Loan. Nilai EMI dianggap bermasalah jika tidak sesuai secara logika, misalnya customer tidak memiliki loan tetapi memiliki EMI, atau nilai EMI lebih besar dari pendapatan bulanan customer.

Pada tahap pengecekan missing value, ditemukan 46 baris dengan nilai Total_EMI_per_month yang missing. Missing value ini tidak langsung dihapus karena masih ada kemungkinan nilai tersebut dapat diperbaiki menggunakan informasi lain yang tersedia.

Langkah pertama yang dilakukan adalah mengecek apakah missing Total_EMI_per_month terjadi pada customer dengan Num_of_Loan = 0. Jika customer tidak memiliki loan, maka secara logika customer tidak memiliki cicilan bulanan. Oleh karena itu, jika ditemukan kondisi Num_of_Loan = 0 dan Total_EMI_per_month missing, nilai EMI dapat diisi menjadi 0.

Setelah itu, sisa missing value pada Total_EMI_per_month dicek menggunakan referensi dari Customer_ID yang sama. Hal ini dilakukan karena dataset memiliki beberapa data bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai EMI valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk memperbaiki missing value.

Karena Total_EMI_per_month merupakan fitur numerik, nilai missing yang memiliki referensi dari Customer_ID yang sama diperbaiki menggunakan median Total_EMI_per_month dari customer tersebut. Median dipilih karena lebih stabil terhadap nilai ekstrem dibandingkan mean.

Setelah proses repair menggunakan referensi dari Customer_ID yang sama, sisa missing value yang masih tidak memiliki referensi valid tidak dipaksa untuk diisi secara manual. Jika tidak ada dasar yang cukup kuat untuk memperbaiki nilai tersebut, maka baris yang masih missing dihapus.

Selain missing value, dilakukan juga validasi logika terhadap nilai EMI. Jika terdapat customer dengan Num_of_Loan = 0 tetapi Total_EMI_per_month > 0, maka nilai EMI diperbaiki menjadi 0 karena customer yang tidak memiliki pinjaman seharusnya tidak memiliki cicilan bulanan. Jika terdapat baris dengan Total_EMI_per_month > Monthly_Inhand_Salary, baris tersebut dapat dihapus karena secara logika nilai cicilan yang lebih besar dari pendapatan bulanan customer sangat tidak wajar dan jumlah kasusnya kecil.

Keputusan akhir untuk kolom Total_EMI_per_month adalah tidak menghapus seluruh outlier berdasarkan IQR. Nilai outlier tetap dipertahankan jika masih masuk akal secara konteks. Penanganan hanya dilakukan pada nilai yang bermasalah secara logika, yaitu missing value, EMI positif pada customer tanpa loan, dan EMI yang lebih besar dari salary bulanan.

Dengan proses ini, kolom Total_EMI_per_month menjadi lebih bersih dan tetap mempertahankan informasi penting mengenai beban cicilan customer. Kolom ini kemudian dapat digunakan sebagai fitur numerik dalam proses modeling.

### 17.  Amount_invested_monthly

In [ ]:
show_missing_rows(
    df,
    "Amount_invested_monthly"
)

In [ ]:
# CHECK, REPAIR, AND DROP MISSING AMOUNT_INVESTED_MONTHLY

before_rows = df.shape[0]
before_missing_invested = df["Amount_invested_monthly"].isna().sum()

# Get valid Amount_invested_monthly reference from the same Customer_ID
valid_invested_reference = (
    df[df["Amount_invested_monthly"].notna()]
    .groupby("Customer_ID")["Amount_invested_monthly"]
    .median()
)

# Get rows with missing Amount_invested_monthly
missing_invested_indices = df[
    df["Amount_invested_monthly"].isna()
].index

# Map missing rows to same Customer_ID reference
suggested_invested_from_customer = df.loc[
    missing_invested_indices,
    "Customer_ID"
].map(valid_invested_reference)

# Rows that can be repaired from same Customer_ID
invested_fillable_from_customer_indices = suggested_invested_from_customer[
    suggested_invested_from_customer.notna()
].index

# Repair fillable rows
df.loc[
    invested_fillable_from_customer_indices,
    "Amount_invested_monthly"
] = suggested_invested_from_customer.loc[
    invested_fillable_from_customer_indices
]

missing_after_repair = df["Amount_invested_monthly"].isna().sum()

# Drop remaining rows without valid Customer_ID reference
df = df.dropna(
    subset=["Amount_invested_monthly"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_invested = df["Amount_invested_monthly"].isna().sum()

print("Missing Amount_invested_monthly sebelum repair:", before_missing_invested)
print("Amount_invested_monthly yang diisi dari Customer_ID sama:", len(invested_fillable_from_customer_indices))
print("Missing Amount_invested_monthly setelah repair Customer_ID:", missing_after_repair)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Amount_invested_monthly setelah drop:", after_missing_invested)

Pada tahap pengecekan missing value, nilai missing pada Amount_invested_monthly tidak langsung dihapus. Terlebih dahulu dilakukan pengecekan apakah nilai tersebut dapat diperbaiki menggunakan referensi dari Customer_ID yang sama.

Pengecekan berdasarkan Customer_ID dilakukan karena dataset memiliki beberapa data bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai Amount_invested_monthly valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk memperbaiki missing value secara lebih aman.

Karena Amount_invested_monthly merupakan fitur numerik, missing value yang memiliki referensi dari Customer_ID yang sama diperbaiki menggunakan median Amount_invested_monthly dari customer tersebut. Median dipilih karena lebih stabil terhadap nilai ekstrem dibandingkan mean, terutama pada fitur keuangan yang dapat memiliki variasi cukup besar antar bulan.

Setelah proses repair menggunakan referensi dari Customer_ID yang sama, masih terdapat kemungkinan beberapa baris tetap missing karena customer tersebut tidak memiliki nilai Amount_invested_monthly valid pada bulan lain. Nilai yang masih missing setelah proses repair tidak dipaksa untuk diisi secara manual karena tidak tersedia referensi yang cukup kuat.

Keputusan akhir untuk kolom Amount_invested_monthly adalah memperbaiki missing value yang memiliki referensi dari Customer_ID yang sama, kemudian menghapus baris yang masih missing setelah proses repair. Penghapusan dilakukan karena sisa missing value tidak memiliki referensi yang aman untuk diperbaiki.

Dengan proses ini, kolom Amount_invested_monthly menjadi lebih bersih, tidak memiliki missing value, dan tetap dapat digunakan sebagai fitur numerik dalam proses modeling.

### 18. Payment_Behaviour

In [ ]:
show_missing_rows(
    df,
    "Payment_Behaviour"
)

In [ ]:
# CHECK, REPAIR, AND DROP MISSING PAYMENT_BEHAVIOUR

before_rows = df.shape[0]

# Treat invalid placeholder as missing if it still exists
df["Payment_Behaviour"] = df["Payment_Behaviour"].replace("!@9#%8", pd.NA)

before_missing_payment_behaviour = df["Payment_Behaviour"].isna().sum()

# Get valid Payment_Behaviour reference from the same Customer_ID
valid_payment_behaviour_reference = (
    df[df["Payment_Behaviour"].notna()]
    .groupby("Customer_ID")["Payment_Behaviour"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA)
)

# Get rows with missing Payment_Behaviour
missing_payment_behaviour_indices = df[
    df["Payment_Behaviour"].isna()
].index

# Map missing rows to same Customer_ID reference
suggested_payment_behaviour_from_customer = df.loc[
    missing_payment_behaviour_indices,
    "Customer_ID"
].map(valid_payment_behaviour_reference)

# Rows that can be repaired from same Customer_ID
payment_behaviour_fillable_from_customer_indices = suggested_payment_behaviour_from_customer[
    suggested_payment_behaviour_from_customer.notna()
].index

# Rows that still do not have valid reference
payment_behaviour_unresolved_indices = suggested_payment_behaviour_from_customer[
    suggested_payment_behaviour_from_customer.isna()
].index

# Repair fillable rows
df.loc[
    payment_behaviour_fillable_from_customer_indices,
    "Payment_Behaviour"
] = suggested_payment_behaviour_from_customer.loc[
    payment_behaviour_fillable_from_customer_indices
]

missing_after_repair = df["Payment_Behaviour"].isna().sum()

# Drop remaining missing Payment_Behaviour rows
df = df.dropna(
    subset=["Payment_Behaviour"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_payment_behaviour = df["Payment_Behaviour"].isna().sum()

print("Missing Payment_Behaviour sebelum repair:", before_missing_payment_behaviour)
print("Payment_Behaviour yang diisi dari Customer_ID sama:", len(payment_behaviour_fillable_from_customer_indices))
print("Payment_Behaviour tanpa referensi Customer_ID:", len(payment_behaviour_unresolved_indices))
print("Missing Payment_Behaviour setelah repair Customer_ID:", missing_after_repair)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Payment_Behaviour setelah drop:", after_missing_payment_behaviour)

Kolom Payment_Behaviour merupakan fitur kategorikal yang menunjukkan pola perilaku pembayaran dan pengeluaran customer. Kolom ini penting dalam analisis risiko kredit karena dapat menggambarkan bagaimana customer mengelola uangnya, apakah cenderung high spending atau low spending, serta bagaimana pola pembayaran yang dilakukan.

Pada tahap awal, nilai tidak valid seperti !@9#%8 diperlakukan sebagai missing value. Nilai tersebut bukan kategori perilaku pembayaran yang valid, sehingga tidak boleh dipertahankan sebagai kategori asli dalam data.

Setelah nilai tidak valid diganti menjadi missing, dilakukan pengecekan jumlah missing value pada kolom Payment_Behaviour. Missing value ini tidak langsung dihapus karena masih ada kemungkinan nilai tersebut dapat diperbaiki menggunakan referensi dari Customer_ID yang sama.

Pengecekan berdasarkan Customer_ID dilakukan karena dataset memiliki beberapa data bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai Payment_Behaviour valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk memperbaiki missing value.

Karena Payment_Behaviour merupakan fitur kategorikal, nilai missing yang memiliki referensi dari Customer_ID yang sama diperbaiki menggunakan modus Payment_Behaviour dari customer tersebut. Modus dipilih karena menunjukkan kategori yang paling sering muncul pada customer yang sama, sehingga lebih representatif dibandingkan mengisi secara acak atau menggunakan kategori dari customer lain.

Setelah proses repair menggunakan referensi dari Customer_ID yang sama, masih terdapat kemungkinan beberapa baris tidak memiliki referensi valid. Nilai yang masih missing setelah proses repair tidak dipaksa untuk diisi secara manual karena tidak tersedia dasar yang cukup kuat.

Keputusan akhir untuk kolom Payment_Behaviour adalah memperbaiki missing value yang memiliki referensi dari Customer_ID yang sama, kemudian menghapus baris yang masih missing setelah proses repair. Penghapusan dilakukan karena sisa missing value tidak memiliki referensi yang aman untuk diperbaiki.

Dengan proses ini, kolom Payment_Behaviour menjadi lebih bersih, tidak memiliki missing value, dan hanya berisi kategori perilaku pembayaran yang valid. Kolom ini tetap digunakan sebagai fitur kategorikal dalam proses modeling karena memiliki informasi penting terkait pola keuangan dan risiko kredit customer.

### 19. Monthly_Balance 


In [ ]:
show_missing_rows(
    df,
    "Monthly_Balance"
)

In [ ]:
# CHECK, REPAIR, AND DROP MISSING MONTHLY_BALANCE

before_rows = df.shape[0]
before_missing_monthly_balance = df["Monthly_Balance"].isna().sum()

# Get valid Monthly_Balance reference from the same Customer_ID
valid_monthly_balance_reference = (
    df[df["Monthly_Balance"].notna()]
    .groupby("Customer_ID")["Monthly_Balance"]
    .median()
)

# Get rows with missing Monthly_Balance
missing_monthly_balance_indices = df[
    df["Monthly_Balance"].isna()
].index

# Map missing rows to same Customer_ID reference
suggested_monthly_balance_from_customer = df.loc[
    missing_monthly_balance_indices,
    "Customer_ID"
].map(valid_monthly_balance_reference)

# Rows that can be repaired from same Customer_ID
monthly_balance_fillable_from_customer_indices = suggested_monthly_balance_from_customer[
    suggested_monthly_balance_from_customer.notna()
].index

# Rows that still do not have valid reference
monthly_balance_unresolved_indices = suggested_monthly_balance_from_customer[
    suggested_monthly_balance_from_customer.isna()
].index

# Repair fillable rows
df.loc[
    monthly_balance_fillable_from_customer_indices,
    "Monthly_Balance"
] = suggested_monthly_balance_from_customer.loc[
    monthly_balance_fillable_from_customer_indices
]

missing_after_repair = df["Monthly_Balance"].isna().sum()

# Drop remaining missing Monthly_Balance rows
df = df.dropna(
    subset=["Monthly_Balance"]
).reset_index(drop=True)

after_rows = df.shape[0]
after_missing_monthly_balance = df["Monthly_Balance"].isna().sum()

print("Missing Monthly_Balance sebelum repair:", before_missing_monthly_balance)
print("Monthly_Balance yang diisi dari Customer_ID sama:", len(monthly_balance_fillable_from_customer_indices))
print("Monthly_Balance tanpa referensi Customer_ID:", len(monthly_balance_unresolved_indices))
print("Missing Monthly_Balance setelah repair Customer_ID:", missing_after_repair)
print("Jumlah baris sebelum drop:", before_rows)
print("Jumlah baris setelah drop:", after_rows)
print("Jumlah baris yang dihapus:", before_rows - after_rows)
print("Missing Monthly_Balance setelah drop:", after_missing_monthly_balance)

ada tahap pengecekan missing value, nilai missing pada Monthly_Balance tidak langsung dihapus. Terlebih dahulu dilakukan pengecekan apakah nilai tersebut dapat diperbaiki menggunakan referensi dari Customer_ID yang sama.

Pengecekan berdasarkan Customer_ID dilakukan karena dataset memiliki beberapa data bulanan untuk customer yang sama. Jika customer yang sama memiliki nilai Monthly_Balance valid pada bulan lain, maka nilai tersebut dapat digunakan sebagai referensi untuk memperbaiki missing value secara lebih aman.

Karena Monthly_Balance merupakan fitur numerik, missing value yang memiliki referensi dari Customer_ID yang sama diperbaiki menggunakan median Monthly_Balance dari customer tersebut. Median dipilih karena lebih stabil terhadap nilai ekstrem dibandingkan mean, terutama pada fitur keuangan yang dapat memiliki variasi besar antarbulan.

Setelah proses repair menggunakan referensi dari Customer_ID yang sama, masih terdapat kemungkinan beberapa baris tetap missing karena customer tersebut tidak memiliki nilai Monthly_Balance valid pada bulan lain. Nilai yang masih missing setelah proses repair tidak dipaksa untuk diisi secara manual karena tidak tersedia referensi yang cukup kuat.

Keputusan akhir untuk kolom Monthly_Balance adalah memperbaiki missing value yang memiliki referensi dari Customer_ID yang sama, kemudian menghapus baris yang masih missing setelah proses repair. Penghapusan dilakukan karena sisa missing value tidak memiliki dasar yang aman untuk diperbaiki.

Setelah proses cleaning, kolom Monthly_Balance dicek kembali untuk memastikan tidak terdapat missing value dan tidak terdapat nilai negatif. Nilai negatif perlu dicek karena secara konteks saldo bulanan yang sangat tidak wajar dapat menunjukkan anomali data.

Dengan proses ini, kolom Monthly_Balance menjadi lebih bersih, tidak memiliki missing value, dan tetap dapat digunakan sebagai fitur numerik dalam proses modeling. Kolom ini dipertahankan karena memiliki informasi penting terkait kondisi keuangan customer dan dapat membantu model dalam memprediksi Credit_Score.

## Summary Kolom Tambahan Missing Value Handling dan Keputusan Cleaning

## Prinsip Umum Cleaning

- Proses cleaning dilakukan dengan tujuan memperbaiki nilai yang masih bisa direkonstruksi secara logis, bukan langsung menghapus semua missing value.
- Karena dataset memiliki beberapa data bulanan untuk customer yang sama, `Customer_ID` digunakan sebagai referensi cleaning sementara untuk mengecek apakah nilai missing dapat diperbaiki dari data customer yang sama pada bulan lain.
- `Customer_ID` hanya digunakan untuk proses cleaning dan validasi, bukan sebagai fitur modeling akhir.
- Untuk fitur numerik, nilai missing yang masih memiliki referensi dari `Customer_ID` yang sama diperbaiki menggunakan median dari customer tersebut.
- Median digunakan karena lebih stabil terhadap nilai ekstrem dibandingkan mean, terutama pada fitur keuangan yang cenderung right-skewed.
- Untuk fitur kategorikal, nilai missing yang masih memiliki referensi dari `Customer_ID` yang sama diperbaiki menggunakan modus dari customer tersebut.
- Modus digunakan karena menunjukkan kategori yang paling sering muncul pada customer yang sama.
- Jika setelah proses repair menggunakan `Customer_ID` masih terdapat missing value dan jumlahnya kecil, baris tersebut dihapus.
- Jika jumlah missing value yang tersisa masih besar, baris tidak dihapus agar dataset tidak kehilangan terlalu banyak informasi penting.
- Missing value yang jumlahnya besar dan tidak memiliki referensi valid akan ditangani pada preprocessing pipeline menggunakan imputer yang sesuai.

## 1. Credit_History_Age

- `Credit_History_Age` adalah fitur numerik yang menunjukkan panjang riwayat kredit customer dalam satuan bulan.
- Fitur ini penting karena riwayat kredit yang lebih panjang dapat memberikan lebih banyak informasi mengenai perilaku kredit customer.
- Pada tahap pengecekan, ditemukan missing value pada kolom `Credit_History_Age`.
- Missing value tidak langsung dihapus karena jumlahnya cukup besar dan fitur ini masih memiliki nilai penting untuk modeling.
- Proses pertama yang dilakukan adalah mengecek apakah missing value dapat diperbaiki menggunakan referensi dari `Customer_ID` yang sama.
- Jika customer yang sama memiliki nilai `Credit_History_Age` valid pada bulan lain, nilai tersebut digunakan sebagai dasar perbaikan.
- Karena `Credit_History_Age` merupakan fitur numerik, nilai missing diperbaiki menggunakan median `Credit_History_Age` dari customer yang sama.
- Berdasarkan hasil cleaning:
  - Missing `Credit_History_Age` sebelum repair: 4.524 baris.
  - Missing yang berhasil diisi dari `Customer_ID` yang sama: 1.615 baris.
  - Missing yang tidak memiliki referensi dari `Customer_ID` yang sama: 2.909 baris.
  - Missing setelah repair dari `Customer_ID`: 2.909 baris.
- Sisa missing value sebanyak 2.909 baris tidak dihapus karena jumlahnya besar.
- Jika 2.909 baris tersebut dihapus, dataset akan kehilangan terlalu banyak informasi dan distribusi data dapat berubah.
- Oleh karena itu, keputusan akhir untuk `Credit_History_Age` adalah:
  - Memperbaiki missing value yang memiliki referensi dari `Customer_ID` yang sama.
  - Membuat fitur indikator missing bernama `Credit_History_Age_Missing`.
  - Membiarkan sisa missing value untuk ditangani oleh median imputer pada preprocessing pipeline.
- Fitur `Credit_History_Age_Missing` digunakan agar model dapat mengetahui apakah nilai `Credit_History_Age` pada baris tersebut awalnya missing atau tidak.
- Missing indicator ini penting karena kondisi missing pada riwayat kredit dapat memiliki pola tertentu terhadap `Credit_Score`.
- Pada tahap modeling, `Credit_History_Age` akan tetap digunakan sebagai fitur numerik.
- Nilai missing pada `Credit_History_Age` akan diimpute menggunakan median di dalam pipeline preprocessing.
- Median imputer harus diterapkan dalam pipeline agar proses preprocessing lebih rapi, konsisten, dan aman saat digunakan pada data baru.
- Setelah cleaning, kolom ini juga dicek secara logika, terutama untuk memastikan tidak ada nilai negatif dan tidak ada nilai `Credit_History_Age` yang lebih besar dari `Age * 12`.

## 2. Total_EMI_per_month

- `Total_EMI_per_month` adalah fitur numerik yang menunjukkan total cicilan bulanan customer.
- Fitur ini penting karena menggambarkan beban pembayaran bulanan customer.
- Nilai EMI yang tinggi dapat menunjukkan tekanan keuangan yang lebih besar, sehingga relevan untuk memprediksi `Credit_Score`.
- Pada pengecekan outlier menggunakan metode IQR, terdapat 1.206 baris yang terdeteksi sebagai outlier.
- Namun, outlier pada kolom ini tidak langsung dihapus karena EMI yang tinggi masih bisa masuk akal jika customer memiliki banyak pinjaman atau pendapatan bulanan yang tinggi.
- Oleh karena itu, validasi untuk `Total_EMI_per_month` tidak hanya berdasarkan IQR, tetapi juga berdasarkan logika hubungan dengan `Num_of_Loan` dan `Monthly_Inhand_Salary`.
- Pada tahap pengecekan missing value, ditemukan missing value pada `Total_EMI_per_month`.
- Proses cleaning yang dilakukan:
  - Jika `Num_of_Loan = 0` dan `Total_EMI_per_month` missing, nilai EMI diisi menjadi 0.
  - Jika `Total_EMI_per_month` missing dan customer yang sama memiliki nilai EMI valid di bulan lain, nilai missing diperbaiki menggunakan median EMI dari `Customer_ID` yang sama.
  - Jika masih missing setelah dicek ke `Customer_ID` yang sama, baris tersebut dihapus.
- Selain missing value, dilakukan juga validasi logika:
  - Jika `Num_of_Loan = 0` tetapi `Total_EMI_per_month > 0`, nilai EMI diperbaiki menjadi 0.
  - Jika `Total_EMI_per_month > Monthly_Inhand_Salary`, baris tersebut dihapus karena cicilan bulanan yang lebih besar dari pendapatan bulanan kurang wajar dan jumlah kasusnya kecil.
- Keputusan akhir untuk `Total_EMI_per_month` adalah:
  - Tidak menghapus seluruh outlier IQR.
  - Memperbaiki nilai yang tidak logis jika masih dapat diperbaiki.
  - Menghapus baris yang masih missing setelah proses repair menggunakan `Customer_ID`.
  - Menghapus baris yang memiliki nilai EMI tidak logis, terutama jika EMI lebih besar dari monthly salary.
- Dengan keputusan ini, informasi penting dari customer dengan cicilan tinggi tetap dipertahankan, sedangkan data yang tidak konsisten tetap dibersihkan.

## 3. Amount_invested_monthly

- `Amount_invested_monthly` adalah fitur numerik yang menunjukkan jumlah uang yang diinvestasikan customer setiap bulan.
- Fitur ini dapat membantu menjelaskan pola pengelolaan keuangan customer.
- Customer dengan pola investasi tertentu dapat memiliki kondisi keuangan yang berbeda, sehingga kolom ini relevan untuk prediksi `Credit_Score`.
- Missing value pada kolom ini tidak langsung dihapus.
- Langkah pertama adalah mengecek apakah nilai missing dapat diperbaiki menggunakan data dari `Customer_ID` yang sama.
- Karena `Amount_invested_monthly` merupakan fitur numerik, nilai missing yang memiliki referensi valid diperbaiki menggunakan median dari customer yang sama.
- Median dipilih karena nilai investasi bulanan dapat bervariasi dan bisa memiliki nilai ekstrem.
- Proses cleaning yang dilakukan:
  - Cek missing value pada `Amount_invested_monthly`.
  - Cek apakah customer yang sama memiliki nilai valid pada bulan lain.
  - Jika ada referensi valid, isi missing value menggunakan median `Amount_invested_monthly` dari `Customer_ID` yang sama.
  - Jika tidak ada referensi valid, baris tersebut dihapus.
- Keputusan akhir untuk `Amount_invested_monthly` adalah:
  - Repair menggunakan median dari `Customer_ID` yang sama jika tersedia.
  - Drop baris yang masih missing setelah proses repair.
- Setelah cleaning, kolom ini dicek kembali untuk memastikan tidak ada missing value dan tidak ada nilai negatif.
- Kolom ini tetap digunakan sebagai fitur numerik dalam modeling karena memiliki informasi penting mengenai kebiasaan keuangan customer.

## 4. Payment_Behaviour

- `Payment_Behaviour` adalah fitur kategorikal yang menunjukkan pola pembayaran dan pengeluaran customer.
- Kolom ini relevan karena menggambarkan bagaimana customer menggunakan uangnya, apakah cenderung high spending atau low spending, serta bagaimana pola nilai pembayaran yang dilakukan.
- Pada tahap awal, nilai tidak valid seperti `!@9#%8` diperlakukan sebagai missing value.
- Nilai tersebut bukan kategori perilaku pembayaran yang valid, sehingga tidak boleh dipertahankan sebagai kategori asli.
- Setelah nilai tidak valid diganti menjadi missing, dilakukan pengecekan apakah missing value dapat diperbaiki dari `Customer_ID` yang sama.
- Karena `Payment_Behaviour` merupakan fitur kategorikal, nilai missing diperbaiki menggunakan modus dari customer yang sama.
- Modus dipilih karena menunjukkan kategori perilaku pembayaran yang paling sering muncul pada customer tersebut.
- Proses cleaning yang dilakukan:
  - Ubah nilai tidak valid `!@9#%8` menjadi missing.
  - Cek missing value pada `Payment_Behaviour`.
  - Cek apakah customer yang sama memiliki nilai valid pada bulan lain.
  - Jika ada referensi valid, isi missing value menggunakan modus `Payment_Behaviour` dari `Customer_ID` yang sama.
  - Jika tidak ada referensi valid, baris tersebut dihapus.
- Keputusan akhir untuk `Payment_Behaviour` adalah:
  - Repair menggunakan modus dari `Customer_ID` yang sama jika tersedia.
  - Drop baris yang masih missing setelah proses repair.
- Setelah cleaning, kolom ini dicek kembali untuk memastikan tidak ada missing value dan tidak ada kategori invalid.
- Kolom ini tetap digunakan sebagai fitur kategorikal dalam modeling karena memiliki informasi penting terkait perilaku pembayaran customer.

## 5. Monthly_Balance

- `Monthly_Balance` adalah fitur numerik yang menunjukkan sisa saldo bulanan customer setelah pendapatan digunakan untuk kebutuhan, cicilan, dan investasi.
- Kolom ini penting karena dapat menggambarkan kondisi keuangan customer pada akhir bulan.
- Customer dengan saldo bulanan rendah dapat memiliki tekanan keuangan lebih tinggi, sedangkan customer dengan saldo bulanan lebih tinggi dapat menunjukkan kondisi keuangan yang lebih stabil.
- Missing value pada `Monthly_Balance` tidak langsung dihapus.
- Langkah pertama adalah mengecek apakah nilai missing dapat diperbaiki menggunakan referensi dari `Customer_ID` yang sama.
- Karena `Monthly_Balance` merupakan fitur numerik, nilai missing yang memiliki referensi valid diperbaiki menggunakan median dari customer yang sama.
- Median dipilih karena lebih stabil terhadap nilai ekstrem dan variasi saldo antarbulan.
- Proses cleaning yang dilakukan:
  - Cek missing value pada `Monthly_Balance`.
  - Cek apakah customer yang sama memiliki nilai valid pada bulan lain.
  - Jika ada referensi valid, isi missing value menggunakan median `Monthly_Balance` dari `Customer_ID` yang sama.
  - Jika tidak ada referensi valid, baris tersebut dihapus.
- Keputusan akhir untuk `Monthly_Balance` adalah:
  - Repair menggunakan median dari `Customer_ID` yang sama jika tersedia.
  - Drop baris yang masih missing setelah proses repair.
- Setelah cleaning, kolom ini dicek kembali untuk memastikan tidak ada missing value dan tidak ada nilai negatif.
- Kolom ini tetap digunakan sebagai fitur numerik dalam modeling karena memberikan informasi penting mengenai stabilitas keuangan customer.

## Ringkasan Keputusan Akhir

- `Credit_History_Age`
  - Missing yang bisa diperbaiki dari `Customer_ID` diisi menggunakan median customer yang sama.
  - Sisa missing tidak di-drop karena jumlahnya besar.
  - Dibuat fitur `Credit_History_Age_Missing` sebagai missing indicator.
  - Sisa missing akan ditangani menggunakan median imputer pada preprocessing pipeline.

- `Total_EMI_per_month`
  - Missing dicek berdasarkan `Num_of_Loan` dan `Customer_ID`.
  - Jika `Num_of_Loan = 0`, EMI missing dapat diisi 0.
  - Jika ada referensi dari `Customer_ID` yang sama, missing diisi menggunakan median EMI customer tersebut.
  - Sisa missing yang tidak memiliki referensi di-drop.
  - Nilai tidak logis seperti EMI lebih besar dari monthly salary di-drop.

- `Amount_invested_monthly`
  - Missing dicek berdasarkan `Customer_ID`.
  - Jika ada referensi dari customer yang sama, missing diisi menggunakan median.
  - Sisa missing yang tidak memiliki referensi di-drop.

- `Payment_Behaviour`
  - Nilai invalid `!@9#%8` diubah menjadi missing.
  - Missing dicek berdasarkan `Customer_ID`.
  - Jika ada referensi dari customer yang sama, missing diisi menggunakan modus.
  - Sisa missing yang tidak memiliki referensi di-drop.

- `Monthly_Balance`
  - Missing dicek berdasarkan `Customer_ID`.
  - Jika ada referensi dari customer yang sama, missing diisi menggunakan median.
  - Sisa missing yang tidak memiliki referensi di-drop.

## Catatan untuk Modeling

- Kolom `Customer_ID` digunakan hanya untuk membantu proses cleaning, bukan sebagai fitur model.
- Kolom identitas seperti `Unnamed: 0`, `ID`, `Customer_ID`, `Name`, dan `SSN` harus dihapus sebelum modeling.
- Sisa missing pada `Credit_History_Age` tidak dihapus, tetapi ditangani menggunakan median imputer dalam pipeline.
- Fitur `Credit_History_Age_Missing` harus dimasukkan sebagai fitur numerik/binary indicator.
- Untuk fitur numerik lainnya yang sudah dibersihkan, pastikan tidak ada missing value sebelum modeling.
- Untuk fitur kategorikal yang sudah dibersihkan, pastikan tidak ada kategori invalid sebelum modeling.
- Seluruh preprocessing akhir sebaiknya dilakukan dalam pipeline agar proses training, evaluation, inference, dan deployment konsisten.


# F. Data Prepocessing(Ide Penangganan Kolom Kolom Numerik Jika Ada Outlier Untuk di Pipeline)

### 1. Age



In [ ]:
show_outlier_rows(df, "Age")

### 2. Annual_Income



In [ ]:
show_outlier_rows(df, "Annual_Income")

Annual_Income berdasarkan IQR tidak dihapus dan tidak di-cap. Nilai tersebut tetap dipertahankan karena masih masuk akal secara konteks dan konsisten dengan informasi pendapatan bulanan. Keputusan ini juga dilakukan agar model tetap dapat mempelajari pola customer dengan pendapatan tinggi, bukan hanya customer dengan pendapatan menengah atau rendah.

### 3. Monthly_Inhand_Salary



In [ ]:
show_outlier_rows(df, "Monthly_Inhand_Salary")

Monthly_Innhand__Salary berdasarkan IQR tidak dihapus dan tidak di-cap. Nilai tersebut tetap dipertahankan karena masih masuk akal secara konteks dan konsisten dengan informasi pendapatan bulanan. Keputusan ini juga dilakukan agar model tetap dapat mempelajari pola customer dengan pendapatan tinggi, bukan hanya customer dengan pendapatan menengah atau rendah.

### 4. Num_Bank_Accounts



In [ ]:
show_outlier_rows(df, "Num_Bank_Accounts")

### 5. Num_Credit_Card




In [ ]:
show_outlier_rows(df, "Num_Credit_Card")

Pada kolom Num_Credit_Card, metode IQR mendeteksi beberapa nilai sebagai outlier karena berada di atas upper bound sebesar 11,5. Namun, nilai outlier yang muncul masih berada dalam rentang yang sudah ditetapkan pada tahap cleaning, yaitu maksimal 25.

Sebelumnya, nilai Num_Credit_Card yang dianggap tidak valid adalah nilai negatif dan nilai yang lebih besar dari 25. Oleh karena itu, nilai seperti 16, 18, dan 24 tidak dihapus meskipun terdeteksi sebagai outlier oleh metode IQR. Nilai tersebut masih dianggap mungkin secara konteks karena masih berada dalam batas valid yang digunakan.

Keputusan akhir untuk outlier Num_Credit_Card adalah mempertahankan nilai yang masih berada pada rentang 0 sampai 25, dan hanya memperbaiki atau menghapus nilai yang berada di luar batas valid tersebut.

### 6. Interest_Rate



In [ ]:
show_outlier_rows(df, "Interest_Rate")

### 7. Num_of_Loan


In [ ]:
show_outlier_rows(df, "Num_of_Loan")

### 8. Delay_from_due_date



In [ ]:
show_outlier_rows(df, "Delay_from_due_date")

In [ ]:
# CHECK DELAY_FROM_DUE_DATE OUTLIER DISTRIBUTION

print("Minimum Delay_from_due_date:", df["Delay_from_due_date"].min())
print("Maximum Delay_from_due_date:", df["Delay_from_due_date"].max())
print("Delay_from_due_date > 55:", (df["Delay_from_due_date"] > 55).sum())
print("Delay_from_due_date < 0:", (df["Delay_from_due_date"] < 0).sum())

display(
    df.groupby("Credit_Score")["Delay_from_due_date"]
    .describe()
)

Hasil pengecekan berdasarkan Credit_Score menunjukkan bahwa Delay_from_due_date memiliki pola yang relevan terhadap kualitas kredit customer. Customer dengan credit score Good memiliki median keterlambatan sebesar 10 hari, customer Standard memiliki median keterlambatan sebesar 17 hari, sedangkan customer Poor memiliki median keterlambatan paling tinggi, yaitu 27 hari.

Pola ini menunjukkan bahwa semakin buruk kategori Credit_Score, semakin tinggi keterlambatan pembayaran customer. Dengan demikian, nilai tinggi pada Delay_from_due_date bukan sekadar outlier statistik, tetapi merupakan sinyal penting dalam analisis risiko kredit.

Meskipun metode IQR mendeteksi beberapa nilai sebagai outlier karena berada di atas upper bound, nilai maksimum yang muncul masih berada pada kisaran yang masuk akal, yaitu hingga 67 hari. Keterlambatan lebih dari 55 hari masih dapat terjadi dalam konteks pembayaran kredit, sehingga nilai tersebut tidak langsung dianggap sebagai kesalahan data.

Selain itu, nilai negatif pada Delay_from_due_date juga tetap dipertahankan karena dapat menunjukkan pembayaran yang dilakukan lebih awal dari tanggal jatuh tempo. Oleh karena itu, outlier pada kolom Delay_from_due_date tidak dihapus dan tidak dilakukan capping.

Keputusan akhir untuk kolom Delay_from_due_date adalah mempertahankan seluruh nilai outlier karena nilai tersebut memiliki makna bisnis dan membantu model dalam membedakan customer dengan risiko kredit yang berbeda.

### 9. Num_of_Delayed_Payment



In [ ]:
show_outlier_rows(df, "Num_of_Delayed_Payment")

### 10. Changed_Credit_Limit



In [ ]:
show_outlier_rows(df, "Changed_Credit_Limit")

In [ ]:
# CHECK CHANGED_CREDIT_LIMIT OUTLIER PATTERN BY CREDIT_SCORE

print("Minimum Changed_Credit_Limit:", df["Changed_Credit_Limit"].min())
print("Maximum Changed_Credit_Limit:", df["Changed_Credit_Limit"].max())
print("Changed_Credit_Limit > 29.425:", (df["Changed_Credit_Limit"] > 29.425).sum())
print("Changed_Credit_Limit < -9.175:", (df["Changed_Credit_Limit"] < -9.175).sum())

display(
    df.groupby("Credit_Score")["Changed_Credit_Limit"]
    .describe()
)

Hasil pengecekan Changed_Credit_Limit berdasarkan Credit_Score menunjukkan bahwa nilai pada kolom ini memiliki pola yang relevan terhadap kategori credit score. Customer dengan Credit_Score Good memiliki median Changed_Credit_Limit sebesar 6,54, sedangkan customer Standard memiliki median 10,37 dan customer Poor memiliki median 9,89.

Pola ini menunjukkan bahwa customer dengan credit score yang lebih rendah cenderung memiliki perubahan limit kredit yang lebih tinggi dibandingkan customer dengan credit score Good. Selain itu, nilai maksimum pada masing-masing kategori masih berada pada kisaran yang masuk akal, yaitu 32,11 untuk Good, 35,30 untuk Standard, dan 35,82 untuk Poor.

Meskipun metode IQR mendeteksi sebagian nilai sebagai outlier, nilai tersebut tidak langsung dianggap sebagai error. Outlier pada Changed_Credit_Limit masih memiliki makna bisnis karena perubahan limit kredit dapat mencerminkan perubahan profil risiko atau kebijakan kredit terhadap customer.

Oleh karena itu, outlier pada kolom Changed_Credit_Limit tidak dihapus dan tidak dilakukan capping. Nilai tersebut tetap dipertahankan karena masih relevan dengan perbedaan Credit_Score dan dapat membantu model mempelajari pola risiko kredit customer.

### 11. Num_Credit_Inquiries



In [ ]:
show_outlier_rows(df, "Num_Credit_Inquiries")

ada kolom Num_Credit_Inquiries, metode IQR mendeteksi 185 baris sebagai outlier karena berada di atas upper bound sebesar 15,5. Nilai outlier yang muncul berada pada kisaran 16 sampai 17.

Outlier ini tidak dihapus karena nilai tersebut masih berada dalam batas valid yang sudah digunakan pada tahap cleaning sebelumnya. Pada tahap cleaning, nilai Num_Credit_Inquiries yang dianggap tidak valid adalah nilai negatif dan nilai yang lebih besar dari 17. Oleh karena itu, nilai 16 dan 17 masih dianggap valid meskipun terdeteksi sebagai outlier oleh metode IQR.

Secara bisnis, jumlah credit inquiry yang tinggi juga memiliki makna penting dalam analisis risiko kredit. Jumlah inquiry yang tinggi dapat menunjukkan bahwa customer sering mengajukan atau dicek untuk fasilitas kredit baru. Kondisi ini dapat berkaitan dengan peningkatan risiko kredit, sehingga nilai tinggi pada kolom ini justru dapat menjadi sinyal penting bagi model.

Selain itu, beberapa baris outlier juga memiliki karakteristik risiko yang kuat, seperti Credit_Mix yang buruk, Interest_Rate tinggi, Delay_from_due_date tinggi, dan Outstanding_Debt yang besar. Hal ini menunjukkan bahwa outlier pada Num_Credit_Inquiries bukan sekadar noise statistik, tetapi dapat mencerminkan perilaku kredit customer.

Keputusan akhir untuk kolom Num_Credit_Inquiries adalah mempertahankan nilai outlier yang masih berada dalam rentang valid 0 sampai 17. Tidak dilakukan drop atau capping karena nilai tersebut masih masuk akal secara konteks dan dapat membantu model mempelajari pola risiko kredit.

### 12. Outstanding_Debt



In [ ]:
show_outlier_rows(df, "Outstanding_Debt")

Pada kolom Outstanding_Debt, metode IQR mendeteksi beberapa baris sebagai outlier karena memiliki nilai utang yang tinggi. Nilai outlier yang muncul berada pada kisaran ribuan, dengan nilai maksimum sekitar 4.997.

Outlier pada kolom ini tidak dihapus karena Outstanding_Debt merupakan fitur yang sangat penting dalam analisis risiko kredit. Kolom ini menunjukkan jumlah utang customer yang masih outstanding. Semakin besar outstanding debt, semakin tinggi beban kredit yang dimiliki customer, sehingga nilai ini dapat menjadi sinyal penting dalam memprediksi Credit_Score.

Dari hasil pengecekan, banyak baris dengan nilai Outstanding_Debt tinggi juga memiliki karakteristik risiko kredit yang kuat, seperti Credit_Mix yang buruk, Interest_Rate tinggi, Delay_from_due_date tinggi, dan jumlah keterlambatan pembayaran yang tinggi. Hal ini menunjukkan bahwa nilai outlier pada Outstanding_Debt bukan sekadar noise statistik, tetapi memiliki makna bisnis yang relevan.

Selain itu, jumlah outlier pada kolom ini cukup banyak. Jika seluruh outlier dihapus, maka banyak informasi penting tentang customer dengan beban utang tinggi akan hilang. Hal ini dapat membuat model kurang mampu mengenali pola customer berisiko tinggi.

Oleh karena itu, outlier pada kolom Outstanding_Debt tidak dihapus dan tidak dilakukan capping. Nilai tersebut tetap dipertahankan karena masih masuk akal secara konteks dan dapat membantu model mempelajari pola risiko kredit customer.

### 13. Credit_Utilization_Ratio



In [ ]:
show_outlier_rows(df, "Credit_Utilization_Ratio")

Credit_Utilization_Ratio Outlier Handling

Pada kolom Credit_Utilization_Ratio, metode IQR mendeteksi 1 baris sebagai outlier karena nilainya berada di atas upper bound sebesar 49,1686. Nilai outlier yang terdeteksi adalah 50,0000.

Outlier ini tidak dihapus karena nilainya masih sangat dekat dengan batas IQR dan masih masuk akal secara konteks. Credit_Utilization_Ratio menunjukkan tingkat penggunaan kredit customer. Nilai 50 berarti customer menggunakan sekitar 50% dari limit kreditnya, sehingga nilai tersebut masih berada dalam rentang yang wajar.

Selain itu, jumlah outlier hanya 1 baris, sehingga tidak menunjukkan pola error sistematis pada kolom ini. Nilai tersebut juga tidak ekstrem karena masih berada dalam rentang logis 0 sampai 100.

Keputusan akhir untuk kolom Credit_Utilization_Ratio adalah mempertahankan outlier tersebut. Tidak dilakukan drop atau capping karena nilai masih valid secara konteks dan dapat tetap digunakan sebagai fitur numerik dalam proses modeling.

### 14. Credit_History_Age


In [ ]:
show_outlier_rows(df, "Credit_History_Age")


### 15. Total_EMI_per_month


In [ ]:
show_outlier_rows(df, "Total_EMI_per_month")

In [ ]:
# FINAL LOGICAL CHECK FOR TOTAL_EMI_PER_MONTH

emi_check = df.copy()

emi_check["EMI_to_Salary_Ratio"] = (
    emi_check["Total_EMI_per_month"] /
    emi_check["Monthly_Inhand_Salary"]
)

print("Missing Total_EMI_per_month:", df["Total_EMI_per_month"].isna().sum())
print("Total_EMI_per_month < 0:", (df["Total_EMI_per_month"] < 0).sum())
print("Total_EMI_per_month > Monthly_Inhand_Salary:", (emi_check["EMI_to_Salary_Ratio"] > 1).sum())
print("Num_of_Loan = 0 tetapi Total_EMI_per_month > 0:", ((df["Num_of_Loan"] == 0) & (df["Total_EMI_per_month"] > 0)).sum())
print("Total_EMI_per_month > 2000:", (df["Total_EMI_per_month"] > 2000).sum())

display(df["Total_EMI_per_month"].describe())

In [ ]:
# FIX REMAINING LOGICAL ISSUES IN TOTAL_EMI_PER_MONTH

before_rows = df.shape[0]

# 1. Set EMI to 0 if Num_of_Loan = 0
no_loan_positive_emi_indices = df[
    (df["Num_of_Loan"] == 0) &
    (df["Total_EMI_per_month"] > 0)
].index

df.loc[
    no_loan_positive_emi_indices,
    "Total_EMI_per_month"
] = 0

print("Num_of_Loan = 0 tetapi EMI > 0 yang diperbaiki menjadi 0:", len(no_loan_positive_emi_indices))


# 2. Drop rows where EMI is greater than monthly salary
emi_greater_than_salary_indices = df[
    df["Total_EMI_per_month"] > df["Monthly_Inhand_Salary"]
].index

df = df.drop(
    index=emi_greater_than_salary_indices
).reset_index(drop=True)

after_rows = df.shape[0]

print("Baris EMI > Monthly_Inhand_Salary yang di-drop:", len(emi_greater_than_salary_indices))
print("Jumlah baris sebelum cleaning:", before_rows)
print("Jumlah baris setelah cleaning:", after_rows)
print("Total baris yang berkurang:", before_rows - after_rows)

In [ ]:
# FINAL CHECK TOTAL_EMI_PER_MONTH AFTER LOGICAL CLEANING

emi_check = df.copy()

emi_check["EMI_to_Salary_Ratio"] = (
    emi_check["Total_EMI_per_month"] /
    emi_check["Monthly_Inhand_Salary"]
)

print("Missing Total_EMI_per_month:", df["Total_EMI_per_month"].isna().sum())
print("Total_EMI_per_month < 0:", (df["Total_EMI_per_month"] < 0).sum())
print("Total_EMI_per_month > Monthly_Inhand_Salary:", (emi_check["EMI_to_Salary_Ratio"] > 1).sum())
print("Num_of_Loan = 0 tetapi Total_EMI_per_month > 0:", ((df["Num_of_Loan"] == 0) & (df["Total_EMI_per_month"] > 0)).sum())
print("Total_EMI_per_month > 2000:", (df["Total_EMI_per_month"] > 2000).sum())

display(df["Total_EMI_per_month"].describe())

Total_EMI_per_month Outlier Handling Process

Pada tahap pengecekan outlier, kolom Total_EMI_per_month diperiksa menggunakan metode IQR. Berdasarkan hasil perhitungan, diperoleh lower bound sebesar -144,5332 dan upper bound sebesar 318,1934. Nilai yang berada di atas upper bound dianggap sebagai outlier oleh metode IQR.

Dari hasil pengecekan, terdapat 1.175 baris yang terdeteksi sebagai outlier pada kolom Total_EMI_per_month. Sebagian besar nilai outlier tersebut berada di atas 318,1934, dengan nilai maksimum sebesar 1.950.

Meskipun jumlah outlier cukup banyak, nilai tersebut tidak langsung dihapus. Hal ini karena Total_EMI_per_month merepresentasikan total cicilan bulanan customer. Dalam konteks keuangan, nilai EMI yang tinggi masih dapat masuk akal, terutama jika customer memiliki beberapa pinjaman atau pendapatan bulanan yang tinggi.

Selain itu, metode IQR hanya melihat penyebaran statistik data, bukan konteks bisnis dari fitur tersebut. Oleh karena itu, nilai yang dianggap outlier secara statistik belum tentu merupakan data yang salah. Pada kolom ini, nilai EMI tinggi justru dapat menjadi informasi penting karena menunjukkan besarnya beban cicilan bulanan customer.

Setelah outlier terdeteksi, proses tidak dilanjutkan dengan menghapus seluruh outlier. Sebaliknya, dilakukan validasi logika dengan membandingkan Total_EMI_per_month terhadap fitur lain, terutama Monthly_Inhand_Salary dan Num_of_Loan.

Validasi pertama dilakukan untuk memastikan tidak ada nilai Total_EMI_per_month yang missing. Hasil pengecekan menunjukkan bahwa missing value pada kolom ini sudah bernilai 0, sehingga tidak ada lagi missing value yang perlu diperbaiki atau dihapus.

Validasi kedua dilakukan untuk memastikan tidak ada nilai Total_EMI_per_month yang bernilai negatif. Hasil pengecekan menunjukkan bahwa jumlah nilai negatif adalah 0. Artinya, tidak terdapat nilai EMI yang tidak valid dari sisi tanda bilangan.

Validasi ketiga dilakukan dengan mengecek apakah terdapat nilai Total_EMI_per_month yang lebih besar dari Monthly_Inhand_Salary. Dari hasil pengecekan, masih ditemukan 4 baris dengan kondisi tersebut. Kondisi ini dianggap tidak wajar karena total cicilan bulanan yang lebih besar dari pendapatan bulanan customer dapat menunjukkan anomali data. Karena jumlah kasusnya sangat kecil, baris tersebut diputuskan untuk dihapus.

Validasi keempat dilakukan dengan mengecek apakah terdapat customer dengan Num_of_Loan = 0 tetapi memiliki Total_EMI_per_month > 0. Dari hasil pengecekan, masih ditemukan 80 baris dengan kondisi tersebut. Kondisi ini tidak konsisten secara logika karena customer yang tidak memiliki loan seharusnya tidak memiliki cicilan bulanan. Oleh karena itu, nilai Total_EMI_per_month pada baris tersebut diperbaiki menjadi 0.

Validasi terakhir dilakukan dengan mengecek apakah masih terdapat nilai Total_EMI_per_month > 2000. Hasil pengecekan menunjukkan bahwa jumlahnya adalah 0. Hal ini menunjukkan bahwa nilai ekstrem yang sangat tidak wajar sudah tidak ada pada dataset.

Berdasarkan seluruh proses tersebut, keputusan akhir untuk kolom Total_EMI_per_month adalah tidak menghapus outlier berdasarkan metode IQR secara otomatis. Outlier tetap dipertahankan apabila masih masuk akal secara konteks bisnis. Pembersihan hanya dilakukan pada data yang tidak konsisten secara logika, yaitu EMI yang lebih besar dari pendapatan bulanan dan EMI positif pada customer yang tidak memiliki loan.

Dengan keputusan ini, informasi penting mengenai beban cicilan customer tetap dipertahankan untuk proses modeling. Kolom Total_EMI_per_month tetap digunakan sebagai fitur numerik karena dapat membantu model memahami kondisi kewajiban pembayaran bulanan customer dan hubungannya dengan Credit_Score.


### 16. Amount_invested_monthly


In [ ]:
show_outlier_rows(df, "Amount_invested_monthly")

In [ ]:
# FINAL LOGICAL CHECK FOR AMOUNT_INVESTED_MONTHLY

investment_check = df.copy()

investment_check["Investment_to_Salary_Ratio"] = (
    investment_check["Amount_invested_monthly"] /
    investment_check["Monthly_Inhand_Salary"]
)

print("Missing Amount_invested_monthly:", df["Amount_invested_monthly"].isna().sum())
print("Amount_invested_monthly < 0:", (df["Amount_invested_monthly"] < 0).sum())
print(
    "Amount_invested_monthly > Monthly_Inhand_Salary:",
    (df["Amount_invested_monthly"] > df["Monthly_Inhand_Salary"]).sum()
)
print(
    "Investment_to_Salary_Ratio > 1:",
    (investment_check["Investment_to_Salary_Ratio"] > 1).sum()
)
print(
    "Investment_to_Salary_Ratio > 0.8:",
    (investment_check["Investment_to_Salary_Ratio"] > 0.8).sum()
)

display(investment_check["Investment_to_Salary_Ratio"].describe())
display(df["Amount_invested_monthly"].describe())

Pada kolom Amount_invested_monthly, metode IQR mendeteksi 1.815 baris sebagai outlier. Berdasarkan hasil perhitungan, lower bound berada pada -174,9023 dan upper bound berada pada 483,8009. Nilai yang berada di atas upper bound dianggap sebagai outlier secara statistik.

Namun, outlier pada kolom Amount_invested_monthly tidak langsung dihapus. Hal ini karena kolom ini merepresentasikan jumlah uang yang diinvestasikan customer setiap bulan. Dalam konteks keuangan, nilai investasi bulanan yang tinggi masih dapat masuk akal, terutama untuk customer dengan pendapatan bulanan yang tinggi.

Metode IQR hanya melihat penyebaran data secara statistik dan tidak mempertimbangkan konteks bisnis. Oleh karena itu, nilai yang melewati upper bound IQR belum tentu merupakan kesalahan data. Pada beberapa baris, customer memiliki Monthly_Inhand_Salary yang besar sehingga nilai Amount_invested_monthly yang tinggi masih proporsional terhadap pendapatannya.

Sebagai contoh, customer dengan pendapatan bulanan yang tinggi dapat memiliki nilai investasi bulanan di atas 483,8009, tetapi nilai tersebut tetap valid apabila masih jauh lebih kecil dari Monthly_Inhand_Salary. Oleh karena itu, penghapusan seluruh outlier berdasarkan IQR dapat menghilangkan informasi penting mengenai perilaku keuangan customer.

Validasi outlier dilakukan dengan membandingkan Amount_invested_monthly terhadap Monthly_Inhand_Salary. Nilai dianggap bermasalah apabila Amount_invested_monthly bernilai negatif atau lebih besar dari pendapatan bulanan customer. Jika nilai investasi masih berada dalam batas yang masuk akal terhadap pendapatan bulanan, maka nilai tersebut dipertahankan.

Keputusan akhir untuk kolom Amount_invested_monthly adalah tidak menghapus outlier hanya berdasarkan metode IQR. Outlier tetap dipertahankan apabila masih logis secara konteks bisnis. Pembersihan hanya dilakukan pada nilai yang tidak valid, seperti nilai negatif, missing value yang tidak dapat diperbaiki, atau nilai investasi yang melebihi pendapatan bulanan.

Dengan keputusan ini, informasi mengenai pola investasi customer tetap dipertahankan dan dapat digunakan sebagai fitur dalam proses modeling untuk membantu memprediksi Credit_Score.


### 17. Monthly_Balance

In [ ]:
show_outlier_rows(df, "Monthly_Balance")

In [ ]:
# FINAL LOGICAL CHECK FOR MONTHLY_BALANCE

balance_check = df.copy()

balance_check["Balance_to_Salary_Ratio"] = (
    balance_check["Monthly_Balance"] /
    balance_check["Monthly_Inhand_Salary"]
)

balance_check["Used_Money_Ratio"] = (
    (
        balance_check["Total_EMI_per_month"] +
        balance_check["Amount_invested_monthly"] +
        balance_check["Monthly_Balance"]
    ) /
    balance_check["Monthly_Inhand_Salary"]
)

print("Missing Monthly_Balance:", df["Monthly_Balance"].isna().sum())
print("Monthly_Balance < 0:", (df["Monthly_Balance"] < 0).sum())
print(
    "Monthly_Balance > Monthly_Inhand_Salary:",
    (df["Monthly_Balance"] > df["Monthly_Inhand_Salary"]).sum()
)
print(
    "Balance_to_Salary_Ratio > 1:",
    (balance_check["Balance_to_Salary_Ratio"] > 1).sum()
)
print(
    "Used_Money_Ratio > 1:",
    (balance_check["Used_Money_Ratio"] > 1).sum()
)

display(balance_check["Balance_to_Salary_Ratio"].describe())
display(balance_check["Used_Money_Ratio"].describe())
display(df["Monthly_Balance"].describe())

Pada kolom Monthly_Balance, metode IQR mendeteksi 1.735 baris sebagai outlier. Berdasarkan hasil perhitungan, lower bound berada pada -34,8464 dan upper bound berada pada 777,1557. Nilai yang berada di atas upper bound dianggap sebagai outlier secara statistik.

Namun, outlier pada kolom Monthly_Balance tidak langsung dihapus. Hal ini karena Monthly_Balance merepresentasikan sisa saldo bulanan customer setelah pendapatan digunakan untuk kebutuhan, cicilan, dan investasi. Dalam konteks keuangan, customer dengan pendapatan bulanan tinggi dapat memiliki saldo bulanan yang lebih tinggi dibandingkan customer lain.

Metode IQR hanya melihat penyebaran data secara statistik dan tidak mempertimbangkan konteks bisnis dari fitur tersebut. Oleh karena itu, nilai yang melewati batas IQR belum tentu merupakan kesalahan data. Pada beberapa baris, customer memiliki Monthly_Inhand_Salary yang tinggi, sehingga nilai Monthly_Balance yang berada di atas 777,1557 masih dapat dianggap masuk akal.

Penghapusan seluruh outlier berdasarkan IQR dapat menyebabkan hilangnya informasi penting mengenai kondisi keuangan customer. Nilai Monthly_Balance yang tinggi dapat menunjukkan kemampuan customer dalam menyisakan dana setelah pengeluaran bulanan, sehingga fitur ini tetap relevan untuk prediksi Credit_Score.

Validasi outlier dilakukan dengan membandingkan Monthly_Balance terhadap Monthly_Inhand_Salary. Nilai dianggap bermasalah apabila Monthly_Balance bernilai negatif atau lebih besar dari pendapatan bulanan customer. Selain itu, dilakukan juga pengecekan tambahan menggunakan rasio gabungan antara Total_EMI_per_month, Amount_invested_monthly, dan Monthly_Balance terhadap Monthly_Inhand_Salary untuk melihat apakah pola cashflow masih masuk akal.

Keputusan akhir untuk kolom Monthly_Balance adalah tidak menghapus outlier hanya berdasarkan metode IQR. Outlier tetap dipertahankan apabila masih logis secara konteks bisnis. Pembersihan hanya dilakukan pada nilai yang tidak valid, seperti nilai negatif, missing value yang tidak dapat diperbaiki, atau nilai saldo bulanan yang tidak masuk akal terhadap pendapatan bulanan.

Dengan keputusan ini, informasi mengenai kondisi saldo bulanan customer tetap dipertahankan dan dapat digunakan sebagai fitur numerik dalam proses modeling untuk membantu memprediksi Credit_Score.

## Summary Outlier Handling pada Dataset Credit Score

Dokumen ini merangkum kolom-kolom yang sudah dicek outlier-nya selama proses data cleaning. Fokus utama dari proses ini adalah membedakan antara **outlier statistik** dan **nilai yang benar-benar tidak valid secara logika bisnis**. Pada dataset credit score, beberapa fitur numerik memang memiliki distribusi yang right-skewed karena berhubungan dengan kondisi finansial customer, sehingga nilai yang melewati batas IQR tidak selalu berarti salah.

---

## Prinsip Umum Outlier Handling

- Outlier dicek menggunakan metode IQR dengan rumus:
  - `Q1 = kuartil pertama`
  - `Q3 = kuartil ketiga`
  - `IQR = Q3 - Q1`
  - `Lower Bound = Q1 - 1.5 * IQR`
  - `Upper Bound = Q3 + 1.5 * IQR`
- Nilai di bawah lower bound atau di atas upper bound dianggap sebagai outlier secara statistik.
- Namun, outlier statistik tidak langsung dihapus.
- Keputusan cleaning dibuat berdasarkan konteks bisnis, hubungan antarfitur, dan validitas logika data.
- Pada fitur keuangan seperti pendapatan, cicilan, investasi, outstanding debt, dan saldo bulanan, nilai tinggi masih bisa valid.
- Nilai hanya diperbaiki atau dihapus jika tidak masuk akal secara logika, seperti:
  - nilai negatif pada fitur yang seharusnya tidak negatif,
  - jumlah loan tidak konsisten dengan tipe loan,
  - customer tanpa loan tetapi memiliki EMI,
  - EMI lebih besar dari pendapatan bulanan,
  - nilai ekstrem yang jelas tidak realistis,
  - missing value yang tidak dapat diperbaiki dari `Customer_ID` yang sama.
- Kolom identitas seperti `Unnamed: 0`, `ID`, `Customer_ID`, `Name`, dan `SSN` tidak digunakan sebagai fitur modeling.
- `Customer_ID` hanya digunakan sementara untuk proses cleaning berbasis riwayat customer, lalu tidak digunakan sebagai fitur model.

---

## 1. Annual_Income

- **Jenis fitur:** Numerik.
- **Makna fitur:** Pendapatan tahunan customer.
- **Masalah awal:** Terdapat nilai ekstrem tidak wajar pada `Annual_Income`, terutama nilai yang sangat besar dan tidak konsisten dengan kondisi customer.
- **Proses yang dilakukan:**
  - Nilai ekstrem yang jelas tidak realistis diperiksa terlebih dahulu.
  - Nilai yang tidak masuk akal diperbaiki menggunakan referensi dari `Customer_ID` yang sama jika tersedia.
  - Jika tidak ada referensi customer yang valid, nilai dapat dibuat missing terlebih dahulu.
  - Missing value kemudian dicek kembali menggunakan hubungan dengan `Monthly_Inhand_Salary`.
  - Jika memungkinkan, `Annual_Income` diperbaiki menggunakan `Monthly_Inhand_Salary × 12`.
  - Sisa missing yang tidak memiliki referensi cukup kuat dihapus.
- **Hasil setelah cleaning:**
  - Nilai maksimum `Annual_Income` menjadi sekitar 179.987,28.
  - Tidak ada missing value yang tersisa pada kolom ini.
- **Outlier IQR:**
  - Masih terdapat nilai tinggi yang melewati upper bound IQR.
  - Nilai tinggi tersebut tidak langsung dihapus karena pendapatan tinggi masih mungkin valid.
- **Keputusan akhir:**
  - Outlier IQR pada `Annual_Income` dipertahankan jika masih berada dalam rentang yang masuk akal.
  - Nilai ekstrem yang tidak realistis sudah diperbaiki sebelumnya.
  - Kolom tetap digunakan sebagai fitur numerik dalam modeling.
- **Alasan:**
  - Pendapatan tinggi dapat menjadi informasi penting dalam prediksi credit score.
  - Menghapus semua outlier IQR dapat menghilangkan customer berpendapatan tinggi yang sebenarnya valid.

---

## 2. Num_Bank_Accounts

- **Jenis fitur:** Numerik diskrit.
- **Makna fitur:** Jumlah rekening bank yang dimiliki customer.
- **Masalah awal:** Terdapat nilai missing dan kemungkinan nilai tidak wajar.
- **Proses yang dilakukan:**
  - Missing value dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Karena tidak tersedia referensi customer yang valid untuk beberapa missing, baris tersebut dihapus.
  - Nilai negatif atau nilai yang terlalu tinggi dianggap tidak valid.
- **Aturan validasi:**
  - Nilai negatif tidak valid.
  - Nilai yang terlalu besar dan tidak realistis dibersihkan.
  - Batas realistis yang digunakan adalah jumlah akun bank yang masih masuk akal secara konteks dataset.
- **Keputusan akhir:**
  - Missing value yang tidak bisa direpair dihapus.
  - Nilai yang tidak realistis dibersihkan.
  - Kolom tetap digunakan sebagai fitur numerik.
- **Alasan:**
  - Jumlah rekening bank dapat berkaitan dengan profil finansial customer.
  - Nilai ekstrem yang tidak realistis dapat mengganggu model.

---

## 3. Num_Credit_Card

- **Jenis fitur:** Numerik diskrit.
- **Makna fitur:** Jumlah kartu kredit yang dimiliki customer.
- **Masalah awal:** Terdapat missing value dan nilai ekstrem.
- **Proses yang dilakukan:**
  - Missing value dicek menggunakan `Customer_ID` yang sama.
  - Missing yang tidak memiliki referensi customer valid dihapus.
  - Nilai negatif atau nilai terlalu besar dibersihkan.
- **Aturan validasi:**
  - Nilai negatif tidak valid.
  - Nilai `Num_Credit_Card > 25` dianggap tidak realistis dan dibersihkan.
- **Outlier IQR:**
  - Beberapa nilai seperti 16, 18, atau 24 dapat terdeteksi sebagai outlier secara statistik.
  - Namun, selama nilainya masih `≤ 25`, nilai tersebut dipertahankan.
- **Keputusan akhir:**
  - Missing yang tidak dapat direpair dihapus.
  - Nilai ekstrem di atas batas realistis dibersihkan.
  - Outlier IQR yang masih dalam batas valid dipertahankan.
- **Alasan:**
  - Banyaknya kartu kredit merupakan sinyal penting dalam analisis risiko kredit.
  - Nilai tinggi yang masih masuk akal tidak boleh langsung dihapus.

---

## 4. Interest_Rate

- **Jenis fitur:** Numerik.
- **Makna fitur:** Tingkat bunga kredit customer.
- **Masalah awal:** Terdapat missing value dan nilai yang terlalu ekstrem.
- **Proses yang dilakukan:**
  - Missing value dicek berdasarkan `Customer_ID` yang sama.
  - Missing yang memiliki referensi customer valid diperbaiki menggunakan nilai dari customer yang sama.
  - Missing yang tidak memiliki referensi dihapus.
  - Nilai negatif atau nilai terlalu tinggi dibersihkan.
- **Aturan validasi:**
  - Nilai negatif tidak valid.
  - Nilai `Interest_Rate > 34` dianggap tidak valid dan dibersihkan.
- **Keputusan akhir:**
  - Missing yang dapat direpair diperbaiki.
  - Missing tanpa referensi dihapus.
  - Nilai di atas batas realistis dibersihkan.
  - Kolom tetap digunakan sebagai fitur numerik.
- **Alasan:**
  - Interest rate merupakan indikator penting untuk membedakan profil risiko customer.
  - Interest rate tinggi masih bisa valid sampai batas tertentu, tetapi nilai yang terlalu ekstrem perlu dibersihkan.

---

## 5. Num_of_Loan

- **Jenis fitur:** Numerik diskrit.
- **Makna fitur:** Jumlah loan yang dimiliki customer.
- **Masalah awal:** Terdapat missing value dan kemungkinan ketidaksesuaian dengan `Type_of_Loan`.
- **Proses yang dilakukan:**
  - `Num_of_Loan` divalidasi dengan `Type_of_Loan`.
  - Jumlah loan dihitung berdasarkan jumlah item loan pada `Type_of_Loan`, bukan berdasarkan jumlah tipe unik.
  - Jika `Type_of_Loan = No Loan`, maka jumlah loan seharusnya 0.
  - Missing `Num_of_Loan` diperbaiki menggunakan jumlah item dari `Type_of_Loan` jika tersedia.
  - Sisa missing yang tidak memiliki dasar perbaikan dihapus.
- **Aturan validasi:**
  - Nilai negatif tidak valid.
  - Nilai `Num_of_Loan > 9` dianggap tidak realistis.
- **Keputusan akhir:**
  - Missing diperbaiki menggunakan informasi dari `Type_of_Loan` jika memungkinkan.
  - Sisa missing dihapus.
  - Nilai tidak valid dibersihkan.
  - Kolom tetap digunakan sebagai fitur numerik.
- **Alasan:**
  - Jumlah loan sangat relevan untuk menilai beban kredit customer.
  - Konsistensi antara `Num_of_Loan` dan `Type_of_Loan` penting agar fitur tidak saling bertentangan.

---

## 6. Delay_from_due_date

- **Jenis fitur:** Numerik.
- **Makna fitur:** Jumlah hari keterlambatan dari tanggal jatuh tempo.
- **Hasil outlier:**
  - Upper bound IQR sekitar 55.
  - Terdapat sekitar 899 outlier dengan nilai sekitar 56 sampai 67.
- **Proses yang dilakukan:**
  - Nilai outlier dicek berdasarkan distribusi dan hubungannya dengan `Credit_Score`.
  - Nilai tinggi pada `Delay_from_due_date` tidak langsung dihapus.
  - Nilai negatif juga tidak langsung dianggap salah karena dapat berarti pembayaran dilakukan lebih awal dari jatuh tempo.
- **Insight berdasarkan `Credit_Score`:**
  - Customer dengan score `Good` cenderung memiliki median delay lebih rendah.
  - Customer dengan score `Poor` cenderung memiliki median delay lebih tinggi.
  - Ini menunjukkan bahwa fitur ini memiliki hubungan yang kuat dengan risiko kredit.
- **Keputusan akhir:**
  - Outlier IQR pada `Delay_from_due_date` dipertahankan.
  - Tidak dilakukan cap atau drop hanya karena melewati upper bound IQR.
- **Alasan:**
  - Keterlambatan pembayaran yang tinggi merupakan sinyal penting dalam prediksi credit score.
  - Menghapus nilai tinggi dapat menghilangkan informasi penting tentang customer berisiko.

---

## 7. Num_of_Delayed_Payment

- **Jenis fitur:** Numerik diskrit.
- **Makna fitur:** Jumlah pembayaran yang terlambat.
- **Masalah awal:** Terdapat missing value dan kemungkinan nilai tidak valid.
- **Proses yang dilakukan:**
  - Missing value dicek berdasarkan `Customer_ID` yang sama.
  - Jika customer yang sama memiliki nilai valid pada bulan lain, missing value diperbaiki menggunakan median customer tersebut.
  - Jika tidak ada referensi customer, sisa missing dihapus.
  - Nilai negatif atau terlalu besar dibersihkan.
- **Aturan validasi:**
  - Nilai negatif tidak valid.
  - Nilai `Num_of_Delayed_Payment > 28` dianggap tidak realistis.
- **Keputusan akhir:**
  - Missing yang bisa direpair diperbaiki.
  - Missing tanpa referensi dihapus.
  - Nilai tidak valid dibersihkan.
  - Kolom tetap digunakan sebagai fitur numerik.
- **Alasan:**
  - Jumlah keterlambatan pembayaran adalah indikator risiko kredit yang penting.
  - Nilai tinggi yang masih realistis perlu dipertahankan karena berpotensi membantu model membedakan customer berisiko.

---

## 8. Changed_Credit_Limit

- **Jenis fitur:** Numerik.
- **Makna fitur:** Perubahan limit kredit customer.
- **Hasil outlier:**
  - Terdapat sekitar 131 outlier berdasarkan IQR.
  - Lower bound sekitar -9,175.
  - Upper bound sekitar 29,425.
  - Nilai outlier berada pada kisaran sekitar 29 sampai 35,82.
- **Proses yang dilakukan:**
  - Missing value yang tidak memiliki referensi customer dihapus.
  - Nilai negatif tidak dianggap otomatis salah karena dapat menunjukkan penurunan limit kredit.
  - Outlier IQR dicek berdasarkan distribusi dan hubungan dengan `Credit_Score`.
- **Insight berdasarkan `Credit_Score`:**
  - Nilai `Changed_Credit_Limit` masih berada dalam rentang yang masuk akal.
  - Perubahan limit kredit dapat berbeda antar customer tergantung profil kredit.
- **Keputusan akhir:**
  - Outlier IQR dipertahankan.
  - Tidak dilakukan cap atau drop pada nilai yang masih masuk akal.
- **Alasan:**
  - Perubahan limit kredit dapat menjadi sinyal penting dalam penilaian risiko kredit.
  - Nilai negatif maupun nilai tinggi masih bisa valid secara bisnis.

---

## 9. Num_Credit_Inquiries

- **Jenis fitur:** Numerik diskrit.
- **Makna fitur:** Jumlah pertanyaan atau pengecekan kredit terhadap customer.
- **Hasil outlier:**
  - Terdapat sekitar 185 outlier berdasarkan IQR.
  - Upper bound sekitar 15,5.
  - Nilai outlier umumnya berada pada 16 sampai 17.
- **Proses yang dilakukan:**
  - Missing value dicek menggunakan `Customer_ID` yang sama.
  - Missing yang dapat direpair diperbaiki dengan median customer yang sama.
  - Missing tanpa referensi dihapus.
  - Nilai ekstrem tidak realistis dibersihkan.
- **Aturan validasi:**
  - Nilai `Num_Credit_Inquiries > 17` dianggap tidak realistis dan dibersihkan.
- **Keputusan akhir:**
  - Nilai outlier IQR 16 sampai 17 dipertahankan karena masih dalam batas valid.
  - Missing tanpa referensi dihapus.
  - Kolom tetap digunakan sebagai fitur numerik.
- **Alasan:**
  - Banyaknya credit inquiry dapat menunjukkan aktivitas pencarian kredit.
  - Aktivitas ini dapat berkaitan dengan risiko kredit customer.
  - Nilai tinggi yang masih realistis tidak boleh langsung dihapus.

---

## 10. Outstanding_Debt

- **Jenis fitur:** Numerik.
- **Makna fitur:** Total utang yang masih belum dibayar oleh customer.
- **Hasil pengecekan:**
  - Ditemukan nilai tinggi pada `Outstanding_Debt`, termasuk nilai mendekati 4.997.
  - Banyak customer dengan outstanding debt tinggi memiliki `Credit_Mix = Bad` atau `Credit_Score = Poor/Standard`.
- **Proses yang dilakukan:**
  - Nilai tinggi diperiksa secara konteks.
  - Tidak ditemukan alasan kuat untuk menganggap nilai tinggi sebagai error jika masih realistis.
  - Nilai tinggi dibandingkan dengan informasi risiko lain seperti `Credit_Mix` dan `Credit_Score`.
- **Keputusan akhir:**
  - Outlier pada `Outstanding_Debt` dipertahankan.
  - Tidak dilakukan cap atau drop hanya berdasarkan IQR.
- **Alasan:**
  - Outstanding debt tinggi adalah sinyal risiko kredit yang sangat penting.
  - Menghapus nilai tinggi dapat menghilangkan pola customer dengan risiko kredit buruk.

---

## 11. Credit_Utilization_Ratio

- **Jenis fitur:** Numerik.
- **Makna fitur:** Rasio pemanfaatan kredit customer.
- **Hasil outlier:**
  - Terdapat 1 outlier berdasarkan IQR.
  - Upper bound sekitar 49,1686.
  - Nilai outlier maksimum adalah 50,0.
- **Proses yang dilakukan:**
  - Nilai dicek apakah masih berada dalam rentang logis.
  - Karena credit utilization ratio berada pada rentang 0 sampai 100, nilai 50 masih valid.
- **Keputusan akhir:**
  - Outlier dipertahankan.
  - Tidak dilakukan cap atau drop.
- **Alasan:**
  - Nilai 50 masih realistis sebagai rasio pemanfaatan kredit.
  - Hanya terdapat 1 outlier, dan nilainya tidak ekstrem secara bisnis.

---

## 12. Credit_History_Age

- **Jenis fitur:** Numerik.
- **Makna fitur:** Lama riwayat kredit customer dalam satuan bulan.
- **Masalah utama:**
  - Terdapat missing value yang cukup besar.
  - Sebelumnya juga dicek apakah `Credit_History_Age > Age × 12`, karena riwayat kredit tidak boleh lebih tua dari usia customer.
- **Proses yang dilakukan:**
  - Nilai yang tidak logis terhadap umur customer diperiksa dan diperbaiki jika memungkinkan.
  - Missing value dicek menggunakan referensi dari `Customer_ID` yang sama.
  - Sebanyak 1.615 missing berhasil diperbaiki menggunakan median `Credit_History_Age` dari customer yang sama.
  - Masih terdapat 2.909 missing yang tidak memiliki referensi customer valid.
- **Keputusan akhir:**
  - Sisa missing tidak dihapus karena jumlahnya besar.
  - Dibuat fitur indikator baru `Credit_History_Age_Missing`.
  - Nilai missing pada `Credit_History_Age` akan ditangani di preprocessing pipeline menggunakan median imputer.
- **Alasan:**
  - Menghapus 2.909 baris akan mengurangi data secara besar.
  - Missing indicator dapat membantu model menangkap pola bahwa ketiadaan informasi riwayat kredit mungkin relevan terhadap `Credit_Score`.
  - Median imputer digunakan agar preprocessing tetap aman dan tidak menyebabkan data leakage.

---

## 13. Total_EMI_per_month

- **Jenis fitur:** Numerik.
- **Makna fitur:** Total cicilan bulanan customer.
- **Hasil outlier IQR terbaru:**
  - Lower bound: -144,5332.
  - Upper bound: 318,1934.
  - Total outlier: 1.175 baris.
  - Nilai maksimum setelah cleaning: 1.950.
- **Proses yang dilakukan dari tahap outlier:**
  - Outlier dicek menggunakan metode IQR.
  - Terdapat banyak nilai di atas upper bound.
  - Nilai tersebut tidak langsung dihapus karena EMI tinggi masih bisa valid untuk customer dengan banyak loan atau pendapatan tinggi.
  - Dilakukan validasi logika dengan `Monthly_Inhand_Salary` dan `Num_of_Loan`.
- **Validasi logika yang dilakukan:**
  - Mengecek missing value pada `Total_EMI_per_month`.
  - Mengecek nilai negatif.
  - Mengecek apakah `Total_EMI_per_month > Monthly_Inhand_Salary`.
  - Mengecek apakah `Num_of_Loan = 0` tetapi `Total_EMI_per_month > 0`.
  - Mengecek apakah terdapat nilai `Total_EMI_per_month > 2000`.
- **Temuan penting:**
  - Missing value sudah dibersihkan menjadi 0 missing.
  - Tidak ada nilai negatif.
  - Tidak ada nilai di atas 2000.
  - Masih ditemukan beberapa kasus EMI lebih besar dari salary dan beberapa kasus customer tanpa loan tetapi memiliki EMI.
- **Tindakan cleaning:**
  - Jika `Num_of_Loan = 0` tetapi `Total_EMI_per_month > 0`, maka EMI diperbaiki menjadi 0.
  - Jika `Total_EMI_per_month > Monthly_Inhand_Salary`, baris tersebut dihapus karena jumlahnya kecil dan tidak logis.
  - Outlier IQR lain yang masih masuk akal dipertahankan.
- **Keputusan akhir:**
  - Outlier IQR tidak dihapus otomatis.
  - Hanya nilai yang tidak logis secara bisnis yang diperbaiki atau dihapus.
  - Kolom tetap digunakan sebagai fitur numerik.
- **Alasan:**
  - EMI tinggi adalah indikator penting tentang beban cicilan customer.
  - Nilai tinggi dapat membantu model mengenali customer dengan tekanan pembayaran lebih besar.

---

## 14. Amount_invested_monthly

- **Jenis fitur:** Numerik.
- **Makna fitur:** Jumlah uang yang diinvestasikan customer setiap bulan.
- **Hasil outlier IQR:**
  - Lower bound: -174,9023.
  - Upper bound: 483,8009.
  - Total outlier: 1.815 baris.
- **Proses yang dilakukan:**
  - Missing value dicek menggunakan `Customer_ID` yang sama.
  - Missing yang bisa direpair diperbaiki menggunakan median customer yang sama.
  - Missing tanpa referensi customer dihapus.
  - Outlier dicek menggunakan IQR.
  - Nilai investasi tinggi tidak langsung dihapus.
- **Validasi logika yang dilakukan:**
  - Mengecek `Amount_invested_monthly < 0`.
  - Mengecek `Amount_invested_monthly > Monthly_Inhand_Salary`.
  - Mengecek rasio `Amount_invested_monthly / Monthly_Inhand_Salary`.
- **Insight:**
  - Beberapa customer memiliki income bulanan tinggi, sehingga investasi bulanan di atas 483,8009 masih dapat dianggap valid.
  - Contoh nilai investasi sekitar 1.900 masih bisa masuk akal jika salary bulanan sekitar 15.000.
- **Keputusan akhir:**
  - Missing value yang tidak dapat direpair dihapus.
  - Outlier IQR tidak dihapus otomatis.
  - Nilai dipertahankan jika masih masuk akal terhadap salary bulanan.
- **Alasan:**
  - Nilai investasi bulanan mencerminkan perilaku pengelolaan keuangan customer.
  - Menghapus seluruh outlier dapat menghilangkan informasi tentang customer dengan pendapatan dan investasi tinggi.

---

## 15. Monthly_Balance

- **Jenis fitur:** Numerik.
- **Makna fitur:** Sisa saldo bulanan customer.
- **Hasil outlier IQR:**
  - Lower bound: -34,8464.
  - Upper bound: 777,1557.
  - Total outlier: 1.735 baris.
- **Proses yang dilakukan:**
  - Missing value dicek menggunakan `Customer_ID` yang sama.
  - Missing yang dapat direpair diperbaiki menggunakan median customer yang sama.
  - Missing tanpa referensi customer dihapus.
  - Outlier dicek menggunakan metode IQR.
  - Nilai saldo bulanan tinggi tidak langsung dihapus.
- **Validasi logika yang dilakukan:**
  - Mengecek `Monthly_Balance < 0`.
  - Mengecek `Monthly_Balance > Monthly_Inhand_Salary`.
  - Mengecek rasio `Monthly_Balance / Monthly_Inhand_Salary`.
  - Mengecek cashflow sederhana menggunakan gabungan `Total_EMI_per_month`, `Amount_invested_monthly`, dan `Monthly_Balance` terhadap `Monthly_Inhand_Salary`.
- **Insight:**
  - Banyak customer dengan `Monthly_Balance` tinggi memiliki `Monthly_Inhand_Salary` yang juga tinggi.
  - Nilai saldo sekitar 1.400 sampai 1.500 masih bisa masuk akal jika salary bulanan berada pada kisaran 13.000 sampai 15.000.
- **Keputusan akhir:**
  - Missing value yang tidak dapat direpair dihapus.
  - Outlier IQR tidak dihapus otomatis.
  - Nilai dipertahankan jika masih logis terhadap salary bulanan.
- **Alasan:**
  - Monthly balance tinggi dapat menunjukkan kondisi keuangan customer yang lebih stabil.
  - Fitur ini penting untuk membantu model memahami kemampuan customer menyisakan dana setelah pengeluaran bulanan.

---

## Ringkasan Keputusan Akhir

| Kolom | Tipe Masalah | Tindakan Akhir |
|---|---|---|
| `Annual_Income` | Nilai ekstrem dan outlier IQR | Nilai ekstrem tidak realistis diperbaiki, outlier valid dipertahankan |
| `Num_Bank_Accounts` | Missing dan nilai tidak valid | Missing tanpa referensi dihapus, nilai tidak realistis dibersihkan |
| `Num_Credit_Card` | Missing dan outlier IQR | Missing tanpa referensi dihapus, outlier valid dipertahankan |
| `Interest_Rate` | Missing dan nilai ekstrem | Missing direpair/drop, nilai > 34 dibersihkan |
| `Num_of_Loan` | Missing dan inkonsistensi dengan `Type_of_Loan` | Direpair menggunakan `Type_of_Loan`, sisa missing dihapus |
| `Delay_from_due_date` | Outlier IQR | Dipertahankan karena merupakan sinyal risiko kredit |
| `Num_of_Delayed_Payment` | Missing dan nilai ekstrem | Missing direpair/drop, nilai tidak valid dibersihkan |
| `Changed_Credit_Limit` | Outlier IQR | Dipertahankan karena masih masuk akal secara bisnis |
| `Num_Credit_Inquiries` | Outlier IQR | Nilai 16-17 dipertahankan karena masih valid |
| `Outstanding_Debt` | Outlier tinggi | Dipertahankan karena penting untuk risiko kredit |
| `Credit_Utilization_Ratio` | 1 outlier IQR | Dipertahankan karena nilai 50 masih valid |
| `Credit_History_Age` | Missing besar dan validasi usia | Missing indicator dibuat, sisa missing diimpute median di pipeline |
| `Total_EMI_per_month` | Outlier IQR dan inkonsistensi logika | Outlier valid dipertahankan, EMI tanpa loan diubah 0, EMI > salary dihapus |
| `Amount_invested_monthly` | Outlier IQR | Missing tanpa referensi dihapus, outlier valid dipertahankan |
| `Monthly_Balance` | Outlier IQR | Missing tanpa referensi dihapus, outlier valid dipertahankan |

---

## Kesimpulan

- Tidak semua outlier statistik dihapus.
- Keputusan cleaning dibuat berdasarkan gabungan antara metode statistik dan konteks bisnis.
- Outlier yang masih masuk akal secara konteks dipertahankan karena dapat memberikan informasi penting untuk model.
- Nilai yang tidak logis secara bisnis diperbaiki atau dihapus.
- Fitur dengan missing besar seperti `Credit_History_Age` tidak langsung dihapus, tetapi ditangani dengan missing indicator dan median imputer pada preprocessing pipeline.
- Pendekatan ini menjaga keseimbangan antara kebersihan data dan pelestarian informasi penting untuk prediksi `Credit_Score`.


## G. Feature Selection dan Feature Engineering(Ide Untuk di Pipeline)

## Feature Selection dan Feature Engineering untuk Model 1 dan Model 2

Pada tahap feature selection dan feature engineering, fitur yang digunakan untuk proses modeling dipilih berdasarkan hasil data cleaning, validasi outlier, dan logika bisnis pada kasus credit scoring. Tujuan utama dari tahap ini adalah memastikan bahwa model hanya menggunakan fitur yang relevan, aman, dan dapat dijelaskan secara bisnis untuk memprediksi `Credit_Score`.

Dalam proyek ini, eksperimen modeling dibagi menjadi dua feature set utama, yaitu **Model 1** dan **Model 2**. Pembagian ini dilakukan agar performa model dapat dibandingkan berdasarkan perbedaan strategi penggunaan fitur.

**Model 1** menggunakan pendekatan **full feature engineering**, yaitu fitur original yang sudah dipilih kemudian ditambah dengan beberapa hasil rekayasa fitur, seperti missing indicator, ratio features, loan type flags, dan pemisahan `Payment_Behaviour` menjadi dua fitur baru.

**Model 2** menggunakan pendekatan **original features + loan feature engineering**, yaitu fitur original dari dataset tetap digunakan lebih lengkap, kemudian informasi dari `Type_of_Loan` diubah menjadi beberapa fitur binary atau loan flags.

Dengan pembagian ini, eksperimen dapat menunjukkan apakah performa model lebih baik ketika menggunakan fitur hasil rekayasa yang lebih proporsional dan terstruktur, atau ketika menggunakan fitur original yang lebih lengkap ditambah informasi jenis pinjaman.


## Model 1: Full Feature Engineering Model

Model 1 menggunakan pendekatan **full feature engineering**. Artinya, model ini tidak hanya memakai fitur original yang sudah dipilih, tetapi juga menambahkan beberapa fitur baru yang dibuat dari kolom asli dataset.

Feature engineering pada Model 1 terdiri dari empat bagian utama, yaitu:

* Missing indicator untuk `Credit_History_Age`
* Ratio features terhadap `Monthly_Inhand_Salary`
* Loan type flags dari `Type_of_Loan`
* Split `Payment_Behaviour` menjadi `Spending_Level` dan `Payment_Value_Level`

Tujuan dari Model 1 adalah membuat representasi fitur yang lebih informatif, proporsional, dan mudah dijelaskan secara bisnis. Model ini tidak hanya melihat nilai mentah dari data, tetapi juga mencoba menangkap hubungan antarfitur, terutama hubungan antara beban finansial customer dengan kemampuan pendapatannya.

Sebagai contoh, `Total_EMI_per_month` tidak hanya dilihat sebagai nominal cicilan, tetapi dibandingkan dengan `Monthly_Inhand_Salary` melalui fitur `EMI_to_Salary_Ratio`. Hal ini penting karena EMI sebesar 500 akan memiliki arti yang berbeda untuk customer dengan salary 1.000 dibanding customer dengan salary 10.000.

Selain itu, Model 1 juga tetap menggunakan informasi jenis pinjaman melalui loan type flags. Hal ini penting karena dua customer dapat memiliki jumlah pinjaman yang sama, tetapi jenis pinjamannya berbeda. Misalnya, customer yang memiliki Payday Loan dapat memiliki profil risiko yang berbeda dibanding customer yang memiliki Mortgage Loan atau Auto Loan.

### Tujuan Model 1

Tujuan dari Model 1 adalah:

* Menggunakan fitur hasil feature engineering yang lebih informatif.
* Membantu model memahami beban keuangan customer secara proporsional.
* Mengubah `Type_of_Loan` menjadi loan flags agar informasi jenis loan dapat dibaca model.
* Memisahkan `Payment_Behaviour` menjadi dua informasi yang lebih sederhana.
* Menambahkan missing indicator untuk menangkap pola missing value pada riwayat kredit.
* Mengurangi ketergantungan model terhadap nilai mentah yang skalanya terlalu besar.
* Membuat feature set yang lebih mudah dijelaskan secara bisnis.

## Fitur yang Digunakan pada Model 1

### Numeric Original Features Model 1

#### 1. `Age`

Fitur `Age` digunakan karena usia customer dapat berhubungan dengan stabilitas finansial, pengalaman kredit, dan kemampuan dalam mengelola kewajiban pembayaran. Customer dengan usia yang lebih tinggi umumnya memiliki potensi riwayat keuangan yang lebih panjang dibanding customer yang lebih muda.

Fitur ini tetap digunakan karena tidak termasuk kolom identitas dan memiliki makna bisnis yang jelas dalam konteks credit scoring.

#### 2. `Monthly_Inhand_Salary`

Fitur `Monthly_Inhand_Salary` digunakan untuk merepresentasikan pendapatan bulanan customer yang benar-benar diterima. Fitur ini penting karena kemampuan membayar pinjaman atau cicilan biasanya lebih relevan jika dilihat dari pendapatan bulanan yang tersedia.

Pada Model 1, fitur ini juga penting karena menjadi dasar dalam pembuatan beberapa fitur rasio, seperti `EMI_to_Salary_Ratio`, `Investment_to_Salary_Ratio`, `Balance_to_Salary_Ratio`, dan `Debt_to_Salary_Ratio`.

#### 3. `Num_Bank_Accounts`

Fitur `Num_Bank_Accounts` digunakan karena jumlah rekening bank dapat menggambarkan hubungan customer dengan sistem perbankan. Jumlah rekening yang terlalu banyak dapat menunjukkan kompleksitas aktivitas finansial customer.

Fitur ini tetap digunakan karena dapat membantu model menangkap pola eksposur finansial customer.

#### 4. `Num_Credit_Card`

Fitur `Num_Credit_Card` digunakan karena jumlah kartu kredit dapat berhubungan dengan tingkat eksposur kredit customer. Customer dengan banyak kartu kredit mungkin memiliki akses kredit yang lebih besar dan potensi penggunaan kredit yang lebih tinggi.

Fitur ini penting untuk credit scoring karena kartu kredit merupakan salah satu instrumen kredit yang dapat memengaruhi risiko pembayaran.

#### 5. `Interest_Rate`

Fitur `Interest_Rate` digunakan karena tingkat bunga dapat mencerminkan tingkat risiko kredit customer. Customer dengan interest rate tinggi sering kali diasosiasikan dengan risiko yang lebih tinggi karena pemberi pinjaman menetapkan bunga lebih besar untuk mengimbangi risiko tersebut.

Fitur ini memiliki hubungan langsung dengan risiko kredit sehingga tetap digunakan dalam modeling.

#### 6. `Num_of_Loan`

Fitur `Num_of_Loan` digunakan karena jumlah pinjaman customer menunjukkan seberapa banyak kewajiban kredit yang sedang dimiliki. Semakin banyak loan yang dimiliki customer, semakin besar kemungkinan beban pembayaran yang harus dikelola.

Pada Model 1, fitur ini digunakan bersama dengan loan type flags. `Num_of_Loan` menunjukkan jumlah pinjaman, sedangkan loan flags menunjukkan jenis pinjaman.

#### 7. `Delay_from_due_date`

Fitur `Delay_from_due_date` digunakan karena menunjukkan keterlambatan pembayaran customer dari tanggal jatuh tempo. Fitur ini sangat penting dalam credit scoring karena keterlambatan pembayaran merupakan salah satu indikator utama risiko kredit.

Nilai negatif pada kolom ini tetap dianggap valid karena dapat menunjukkan pembayaran yang dilakukan lebih awal dari tanggal jatuh tempo.

#### 8. `Num_of_Delayed_Payment`

Fitur `Num_of_Delayed_Payment` digunakan karena menunjukkan jumlah keterlambatan pembayaran yang pernah dilakukan customer. Semakin sering customer terlambat membayar, semakin tinggi potensi risiko kreditnya.

Fitur ini berbeda dari `Delay_from_due_date`. `Delay_from_due_date` menunjukkan besar keterlambatan pada periode tertentu, sedangkan `Num_of_Delayed_Payment` menunjukkan frekuensi keterlambatan.

#### 9. `Changed_Credit_Limit`

Fitur `Changed_Credit_Limit` digunakan karena menunjukkan perubahan limit kredit customer. Perubahan limit kredit dapat mencerminkan perubahan kepercayaan lembaga keuangan terhadap customer.

Nilai negatif pada fitur ini tetap valid karena dapat berarti limit kredit customer mengalami penurunan. Oleh karena itu, nilai negatif tidak langsung dianggap sebagai error selama masih masuk akal secara konteks.

#### 10. `Num_Credit_Inquiries`

Fitur `Num_Credit_Inquiries` digunakan karena jumlah inquiry kredit dapat menunjukkan seberapa sering customer mengajukan atau mencari fasilitas kredit. Jumlah inquiry yang tinggi dapat menjadi sinyal bahwa customer sedang membutuhkan banyak akses kredit.

Fitur ini relevan untuk credit scoring karena banyaknya inquiry dapat berkaitan dengan peningkatan risiko kredit.

#### 11. `Outstanding_Debt`

Fitur `Outstanding_Debt` digunakan karena menunjukkan total utang customer yang masih belum dibayar. Fitur ini sangat penting karena semakin besar outstanding debt, semakin besar beban kewajiban customer.

Walaupun Model 1 juga menggunakan `Debt_to_Salary_Ratio`, nilai debt mentah tetap dapat digunakan karena memberikan informasi absolut mengenai total utang customer.

#### 12. `Credit_Utilization_Ratio`

Fitur `Credit_Utilization_Ratio` digunakan karena menunjukkan seberapa besar penggunaan kredit customer dibandingkan limit kredit yang tersedia. Rasio utilisasi kredit yang tinggi dapat menunjukkan bahwa customer sangat bergantung pada fasilitas kredit.

Fitur ini sering digunakan dalam konteks credit scoring karena berkaitan langsung dengan perilaku penggunaan kredit.

#### 13. `Credit_History_Age`

Fitur `Credit_History_Age` digunakan karena menunjukkan panjang riwayat kredit customer dalam satuan bulan. Riwayat kredit yang lebih panjang dapat memberikan lebih banyak informasi mengenai konsistensi perilaku pembayaran customer.

Kolom ini tetap digunakan meskipun masih memiliki sebagian missing value, karena missing value akan ditangani pada preprocessing pipeline menggunakan imputer.

#### 14. `Credit_History_Age_Missing`

Fitur `Credit_History_Age_Missing` digunakan sebagai missing indicator untuk kolom `Credit_History_Age`. Fitur ini bernilai 1 jika `Credit_History_Age` sebelumnya missing, dan bernilai 0 jika datanya tersedia.

Fitur ini penting karena missing value pada riwayat kredit bisa saja mengandung pola tertentu. Misalnya, customer yang tidak memiliki data riwayat kredit dapat memiliki karakteristik risiko yang berbeda dibanding customer yang memiliki riwayat kredit lengkap.

### Categorical Original Features Model 1

#### 15. `Occupation`

Fitur `Occupation` digunakan karena jenis pekerjaan customer dapat berkaitan dengan stabilitas pendapatan dan pola keuangan. Beberapa pekerjaan mungkin memiliki pendapatan yang lebih stabil, sedangkan pekerjaan lain dapat memiliki pendapatan yang lebih fluktuatif.

Fitur ini digunakan sebagai categorical feature dan akan diproses menggunakan encoding pada tahap preprocessing.

#### 16. `Credit_Mix`

Fitur `Credit_Mix` digunakan karena menunjukkan kualitas atau variasi portofolio kredit customer. Customer dengan credit mix yang baik dapat menunjukkan pengalaman kredit yang lebih sehat, sedangkan credit mix yang buruk dapat menjadi sinyal risiko.

Fitur ini sangat relevan untuk credit scoring dan tetap digunakan sebagai categorical feature.

#### 17. `Payment_of_Min_Amount`

Fitur `Payment_of_Min_Amount` digunakan karena menunjukkan apakah customer sering membayar minimum amount. Customer yang hanya membayar minimum amount dapat memiliki risiko lebih tinggi karena menunjukkan potensi keterbatasan cashflow atau kebiasaan membayar kewajiban secara minimum.

Nilai `NM` tidak dihapus karena dapat mengandung informasi tersendiri. Oleh karena itu, nilai tersebut tetap dipertahankan sebagai kategori.

### Loan Type Flags Model 1

Kolom `Type_of_Loan` tidak digunakan secara langsung karena bentuknya berupa teks panjang yang dapat berisi beberapa jenis pinjaman dalam satu baris. Jika digunakan secara langsung sebagai kategori mentah, jumlah kombinasi kategorinya dapat menjadi terlalu banyak dan sulit dipelajari oleh model.

Oleh karena itu, `Type_of_Loan` diubah menjadi beberapa fitur binary atau loan flags. Setiap fitur menunjukkan apakah customer memiliki jenis loan tertentu atau tidak.

#### 18. `Has_Payday_Loan`

Fitur ini bernilai 1 jika customer memiliki Payday Loan, dan 0 jika tidak.

Fitur ini penting karena Payday Loan sering diasosiasikan dengan kebutuhan dana jangka pendek dan potensi tekanan finansial. Dalam konteks credit risk, keberadaan Payday Loan dapat menjadi sinyal risiko yang perlu diperhatikan.

#### 19. `Has_Mortgage_Loan`

Fitur ini bernilai 1 jika customer memiliki Mortgage Loan, dan 0 jika tidak.

Mortgage Loan dapat menunjukkan jenis pinjaman jangka panjang dengan nilai besar. Jenis pinjaman ini memiliki karakteristik risiko yang berbeda dibanding loan jangka pendek.

#### 20. `Has_Student_Loan`

Fitur ini bernilai 1 jika customer memiliki Student Loan, dan 0 jika tidak.

Student Loan memiliki pola risiko yang berbeda karena biasanya berkaitan dengan pembiayaan pendidikan. Fitur ini dapat membantu model membedakan jenis kewajiban kredit customer.

#### 21. `Has_Credit_Builder_Loan`

Fitur ini bernilai 1 jika customer memiliki Credit-Builder Loan, dan 0 jika tidak.

Credit-Builder Loan dapat berkaitan dengan customer yang sedang membangun atau memperbaiki riwayat kredit. Fitur ini relevan karena dapat menunjukkan profil customer yang berbeda dari pemilik loan lain.

#### 22. `Has_Debt_Consolidation_Loan`

Fitur ini bernilai 1 jika customer memiliki Debt Consolidation Loan, dan 0 jika tidak.

Debt Consolidation Loan dapat menunjukkan bahwa customer menggabungkan beberapa utang menjadi satu pinjaman. Hal ini dapat menjadi sinyal bahwa customer memiliki riwayat atau beban utang yang cukup kompleks.

#### 23. `Has_Personal_Loan`

Fitur ini bernilai 1 jika customer memiliki Personal Loan, dan 0 jika tidak.

Personal Loan merupakan jenis pinjaman yang umum dan dapat memiliki berbagai tujuan. Fitur ini tetap digunakan karena dapat membantu model memahami komposisi jenis pinjaman customer.

#### 24. `Has_Auto_Loan`

Fitur ini bernilai 1 jika customer memiliki Auto Loan, dan 0 jika tidak.

Auto Loan berkaitan dengan pembiayaan kendaraan. Jenis pinjaman ini memiliki karakteristik risiko yang berbeda karena biasanya berkaitan dengan aset pembiayaan yang spesifik.

#### 25. `Has_Home_Equity_Loan`

Fitur ini bernilai 1 jika customer memiliki Home Equity Loan, dan 0 jika tidak.

Home Equity Loan berkaitan dengan aset properti dan dapat menunjukkan pola pembiayaan yang berbeda. Fitur ini dapat membantu model mengenali jenis eksposur kredit customer.

#### 26. `Has_Not_Specified_Loan`

Fitur ini bernilai 1 jika customer memiliki loan dengan jenis `Not Specified`, dan 0 jika tidak.

Fitur ini tetap dipertahankan karena nilai `Not Specified` dapat mengandung informasi. Menghapusnya dapat membuat model kehilangan pola dari customer yang memiliki jenis loan yang tidak teridentifikasi secara spesifik.

### Payment Behaviour Split Model 1

Kolom `Payment_Behaviour` tidak digunakan secara langsung pada Model 1 karena kolom ini merupakan gabungan dari dua informasi, yaitu pola pengeluaran dan nilai pembayaran. Contoh isi kolom ini adalah `Low_spent_Large_value_payments`.

Agar informasi di dalamnya lebih jelas, kolom `Payment_Behaviour` dipisahkan menjadi dua fitur baru, yaitu `Spending_Level` dan `Payment_Value_Level`.

#### 27. `Spending_Level`

Fitur `Spending_Level` menunjukkan apakah customer memiliki pola pengeluaran tinggi atau rendah. Nilainya dapat berupa `High`, `Low`, atau `Unknown`.

Fitur ini digunakan karena tingkat pengeluaran customer dapat berkaitan dengan kemampuan mengelola keuangan. Customer dengan spending tinggi belum tentu memiliki risiko buruk, tetapi pola tersebut perlu dianalisis bersama dengan salary, EMI, investasi, balance, dan debt.

#### 28. `Payment_Value_Level`

Fitur `Payment_Value_Level` menunjukkan nilai pembayaran customer, misalnya `Small`, `Medium`, atau `Large`.

Fitur ini digunakan karena nilai pembayaran dapat menggambarkan pola customer dalam memenuhi kewajibannya. Customer dengan pembayaran bernilai besar dapat memiliki karakteristik risiko yang berbeda dibanding customer dengan pembayaran kecil.

Dengan memisahkan `Payment_Behaviour`, Model 1 dapat membaca informasi secara lebih jelas dibanding menggunakan gabungan teks mentah.

### Ratio Features Model 1

Ratio features dibuat untuk membandingkan nilai keuangan customer terhadap pendapatan bulanannya. Hal ini penting karena nilai absolut yang besar belum tentu buruk jika customer juga memiliki pendapatan yang tinggi.

#### 29. `EMI_to_Salary_Ratio`

Formula:

`Total_EMI_per_month / Monthly_Inhand_Salary`

Fitur ini menunjukkan proporsi pendapatan bulanan customer yang digunakan untuk membayar cicilan. Fitur ini sangat penting karena cicilan bulanan yang besar dapat mengurangi kemampuan customer untuk memenuhi kebutuhan lain.

Fitur ini lebih informatif dibanding menggunakan `Total_EMI_per_month` mentah karena EMI yang sama dapat memiliki dampak berbeda tergantung besar pendapatan customer.

#### 30. `Investment_to_Salary_Ratio`

Formula:

`Amount_invested_monthly / Monthly_Inhand_Salary`

Fitur ini menunjukkan proporsi pendapatan bulanan yang digunakan customer untuk investasi. Fitur ini penting karena menggambarkan pola pengelolaan keuangan customer.

Nilai investasi bulanan yang tinggi tidak otomatis buruk. Jika customer memiliki pendapatan tinggi, investasi yang tinggi masih bisa masuk akal. Oleh karena itu, rasio terhadap salary lebih adil dibanding nilai investasi mentah.

#### 31. `Balance_to_Salary_Ratio`

Formula:

`Monthly_Balance / Monthly_Inhand_Salary`

Fitur ini menunjukkan proporsi saldo bulanan customer terhadap pendapatan bulanannya. Fitur ini penting untuk melihat seberapa besar sisa dana customer setelah pengeluaran bulanan.

Customer dengan balance ratio rendah dapat menunjukkan tekanan finansial yang lebih besar, sedangkan customer dengan balance ratio tinggi dapat menunjukkan kondisi keuangan yang lebih stabil.

#### 32. `Debt_to_Salary_Ratio`

Formula:

`Outstanding_Debt / Monthly_Inhand_Salary`

Fitur ini menunjukkan besar outstanding debt customer dibandingkan pendapatan bulanan. Fitur ini penting karena outstanding debt yang besar belum tentu memiliki dampak risiko yang sama untuk semua customer.

Dengan membandingkan debt terhadap salary, Model 1 dapat memahami beban utang customer secara lebih proporsional.

## Final Feature Set Model 1

### Numeric Features Model 1

* `Age`
* `Monthly_Inhand_Salary`
* `Num_Bank_Accounts`
* `Num_Credit_Card`
* `Interest_Rate`
* `Num_of_Loan`
* `Delay_from_due_date`
* `Num_of_Delayed_Payment`
* `Changed_Credit_Limit`
* `Num_Credit_Inquiries`
* `Outstanding_Debt`
* `Credit_Utilization_Ratio`
* `Credit_History_Age`
* `Credit_History_Age_Missing`
* `EMI_to_Salary_Ratio`
* `Investment_to_Salary_Ratio`
* `Balance_to_Salary_Ratio`
* `Debt_to_Salary_Ratio`
* `Has_Payday_Loan`
* `Has_Mortgage_Loan`
* `Has_Student_Loan`
* `Has_Credit_Builder_Loan`
* `Has_Debt_Consolidation_Loan`
* `Has_Personal_Loan`
* `Has_Auto_Loan`
* `Has_Home_Equity_Loan`
* `Has_Not_Specified_Loan`

### Categorical Features Model 1

* `Occupation`
* `Credit_Mix`
* `Payment_of_Min_Amount`
* `Spending_Level`
* `Payment_Value_Level`

## Alasan Model 1 Aman Digunakan

Model 1 aman digunakan karena tidak memakai target `Credit_Score` sebagai input, tidak menggunakan kolom identitas customer, dan tidak menggunakan kolom teks panjang secara langsung. Kolom `Type_of_Loan` diubah menjadi loan flags, sehingga informasi jenis pinjaman menjadi lebih terstruktur dan dapat dibaca oleh model.

Model ini juga menggunakan ratio features yang lebih proporsional terhadap kemampuan finansial customer. Misalnya, EMI dan outstanding debt tidak hanya dilihat dari nominalnya, tetapi dibandingkan dengan pendapatan bulanan customer. Dengan cara ini, model dapat memahami beban finansial customer secara lebih adil.

Selain itu, pemisahan `Payment_Behaviour` menjadi `Spending_Level` dan `Payment_Value_Level` membuat informasi perilaku pembayaran lebih mudah dipelajari. Model tidak perlu membaca satu kategori gabungan yang terlalu panjang, tetapi dapat mempelajari dua informasi yang lebih sederhana, yaitu tingkat pengeluaran dan nilai pembayaran.

## Model 2: Original Features + Loan Feature Engineering Model

Model 2 menggunakan pendekatan **original features + loan feature engineering**. Pada model ini, fitur original dari dataset dipertahankan lebih lengkap, kemudian informasi dari kolom `Type_of_Loan` diubah menjadi beberapa fitur binary atau loan flags.

Berbeda dengan Model 1, Model 2 tidak berfokus pada ratio features dan split `Payment_Behaviour`. Model 2 tetap menggunakan beberapa fitur keuangan dalam bentuk original, seperti `Annual_Income`, `Total_EMI_per_month`, `Amount_invested_monthly`, dan `Monthly_Balance`.

Tujuan Model 2 adalah melihat apakah model dapat memperoleh performa yang lebih baik ketika diberikan informasi original yang lebih lengkap dan detail jenis pinjaman customer. Dengan kata lain, Model 2 menguji apakah data mentah yang lebih luas ditambah loan flags sudah cukup kuat untuk memprediksi `Credit_Score`.

Model 2 juga penting karena jenis pinjaman customer dapat mengandung informasi risiko yang tidak sepenuhnya tertangkap oleh jumlah pinjaman saja. Dua customer dapat memiliki `Num_of_Loan` yang sama, tetapi jenis loan yang dimiliki bisa berbeda. Misalnya, customer yang memiliki Payday Loan dapat memiliki profil risiko yang berbeda dibanding customer yang memiliki Mortgage Loan atau Auto Loan.

### Tujuan Model 2

Tujuan dari Model 2 adalah:

* Menggunakan original features yang lebih lengkap dari dataset.
* Mempertahankan fitur keuangan original seperti `Annual_Income`, `Total_EMI_per_month`, `Amount_invested_monthly`, dan `Monthly_Balance`.
* Mengubah `Type_of_Loan` menjadi loan type flags.
* Membantu model memahami bukan hanya jumlah loan, tetapi juga jenis loan yang dimiliki customer.
* Membandingkan apakah original features + loan feature engineering memberikan performa lebih baik dibanding full feature engineering pada Model 1.

## Fitur yang Digunakan pada Model 2

### Numeric Original Features Model 2

#### 1. `Age`

Fitur `Age` digunakan karena usia customer dapat berhubungan dengan stabilitas finansial, pengalaman kredit, dan kemampuan dalam mengelola kewajiban pembayaran.

Fitur ini tetap digunakan karena memiliki makna bisnis yang jelas dan tidak termasuk kolom identitas.

#### 2. `Annual_Income`

Fitur `Annual_Income` digunakan pada Model 2 karena model ini mempertahankan original features yang lebih lengkap. Annual income memberikan informasi mengenai total pendapatan customer dalam satu tahun.

Fitur ini dapat membantu model memahami kapasitas finansial customer secara lebih luas. Walaupun memiliki hubungan dengan `Monthly_Inhand_Salary`, Model 2 tetap memasukkan fitur ini karena tujuan model adalah menguji apakah informasi original yang lebih lengkap dapat meningkatkan performa.

#### 3. `Monthly_Inhand_Salary`

Fitur `Monthly_Inhand_Salary` digunakan karena menunjukkan pendapatan bulanan yang benar-benar diterima customer. Fitur ini penting untuk memahami kemampuan pembayaran customer dalam periode bulanan.

Pada Model 2, fitur ini digunakan bersama dengan `Annual_Income` karena keduanya dapat memberikan perspektif pendapatan yang berbeda, yaitu pendapatan tahunan dan pendapatan bulanan.

#### 4. `Num_Bank_Accounts`

Fitur `Num_Bank_Accounts` digunakan karena jumlah rekening bank dapat menunjukkan kompleksitas hubungan customer dengan sistem perbankan.

Jumlah rekening yang terlalu banyak dapat mencerminkan aktivitas finansial yang lebih kompleks, sehingga fitur ini relevan untuk credit scoring.

#### 5. `Num_Credit_Card`

Fitur `Num_Credit_Card` digunakan karena jumlah kartu kredit berhubungan dengan eksposur kredit customer. Customer dengan banyak kartu kredit memiliki akses terhadap fasilitas kredit yang lebih besar.

Fitur ini penting karena kartu kredit dapat memengaruhi risiko pembayaran customer.

#### 6. `Interest_Rate`

Fitur `Interest_Rate` digunakan karena tingkat bunga dapat mencerminkan risiko kredit customer. Interest rate yang tinggi dapat menunjukkan bahwa customer dianggap memiliki risiko lebih tinggi oleh pemberi pinjaman.

Fitur ini tetap digunakan karena memiliki hubungan langsung dengan konteks credit scoring.

#### 7. `Num_of_Loan`

Fitur `Num_of_Loan` digunakan karena menunjukkan jumlah pinjaman customer. Fitur ini memberikan informasi mengenai banyaknya kewajiban kredit yang sedang dimiliki.

Pada Model 2, fitur ini digunakan bersama dengan loan type flags. `Num_of_Loan` menunjukkan jumlah pinjaman, sedangkan loan flags menunjukkan jenis pinjaman.

#### 8. `Delay_from_due_date`

Fitur `Delay_from_due_date` digunakan karena menunjukkan keterlambatan pembayaran dari tanggal jatuh tempo. Fitur ini merupakan salah satu indikator utama risiko kredit.

Customer yang sering terlambat membayar atau memiliki keterlambatan yang besar cenderung memiliki risiko kredit yang lebih tinggi.

#### 9. `Num_of_Delayed_Payment`

Fitur `Num_of_Delayed_Payment` digunakan karena menunjukkan frekuensi keterlambatan pembayaran customer.

Fitur ini penting karena frekuensi keterlambatan dapat menunjukkan kebiasaan pembayaran customer dalam jangka waktu tertentu.

#### 10. `Changed_Credit_Limit`

Fitur `Changed_Credit_Limit` digunakan karena menunjukkan perubahan limit kredit customer. Perubahan limit dapat mencerminkan perubahan penilaian risiko dari pihak pemberi kredit terhadap customer.

Nilai negatif tetap valid karena dapat menunjukkan penurunan limit kredit.

#### 11. `Num_Credit_Inquiries`

Fitur `Num_Credit_Inquiries` digunakan karena menunjukkan jumlah inquiry kredit customer. Jumlah inquiry yang tinggi dapat menandakan bahwa customer sering mengajukan atau mencari fasilitas kredit baru.

Fitur ini relevan karena banyaknya inquiry dapat berkaitan dengan peningkatan risiko kredit.

#### 12. `Outstanding_Debt`

Fitur `Outstanding_Debt` digunakan karena menunjukkan total utang customer yang masih belum dibayar. Semakin besar outstanding debt, semakin besar beban kewajiban customer.

Fitur ini sangat penting dalam credit scoring karena beban utang merupakan salah satu komponen utama dalam penilaian risiko.

#### 13. `Credit_Utilization_Ratio`

Fitur `Credit_Utilization_Ratio` digunakan karena menunjukkan tingkat penggunaan kredit customer dibandingkan limit kredit yang tersedia.

Customer dengan credit utilization ratio tinggi dapat menunjukkan ketergantungan yang besar terhadap fasilitas kredit, sehingga fitur ini penting untuk memprediksi credit score.

#### 14. `Credit_History_Age`

Fitur `Credit_History_Age` digunakan karena menunjukkan panjang riwayat kredit customer. Customer dengan riwayat kredit yang lebih panjang dapat memberikan lebih banyak informasi mengenai konsistensi pembayaran dan perilaku kredit.

Missing value pada fitur ini akan ditangani pada preprocessing pipeline menggunakan imputer.

#### 15. `Total_EMI_per_month`

Fitur `Total_EMI_per_month` digunakan pada Model 2 karena model ini mempertahankan original features dalam bentuk mentah. Fitur ini menunjukkan total cicilan bulanan yang harus dibayar customer.

Fitur ini penting karena EMI yang tinggi dapat mengurangi kemampuan customer untuk memenuhi kebutuhan lain dan meningkatkan risiko gagal bayar.

#### 16. `Amount_invested_monthly`

Fitur `Amount_invested_monthly` digunakan karena menunjukkan jumlah uang yang diinvestasikan customer setiap bulan. Fitur ini dapat menggambarkan pola pengelolaan keuangan customer.

Customer yang memiliki investasi bulanan tertentu mungkin memiliki perilaku finansial yang berbeda dibanding customer yang tidak memiliki investasi atau memiliki investasi sangat rendah.

#### 17. `Monthly_Balance`

Fitur `Monthly_Balance` digunakan karena menunjukkan sisa saldo bulanan customer. Fitur ini dapat menggambarkan kondisi cashflow customer setelah pendapatan digunakan untuk kebutuhan, cicilan, dan investasi.

Monthly balance yang rendah dapat menjadi sinyal adanya tekanan finansial.

### Categorical Original Features Model 2

#### 18. `Occupation`

Fitur `Occupation` digunakan karena jenis pekerjaan customer dapat berkaitan dengan stabilitas pendapatan dan kemampuan membayar kewajiban kredit.

Fitur ini diproses sebagai categorical feature menggunakan encoding.

#### 19. `Credit_Mix`

Fitur `Credit_Mix` digunakan karena menunjukkan kualitas atau variasi portofolio kredit customer. Credit mix yang baik dapat menunjukkan pengelolaan kredit yang lebih sehat, sedangkan credit mix yang buruk dapat menjadi sinyal risiko.

Fitur ini memiliki hubungan kuat dengan credit scoring sehingga tetap digunakan.

#### 20. `Payment_of_Min_Amount`

Fitur `Payment_of_Min_Amount` digunakan karena menunjukkan apakah customer sering membayar minimum amount. Customer yang hanya membayar minimum amount dapat memiliki potensi risiko lebih tinggi karena menunjukkan keterbatasan cashflow atau kebiasaan membayar kewajiban dalam jumlah minimum.

Nilai `NM` tetap dipertahankan sebagai kategori karena dapat mengandung informasi tertentu.

#### 21. `Payment_Behaviour`

Pada Model 2, `Payment_Behaviour` digunakan sebagai categorical original feature karena model ini mempertahankan original features yang lebih lengkap. Fitur ini menunjukkan pola pengeluaran dan nilai pembayaran customer dalam satu kolom gabungan.

Walaupun pada Model 1 fitur ini dipisahkan menjadi `Spending_Level` dan `Payment_Value_Level`, pada Model 2 fitur ini tetap digunakan dalam bentuk original untuk menguji apakah representasi categorical asli dari dataset dapat membantu performa model.

### Loan Feature Engineering Model 2

Sama seperti Model 1, Model 2 juga menggunakan loan type flags dari `Type_of_Loan`. Kolom `Type_of_Loan` tidak digunakan secara langsung karena bentuknya berupa teks panjang yang dapat berisi beberapa jenis loan dalam satu baris.

Loan type flags dibuat agar informasi jenis pinjaman dapat dibaca oleh model secara lebih sederhana dan terstruktur.

#### 22. `Has_Payday_Loan`

Fitur ini bernilai 1 jika customer memiliki Payday Loan, dan 0 jika tidak.

Fitur ini penting karena Payday Loan sering diasosiasikan dengan kebutuhan dana jangka pendek dan potensi tekanan finansial. Dalam konteks credit risk, keberadaan Payday Loan dapat menjadi sinyal risiko yang perlu diperhatikan.

#### 23. `Has_Mortgage_Loan`

Fitur ini bernilai 1 jika customer memiliki Mortgage Loan, dan 0 jika tidak.

Mortgage Loan dapat menunjukkan jenis pinjaman jangka panjang dengan nilai besar. Jenis pinjaman ini memiliki karakteristik risiko yang berbeda dibanding loan jangka pendek.

#### 24. `Has_Student_Loan`

Fitur ini bernilai 1 jika customer memiliki Student Loan, dan 0 jika tidak.

Student Loan memiliki pola risiko yang berbeda karena biasanya berkaitan dengan pembiayaan pendidikan. Fitur ini dapat membantu model membedakan jenis kewajiban kredit customer.

#### 25. `Has_Credit_Builder_Loan`

Fitur ini bernilai 1 jika customer memiliki Credit-Builder Loan, dan 0 jika tidak.

Credit-Builder Loan dapat berkaitan dengan customer yang sedang membangun atau memperbaiki riwayat kredit. Fitur ini relevan karena dapat menunjukkan profil customer yang berbeda dari pemilik loan lain.

#### 26. `Has_Debt_Consolidation_Loan`

Fitur ini bernilai 1 jika customer memiliki Debt Consolidation Loan, dan 0 jika tidak.

Debt Consolidation Loan dapat menunjukkan bahwa customer menggabungkan beberapa utang menjadi satu pinjaman. Hal ini dapat menjadi sinyal bahwa customer memiliki riwayat atau beban utang yang cukup kompleks.

#### 27. `Has_Personal_Loan`

Fitur ini bernilai 1 jika customer memiliki Personal Loan, dan 0 jika tidak.

Personal Loan merupakan jenis pinjaman yang umum dan dapat memiliki berbagai tujuan. Fitur ini tetap digunakan karena dapat membantu model memahami komposisi jenis pinjaman customer.

#### 28. `Has_Auto_Loan`

Fitur ini bernilai 1 jika customer memiliki Auto Loan, dan 0 jika tidak.

Auto Loan berkaitan dengan pembiayaan kendaraan. Jenis pinjaman ini memiliki karakteristik risiko yang berbeda karena biasanya berkaitan dengan aset pembiayaan yang spesifik.

#### 29. `Has_Home_Equity_Loan`

Fitur ini bernilai 1 jika customer memiliki Home Equity Loan, dan 0 jika tidak.

Home Equity Loan berkaitan dengan aset properti dan dapat menunjukkan pola pembiayaan yang berbeda. Fitur ini dapat membantu model mengenali jenis eksposur kredit customer.

#### 30. `Has_Not_Specified_Loan`

Fitur ini bernilai 1 jika customer memiliki loan dengan jenis `Not Specified`, dan 0 jika tidak.

Fitur ini tetap dipertahankan karena nilai `Not Specified` dapat mengandung informasi. Menghapusnya dapat membuat model kehilangan pola dari customer yang memiliki jenis loan yang tidak teridentifikasi secara spesifik.

## Final Feature Set Model 2

### Numeric Features Model 2

* `Age`
* `Annual_Income`
* `Monthly_Inhand_Salary`
* `Num_Bank_Accounts`
* `Num_Credit_Card`
* `Interest_Rate`
* `Num_of_Loan`
* `Delay_from_due_date`
* `Num_of_Delayed_Payment`
* `Changed_Credit_Limit`
* `Num_Credit_Inquiries`
* `Outstanding_Debt`
* `Credit_Utilization_Ratio`
* `Credit_History_Age`
* `Total_EMI_per_month`
* `Amount_invested_monthly`
* `Monthly_Balance`
* `Has_Payday_Loan`
* `Has_Mortgage_Loan`
* `Has_Student_Loan`
* `Has_Credit_Builder_Loan`
* `Has_Debt_Consolidation_Loan`
* `Has_Personal_Loan`
* `Has_Auto_Loan`
* `Has_Home_Equity_Loan`
* `Has_Not_Specified_Loan`

### Categorical Features Model 2

* `Occupation`
* `Credit_Mix`
* `Payment_of_Min_Amount`
* `Payment_Behaviour`

## Alasan Model 2 Aman Digunakan

Model 2 aman digunakan karena tidak menggunakan target `Credit_Score` sebagai input dan tidak menggunakan kolom identitas customer. Kolom `Type_of_Loan` juga tidak digunakan secara langsung, tetapi diubah menjadi loan flags sehingga informasi jenis loan menjadi lebih terstruktur dan mudah dipelajari oleh model.

Model 2 mempertahankan original features yang lebih lengkap sehingga model dapat mempelajari informasi mentah dari dataset, seperti pendapatan tahunan, pendapatan bulanan, total EMI, investasi bulanan, saldo bulanan, jumlah pinjaman, dan perilaku pembayaran. Dengan tambahan loan flags, model juga dapat memahami jenis pinjaman yang dimiliki customer, bukan hanya jumlah pinjamannya.

Namun, karena Model 2 menggunakan lebih banyak original features, terdapat kemungkinan adanya fitur yang memiliki informasi saling berdekatan, seperti `Annual_Income` dan `Monthly_Inhand_Salary`. Oleh karena itu, preprocessing dan evaluasi model tetap perlu dilakukan dengan hati-hati. Untuk model linear seperti Logistic Regression, potensi multikolinearity perlu diperhatikan. Untuk model tree-based seperti Random Forest dan XGBoost, fitur yang lebih banyak masih dapat digunakan karena model tersebut lebih tahan terhadap hubungan non-linear dan interaksi antarfitur.

## Fitur yang Tidak Digunakan pada Kedua Model

Beberapa fitur tidak digunakan pada kedua model karena alasan identitas, target leakage, atau hanya berfungsi sebagai helper pada tahap cleaning.

### Kolom Identitas yang Tidak Digunakan

Kolom berikut tidak digunakan karena hanya berfungsi sebagai identitas dan tidak memiliki makna prediktif yang valid untuk customer baru:

* `Unnamed: 0`
* `ID`
* `Customer_ID`
* `Name`
* `SSN`

Kolom-kolom tersebut dapat membuat model menghafal data tertentu, bukan mempelajari pola umum yang dapat digunakan untuk prediksi customer baru.

### Target yang Tidak Digunakan sebagai Fitur

* `Credit_Score`

Kolom `Credit_Score` adalah target yang ingin diprediksi, sehingga tidak boleh digunakan sebagai input feature. Jika target digunakan sebagai fitur, maka akan terjadi target leakage dan hasil evaluasi model menjadi tidak valid.

### Kolom Waktu yang Tidak Digunakan

* `Month`

Kolom `Month` tidak digunakan karena tujuan model adalah memprediksi credit score berdasarkan profil dan perilaku kredit customer, bukan berdasarkan pola musiman. Selain itu, penggunaan `Month` dapat menambah kompleksitas tanpa memberikan makna yang kuat untuk model final.

### Kolom Helper yang Tidak Digunakan

Kolom berikut tidak digunakan karena hanya dipakai pada tahap cleaning dan validasi:

* `Loan_List_From_Type`
* `Loan_Item_Count`
* `Unique_Loan_Type_Count`

Kolom ini membantu memeriksa konsistensi antara `Type_of_Loan` dan `Num_of_Loan`, tetapi tidak digunakan sebagai fitur final karena informasi utamanya sudah diwakili oleh `Num_of_Loan` dan loan type flags.

## Perbedaan Utama Model 1 dan Model 2

Perbedaan utama antara Model 1 dan Model 2 bukan pada penggunaan loan flags, karena kedua model sama-sama menggunakan loan type flags dari `Type_of_Loan`.

Perbedaan utamanya adalah strategi feature engineering yang digunakan.

Model 1 menggunakan pendekatan full feature engineering. Model ini menambahkan missing indicator, ratio features, loan flags, dan split `Payment_Behaviour`. Model 1 lebih menekankan fitur yang sudah ditransformasi agar lebih proporsional, lebih terstruktur, dan lebih mudah dijelaskan secara bisnis.

Model 2 menggunakan pendekatan original features + loan feature engineering. Model ini mempertahankan fitur original yang lebih lengkap, seperti `Annual_Income`, `Total_EMI_per_month`, `Amount_invested_monthly`, `Monthly_Balance`, dan `Payment_Behaviour`, lalu menambahkan loan flags dari `Type_of_Loan`.

Dengan kata lain, Model 1 menguji apakah fitur hasil rekayasa yang lebih ringkas dan proporsional dapat meningkatkan performa model, sedangkan Model 2 menguji apakah fitur original yang lebih lengkap ditambah loan flags dapat memberikan performa yang lebih baik.

## Kesimpulan Feature Selection

Secara keseluruhan, Model 1 dan Model 2 dirancang untuk membandingkan dua pendekatan feature set yang berbeda dalam credit scoring.

Model 1 berfokus pada full feature engineering, yaitu penggunaan fitur original terpilih yang ditambah dengan missing indicator, ratio features, loan type flags, dan split payment behaviour. Model ini bertujuan membantu model memahami beban keuangan customer secara lebih proporsional dan terstruktur.

Model 2 berfokus pada original features + loan feature engineering. Model ini mempertahankan lebih banyak fitur original dari dataset, kemudian menambahkan loan type flags agar informasi jenis pinjaman tetap dapat digunakan dalam modeling.

Kedua model tetap aman digunakan karena tidak memakai target sebagai input, tidak memakai kolom identitas, dan tidak menggunakan kolom teks panjang secara langsung tanpa proses transformasi. Perbandingan performa antara Model 1 dan Model 2 akan membantu menentukan apakah pendekatan full feature engineering atau original features + loan feature engineering lebih efektif untuk memprediksi `Credit_Score` pada dataset credit scoring ini.


# H. Scalling(Ide Untuk di Modelling)



## 1. Untuk Logistic Regression

Gunakan:

`SimpleImputer(strategy="median") + RobustScaler()`

Alasannya:

- Logistic Regression sensitif terhadap skala fitur.
- Fitur numerik kamu punya skala yang sangat berbeda, misalnya:
  - `Age`
  - `Monthly_Inhand_Salary`
  - `Outstanding_Debt`
  - `Credit_History_Age`
  - `Interest_Rate`
  - ratio features
- Banyak outlier valid yang sengaja kita pertahankan, seperti:
  - `Outstanding_Debt`
  - `Total_EMI_per_month`
  - `Amount_invested_monthly`
  - `Monthly_Balance`
  - `Delay_from_due_date`
- Karena outlier valid tetap dipakai, RobustScaler lebih cocok daripada StandardScaler.
- RobustScaler memakai median dan IQR, jadi lebih tahan terhadap outlier.

Jadi untuk model linear, pilihan terbaik:

`RobustScaler`

## 2. Untuk Random Forest, XGBoost, dan LightGBM

Gunakan:

`SimpleImputer(strategy="median")` saja untuk numeric features, tanpa scaling.

Alasannya:

- Tree-based model tidak sensitif terhadap skala fitur.
- Random Forest, XGBoost, dan LightGBM bekerja berdasarkan split, bukan jarak antar nilai.
- Scaling tidak wajib dan biasanya tidak meningkatkan performa tree-based model.
- Dengan tidak scaling tree-based model, interpretasi nilai threshold juga lebih natural.

Jadi untuk model tree-based:

No scaling / passthrough numeric feature


# I. Proses Modelling 1(Menggunakan Tambahan Hasil Feature Engineering) 

In [ ]:
# STANDARD LIBRARY
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

# SKLEARN
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# PARAMETER DISTRIBUTION
from scipy.stats import randint, uniform, loguniform

# OPTIONAL MODELS
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

RANDOM_STATE = 42

### 1. Konfigurasi Kolom Final

In [ ]:
# TARGET AND GROUP COLUMN

target_column = "Credit_Score"
group_column = "Customer_ID"

# NUMERIC ORIGINAL FEATURES USED FOR MODELING
numeric_features = [
    "Age",
    "Monthly_Inhand_Salary",
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Delay_from_due_date",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Num_Credit_Inquiries",
    "Outstanding_Debt",
    "Credit_Utilization_Ratio",
    "Credit_History_Age",

    # SELECTED RATIO FEATURES
    "EMI_to_Salary_Ratio",
    "Investment_to_Salary_Ratio",
    "Balance_to_Salary_Ratio",
    "Debt_to_Salary_Ratio"
]

# BINARY FEATURES
binary_features = [
    "Credit_History_Age_Missing",
    "Has_Payday_Loan",
    "Has_Mortgage_Loan",
    "Has_Student_Loan",
    "Has_Credit_Builder_Loan",
    "Has_Debt_Consolidation_Loan",
    "Has_Personal_Loan",
    "Has_Auto_Loan",
    "Has_Home_Equity_Loan",
    "Has_Not_Specified_Loan"
]

# CATEGORICAL FEATURES
categorical_features = [
    "Occupation",
    "Credit_Mix",
    "Payment_of_Min_Amount",
    "Spending_Level",
    "Payment_Value_Level"
]

# FINAL FEATURES
final_model_features = (
    numeric_features
    + binary_features
    + categorical_features
)

### 2. Class Feature Engineering

In [ ]:
class CreditFeatureEngineer:
    def __init__(self):
        self.loan_types = {
            "Has_Payday_Loan": "Payday Loan",
            "Has_Mortgage_Loan": "Mortgage Loan",
            "Has_Student_Loan": "Student Loan",
            "Has_Credit_Builder_Loan": "Credit-Builder Loan",
            "Has_Debt_Consolidation_Loan": "Debt Consolidation Loan",
            "Has_Personal_Loan": "Personal Loan",
            "Has_Auto_Loan": "Auto Loan",
            "Has_Home_Equity_Loan": "Home Equity Loan",
            "Has_Not_Specified_Loan": "Not Specified"
        }

    def safe_divide(self, numerator, denominator):
        numerator = pd.to_numeric(numerator, errors="coerce")
        denominator = pd.to_numeric(denominator, errors="coerce")

        result = np.where(
            denominator != 0,
            numerator / denominator,
            np.nan
        )

        return result

    def add_missing_indicator(self, df):
        df["Credit_History_Age_Missing"] = (
            df["Credit_History_Age"].isna().astype(int)
        )

        return df

    def add_ratio_features(self, df):
        df["EMI_to_Salary_Ratio"] = self.safe_divide(
            df["Total_EMI_per_month"],
            df["Monthly_Inhand_Salary"]
        )

        df["Investment_to_Salary_Ratio"] = self.safe_divide(
            df["Amount_invested_monthly"],
            df["Monthly_Inhand_Salary"]
        )

        df["Balance_to_Salary_Ratio"] = self.safe_divide(
            df["Monthly_Balance"],
            df["Monthly_Inhand_Salary"]
        )

        df["Debt_to_Salary_Ratio"] = self.safe_divide(
            df["Outstanding_Debt"],
            df["Monthly_Inhand_Salary"]
        )

        return df

    def add_loan_type_flags(self, df):
        loan_text = df["Type_of_Loan"].fillna("No Loan").astype(str)

        for new_column, loan_name in self.loan_types.items():
            df[new_column] = (
                loan_text
                .str.contains(loan_name, case=False, regex=False)
                .astype(int)
            )

        return df

    def split_payment_behaviour(self, df):
        payment_behaviour = (
            df["Payment_Behaviour"]
            .fillna("Unknown")
            .astype(str)
        )

        df["Spending_Level"] = np.select(
            [
                payment_behaviour.str.startswith("High_spent"),
                payment_behaviour.str.startswith("Low_spent")
            ],
            [
                "High",
                "Low"
            ],
            default="Unknown"
        )

        df["Payment_Value_Level"] = np.select(
            [
                payment_behaviour.str.contains("Small_value", regex=False),
                payment_behaviour.str.contains("Medium_value", regex=False),
                payment_behaviour.str.contains("Large_value", regex=False)
            ],
            [
                "Small",
                "Medium",
                "Large"
            ],
            default="Unknown"
        )

        return df

    def transform(self, df):
        df_fe = df.copy()

        df_fe = self.add_missing_indicator(df_fe)
        df_fe = self.add_ratio_features(df_fe)
        df_fe = self.add_loan_type_flags(df_fe)
        df_fe = self.split_payment_behaviour(df_fe)

        return df_fe

### 3. Apply Feature Engineering

In [ ]:
# APPLY FEATURE ENGINEERING

feature_engineer = CreditFeatureEngineer()
df_fe = feature_engineer.transform(df)

print("Shape sebelum feature engineering:", df.shape)
print("Shape setelah feature engineering:", df_fe.shape)

display(df_fe[final_model_features + [target_column, group_column]].head())

### 4. Cek Final Feature dan Missing Value

In [ ]:
# CHECK FINAL FEATURES

missing_features = [
    feature for feature in final_model_features
    if feature not in df_fe.columns
]

print("Missing feature columns:", missing_features)

print("\nMissing value in final features:")
display(
    df_fe[final_model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nTarget distribution:")
display(df_fe[target_column].value_counts())

In [ ]:
# FIX PANDAS NA BEFORE MODELING

df_fe = df_fe.copy()

# Replace pandas pd.NA with numpy np.nan because sklearn handles np.nan better
df_fe[final_model_features] = df_fe[final_model_features].replace({pd.NA: np.nan})

# Ensure numeric features are numeric float
for column in numeric_features:
    df_fe[column] = pd.to_numeric(
        df_fe[column],
        errors="coerce"
    ).astype(float)

# Ensure binary features are numeric
for column in binary_features:
    df_fe[column] = pd.to_numeric(
        df_fe[column],
        errors="coerce"
    )

# Ensure categorical features are object/string-compatible
for column in categorical_features:
    df_fe[column] = df_fe[column].astype("object")

print("Missing value after pd.NA to np.nan conversion:")
display(
    df_fe[final_model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nDtypes after conversion:")
display(df_fe[final_model_features].dtypes)

### 5. Class Data Splitter

In [ ]:
class CreditDataSplitter:
    def __init__(self, n_splits=5, random_state=42):
        self.n_splits = n_splits
        self.random_state = random_state

    def split(self, X, y, groups):
        splitter = StratifiedGroupKFold(
            n_splits=self.n_splits,
            shuffle=True,
            random_state=self.random_state
        )

        train_index, test_index = next(
            splitter.split(X, y, groups=groups)
        )

        X_train = X.iloc[train_index].copy()
        X_test = X.iloc[test_index].copy()

        y_train = y.iloc[train_index].copy()
        y_test = y.iloc[test_index].copy()

        groups_train = groups.iloc[train_index].copy()
        groups_test = groups.iloc[test_index].copy()

        return X_train, X_test, y_train, y_test, groups_train, groups_test

    def check_split(self, y_train, y_test, groups_train, groups_test):
        train_customers = set(groups_train)
        test_customers = set(groups_test)
        overlap_customers = train_customers.intersection(test_customers)

        print("Total train customers:", len(train_customers))
        print("Total test customers :", len(test_customers))
        print("Overlapping customers:", len(overlap_customers))

        print("\nTrain target distribution:")
        display(y_train.value_counts(normalize=True))

        print("\nTest target distribution:")
        display(y_test.value_counts(normalize=True))

### Penjelasan Kode `CreditDataSplitter`

Kode `CreditDataSplitter` dibuat untuk membagi dataset menjadi data training dan data testing dengan cara yang aman. Karena dataset memiliki banyak baris dari customer yang sama pada beberapa bulan, proses split tidak boleh dilakukan secara acak biasa. Jika split dilakukan secara acak biasa, ada kemungkinan `Customer_ID` yang sama muncul di data training dan data testing sekaligus. Kondisi tersebut dapat menyebabkan data leakage karena model sudah pernah melihat pola customer yang sama saat proses training.

### Tujuan Class

Class `CreditDataSplitter` memiliki tujuan utama untuk:

- Membagi data menjadi train dan test.
- Menjaga distribusi target `Credit_Score` tetap seimbang antara train dan test.
- Memastikan `Customer_ID` yang sama tidak muncul di train dan test secara bersamaan.
- Mengecek hasil split agar tidak terjadi customer overlap.

### Bagian `__init__`

Pada bagian `__init__`, terdapat dua parameter utama, yaitu `n_splits` dan `random_state`.

- `n_splits` digunakan untuk menentukan jumlah fold dalam proses split.
- `random_state` digunakan agar hasil split dapat direproduksi atau menghasilkan output yang sama ketika kode dijalankan ulang.

Karena `n_splits=5`, data dibagi menjadi 5 fold. Satu fold digunakan sebagai test set, sedangkan empat fold lainnya digunakan sebagai train set. Dengan demikian, rasio split yang dihasilkan mendekati 80% data training dan 20% data testing.

### Bagian `split()`

Fungsi `split()` digunakan untuk melakukan proses pembagian data. Fungsi ini menerima tiga input utama:

- `X`, yaitu data fitur yang digunakan untuk modeling.
- `y`, yaitu target yang akan diprediksi, yaitu `Credit_Score`.
- `groups`, yaitu `Customer_ID` yang digunakan untuk menjaga agar customer yang sama tidak masuk ke train dan test sekaligus.

Di dalam fungsi ini digunakan `StratifiedGroupKFold`. Metode ini dipilih karena menggabungkan dua kebutuhan penting:

- `Stratified`, yaitu menjaga distribusi kelas target agar tetap seimbang antara train dan test.
- `Group`, yaitu menjaga agar data dengan `Customer_ID` yang sama tetap berada pada split yang sama.

Dengan demikian, split yang dihasilkan tidak hanya mempertahankan distribusi target, tetapi juga aman dari data leakage berbasis customer.

### Proses Pengambilan Index

Kode berikut digunakan untuk mengambil index train dan test:

`train_index, test_index = next(splitter.split(X, y, groups=groups))`

Hasil dari `splitter.split()` adalah index baris untuk data training dan testing. Index tersebut kemudian digunakan untuk mengambil data dari `X`, `y`, dan `groups`.

### Pembentukan Data Train dan Test

Setelah index diperoleh, data dibagi menjadi:

- `X_train`, yaitu fitur untuk training.
- `X_test`, yaitu fitur untuk testing.
- `y_train`, yaitu target untuk training.
- `y_test`, yaitu target untuk testing.
- `groups_train`, yaitu `Customer_ID` pada data training.
- `groups_test`, yaitu `Customer_ID` pada data testing.

Penggunaan `.copy()` dilakukan agar data hasil split menjadi salinan baru dan tidak mengubah dataframe asli secara tidak sengaja.

### Bagian `check_split()`

Fungsi `check_split()` digunakan untuk mengecek apakah hasil split sudah benar. Pengecekan dilakukan dengan membandingkan daftar customer pada train dan test.

Di dalam fungsi ini, `groups_train` dan `groups_test` diubah menjadi set. Setelah itu, dilakukan pengecekan intersection untuk melihat apakah ada customer yang muncul pada kedua split.

Jika hasil `Overlapping customers` adalah 0, berarti tidak ada `Customer_ID` yang muncul di train dan test secara bersamaan. Ini menunjukkan bahwa split sudah aman dari data leakage.

Selain itu, fungsi ini juga menampilkan distribusi target pada data training dan testing. Hal ini penting untuk memastikan bahwa proporsi kelas `Credit_Score` tetap representatif setelah proses split.




### 6. Split Data dengan StratifiedGroupKFold

In [ ]:
# PREPARE X, y, AND GROUPS

X = df_fe[final_model_features].copy()
y = df_fe[target_column].copy()
groups = df_fe[group_column].copy()

# SPLIT DATA
data_splitter = CreditDataSplitter(
    n_splits=5,
    random_state=RANDOM_STATE
)

X_train, X_test, y_train, y_test, groups_train, groups_test = data_splitter.split(
    X=X,
    y=y,
    groups=groups
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

data_splitter.check_split(
    y_train=y_train,
    y_test=y_test,
    groups_train=groups_train,
    groups_test=groups_test
)

### Data Splitting dengan StratifiedGroupKFold

Pada tahap modeling, data dibagi menjadi data training dan data testing sebelum masuk ke proses preprocessing dan training model. Split data dilakukan menggunakan `StratifiedGroupKFold` agar pembagian data tetap aman dan sesuai dengan karakteristik dataset.

Dataset ini memiliki beberapa baris dari customer yang sama karena satu `Customer_ID` dapat muncul dalam beberapa bulan. Oleh karena itu, split biasa seperti `train_test_split` kurang aman digunakan karena dapat menyebabkan customer yang sama muncul di data training dan testing sekaligus. Jika hal tersebut terjadi, model dapat mempelajari pola customer tertentu dari data training dan terlihat lebih bagus saat diuji pada data testing, padahal sebenarnya terjadi data leakage.

Untuk menghindari masalah tersebut, digunakan `Customer_ID` sebagai `groups` pada proses splitting.

### Proses Pembentukan X, y, dan Groups

Sebelum split dilakukan, data dipisahkan menjadi tiga bagian utama:

- `X` berisi seluruh fitur final yang digunakan untuk modeling.
- `y` berisi target yang akan diprediksi, yaitu `Credit_Score`.
- `groups` berisi `Customer_ID` yang digunakan untuk memastikan customer yang sama tidak masuk ke train dan test secara bersamaan.

Fitur yang digunakan pada `X` adalah 32 fitur final hasil seleksi dan feature engineering. Target `y` adalah kolom `Credit_Score`, sedangkan `groups` hanya digunakan untuk proses split dan tidak dimasukkan sebagai fitur model.

### Metode Split yang Digunakan

Metode split yang digunakan adalah `StratifiedGroupKFold` dengan `n_splits=5`.

Karena jumlah fold adalah 5, maka satu fold digunakan sebagai data testing dan empat fold lainnya digunakan sebagai data training. Dengan demikian, pembagian data menjadi sekitar:

- 80% data training
- 20% data testing

Metode ini dipilih karena memiliki dua keunggulan utama. Pertama, proses split tetap mempertahankan distribusi target `Credit_Score` agar proporsi kelas pada train dan test tetap seimbang. Kedua, proses split juga menjaga agar `Customer_ID` yang sama tidak muncul di train dan test sekaligus.

### Hasil Split Data

Berdasarkan hasil split, diperoleh ukuran data sebagai berikut:

- `X_train`: 17.835 baris dan 32 fitur
- `X_test`: 4.577 baris dan 32 fitur
- `y_train`: 17.835 data target
- `y_test`: 4.577 data target

Hasil ini menunjukkan bahwa data berhasil dibagi menjadi data training dan testing dengan rasio mendekati 80:20.

### Validasi Customer Overlap

Setelah split dilakukan, dilakukan pengecekan jumlah customer pada masing-masing data. Hasilnya adalah:

- Total customer pada data training: 7.212 customer
- Total customer pada data testing: 1.830 customer
- Customer yang overlap antara train dan test: 0 customer

Hasil overlap sebesar 0 menunjukkan bahwa tidak ada `Customer_ID` yang muncul di train dan test secara bersamaan. Hal ini sangat penting karena membuktikan bahwa proses split sudah aman dari data leakage berbasis customer.

Dengan tidak adanya overlap customer, model akan diuji pada customer yang benar-benar berbeda dari customer yang digunakan saat training. Hal ini membuat evaluasi model menjadi lebih realistis karena model diuji pada data customer baru.

### Distribusi Target pada Data Training

Distribusi target pada data training adalah sebagai berikut:

- `Standard`: 53,22%
- `Poor`: 29,00%
- `Good`: 17,77%

Distribusi ini menunjukkan bahwa kelas `Standard` merupakan kelas mayoritas, diikuti oleh kelas `Poor`, kemudian kelas `Good`. Karena dataset memiliki distribusi kelas yang tidak sepenuhnya seimbang, penggunaan stratified split menjadi penting agar proporsi kelas pada data training dan testing tetap representatif.

Split data ini sudah tepat digunakan karena:

- Rasio train-test mendekati 80:20.
- Distribusi target tetap dijaga menggunakan stratification.
- `Customer_ID` yang sama tidak muncul pada train dan test.
- Risiko data leakage antar customer dapat dihindari.
- Data training dapat digunakan untuk proses cross-validation dan hyperparameter tuning.
- Data testing hanya digunakan untuk evaluasi akhir model.


### 7. Encode Target

In [ ]:
# ENCODE TARGET LABEL

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

class_names = label_encoder.classes_

print("Class mapping:")
for index, class_name in enumerate(class_names):
    print(index, "=", class_name)

Target Credit_Score perlu di-encode karena masih berbentuk kategori teks, yaitu Good, Standard, dan Poor. Sebagian model machine learning, terutama XGBoost dan LightGBM, membutuhkan target dalam bentuk angka agar dapat diproses saat training.

### 8. Class Preprocessing

In [ ]:
class CreditPreprocessor:
    def __init__(self, numeric_features, binary_features, categorical_features):
        self.numeric_features = numeric_features
        self.binary_features = binary_features
        self.categorical_features = categorical_features

    def build_linear_preprocessor(self):
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", RobustScaler())
            ]
        )

        binary_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]
        )

        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]
        )

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, self.numeric_features),
                ("bin", binary_transformer, self.binary_features),
                ("cat", categorical_transformer, self.categorical_features)
            ]
        )

        return preprocessor

    def build_tree_preprocessor(self):
        numeric_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median"))
            ]
        )

        binary_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent"))
            ]
        )

        categorical_transformer = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]
        )

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, self.numeric_features),
                ("bin", binary_transformer, self.binary_features),
                ("cat", categorical_transformer, self.categorical_features)
            ]
        )

        return preprocessor

### Penjelasan Kode CreditPreprocessor

Kode `CreditPreprocessor` digunakan untuk membuat preprocessing pipeline sebelum data masuk ke model machine learning. Class ini dibuat agar proses preprocessing menjadi lebih rapi, konsisten, dan sesuai dengan jenis model yang digunakan.

Class ini menerima tiga kelompok fitur utama, yaitu:

- `numeric_features`: fitur numerik seperti `Age`, `Monthly_Inhand_Salary`, `Outstanding_Debt`, dan fitur rasio.
- `binary_features`: fitur bernilai 0 atau 1 seperti loan type flags dan missing indicator.
- `categorical_features`: fitur kategorikal seperti `Occupation`, `Credit_Mix`, `Payment_of_Min_Amount`, `Spending_Level`, dan `Payment_Value_Level`.

### build_linear_preprocessor

Method `build_linear_preprocessor()` digunakan untuk membuat preprocessing khusus model linear, yaitu Logistic Regression.

Pada bagian numeric features, digunakan:

- `SimpleImputer(strategy="median")` untuk mengisi missing value numerik menggunakan median.
- `RobustScaler()` untuk melakukan scaling pada fitur numerik.

Median digunakan karena lebih tahan terhadap outlier dibandingkan mean. `RobustScaler` digunakan karena Logistic Regression sensitif terhadap skala fitur, sedangkan dataset ini memiliki banyak outlier valid yang tetap dipertahankan.

Pada bagian binary features, digunakan:

- `SimpleImputer(strategy="most_frequent")`

Binary features tidak perlu scaling karena nilainya hanya 0 dan 1. Jika terdapat missing value, nilai tersebut diisi dengan nilai yang paling sering muncul.

Pada bagian categorical features, digunakan:

- `SimpleImputer(strategy="constant", fill_value="Missing")`
- `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`

Missing value pada categorical features diisi dengan kategori `Missing`, sehingga informasi missing tetap dipertahankan sebagai kategori tersendiri. Setelah itu, fitur kategorikal diubah menjadi bentuk numerik menggunakan One Hot Encoding.

`handle_unknown="ignore"` digunakan agar saat ada kategori baru pada data test atau data inference, pipeline tidak error.

### build_tree_preprocessor

Method `build_tree_preprocessor()` digunakan untuk membuat preprocessing khusus model tree-based seperti Random Forest, XGBoost, dan LightGBM.

Pada bagian numeric features, digunakan:

- `SimpleImputer(strategy="median")`

Berbeda dengan Logistic Regression, pada tree-based model fitur numerik tidak menggunakan scaling. Hal ini karena model berbasis tree tidak sensitif terhadap skala fitur. Model tree-based bekerja berdasarkan split pada nilai fitur, bukan berdasarkan jarak antar nilai.

Pada bagian binary features, digunakan:

- `SimpleImputer(strategy="most_frequent")`

Binary features tetap tidak perlu scaling dan hanya perlu ditangani jika terdapat missing value.

Pada bagian categorical features, prosesnya sama seperti pada model linear, yaitu:

- Missing value diisi dengan kategori `Missing`.
- Fitur kategorikal diubah menjadi numerik menggunakan One Hot Encoding.
- `handle_unknown="ignore"` digunakan agar pipeline aman saat menghadapi kategori baru.

### ColumnTransformer

Pada kedua method, `ColumnTransformer` digunakan untuk menggabungkan seluruh proses preprocessing berdasarkan jenis fitur.

Artinya:

- Numeric features diproses menggunakan numeric transformer.
- Binary features diproses menggunakan binary transformer.
- Categorical features diproses menggunakan categorical transformer.

Dengan `ColumnTransformer`, setiap kelompok fitur mendapat perlakuan preprocessing yang sesuai dengan karakteristiknya.

### Kesimpulan

Class `CreditPreprocessor` dibuat untuk memastikan preprocessing dilakukan secara konsisten dan berbasis OOP. Terdapat dua jenis preprocessor:

- `build_linear_preprocessor()` untuk Logistic Regression, dengan median imputer dan RobustScaler pada fitur numerik.
- `build_tree_preprocessor()` untuk Random Forest, XGBoost, dan LightGBM, dengan median imputer saja pada fitur numerik tanpa scaling.

Pendekatan ini sesuai dengan strategi preprocessing yang sudah dirancang sebelumnya, yaitu model linear membutuhkan scaling, sedangkan model tree-based tidak membutuhkan scaling.


### 9. Build Preprocessor

In [ ]:
# BUILD PREPROCESSORS

preprocessor_builder = CreditPreprocessor(
    numeric_features=numeric_features,
    binary_features=binary_features,
    categorical_features=categorical_features
)

linear_preprocessor = preprocessor_builder.build_linear_preprocessor()
tree_preprocessor = preprocessor_builder.build_tree_preprocessor()

### 10. Class Model Trainer

In [ ]:
class CreditModelTrainer:
    def __init__(self, linear_preprocessor, tree_preprocessor, random_state=42):
        self.linear_preprocessor = linear_preprocessor
        self.tree_preprocessor = tree_preprocessor
        self.random_state = random_state

    def get_models(self):
        models = {}

        models["Logistic Regression"] = Pipeline(
            steps=[
                ("preprocessor", self.linear_preprocessor),
                ("model", LogisticRegression(
                    max_iter=3000,
                    class_weight="balanced",
                    random_state=self.random_state
                ))
            ]
        )

        models["Random Forest"] = Pipeline(
            steps=[
                ("preprocessor", self.tree_preprocessor),
                ("model", RandomForestClassifier(
                    random_state=self.random_state,
                    class_weight="balanced",
                    n_jobs=-1
                ))
            ]
        )

        if XGBOOST_AVAILABLE:
            models["XGBoost"] = Pipeline(
                steps=[
                    ("preprocessor", self.tree_preprocessor),
                    ("model", XGBClassifier(
                        objective="multi:softprob",
                        eval_metric="mlogloss",
                        random_state=self.random_state,
                        n_jobs=-1,
                        tree_method="hist"
                    ))
                ]
            )

        if LIGHTGBM_AVAILABLE:
            models["LightGBM"] = Pipeline(
                steps=[
                    ("preprocessor", self.tree_preprocessor),
                    ("model", LGBMClassifier(
                        random_state=self.random_state,
                        class_weight="balanced",
                        n_jobs=-1,
                        verbose=-1
                    ))
                ]
            )

        return models

    def get_param_distributions(self):
        param_distributions = {}

        param_distributions["Logistic Regression"] = {
            "model__C": loguniform(0.001, 10),
            "model__solver": ["lbfgs", "saga"],
            "model__penalty": ["l2"]
        }

        param_distributions["Random Forest"] = {
            "model__n_estimators": randint(200, 700),
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_split": randint(2, 10),
            "model__min_samples_leaf": randint(1, 5),
            "model__max_features": ["sqrt", "log2", None]
        }

        if XGBOOST_AVAILABLE:
            param_distributions["XGBoost"] = {
                "model__n_estimators": randint(200, 700),
                "model__max_depth": randint(3, 9),
                "model__learning_rate": loguniform(0.01, 0.2),
                "model__subsample": uniform(0.7, 0.3),
                "model__colsample_bytree": uniform(0.7, 0.3),
                "model__reg_alpha": loguniform(0.0001, 5),
                "model__reg_lambda": loguniform(0.01, 10)
            }

        if LIGHTGBM_AVAILABLE:
            param_distributions["LightGBM"] = {
                "model__n_estimators": randint(200, 700),
                "model__learning_rate": loguniform(0.01, 0.2),
                "model__num_leaves": randint(15, 80),
                "model__max_depth": [-1, 5, 8, 12, 20],
                "model__min_child_samples": randint(10, 60),
                "model__subsample": uniform(0.7, 0.3),
                "model__colsample_bytree": uniform(0.7, 0.3),
                "model__reg_lambda": loguniform(0.01, 10)
            }

        return param_distributions

    def tune_models(
        self,
        X_train,
        y_train,
        groups_train,
        n_iter=20,
        cv_splits=5,
        scoring="f1_macro"
    ):
        models = self.get_models()
        param_distributions = self.get_param_distributions()

        cv = StratifiedGroupKFold(
            n_splits=cv_splits,
            shuffle=True,
            random_state=self.random_state
        )

        tuned_models = {}
        tuning_results = []

        for model_name, pipeline in models.items():
            print("=" * 80)
            print("Training model:", model_name)
            print("=" * 80)

            random_search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_distributions[model_name],
                n_iter=n_iter,
                scoring=scoring,
                cv=cv,
                random_state=self.random_state,
                n_jobs=-1,
                verbose=1,
                refit=True
            )

            random_search.fit(
                X_train,
                y_train,
                groups=groups_train
            )

            tuned_models[model_name] = random_search.best_estimator_

            tuning_results.append({
                "model_name": model_name,
                "best_cv_score": random_search.best_score_,
                "best_params": random_search.best_params_
            })

            print("Best CV score:", random_search.best_score_)
            print("Best params:", random_search.best_params_)

        tuning_results_df = pd.DataFrame(tuning_results)

        return tuned_models, tuning_results_df

# Penjelasan Pemilihan Model dan CreditModelTrainer

## Tujuan Tahap Modeling

Tahap modeling dilakukan untuk membangun model klasifikasi yang dapat memprediksi `Credit_Score` customer ke dalam beberapa kelas, yaitu `Good`, `Standard`, dan `Poor`.

Berdasarkan hasil EDA, preprocessing, dan cleaning yang sudah dilakukan, dataset ini memiliki karakteristik sebagai berikut:

- Target `Credit_Score` merupakan masalah klasifikasi multi-class.
- Distribusi target tidak sepenuhnya seimbang, karena kelas `Standard` menjadi kelas mayoritas, sedangkan `Good` dan `Poor` memiliki proporsi lebih kecil.
- Dataset berisi campuran fitur numerik, kategorikal, binary feature, dan fitur hasil feature engineering.
- Terdapat beberapa customer yang sama muncul dalam beberapa bulan, sehingga proses split dan cross-validation harus memperhatikan `Customer_ID`.
- Banyak outlier valid yang tetap dipertahankan karena masih masuk akal secara konteks bisnis.
- Beberapa fitur numerik memiliki skala yang sangat berbeda, seperti `Age`, `Monthly_Inhand_Salary`, `Outstanding_Debt`, `Credit_History_Age`, dan ratio features.
- Beberapa fitur kategorikal perlu diproses menggunakan encoding.
- Beberapa missing value tidak dihapus, tetapi ditangani di pipeline menggunakan imputer.

Berdasarkan karakteristik tersebut, model yang dipilih harus mampu menangani data tabular, fitur numerik dan kategorikal hasil encoding, target multi-class, imbalance antar kelas, serta pola hubungan yang kemungkinan tidak sepenuhnya linear.

## Alasan Menggunakan Beberapa Model

Pada project ini, beberapa model dibandingkan agar performa tidak hanya bergantung pada satu algoritma. Model yang digunakan adalah:

- Logistic Regression
- Random Forest
- XGBoost
- LightGBM

Pemilihan beberapa model ini dilakukan agar dapat membandingkan model linear sederhana dengan model tree-based yang lebih kompleks. Dengan cara ini, hasil akhir tidak hanya menunjukkan satu model, tetapi juga menunjukkan proses eksperimen yang lebih kuat dan sesuai dengan kebutuhan project machine learning.

## 1. Logistic Regression

### Alasan Dipilih

`Logistic Regression` digunakan sebagai baseline model.

Model ini dipilih karena:

- Sederhana dan cepat dilatih.
- Cocok digunakan sebagai pembanding awal sebelum menggunakan model yang lebih kompleks.
- Mudah diinterpretasikan.
- Dapat digunakan untuk klasifikasi multi-class.
- Cocok untuk melihat apakah hubungan fitur terhadap target masih dapat dijelaskan secara linear.

Dalam project credit scoring, baseline model penting karena dapat menunjukkan apakah fitur-fitur yang sudah dibersihkan dan dibuat melalui feature engineering sudah cukup informatif bahkan untuk model sederhana.

### Hubungan dengan Hasil Preprocessing

Karena `Logistic Regression` merupakan model linear, model ini sensitif terhadap skala fitur. Dataset memiliki fitur numerik dengan skala yang sangat berbeda, seperti:

- `Age`
- `Monthly_Inhand_Salary`
- `Outstanding_Debt`
- `Credit_History_Age`
- `Interest_Rate`
- `EMI_to_Salary_Ratio`
- `Investment_to_Salary_Ratio`
- `Balance_to_Salary_Ratio`
- `Debt_to_Salary_Ratio`

Oleh karena itu, Logistic Regression menggunakan `linear_preprocessor`, yaitu preprocessing khusus yang terdiri dari:

- numeric imputation menggunakan median,
- scaling menggunakan `RobustScaler`,
- categorical encoding menggunakan `OneHotEncoder`.

`RobustScaler` dipilih karena dataset memiliki outlier valid yang tetap dipertahankan. Scaler ini lebih tahan terhadap outlier karena menggunakan median dan IQR.

### Pengaturan Model

Pada kode, Logistic Regression dibuat dengan:

- `max_iter=3000` agar proses optimasi memiliki cukup iterasi untuk konvergen.
- `class_weight="balanced"` untuk membantu menangani distribusi kelas yang tidak seimbang.
- `random_state` agar hasil training lebih reproducible.

## 2. Random Forest

### Alasan Dipilih

`Random Forest` digunakan karena merupakan model ensemble berbasis decision tree yang kuat untuk data tabular.

Model ini dipilih karena:

- Mampu menangkap hubungan non-linear antar fitur.
- Lebih tahan terhadap outlier dibanding model linear.
- Dapat menangani interaksi antar fitur tanpa perlu dibuat secara manual.
- Cocok untuk dataset dengan banyak fitur numerik, binary, dan kategorikal hasil encoding.
- Dapat menjadi pembanding yang kuat terhadap Logistic Regression.

Berdasarkan hasil EDA, beberapa fitur seperti `Outstanding_Debt`, `Delay_from_due_date`, `Num_of_Delayed_Payment`, `Credit_Mix`, dan ratio features kemungkinan memiliki hubungan non-linear terhadap `Credit_Score`. Karena itu, model tree-based seperti Random Forest cocok digunakan.

### Hubungan dengan Hasil Preprocessing

Random Forest tidak membutuhkan scaling karena model ini bekerja berdasarkan split pada nilai fitur, bukan berdasarkan jarak antar nilai. Oleh karena itu, Random Forest menggunakan `tree_preprocessor`, yaitu:

- numeric imputation menggunakan median,
- binary feature passthrough setelah imputasi,
- categorical feature diimpute dan di-encode.

Tidak menggunakan scaling pada Random Forest membuat nilai fitur tetap lebih natural dan sesuai dengan karakter model tree-based.

### Pengaturan Model

Pada kode, Random Forest dibuat dengan:

- `class_weight="balanced"` untuk membantu menangani imbalance target.
- `n_jobs=-1` agar training dapat menggunakan seluruh core CPU yang tersedia.
- `random_state` agar hasil eksperimen lebih konsisten.

## 3. XGBoost

### Alasan Dipilih

`XGBoost` digunakan karena merupakan model gradient boosting yang sangat kuat untuk data tabular.

Model ini dipilih karena:

- Sering memberikan performa tinggi pada data tabular.
- Mampu menangkap pola non-linear yang kompleks.
- Membangun model secara bertahap dengan memperbaiki kesalahan model sebelumnya.
- Cocok untuk fitur hasil feature engineering seperti ratio features, loan type flags, dan payment behaviour split.
- Dapat menangani kombinasi fitur numerik dan hasil encoding dengan baik.

Pada dataset credit scoring, pola antar fitur kemungkinan cukup kompleks. Misalnya, risiko kredit tidak hanya ditentukan oleh nilai debt yang tinggi, tetapi juga oleh salary, EMI ratio, payment delay, credit mix, dan jenis loan. XGBoost cocok digunakan karena mampu mempelajari kombinasi pola tersebut.

### Hubungan dengan Hasil Preprocessing

XGBoost termasuk model tree-based, sehingga tidak membutuhkan scaling. Karena itu, XGBoost menggunakan `tree_preprocessor`.

Preprocessing tetap diperlukan untuk:

- mengisi missing numeric dengan median,
- mengisi missing categorical dengan `"Missing"`,
- mengubah categorical feature menjadi numeric melalui `OneHotEncoder`.

### Pengaturan Model

Pada kode, XGBoost dibuat dengan:

- `objective="multi:softprob"` karena target terdiri dari lebih dari dua kelas.
- `eval_metric="mlogloss"` sebagai metrik internal untuk multi-class classification.
- `tree_method="hist"` agar training lebih efisien.
- `n_jobs=-1` agar proses training lebih cepat.
- `random_state` untuk reproducibility.



## 4. LightGBM

### Alasan Dipilih

`LightGBM` digunakan sebagai model boosting lain yang efisien untuk data tabular.

Model ini dipilih karena:

- Cepat dan efisien untuk dataset tabular.
- Mampu menangkap hubungan non-linear.
- Biasanya memiliki performa tinggi pada fitur numerik dan kategorikal hasil encoding.
- Dapat menjadi pembanding langsung dengan XGBoost.
- Cocok untuk eksperimen model deployment karena training relatif cepat.

LightGBM sering digunakan pada masalah klasifikasi tabular karena efisiensinya. Pada dataset ini, LightGBM cocok untuk mempelajari kombinasi fitur credit risk seperti delayed payment, debt, salary ratio, credit mix, dan loan type flags.

### Hubungan dengan Hasil Preprocessing

Sama seperti Random Forest dan XGBoost, LightGBM tidak membutuhkan scaling karena termasuk model tree-based. Oleh karena itu, LightGBM juga menggunakan `tree_preprocessor`.

Preprocessing tetap dilakukan untuk menjaga proses training, evaluation, dan inference tetap konsisten.

### Pengaturan Model

Pada kode, LightGBM dibuat dengan:

- `class_weight="balanced"` untuk membantu mengatasi distribusi target yang tidak seimbang.
- `n_jobs=-1` agar training lebih cepat.
- `verbose=-1` agar output training tidak terlalu ramai.
- `random_state` agar hasil lebih reproducible.

LightGBM hanya digunakan jika library `lightgbm` tersedia di environment.

## Alasan Menggunakan Pipeline

Setiap model dibuat dalam bentuk `Pipeline` yang berisi:

- preprocessing,
- model machine learning.

Contoh struktur pipeline:

- `("preprocessor", self.linear_preprocessor)`
- `("model", LogisticRegression(...))`

atau

- `("preprocessor", self.tree_preprocessor)`
- `("model", RandomForestClassifier(...))`

Pipeline digunakan agar preprocessing selalu dilakukan dengan cara yang sama pada training, cross-validation, testing, dan inference. Hal ini penting untuk mencegah data leakage.

Dengan pipeline, imputer, scaler, dan encoder hanya di-fit pada data training dalam setiap fold cross-validation. Data validation atau test hanya ditransform menggunakan preprocessing yang sudah dipelajari dari train.

## Alasan Menggunakan Hyperparameter Tuning

Setiap model memiliki hyperparameter yang dapat memengaruhi performa. Oleh karena itu, digunakan `RandomizedSearchCV` untuk mencari kombinasi hyperparameter terbaik.

Randomized search dipilih karena:

- Lebih cepat dibanding grid search penuh.
- Cocok untuk mencoba beberapa kombinasi parameter tanpa membuat proses training terlalu berat.
- Tetap memberikan peluang menemukan kombinasi parameter yang baik.
- Sesuai untuk eksperimen beberapa model sekaligus.

Pada kode, jumlah kombinasi parameter dikontrol melalui `n_iter`. Misalnya, jika `n_iter=20`, maka setiap model akan mencoba 20 kombinasi hyperparameter secara acak.

## Alasan Menggunakan StratifiedGroupKFold saat Tuning

Cross-validation pada tahap tuning menggunakan `StratifiedGroupKFold`.

Metode ini dipilih karena dataset memiliki dua kondisi penting:

- Target `Credit_Score` tidak sepenuhnya seimbang.
- Satu `Customer_ID` dapat muncul pada beberapa bulan.

Jika menggunakan KFold biasa, customer yang sama bisa muncul di train dan validation fold sekaligus. Hal ini dapat menyebabkan data leakage karena model melihat pola customer yang sama saat training dan validation.

Dengan `StratifiedGroupKFold`, proses tuning menjadi lebih aman karena:

- distribusi target tetap dijaga pada setiap fold,
- customer yang sama tidak masuk ke train dan validation fold sekaligus,
- evaluasi cross-validation menjadi lebih realistis.

## Alasan Menggunakan F1-Score sebagai Metrik Utama

Untuk kasus ini, F1-score lebih cocok dibanding hanya menggunakan precision atau hanya menggunakan recall karena target `Credit_Score` adalah target multi-class dan distribusi kelasnya tidak seimbang.

Target yang diprediksi terdiri dari tiga kelas, yaitu:

- `Good`
- `Standard`
- `Poor`

Berdasarkan hasil EDA, kelas `Standard` merupakan kelas dengan jumlah data paling banyak, sedangkan kelas `Good` merupakan kelas dengan jumlah data paling sedikit. Kondisi ini menunjukkan bahwa dataset memiliki class imbalance. Jika evaluasi model hanya menggunakan accuracy, model bisa terlihat memiliki performa tinggi hanya karena sering memprediksi kelas mayoritas, yaitu `Standard`.

Oleh karena itu, diperlukan metrik yang lebih adil untuk mengevaluasi performa model pada seluruh kelas, bukan hanya pada kelas mayoritas.

## Mengapa Tidak Menggunakan Precision Saja

Precision mengukur seberapa tepat prediksi model untuk suatu kelas.

Contohnya, jika model memprediksi seorang customer sebagai `Poor`, precision melihat berapa banyak dari prediksi `Poor` tersebut yang benar-benar merupakan customer `Poor`.

Precision penting karena dapat menunjukkan apakah model terlalu sering memberikan prediksi positif atau prediksi kelas tertentu secara salah. Namun, precision saja tidak cukup karena precision tidak terlalu memperhatikan berapa banyak data asli dari suatu kelas yang gagal ditemukan oleh model.

Dalam konteks credit scoring, jika hanya fokus pada precision, model bisa menjadi terlalu berhati-hati dalam memprediksi kelas tertentu. Akibatnya, beberapa customer yang sebenarnya berisiko dapat gagal terdeteksi.

## Mengapa Tidak Menggunakan Recall Saja

Recall mengukur seberapa banyak data asli dari suatu kelas yang berhasil ditemukan oleh model.

Contohnya, dari semua customer yang sebenarnya termasuk kelas `Poor`, recall melihat berapa banyak yang berhasil diprediksi sebagai `Poor` oleh model.

Recall penting karena dapat menunjukkan kemampuan model dalam menangkap data dari suatu kelas. Namun, recall saja juga tidak cukup karena recall tidak terlalu memperhatikan apakah model terlalu sering salah memasukkan customer dari kelas lain ke kelas tersebut.

Dalam konteks credit scoring, jika hanya fokus pada recall, model bisa terlalu sering memberi label berisiko kepada customer yang sebenarnya tidak berisiko. Hal ini dapat menyebabkan terlalu banyak false alarm.

## Mengapa F1-Score Lebih Cocok

F1-score menggabungkan precision dan recall ke dalam satu metrik. Dengan demikian, F1-score dapat menilai keseimbangan antara ketepatan prediksi dan kemampuan model dalam menemukan data dari setiap kelas.

Formula F1-score adalah:

`F1-score = 2 × (Precision × Recall) / (Precision + Recall)`

F1-score akan tinggi jika precision dan recall sama-sama baik. Jika salah satu dari precision atau recall rendah, maka nilai F1-score juga akan turun.

Hal ini membuat F1-score lebih cocok untuk dataset credit scoring karena model tidak hanya dituntut untuk tepat saat memberikan prediksi, tetapi juga harus mampu mengenali seluruh kelas dengan baik.

## Mengapa Menggunakan F1 Macro

Pada project ini, metrik yang digunakan untuk hyperparameter tuning adalah `f1_macro`.

F1 Macro menghitung F1-score untuk setiap kelas secara terpisah, lalu mengambil rata-ratanya tanpa mempertimbangkan jumlah data pada masing-masing kelas.

Artinya, setiap kelas diberi bobot yang sama, baik kelas mayoritas maupun kelas minoritas.

Hal ini penting karena target `Credit_Score` memiliki distribusi kelas yang tidak seimbang. Jika menggunakan metrik yang terlalu dipengaruhi oleh jumlah data, model bisa terlihat bagus hanya karena kuat dalam memprediksi kelas mayoritas, tetapi lemah pada kelas minoritas.

Dengan menggunakan F1 Macro, performa model pada kelas `Good`, `Standard`, dan `Poor` tetap diperhatikan secara seimbang.

## Hubungan dengan Tujuan Model

Tujuan model adalah memprediksi `Credit_Score` customer ke dalam kelas `Good`, `Standard`, atau `Poor`. Dalam konteks ini, model harus mampu membedakan kualitas kredit customer secara cukup seimbang.

Model tidak boleh hanya bagus dalam mengenali customer `Standard`, karena kelas tersebut merupakan kelas mayoritas. Model juga harus tetap mampu mengenali customer `Good` dan `Poor` dengan baik.

Khususnya untuk kelas `Poor`, model perlu cukup sensitif karena kelas ini dapat menunjukkan customer dengan risiko kredit lebih tinggi. Namun, model juga tidak boleh sembarangan memberi label `Poor` kepada customer lain karena hal tersebut dapat menurunkan precision.

Karena itu, F1-score lebih sesuai dibanding hanya precision atau recall saja. F1-score menjaga agar model tetap seimbang antara ketepatan prediksi dan kemampuan mendeteksi setiap kelas.


## Penjelasan Singkat Kode `CreditModelTrainer`

### Method `__init__`

Method ini digunakan untuk menyimpan:

- `linear_preprocessor`
- `tree_preprocessor`
- `random_state`

`linear_preprocessor` digunakan untuk Logistic Regression, sedangkan `tree_preprocessor` digunakan untuk Random Forest, XGBoost, dan LightGBM.

### Method `get_models`

Method ini membuat dictionary berisi seluruh model yang akan dibandingkan.

Model yang dibuat adalah:

- Logistic Regression
- Random Forest
- XGBoost jika library tersedia
- LightGBM jika library tersedia

Setiap model dibungkus dalam pipeline agar preprocessing dan training model berjalan dalam satu alur yang konsisten.

### Method `get_param_distributions`

Method ini membuat dictionary berisi ruang pencarian hyperparameter untuk setiap model.

Contohnya:

- Logistic Regression mencari nilai terbaik untuk `C`, `solver`, dan `penalty`.
- Random Forest mencari nilai terbaik untuk `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, dan `max_features`.
- XGBoost mencari nilai terbaik untuk `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, `reg_alpha`, dan `reg_lambda`.
- LightGBM mencari nilai terbaik untuk `n_estimators`, `learning_rate`, `num_leaves`, `max_depth`, `min_child_samples`, `subsample`, `colsample_bytree`, dan `reg_lambda`.

### Method `tune_models`

Method ini menjalankan proses training dan hyperparameter tuning untuk semua model.

Langkah yang dilakukan:

- mengambil daftar model dari `get_models()`,
- mengambil daftar parameter dari `get_param_distributions()`,
- membuat cross-validation menggunakan `StratifiedGroupKFold`,
- menjalankan `RandomizedSearchCV` untuk setiap model,
- menyimpan best estimator dari setiap model,
- menyimpan hasil tuning ke dalam dataframe.

Output dari method ini adalah:

- `tuned_models`
- `tuning_results_df`

`tuned_models` berisi pipeline model terbaik untuk setiap algoritma. `tuning_results_df` berisi ringkasan nama model, skor cross-validation terbaik, dan parameter terbaik.

## Kesimpulan

Model yang dipilih sudah sesuai dengan karakteristik dataset dan hasil preprocessing. Logistic Regression digunakan sebagai baseline yang sederhana dan mudah diinterpretasikan. Random Forest digunakan untuk menangkap pola non-linear dengan pendekatan ensemble. XGBoost dan LightGBM digunakan karena keduanya merupakan boosting model yang kuat untuk data tabular.

Penggunaan pipeline, `RandomizedSearchCV`, dan `StratifiedGroupKFold` membuat proses modeling lebih rapi, aman dari data leakage, dan sesuai dengan struktur dataset yang memiliki customer berulang dalam beberapa bulan. Pemilihan `f1_macro` sebagai metrik tuning juga sesuai karena target memiliki distribusi kelas yang tidak seimbang.




# Analisis Data sebagai Dasar Hyperparameter Tuning

Sebelum melakukan hyperparameter tuning, karakteristik dataset perlu dipahami terlebih dahulu agar parameter yang digunakan sesuai dengan kondisi data. Berdasarkan hasil EDA dan preprocessing, target `Credit_Score` terdiri dari tiga kelas, yaitu `Good`, `Standard`, dan `Poor`, sehingga masalah ini termasuk **multi-class classification**.

Distribusi target juga tidak sepenuhnya seimbang. Kelas `Standard` menjadi kelas mayoritas, sedangkan kelas `Good` dan `Poor` memiliki jumlah data yang lebih kecil. Karena itu, tuning tidak hanya menggunakan accuracy, tetapi menggunakan `f1_macro` agar performa setiap kelas dinilai secara lebih adil.

Dataset juga terdiri dari berbagai jenis fitur, yaitu fitur numerik, kategorikal, binary loan flags, dan fitur hasil feature engineering. Beberapa fitur numerik memiliki skala yang berbeda jauh, seperti `Age`, `Monthly_Inhand_Salary`, `Outstanding_Debt`, dan ratio features. Selain itu, terdapat outlier valid yang tetap dipertahankan karena masih masuk akal secara bisnis, misalnya customer dengan debt tinggi, salary tinggi, atau jumlah keterlambatan pembayaran yang besar.

Dataset juga memiliki customer yang muncul pada beberapa bulan berbeda. Oleh karena itu, cross-validation menggunakan `StratifiedGroupKFold` agar customer yang sama tidak masuk ke train dan validation fold sekaligus. Hal ini penting untuk mencegah data leakage dan membuat evaluasi tuning lebih realistis.

Berdasarkan karakteristik tersebut, model yang digunakan perlu mampu menangani data tabular, class imbalance, fitur campuran, outlier valid, dan hubungan antarfitur yang kemungkinan tidak sepenuhnya linear. Karena itu, tuning dilakukan pada Logistic Regression, Random Forest, XGBoost, dan LightGBM.

## Tujuan `get_param_distributions`

Method `get_param_distributions` digunakan untuk menentukan ruang pencarian hyperparameter untuk setiap model. Parameter ditulis dengan awalan `model__` karena model berada di dalam `Pipeline`.

Contohnya, `model__C` berarti parameter `C` pada model Logistic Regression, sedangkan `model__n_estimators` berarti parameter `n_estimators` pada model tree-based.

Hyperparameter tuning dilakukan menggunakan `RandomizedSearchCV` karena jumlah kombinasi parameter cukup banyak. Dibandingkan Grid Search, Randomized Search lebih efisien karena tidak mencoba semua kombinasi, tetapi tetap dapat mencari kombinasi parameter yang baik dalam waktu yang lebih masuk akal.

## Logistic Regression Hyperparameter

Logistic Regression digunakan sebagai baseline model karena sederhana, cepat, dan mudah diinterpretasikan. Model ini cocok untuk melihat apakah fitur yang sudah diproses memiliki sinyal yang cukup kuat untuk memprediksi `Credit_Score`.

Ruang pencarian Logistic Regression adalah:

* `model__C`: `loguniform(0.001, 10)`
* `model__solver`: `["lbfgs", "saga"]`
* `model__penalty`: `["l2"]`

### `model__C`

Parameter `C` mengatur kekuatan regularisasi. Nilai `C` kecil berarti regularisasi lebih kuat, sedangkan nilai `C` besar berarti regularisasi lebih lemah.

Rentang `0.001` sampai `10` digunakan agar model dapat mencoba regularisasi dari yang kuat sampai yang lebih fleksibel. Hal ini penting karena dataset memiliki banyak fitur setelah encoding dan feature engineering. Jika regularisasi terlalu lemah, model bisa overfit. Jika terlalu kuat, model bisa underfit.

### `model__solver`

Solver menentukan algoritma optimasi yang digunakan.

`lbfgs` digunakan karena stabil dan umum digunakan untuk Logistic Regression dengan penalty `l2`. `saga` digunakan karena lebih fleksibel untuk data dengan banyak fitur, terutama setelah categorical features diproses menggunakan OneHotEncoder.

### `model__penalty`

Penalty yang digunakan adalah `l2`. Regularisasi `l2` dipilih karena stabil, cocok untuk baseline, dan membantu mengecilkan koefisien agar tidak terlalu ekstrem tanpa menghapus fitur sepenuhnya.

## Random Forest Hyperparameter

Random Forest digunakan karena mampu menangkap hubungan non-linear dan cocok untuk data tabular. Model ini juga lebih tahan terhadap outlier dibanding model linear.

Ruang pencarian Random Forest adalah:

* `model__n_estimators`: `randint(200, 700)`
* `model__max_depth`: `[None, 10, 20, 30]`
* `model__min_samples_split`: `randint(2, 10)`
* `model__min_samples_leaf`: `randint(1, 5)`
* `model__max_features`: `["sqrt", "log2", None]`

### `model__n_estimators`

Parameter ini menentukan jumlah tree dalam Random Forest. Rentang 200 sampai 700 digunakan agar model memiliki cukup banyak tree untuk menghasilkan prediksi yang stabil, tetapi tetap tidak terlalu berat untuk training.

### `model__max_depth`

Parameter ini mengatur kedalaman maksimum setiap tree. Nilai `None` membuat tree tumbuh tanpa batas, sedangkan nilai 10, 20, dan 30 membatasi kompleksitas tree.

Parameter ini penting karena hubungan antarfitur dalam credit scoring bisa kompleks, tetapi model tetap perlu dikontrol agar tidak overfit terhadap outlier atau pola yang terlalu spesifik.

### `model__min_samples_split`

Parameter ini menentukan jumlah minimum sample yang dibutuhkan agar sebuah node dapat dipecah. Rentang 2 sampai 9 digunakan agar tuning dapat memilih apakah tree perlu lebih fleksibel atau lebih konservatif.

Nilai kecil membuat model lebih detail, sedangkan nilai besar membantu mengurangi overfitting.

### `model__min_samples_leaf`

Parameter ini menentukan jumlah minimum sample pada leaf node. Rentang 1 sampai 4 digunakan untuk menjaga keseimbangan antara fleksibilitas dan stabilitas model.

Jika leaf terlalu kecil, model dapat menghafal outlier. Jika leaf terlalu besar, model bisa kehilangan pola penting pada kelas minoritas.

### `model__max_features`

Parameter ini menentukan jumlah fitur yang dipertimbangkan saat split. `sqrt` dan `log2` membuat model lebih acak dan dapat mengurangi overfitting, sedangkan `None` membuat semua fitur dipertimbangkan.

Parameter ini penting karena dataset memiliki banyak fitur setelah encoding dan feature engineering.

## XGBoost Hyperparameter

XGBoost digunakan karena merupakan boosting model yang kuat untuk data tabular. Model ini membangun tree secara bertahap, di mana tree berikutnya memperbaiki kesalahan tree sebelumnya.

Ruang pencarian XGBoost adalah:

* `model__n_estimators`: `randint(200, 700)`
* `model__max_depth`: `randint(3, 9)`
* `model__learning_rate`: `loguniform(0.01, 0.2)`
* `model__subsample`: `uniform(0.7, 0.3)`
* `model__colsample_bytree`: `uniform(0.7, 0.3)`
* `model__reg_alpha`: `loguniform(0.0001, 5)`
* `model__reg_lambda`: `loguniform(0.01, 10)`

### `model__n_estimators`

Parameter ini menentukan jumlah tree. Rentang 200 sampai 700 digunakan agar model memiliki cukup banyak boosting rounds untuk mempelajari pola data, tetapi tetap efisien untuk proses tuning.

### `model__max_depth`

Parameter ini menentukan kedalaman tree. Rentang 3 sampai 8 digunakan agar model dapat menangkap pola non-linear tanpa membuat tree terlalu dalam.

Nilai kecil membuat model lebih sederhana, sedangkan nilai besar memungkinkan model menangkap interaksi fitur yang lebih kompleks.

### `model__learning_rate`

Parameter ini mengatur seberapa besar kontribusi setiap tree baru. Rentang 0.01 sampai 0.2 digunakan karena learning rate perlu diseimbangkan dengan jumlah tree.

Learning rate kecil membuat proses belajar lebih stabil, sedangkan learning rate besar membuat model belajar lebih cepat tetapi lebih berisiko overfit.

### `model__subsample`

Parameter ini menentukan proporsi data yang digunakan untuk setiap tree. Rentang 0.7 sampai 1.0 digunakan untuk mengurangi risiko overfitting.

Dengan tidak selalu menggunakan seluruh data pada setiap tree, model menjadi lebih general dan tidak terlalu bergantung pada pola sample tertentu.

### `model__colsample_bytree`

Parameter ini menentukan proporsi fitur yang digunakan untuk setiap tree. Rentang 0.7 sampai 1.0 digunakan karena dataset memiliki banyak fitur setelah encoding dan feature engineering.

Sampling fitur membantu model tidak terlalu bergantung pada fitur tertentu yang terlalu dominan.

### `model__reg_alpha`

Parameter ini adalah regularisasi L1. Rentang 0.0001 sampai 5 digunakan untuk membantu model lebih selektif terhadap fitur dan mengurangi pengaruh fitur yang kurang penting.

### `model__reg_lambda`

Parameter ini adalah regularisasi L2. Rentang 0.01 sampai 10 digunakan untuk mengontrol kompleksitas model dan mencegah bobot model menjadi terlalu ekstrem, terutama karena dataset memiliki outlier valid.

## LightGBM Hyperparameter

LightGBM digunakan karena efisien dan kuat untuk data tabular. Model ini menggunakan pendekatan leaf-wise, sehingga dapat menangkap pola kompleks dengan cepat, tetapi perlu dikontrol agar tidak overfit.

Ruang pencarian LightGBM adalah:

* `model__n_estimators`: `randint(200, 700)`
* `model__learning_rate`: `loguniform(0.01, 0.2)`
* `model__num_leaves`: `randint(15, 80)`
* `model__max_depth`: `[-1, 5, 8, 12, 20]`
* `model__min_child_samples`: `randint(10, 60)`
* `model__subsample`: `uniform(0.7, 0.3)`
* `model__colsample_bytree`: `uniform(0.7, 0.3)`
* `model__reg_lambda`: `loguniform(0.01, 10)`

### `model__n_estimators`

Parameter ini menentukan jumlah tree. Rentang 200 sampai 700 digunakan agar LightGBM memiliki cukup banyak tree untuk belajar, tetapi tetap efisien saat tuning.

### `model__learning_rate`

Parameter ini mengatur besar kontribusi setiap tree. Rentang 0.01 sampai 0.2 digunakan agar model dapat mencoba proses belajar yang lebih hati-hati maupun lebih cepat.

### `model__num_leaves`

Parameter ini menentukan jumlah maksimum leaves dalam satu tree. Rentang 15 sampai 80 digunakan karena LightGBM sangat dipengaruhi oleh jumlah leaves.

Jumlah leaves kecil membuat model lebih sederhana, sedangkan jumlah leaves besar membuat model lebih mampu menangkap pola kompleks. Namun, jika terlalu besar, model dapat overfit.

### `model__max_depth`

Parameter ini membatasi kedalaman tree. Nilai `-1` berarti tidak ada batas kedalaman, sedangkan 5, 8, 12, dan 20 memberikan batas kompleksitas.

Parameter ini digunakan bersama `num_leaves` untuk mengontrol agar tree tidak terlalu kompleks.

### `model__min_child_samples`

Parameter ini menentukan jumlah minimum data pada leaf. Rentang 10 sampai 60 digunakan untuk mencegah model membuat leaf yang terlalu kecil.

Nilai kecil membuat model lebih fleksibel, sedangkan nilai besar membuat model lebih stabil dan mengurangi overfitting.

### `model__subsample`

Parameter ini menentukan proporsi data yang digunakan pada setiap tree. Rentang 0.7 sampai 1.0 digunakan agar model tidak selalu belajar dari seluruh data yang sama.

Hal ini membantu meningkatkan generalisasi model.

### `model__colsample_bytree`

Parameter ini menentukan proporsi fitur yang digunakan pada setiap tree. Rentang 0.7 sampai 1.0 digunakan agar model tidak terlalu bergantung pada semua fitur sekaligus.

Parameter ini penting karena dataset memiliki banyak fitur hasil encoding dan feature engineering.

### `model__reg_lambda`

Parameter ini adalah regularisasi L2. Rentang 0.01 sampai 10 digunakan untuk mengontrol kompleksitas LightGBM agar tidak terlalu overfit terhadap outlier atau pola training yang terlalu spesifik.

## Hubungan Hyperparameter dengan Hasil EDA

Pemilihan hyperparameter disesuaikan dengan hasil EDA. Karena target tidak seimbang, tuning menggunakan `f1_macro` dan beberapa model menggunakan `class_weight="balanced"`. Karena fitur numerik memiliki skala berbeda, Logistic Regression membutuhkan scaling dan regularisasi. Karena terdapat outlier valid, model tree-based perlu parameter pembatas seperti `max_depth`, `min_samples_leaf`, `min_child_samples`, dan regularisasi.

Karena dataset memiliki banyak fitur setelah encoding dan feature engineering, parameter seperti `max_features`, `subsample`, dan `colsample_bytree` digunakan untuk mengurangi risiko overfitting. Karena hubungan antarfitur kemungkinan non-linear, model seperti Random Forest, XGBoost, dan LightGBM digunakan agar dapat menangkap pola yang lebih kompleks dibanding Logistic Regression.

## Kesimpulan

Ruang pencarian hyperparameter pada `get_param_distributions` sudah sesuai dengan karakteristik dataset credit scoring.

Logistic Regression digunakan sebagai baseline yang stabil dan mudah diinterpretasikan. Random Forest digunakan untuk menangkap hubungan non-linear dengan pendekatan ensemble. XGBoost dan LightGBM digunakan karena keduanya kuat untuk data tabular dan mampu mempelajari pola kompleks.

Dengan menggunakan `RandomizedSearchCV`, `StratifiedGroupKFold`, dan `f1_macro`, proses tuning menjadi lebih efisien, lebih aman dari data leakage, dan lebih adil terhadap seluruh kelas `Credit_Score`.


### 11. Train dan Hyperparameter Tuning Semua Model

In [ ]:
# TRAIN AND TUNE ALL MODELS

trainer = CreditModelTrainer(
    linear_preprocessor=linear_preprocessor,
    tree_preprocessor=tree_preprocessor,
    random_state=RANDOM_STATE
)

tuned_models, tuning_results_df = trainer.tune_models(
    X_train=X_train,
    y_train=y_train_encoded,
    groups_train=groups_train,
    n_iter=20,
    cv_splits=5,
    scoring="f1_macro"
)

display(tuning_results_df.sort_values("best_cv_score", ascending=False))

### 12. Class Model Evaluation

In [ ]:
class CreditModelEvaluator:
    def __init__(self, class_names):
        self.class_names = class_names

    def evaluate_single_model(self, model_name, model, X_test, y_test):
        y_pred = model.predict(X_test)

        result = {
            "model_name": model_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0)
        }

        if hasattr(model, "predict_proba"):
            try:
                y_proba = model.predict_proba(X_test)

                result["roc_auc_ovr_macro"] = roc_auc_score(
                    y_test,
                    y_proba,
                    multi_class="ovr",
                    average="macro"
                )

                result["roc_auc_ovr_weighted"] = roc_auc_score(
                    y_test,
                    y_proba,
                    multi_class="ovr",
                    average="weighted"
                )

            except Exception:
                result["roc_auc_ovr_macro"] = np.nan
                result["roc_auc_ovr_weighted"] = np.nan

        return result

    def evaluate_all_models(self, models, X_test, y_test):
        evaluation_results = []

        for model_name, model in models.items():
            result = self.evaluate_single_model(
                model_name=model_name,
                model=model,
                X_test=X_test,
                y_test=y_test
            )

            evaluation_results.append(result)

        evaluation_df = pd.DataFrame(evaluation_results)

        return evaluation_df

    def show_classification_report(self, model_name, model, X_test, y_test):
        y_pred = model.predict(X_test)

        print("=" * 80)
        print("Classification Report:", model_name)
        print("=" * 80)

        print(
            classification_report(
                y_test,
                y_pred,
                target_names=self.class_names,
                zero_division=0
            )
        )

    def show_confusion_matrix(self, model_name, model, X_test, y_test):
        y_pred = model.predict(X_test)

        cm = confusion_matrix(y_test, y_pred)

        cm_df = pd.DataFrame(
            cm,
            index=[f"Actual {label}" for label in self.class_names],
            columns=[f"Predicted {label}" for label in self.class_names]
        )

        print("=" * 80)
        print("Confusion Matrix:", model_name)
        print("=" * 80)

        display(cm_df)

### 13. Evaluasi Semua Model

In [ ]:
# EVALUATE ALL MODELS

evaluator = CreditModelEvaluator(
    class_names=class_names
)

evaluation_df = evaluator.evaluate_all_models(
    models=tuned_models,
    X_test=X_test,
    y_test=y_test_encoded
)

evaluation_df = evaluation_df.sort_values(
    by="f1_macro",
    ascending=False
).reset_index(drop=True)

display(evaluation_df)

### 14. Classification Report Semua Model

In [ ]:
# SHOW CLASSIFICATION REPORT FOR EACH MODEL

for model_name, model in tuned_models.items():
    evaluator.show_classification_report(
        model_name=model_name,
        model=model,
        X_test=X_test,
        y_test=y_test_encoded
    )

### 15. Confusion Matrix Semua Mode

In [ ]:
# SHOW CONFUSION MATRIX FOR EACH MODEL

for model_name, model in tuned_models.items():
    evaluator.show_confusion_matrix(
        model_name=model_name,
        model=model,
        X_test=X_test,
        y_test=y_test_encoded
    )

### 16. Pilih Best Model

In [ ]:
# SELECT BEST MODEL BASED ON F1 MACRO

best_model_name = evaluation_df.iloc[0]["model_name"]
best_model = tuned_models[best_model_name]

print("Best model:", best_model_name)
print("Best model F1 Macro:", evaluation_df.iloc[0]["f1_macro"])
print("Best model Balanced Accuracy:", evaluation_df.iloc[0]["balanced_accuracy"])

display(evaluation_df.head())

# J. Proses Modelling 2(Menggunakan Kolom Original Aja) 

### 1. Define Original Features Only(Tapi Type Of Loan Nya mengikuti Feature Engineering)

In [ ]:
# ORIGINAL FEATURES + LOAN TYPE FLAGS CONFIGURATION

original_loan_numeric_features = [
    "Age",
    "Annual_Income",
    "Monthly_Inhand_Salary",
    "Num_Bank_Accounts",
    "Num_Credit_Card",
    "Interest_Rate",
    "Num_of_Loan",
    "Delay_from_due_date",
    "Num_of_Delayed_Payment",
    "Changed_Credit_Limit",
    "Num_Credit_Inquiries",
    "Outstanding_Debt",
    "Credit_Utilization_Ratio",
    "Credit_History_Age",
    "Total_EMI_per_month",
    "Amount_invested_monthly",
    "Monthly_Balance"
]

original_loan_binary_features = [
    "Has_Payday_Loan",
    "Has_Mortgage_Loan",
    "Has_Student_Loan",
    "Has_Credit_Builder_Loan",
    "Has_Debt_Consolidation_Loan",
    "Has_Personal_Loan",
    "Has_Auto_Loan",
    "Has_Home_Equity_Loan",
    "Has_Not_Specified_Loan"
]

original_loan_categorical_features = [
    "Month",
    "Occupation",
    "Credit_Mix",
    "Payment_of_Min_Amount",
    "Payment_Behaviour"
]

original_loan_model_features = (
    original_loan_numeric_features
    + original_loan_binary_features
    + original_loan_categorical_features
)

target_column = "Credit_Score"
group_column = "Customer_ID"

print("Total features:", len(original_loan_model_features))
print("Numeric features:", len(original_loan_numeric_features))
print("Binary loan flag features:", len(original_loan_binary_features))
print("Categorical features:", len(original_loan_categorical_features))

### 2. Create Loan Type Flags

In [ ]:
# CREATE LOAN TYPE FLAGS FOR ORIGINAL FEATURES EXPERIMENT

df_original_loan = df_fe.copy()

df_original_loan["Type_of_Loan"] = (
    df_original_loan["Type_of_Loan"]
    .fillna("No Loan")
    .astype(str)
)

loan_type_mapping = {
    "Has_Payday_Loan": "Payday Loan",
    "Has_Mortgage_Loan": "Mortgage Loan",
    "Has_Student_Loan": "Student Loan",
    "Has_Credit_Builder_Loan": "Credit-Builder Loan",
    "Has_Debt_Consolidation_Loan": "Debt Consolidation Loan",
    "Has_Personal_Loan": "Personal Loan",
    "Has_Auto_Loan": "Auto Loan",
    "Has_Home_Equity_Loan": "Home Equity Loan",
    "Has_Not_Specified_Loan": "Not Specified"
}

for new_column, loan_name in loan_type_mapping.items():
    df_original_loan[new_column] = (
        df_original_loan["Type_of_Loan"]
        .str.contains(loan_name, case=False, regex=False)
        .astype(int)
    )

print("Loan type flags created:")
display(df_original_loan[original_loan_binary_features].head())

### 2. Prepare Original-Only Dataset

In [ ]:
# PREPARE ORIGINAL + LOAN FLAGS DATASET

df_original_loan = df_original_loan.copy()

# Replace invalid Payment_Behaviour placeholder if still exists
df_original_loan["Payment_Behaviour"] = df_original_loan["Payment_Behaviour"].replace(
    "!@9#%8",
    pd.NA
)

# Convert pd.NA to np.nan for sklearn compatibility
df_original_loan[original_loan_model_features] = df_original_loan[
    original_loan_model_features
].replace({pd.NA: np.nan})

# Ensure numeric columns are numeric
for column in original_loan_numeric_features:
    df_original_loan[column] = pd.to_numeric(
        df_original_loan[column],
        errors="coerce"
    ).astype(float)

# Ensure binary columns are numeric
for column in original_loan_binary_features:
    df_original_loan[column] = pd.to_numeric(
        df_original_loan[column],
        errors="coerce"
    ).fillna(0).astype(int)

# Ensure categorical columns are object type
for column in original_loan_categorical_features:
    df_original_loan[column] = df_original_loan[column].astype("object")

missing_columns = [
    column for column in original_loan_model_features
    if column not in df_original_loan.columns
]

print("Missing columns:", missing_columns)

print("\nMissing value in selected features:")
display(
    df_original_loan[original_loan_model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nTarget distribution:")
display(df_original_loan[target_column].value_counts())

### 3. Split Original-Only Data 80:20

In [ ]:
# SPLIT ORIGINAL + LOAN FLAGS DATA

X_original_loan = df_original_loan[original_loan_model_features].copy()
y_original_loan = df_original_loan[target_column].copy()
groups_original_loan = df_original_loan[group_column].copy()

original_loan_data_splitter = CreditDataSplitter(
    n_splits=5,
    random_state=RANDOM_STATE
)

(
    X_train_original_loan,
    X_test_original_loan,
    y_train_original_loan,
    y_test_original_loan,
    groups_train_original_loan,
    groups_test_original_loan
) = original_loan_data_splitter.split(
    X=X_original_loan,
    y=y_original_loan,
    groups=groups_original_loan
)

print("X_train_original_loan:", X_train_original_loan.shape)
print("X_test_original_loan :", X_test_original_loan.shape)
print("y_train_original_loan:", y_train_original_loan.shape)
print("y_test_original_loan :", y_test_original_loan.shape)

original_loan_data_splitter.check_split(
    y_train=y_train_original_loan,
    y_test=y_test_original_loan,
    groups_train=groups_train_original_loan,
    groups_test=groups_test_original_loan
)

### 4. Encode Target Original-Only

In [ ]:
# ENCODE TARGET FOR ORIGINAL + LOAN FLAGS MODELING

label_encoder_original_loan = LabelEncoder()

y_train_original_loan_encoded = label_encoder_original_loan.fit_transform(
    y_train_original_loan
)

y_test_original_loan_encoded = label_encoder_original_loan.transform(
    y_test_original_loan
)

class_names_original_loan = label_encoder_original_loan.classes_

print("Class mapping:")
for index, class_name in enumerate(class_names_original_loan):
    print(index, "=", class_name)

### 5.Build Original-Only Preprocessor

In [ ]:
# BUILD PREPROCESSOR FOR ORIGINAL + LOAN FLAGS MODELING

original_loan_preprocessor_builder = CreditPreprocessor(
    numeric_features=original_loan_numeric_features,
    binary_features=original_loan_binary_features,
    categorical_features=original_loan_categorical_features
)

original_loan_linear_preprocessor = (
    original_loan_preprocessor_builder
    .build_linear_preprocessor()
)

original_loan_tree_preprocessor = (
    original_loan_preprocessor_builder
    .build_tree_preprocessor()
)

print("Preprocessors for original + loan flags are ready.")

### 6. Train and Tune Original-Only Models

In [ ]:
# TRAIN AND TUNE ORIGINAL + LOAN FLAGS MODELS

original_loan_trainer = CreditModelTrainer(
    linear_preprocessor=original_loan_linear_preprocessor,
    tree_preprocessor=original_loan_tree_preprocessor,
    random_state=RANDOM_STATE
)

original_loan_tuned_models, original_loan_tuning_results_df = (
    original_loan_trainer.tune_models(
        X_train=X_train_original_loan,
        y_train=y_train_original_loan_encoded,
        groups_train=groups_train_original_loan,
        n_iter=10,
        cv_splits=3,
        scoring="f1_macro"
    )
)

display(
    original_loan_tuning_results_df
    .sort_values("best_cv_score", ascending=False)
    .reset_index(drop=True)
)

### 7. Evaluate Original-Only Models

In [ ]:
# EVALUATE ORIGINAL + LOAN FLAGS MODELS

original_loan_evaluator = CreditModelEvaluator(
    class_names=class_names_original_loan
)

original_loan_evaluation_df = original_loan_evaluator.evaluate_all_models(
    models=original_loan_tuned_models,
    X_test=X_test_original_loan,
    y_test=y_test_original_loan_encoded
)

original_loan_evaluation_df = (
    original_loan_evaluation_df
    .sort_values(by="f1_macro", ascending=False)
    .reset_index(drop=True)
)

display(original_loan_evaluation_df)

### 8. Classification Report Original-Only Models

In [ ]:
# CLASSIFICATION REPORT FOR ORIGINAL + LOAN FLAGS MODELS

for model_name, model in original_loan_tuned_models.items():
    original_loan_evaluator.show_classification_report(
        model_name=model_name,
        model=model,
        X_test=X_test_original_loan,
        y_test=y_test_original_loan_encoded
    )

### 9. Confusion Matrix Original-Only Models

In [ ]:
# CONFUSION MATRIX FOR ORIGINAL + LOAN FLAGS MODELS

for model_name, model in original_loan_tuned_models.items():
    original_loan_evaluator.show_confusion_matrix(
        model_name=model_name,
        model=model,
        X_test=X_test_original_loan,
        y_test=y_test_original_loan_encoded
    )

### 10. Select Best Original-Only Model

In [ ]:
# SELECT BEST ORIGINAL + LOAN FLAGS MODEL

best_original_loan_model_name = original_loan_evaluation_df.iloc[0]["model_name"]
best_original_loan_model = original_loan_tuned_models[best_original_loan_model_name]

print("Best original + loan flags model:", best_original_loan_model_name)
print("Best F1 Macro:", original_loan_evaluation_df.iloc[0]["f1_macro"])
print("Best Balanced Accuracy:", original_loan_evaluation_df.iloc[0]["balanced_accuracy"])

display(original_loan_evaluation_df.head())

# K. Perbandingan Antara Model1(Pakai Kolom Feature Engineering) dengan Model2(Pakai Kolom Original + Kolom Type Of Loan Feature Engineering)

In [ ]:
# COMPARE ORIGINAL + LOAN FLAGS WITH ORIGINAL + FEATURE ENGINEERING

original_loan_compare_df = original_loan_evaluation_df.copy()
original_loan_compare_df["feature_set"] = "Original + Loan Flags"

feature_engineering_compare_df = evaluation_df.copy()
feature_engineering_compare_df["feature_set"] = "Original + Feature Engineering"

comparison_df = pd.concat(
    [
        feature_engineering_compare_df,
        original_loan_compare_df
    ],
    ignore_index=True
)

comparison_df = comparison_df[
    [
        "feature_set",
        "model_name",
        "accuracy",
        "balanced_accuracy",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "f1_weighted",
        "roc_auc_ovr_macro",
        "roc_auc_ovr_weighted"
    ]
]

comparison_df = comparison_df.sort_values(
    by=["f1_macro", "accuracy"],
    ascending=False
).reset_index(drop=True)

display(comparison_df)

In [ ]:
# FINAL SUMMARY ORIGINAL + LOAN FLAGS MODELLING

print("Original + Loan Flags modelling summary")
print("-" * 80)
print("Total features:", len(original_loan_model_features))
print("Total train rows:", X_train_original_loan.shape[0])
print("Total test rows:", X_test_original_loan.shape[0])
print("Best model:", best_original_loan_model_name)

print("\nModels compared:")
for model_name in original_loan_tuned_models.keys():
    print("-", model_name)

print("\nOriginal + Loan Flags evaluation result:")
display(original_loan_evaluation_df)

print("\nComparison with original + feature engineering:")
display(comparison_df)

### Melihat HyperParemeter Terbaik Model Terbaik

In [ ]:
# VIEW BEST MODEL AND BEST HYPERPARAMETERS

import pandas as pd
import ast

# Show all tuning results first
display(tuning_results_df)

# Automatically detect important column names
model_col_candidates = ["model_name", "Model", "model", "Model Name"]
score_col_candidates = ["best_cv_score", "Best CV Score", "best_score", "Best F1 Macro", "cv_score"]
params_col_candidates = ["best_params", "Best Parameters", "best_parameters", "params"]

model_col = next(col for col in model_col_candidates if col in tuning_results_df.columns)
score_col = next(col for col in score_col_candidates if col in tuning_results_df.columns)
params_col = next(col for col in params_col_candidates if col in tuning_results_df.columns)

# Find best model based on highest CV score
best_index = tuning_results_df[score_col].idxmax()
best_row = tuning_results_df.loc[best_index]

best_model_name = best_row[model_col]
best_cv_score = best_row[score_col]
best_params = best_row[params_col]

# Convert best_params if it is stored as string
if isinstance(best_params, str):
    best_params = ast.literal_eval(best_params)

# Remove "model__" prefix for cleaner display
clean_best_params = {
    key.replace("model__", ""): value
    for key, value in best_params.items()
}

print("BEST MODEL")
print("=" * 50)
print(f"Model Name      : {best_model_name}")
print(f"Best CV Score   : {best_cv_score:.4f}")
print("=" * 50)

print("\nBEST HYPERPARAMETERS")
print("=" * 50)

best_params_df = pd.DataFrame(
    clean_best_params.items(),
    columns=["Hyperparameter", "Best Value"]
)

display(best_params_df)

## Interpretasi Hyperparameter Terbaik Model Terbaik

Berdasarkan hasil hyperparameter tuning, model terbaik yang diperoleh adalah **XGBoost**. Hal ini dapat dilihat dari hyperparameter terbaik yang muncul, yaitu `colsample_bytree`, `learning_rate`, `max_depth`, `n_estimators`, `reg_alpha`, `reg_lambda`, dan `subsample`. Parameter-parameter tersebut merupakan parameter utama pada model XGBoost.

Hasil hyperparameter terbaik adalah sebagai berikut:

* `colsample_bytree`: 0.8173
* `learning_rate`: 0.0173
* `max_depth`: 6
* `n_estimators`: 201
* `reg_alpha`: 0.0099
* `reg_lambda`: 0.0421
* `subsample`: 0.8703

Secara umum, kombinasi hyperparameter ini menunjukkan bahwa model terbaik cenderung menggunakan strategi belajar yang cukup hati-hati, tidak terlalu kompleks, dan tetap memiliki mekanisme regularisasi untuk mengurangi risiko overfitting.

## Hubungan dengan Hasil EDA dan Karakteristik Data

Berdasarkan hasil EDA, dataset credit scoring memiliki karakteristik yang cukup kompleks. Target `Credit_Score` terdiri dari beberapa kelas, yaitu `Good`, `Standard`, dan `Poor`, sehingga masalah ini termasuk multi-class classification. Selain itu, distribusi kelas tidak sepenuhnya seimbang, karena kelas `Standard` menjadi kelas mayoritas, sedangkan kelas `Good` dan `Poor` memiliki proporsi yang lebih kecil.

Dataset juga memiliki kombinasi berbagai jenis fitur, seperti fitur numerik, fitur kategorikal, binary loan flags, dan fitur hasil feature engineering. Beberapa fitur numerik memiliki skala yang berbeda jauh, seperti `Monthly_Inhand_Salary`, `Outstanding_Debt`, `Credit_History_Age`, dan ratio features. Selain itu, terdapat outlier valid yang tetap dipertahankan karena masih masuk akal secara bisnis, misalnya customer dengan debt tinggi, salary tinggi, atau jumlah keterlambatan pembayaran yang besar.

Karakteristik tersebut membuat XGBoost cocok digunakan karena model ini mampu menangkap hubungan non-linear dan interaksi antarfitur. Dalam credit scoring, risiko customer tidak hanya ditentukan oleh satu fitur, tetapi oleh kombinasi beberapa fitur, seperti keterlambatan pembayaran, jumlah delayed payment, outstanding debt, credit utilization ratio, credit mix, income, loan type, dan payment behaviour.

## Interpretasi `learning_rate = 0.0173`

Nilai `learning_rate` terbaik adalah 0.0173. Nilai ini termasuk kecil, sehingga model belajar secara perlahan dari setiap tree yang dibangun.

Learning rate yang kecil menunjukkan bahwa model tidak langsung melakukan update besar pada setiap iterasi. Hal ini baik untuk dataset ini karena data memiliki outlier valid dan pola antarfitur yang cukup kompleks. Jika learning rate terlalu besar, model bisa terlalu cepat mengikuti pola training data, termasuk pola ekstrem dari outlier, sehingga risiko overfitting menjadi lebih tinggi.

Dengan learning rate kecil, XGBoost membangun model secara lebih hati-hati. Setiap tree hanya memberikan kontribusi kecil terhadap prediksi akhir. Strategi ini membantu model menghasilkan generalisasi yang lebih baik pada data baru.

## Interpretasi `n_estimators = 201`

Nilai `n_estimators` terbaik adalah 201, yang berarti model menggunakan sekitar 201 tree dalam proses boosting.

Jumlah ini menunjukkan bahwa model tidak membutuhkan tree yang terlalu banyak untuk mencapai performa terbaik. Karena learning rate yang digunakan cukup kecil, jumlah tree 201 masih masuk akal untuk membantu model belajar secara bertahap. Namun, jumlah ini juga tidak terlalu besar, sehingga model tetap efisien dan tidak terlalu berat untuk kebutuhan deployment.

Hasil ini menunjukkan bahwa setelah sekitar 201 tree, model sudah mampu menangkap pola penting dari data. Menambah terlalu banyak tree kemungkinan tidak memberikan peningkatan signifikan dan justru dapat meningkatkan risiko overfitting atau memperlama waktu training.

## Interpretasi `max_depth = 6`

Nilai `max_depth` terbaik adalah 6. Parameter ini menunjukkan kedalaman maksimum setiap tree dalam XGBoost.

Kedalaman 6 menunjukkan bahwa model membutuhkan tree dengan kompleksitas sedang. Tree tidak terlalu dangkal, sehingga masih mampu menangkap interaksi antarfitur. Namun, tree juga tidak terlalu dalam, sehingga risiko model menghafal data training dapat dikurangi.

Hal ini sesuai dengan hasil EDA karena hubungan antara fitur dan `Credit_Score` kemungkinan tidak sepenuhnya linear. Misalnya, customer dengan outstanding debt tinggi belum tentu selalu memiliki credit score buruk jika pendapatannya juga tinggi. Begitu juga customer dengan jumlah loan tertentu perlu dilihat bersama dengan jenis loan, credit utilization, dan riwayat keterlambatan pembayaran.

Dengan `max_depth = 6`, model dapat mempelajari kombinasi fitur seperti debt, salary, loan flags, credit mix, dan payment delay tanpa membuat tree terlalu kompleks.

## Interpretasi `subsample = 0.8703`

Nilai `subsample` terbaik adalah 0.8703. Artinya, setiap tree hanya menggunakan sekitar 87,03% data training.

Nilai ini menunjukkan bahwa model tidak selalu menggunakan seluruh data untuk membangun setiap tree. Strategi ini membantu mengurangi risiko overfitting karena model tidak terus-menerus belajar dari kombinasi data yang sama.

Hal ini penting karena dataset memiliki customer yang dapat muncul dalam beberapa bulan berbeda. Walaupun proses cross-validation sudah menggunakan group-based split, penggunaan `subsample` tetap membantu model agar tidak terlalu bergantung pada pola sample tertentu.

Dengan menggunakan sebagian besar data, yaitu sekitar 87%, model masih mendapatkan informasi yang cukup kuat dari dataset, tetapi tetap memiliki variasi antar tree yang membantu generalisasi.

## Interpretasi `colsample_bytree = 0.8173`

Nilai `colsample_bytree` terbaik adalah 0.8173. Artinya, setiap tree hanya menggunakan sekitar 81,73% fitur yang tersedia.

Parameter ini penting karena setelah preprocessing dan feature engineering, jumlah fitur menjadi lebih banyak. Fitur kategorikal diubah menggunakan encoding, fitur loan type diubah menjadi binary flags, dan beberapa fitur baru seperti ratio features juga ditambahkan.

Jika setiap tree selalu menggunakan semua fitur, model dapat terlalu bergantung pada fitur tertentu yang sangat dominan, misalnya `Credit_Mix`, `Outstanding_Debt`, atau fitur keterlambatan pembayaran. Dengan hanya menggunakan sekitar 81,73% fitur pada setiap tree, model dipaksa untuk belajar dari kombinasi fitur yang lebih beragam.

Hal ini membantu mengurangi overfitting dan membuat model lebih stabil saat digunakan pada data baru.

## Interpretasi `reg_alpha = 0.0099`

Nilai `reg_alpha` terbaik adalah 0.0099. Parameter ini merupakan regularisasi L1 pada XGBoost.

Nilai ini cukup kecil, sehingga regularisasi L1 yang digunakan tidak terlalu kuat. Artinya, model tidak terlalu agresif dalam menekan kontribusi fitur menjadi nol. Hal ini masuk akal karena banyak fitur dalam dataset masih memiliki makna bisnis yang relevan, seperti payment delay, debt, salary, credit mix, dan loan flags.

Meskipun kecil, nilai `reg_alpha` tetap membantu model menjadi sedikit lebih selektif terhadap fitur. Ini penting karena setelah preprocessing, jumlah fitur bertambah akibat encoding dan feature engineering.

## Interpretasi `reg_lambda = 0.0421`

Nilai `reg_lambda` terbaik adalah 0.0421. Parameter ini merupakan regularisasi L2 pada XGBoost.

Nilai ini juga termasuk kecil, sehingga model tetap diberi fleksibilitas untuk mempelajari pola data. Regularisasi L2 membantu menjaga agar bobot model tidak terlalu ekstrem, terutama karena dataset memiliki outlier valid dan fitur dengan variasi nilai yang besar.

Nilai regularisasi yang kecil menunjukkan bahwa berdasarkan hasil tuning, model tidak membutuhkan regularisasi yang terlalu kuat. Kemungkinan besar fitur-fitur yang digunakan sudah cukup informatif setelah proses cleaning, preprocessing, dan feature engineering.

## Hubungan dengan Preprocessing

Hasil hyperparameter ini juga sesuai dengan preprocessing yang digunakan. Karena XGBoost adalah model tree-based, model ini tidak membutuhkan scaling seperti Logistic Regression. Namun, preprocessing tetap penting untuk menangani missing value, encoding fitur kategorikal, dan memastikan format input konsisten antara training, validation, testing, dan inference.

Fitur kategorikal seperti `Occupation`, `Credit_Mix`, dan `Payment_of_Min_Amount` perlu diubah menjadi bentuk numerik melalui encoding. Fitur seperti `Type_of_Loan` juga tidak digunakan langsung sebagai teks panjang, tetapi diubah menjadi loan flags agar informasi jenis pinjaman lebih terstruktur.

Dengan preprocessing yang konsisten dalam pipeline, hyperparameter tuning dapat dilakukan dengan lebih aman karena semua transformasi hanya dipelajari dari data training pada setiap fold. Hal ini membantu mencegah data leakage.

## Kesimpulan

Kombinasi hyperparameter terbaik menunjukkan bahwa model XGBoost yang terpilih memiliki karakter sebagai model yang cukup hati-hati dan terkontrol.

Nilai `learning_rate` yang kecil menunjukkan bahwa model belajar secara perlahan agar tidak terlalu cepat mengikuti noise atau outlier. Nilai `n_estimators` sebesar 201 menunjukkan bahwa jumlah tree yang dibutuhkan tidak terlalu besar, sehingga model tetap efisien. Nilai `max_depth` sebesar 6 menunjukkan bahwa model memiliki kompleksitas sedang dan mampu menangkap hubungan non-linear antarfitur tanpa terlalu mudah overfit.

Selain itu, nilai `subsample` dan `colsample_bytree` yang berada di bawah 1 menunjukkan bahwa model menggunakan sebagian data dan sebagian fitur pada setiap tree. Strategi ini membantu meningkatkan generalisasi. Regularisasi `reg_alpha` dan `reg_lambda` yang kecil menunjukkan bahwa model tetap fleksibel, tetapi masih memiliki kontrol untuk mengurangi risiko overfitting.

Dengan hasil ini, XGBoost menjadi model terbaik karena mampu mempelajari pola kompleks pada data credit scoring, terutama pola yang berasal dari kombinasi fitur numerik, kategorikal, loan flags, dan hasil feature engineering. Model ini sesuai dengan karakteristik dataset yang memiliki class imbalance, outlier valid, fitur campuran, serta hubungan antarfitur yang tidak sepenuhnya linear.


### Melihat Confusion Matrix Kedua Model Xgboost

Saya membandingkan kedua model terbaik ini menggunakan confusion matrix karena confusion matrix dapat menunjukkan performa model secara lebih detail pada setiap kelas, bukan hanya dari nilai accuracy secara umum. Melalui confusion matrix, saya dapat melihat jumlah prediksi yang benar pada kelas Good, Poor, dan Standard, serta melihat pola kesalahan prediksi antar kelas. Perbandingan ini penting karena dalam kasus credit scoring, model tidak hanya harus memiliki jumlah prediksi benar yang tinggi, tetapi juga harus mampu membedakan kelas risiko dengan baik, terutama agar customer Poor tidak terlalu banyak salah diprediksi sebagai Good. Dari hasil confusion matrix, kedua model memiliki performa yang cukup mirip, tetapi model XGBoost - Original + Loan Flags menghasilkan jumlah prediksi benar yang sedikit lebih tinggi pada seluruh kelas dibanding XGBoost - Original + Feature Engineering, sehingga model tersebut dapat dianggap sedikit lebih baik berdasarkan hasil evaluasi confusion matrix.

In [ ]:
# GET BOTH XGBOOST MODELS

xgb_feature_engineering_model = tuned_models["XGBoost"]
xgb_original_loan_model = original_loan_tuned_models["XGBoost"]

print("XGBoost models are ready.")

In [ ]:
# PREDICT USING BOTH XGBOOST MODELS

# XGBoost with original + feature engineering
y_pred_xgb_fe = xgb_feature_engineering_model.predict(X_test)

# XGBoost with original + loan flags
y_pred_xgb_original_loan = xgb_original_loan_model.predict(X_test_original_loan)

print("Prediction completed.")

In [ ]:
# CONFUSION MATRIX FOR BOTH XGBOOST MODELS

from sklearn.metrics import confusion_matrix

cm_xgb_fe = confusion_matrix(
    y_test_encoded,
    y_pred_xgb_fe
)

cm_xgb_original_loan = confusion_matrix(
    y_test_original_loan_encoded,
    y_pred_xgb_original_loan
)

cm_xgb_fe_df = pd.DataFrame(
    cm_xgb_fe,
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
)

cm_xgb_original_loan_df = pd.DataFrame(
    cm_xgb_original_loan,
    index=[f"Actual {label}" for label in class_names_original_loan],
    columns=[f"Predicted {label}" for label in class_names_original_loan]
)

print("Confusion Matrix: XGBoost - Original + Feature Engineering")
display(cm_xgb_fe_df)

print("\nConfusion Matrix: XGBoost - Original + Loan Flags")
display(cm_xgb_original_loan_df)

## Analisis Confusion Matrix XGBoost

Confusion matrix digunakan untuk melihat bagaimana model memprediksi setiap kelas pada data test. Baris pada confusion matrix menunjukkan kelas aktual, sedangkan kolom menunjukkan kelas hasil prediksi model. Nilai diagonal menunjukkan prediksi yang benar, sedangkan nilai di luar diagonal menunjukkan kesalahan prediksi.

Pada hasil ini, terdapat dua model XGBoost yang dibandingkan:

* **XGBoost - Original + Feature Engineering**
* **XGBoost - Original + Loan Flags**

Kedua model dievaluasi pada jumlah data test yang sama, yaitu:

* Actual `Good`: 790 data
* Actual `Poor`: 1.338 data
* Actual `Standard`: 2.449 data
* Total data test: 4.577 data

## XGBoost - Original + Feature Engineering

Pada model **Original + Feature Engineering**, hasil prediksi benar berada pada diagonal matrix:

* Actual `Good` yang diprediksi benar sebagai `Good`: 562 data
* Actual `Poor` yang diprediksi benar sebagai `Poor`: 894 data
* Actual `Standard` yang diprediksi benar sebagai `Standard`: 1.768 data

Total prediksi benar adalah:

`562 + 894 + 1768 = 3224`

Dari total 4.577 data test, model ini menghasilkan accuracy sekitar:

`3224 / 4577 = 0.7044`

Artinya, model berhasil memprediksi sekitar **70,44%** data test dengan benar.

Untuk kelas `Good`, dari 790 data aktual `Good`, model berhasil memprediksi benar 562 data. Namun, masih terdapat 204 data `Good` yang diprediksi sebagai `Standard` dan 24 data yang diprediksi sebagai `Poor`. Kesalahan terbesar pada kelas `Good` adalah model masih cukup sering menganggap customer `Good` sebagai `Standard`.

Untuk kelas `Poor`, dari 1.338 data aktual `Poor`, model berhasil memprediksi benar 894 data. Namun, terdapat 349 data `Poor` yang diprediksi sebagai `Standard` dan 95 data yang diprediksi sebagai `Good`. Kesalahan terbesar pada kelas `Poor` adalah model masih cukup sering menganggap customer `Poor` sebagai `Standard`.

Untuk kelas `Standard`, dari 2.449 data aktual `Standard`, model berhasil memprediksi benar 1.768 data. Namun, terdapat 350 data `Standard` yang diprediksi sebagai `Good` dan 331 data yang diprediksi sebagai `Poor`. Karena kelas `Standard` merupakan kelas mayoritas, jumlah kesalahan pada kelas ini juga cukup besar secara absolut.

## XGBoost - Original + Loan Flags

Pada model **Original + Loan Flags**, hasil prediksi benar adalah:

* Actual `Good` yang diprediksi benar sebagai `Good`: 566 data
* Actual `Poor` yang diprediksi benar sebagai `Poor`: 896 data
* Actual `Standard` yang diprediksi benar sebagai `Standard`: 1.778 data

Total prediksi benar adalah:

`566 + 896 + 1778 = 3240`

Dari total 4.577 data test, model ini menghasilkan accuracy sekitar:

`3240 / 4577 = 0.7079`

Artinya, model berhasil memprediksi sekitar **70,79%** data test dengan benar.

Untuk kelas `Good`, dari 790 data aktual `Good`, model berhasil memprediksi benar 566 data. Kesalahan prediksi pada kelas ini adalah 204 data diprediksi sebagai `Standard` dan 20 data diprediksi sebagai `Poor`.

Untuk kelas `Poor`, dari 1.338 data aktual `Poor`, model berhasil memprediksi benar 896 data. Kesalahan prediksi pada kelas ini adalah 349 data diprediksi sebagai `Standard` dan 93 data diprediksi sebagai `Good`.

Untuk kelas `Standard`, dari 2.449 data aktual `Standard`, model berhasil memprediksi benar 1.778 data. Kesalahan prediksi pada kelas ini adalah 346 data diprediksi sebagai `Good` dan 325 data diprediksi sebagai `Poor`.

## Perbandingan Kedua Model

Jika dibandingkan, model **Original + Loan Flags** sedikit lebih baik dibanding model **Original + Feature Engineering**.

Perbedaannya dapat dilihat dari jumlah prediksi benar:

* Original + Feature Engineering: 3.224 prediksi benar
* Original + Loan Flags: 3.240 prediksi benar

Model Original + Loan Flags menghasilkan **16 prediksi benar lebih banyak** dibanding model Original + Feature Engineering.

Perbaikan ini terlihat pada semua kelas diagonal:

* Kelas `Good`: naik dari 562 menjadi 566
* Kelas `Poor`: naik dari 894 menjadi 896
* Kelas `Standard`: naik dari 1.768 menjadi 1.778

Artinya, penambahan atau penggunaan loan flags membantu model sedikit lebih baik dalam mengenali pola pada seluruh kelas, terutama kelas `Standard`.

## Analisis Kesalahan Prediksi

Kesalahan terbesar pada kedua model terjadi ketika model membedakan kelas `Good` dan `Poor` dari kelas `Standard`.

Pada kedua confusion matrix, banyak data aktual `Good` dan `Poor` yang diprediksi sebagai `Standard`. Hal ini masuk akal karena kelas `Standard` biasanya berada di tengah antara `Good` dan `Poor`. Dalam konteks credit scoring, customer dengan karakteristik mendekati batas antar kelas dapat lebih sulit dibedakan.

Contohnya, pada model Original + Loan Flags:

* 204 data aktual `Good` diprediksi sebagai `Standard`
* 349 data aktual `Poor` diprediksi sebagai `Standard`

Hal ini menunjukkan bahwa model lebih sering salah ke kelas tengah, bukan langsung salah ekstrem. Kesalahan seperti ini masih lebih wajar dibanding jika banyak customer `Good` langsung diprediksi sebagai `Poor` atau sebaliknya.

Namun, masih terdapat juga kesalahan ekstrem:

* 20 data aktual `Good` diprediksi sebagai `Poor`
* 93 data aktual `Poor` diprediksi sebagai `Good`

Kesalahan ini perlu diperhatikan karena dalam credit scoring, salah memprediksi customer `Poor` sebagai `Good` bisa lebih berisiko secara bisnis. Pada model Original + Loan Flags, jumlah actual `Poor` yang salah diprediksi sebagai `Good` adalah 93 data, sedikit lebih rendah dibanding model Original + Feature Engineering yang berjumlah 95 data. Ini menunjukkan sedikit perbaikan dari sisi risiko bisnis.



## Kesimpulan

Berdasarkan confusion matrix, kedua model XGBoost memiliki performa yang cukup mirip, tetapi model **XGBoost - Original + Loan Flags** memberikan hasil yang sedikit lebih baik.

Model Original + Loan Flags menghasilkan:

* prediksi benar lebih banyak,
* accuracy lebih tinggi,
* peningkatan prediksi benar pada kelas `Good`, `Poor`, dan `Standard`,
* serta sedikit penurunan kesalahan actual `Poor` yang diprediksi sebagai `Good`.

Secara keseluruhan, hasil ini menunjukkan bahwa fitur loan flags bermanfaat untuk membantu model memahami profil risiko customer. Namun, masih terdapat tantangan dalam membedakan kelas `Good` dan `Poor` dari kelas `Standard`, karena kelas `Standard` merupakan kelas tengah dan juga kelas mayoritas dalam dataset.

Oleh karena itu, jika hanya berdasarkan confusion matrix ini, model **XGBoost - Original + Loan Flags** dapat dianggap sedikit lebih baik dan lebih layak dipilih dibanding model XGBoost Original + Feature Engineering.


### Pilih dan Simpan Best Model dari Semua Model

In [ ]:
# SELECT AND SAVE BEST MODEL FROM ALL AVAILABLE MODELING EXPERIMENTS

all_model_candidates = []

# 1. Original + Feature Engineering models
if "tuned_models" in globals() and "evaluation_df" in globals():
    for model_name, model in tuned_models.items():
        model_score = evaluation_df.loc[
            evaluation_df["model_name"] == model_name,
            "f1_macro"
        ].values[0]

        all_model_candidates.append({
            "feature_set": "Original + Feature Engineering",
            "model_name": model_name,
            "f1_macro": model_score,
            "model": model,
            "label_encoder": label_encoder,
            "model_features": final_model_features,
            "numeric_features": numeric_features,
            "binary_features": binary_features,
            "categorical_features": categorical_features,
            "X_test": X_test,
            "y_test": y_test_encoded
        })


# 2. Original + Loan Flags models
if "original_loan_tuned_models" in globals() and "original_loan_evaluation_df" in globals():
    for model_name, model in original_loan_tuned_models.items():
        model_score = original_loan_evaluation_df.loc[
            original_loan_evaluation_df["model_name"] == model_name,
            "f1_macro"
        ].values[0]

        all_model_candidates.append({
            "feature_set": "Original + Loan Flags",
            "model_name": model_name,
            "f1_macro": model_score,
            "model": model,
            "label_encoder": label_encoder_original_loan,
            "model_features": original_loan_model_features,
            "numeric_features": original_loan_numeric_features,
            "binary_features": original_loan_binary_features,
            "categorical_features": original_loan_categorical_features,
            "X_test": X_test_original_loan,
            "y_test": y_test_original_loan_encoded
        })


# 3. Original-only models, if still available
if "original_tuned_models" in globals() and "original_evaluation_df" in globals():
    for model_name, model in original_tuned_models.items():
        model_score = original_evaluation_df.loc[
            original_evaluation_df["model_name"] == model_name,
            "f1_macro"
        ].values[0]

        all_model_candidates.append({
            "feature_set": "Original Only",
            "model_name": model_name,
            "f1_macro": model_score,
            "model": model,
            "label_encoder": label_encoder_original,
            "model_features": original_model_features,
            "numeric_features": original_numeric_features,
            "binary_features": original_binary_features,
            "categorical_features": original_categorical_features,
            "X_test": X_test_original,
            "y_test": y_test_original_encoded
        })


# Convert candidates to dataframe
all_model_summary_df = pd.DataFrame([
    {
        "feature_set": candidate["feature_set"],
        "model_name": candidate["model_name"],
        "f1_macro": candidate["f1_macro"]
    }
    for candidate in all_model_candidates
])

all_model_summary_df = all_model_summary_df.sort_values(
    by="f1_macro",
    ascending=False
).reset_index(drop=True)

display(all_model_summary_df)

# Select best model based on F1 Macro
best_candidate = max(
    all_model_candidates,
    key=lambda candidate: candidate["f1_macro"]
)

final_best_model_name = best_candidate["model_name"]
final_best_feature_set = best_candidate["feature_set"]
final_best_model = best_candidate["model"]
final_best_label_encoder = best_candidate["label_encoder"]
final_best_features = best_candidate["model_features"]

print("Final best feature set:", final_best_feature_set)
print("Final best model:", final_best_model_name)
print("Final best F1 Macro:", best_candidate["f1_macro"])


# Save final best model artifact
final_model_artifact = {
    "feature_set": final_best_feature_set,
    "best_model_name": final_best_model_name,
    "model": final_best_model,
    "label_encoder": final_best_label_encoder,
    "model_features": final_best_features,
    "numeric_features": best_candidate["numeric_features"],
    "binary_features": best_candidate["binary_features"],
    "categorical_features": best_candidate["categorical_features"]
}

joblib.dump(
    final_model_artifact,
    "final_best_credit_score_model.pkl"
)

print("Final best model saved as final_best_credit_score_model.pkl")

## Summary Hasil Modeling

Pada tahap modeling, tujuan utama project ini adalah membangun model klasifikasi untuk memprediksi Credit_Score customer ke dalam tiga kelas, yaitu Good, Poor, dan Standard. Berdasarkan hasil EDA, dataset memiliki karakteristik data tabular dengan fitur numerik, kategorikal, binary loan flags, dan fitur hasil feature engineering. Selain itu, target tidak sepenuhnya seimbang karena kelas Standard memiliki jumlah data paling besar dibanding kelas lainnya. Oleh karena itu, proses modeling tidak hanya berfokus pada accuracy, tetapi juga menggunakan pendekatan evaluasi yang lebih adil terhadap semua kelas, yaitu f1_macro.

Dalam proses modeling, beberapa algoritma dibandingkan, yaitu Logistic Regression, Random Forest, XGBoost, dan LightGBM. Logistic Regression digunakan sebagai baseline karena model ini sederhana, cepat, dan mudah diinterpretasikan. Random Forest digunakan karena mampu menangkap hubungan non-linear dan lebih tahan terhadap outlier. XGBoost dan LightGBM digunakan karena keduanya merupakan boosting model yang kuat untuk data tabular dan mampu mempelajari interaksi antar fitur dengan lebih baik.

Seluruh model dilatih menggunakan pipeline agar proses preprocessing dan modeling berjalan konsisten. Pipeline ini penting karena dataset memiliki missing value, fitur numerik dengan skala berbeda, serta fitur kategorikal yang perlu di-encoding. Untuk model linear seperti Logistic Regression, preprocessing menggunakan scaling karena model tersebut sensitif terhadap skala fitur. Sementara itu, untuk model tree-based seperti Random Forest, XGBoost, dan LightGBM, scaling tidak menjadi fokus utama karena model berbasis tree tidak bergantung pada jarak antar nilai fitur.

Eksperimen dilakukan dengan membandingkan dua pendekatan feature set utama. Pendekatan pertama adalah Original + Feature Engineering, yaitu feature set yang menggunakan fitur original terpilih, missing indicator, ratio features, loan flags, dan split Payment_Behaviour menjadi Spending_Level dan Payment_Value_Level. Pendekatan kedua adalah Original + Loan Flags, yaitu feature set yang lebih menekankan penggunaan fitur original yang lebih lengkap, seperti Annual_Income, Total_EMI_per_month, Amount_invested_monthly, Monthly_Balance, dan Payment_Behaviour, kemudian ditambah dengan loan flags dari Type_of_Loan.

Perlu ditekankan bahwa perbedaan kedua pendekatan ini bukan pada ada atau tidaknya loan flags, karena kedua pendekatan sama-sama menggunakan loan flags. Perbedaannya terletak pada strategi penggunaan fitur. Model Original + Feature Engineering lebih banyak menggunakan fitur hasil transformasi seperti ratio features dan split payment behaviour, sedangkan model Original + Loan Flags lebih mempertahankan fitur original yang lebih lengkap dan menambahkan loan flags sebagai informasi jenis pinjaman.

Proses hyperparameter tuning menunjukkan bahwa XGBoost menjadi model terbaik. Hal ini sesuai dengan karakteristik dataset karena XGBoost mampu menangkap hubungan non-linear antara fitur dan target. Dalam credit scoring, prediksi Credit_Score tidak hanya dipengaruhi oleh satu fitur, tetapi oleh kombinasi banyak faktor, seperti keterlambatan pembayaran, jumlah delayed payment, outstanding debt, credit utilization ratio, salary, credit mix, payment behaviour, dan jenis pinjaman customer.

Hyperparameter terbaik pada model XGBoost adalah learning_rate = 0.0173, n_estimators = 201, max_depth = 6, subsample = 0.8703, colsample_bytree = 0.8173, reg_alpha = 0.0099, dan reg_lambda = 0.0421. Kombinasi ini menunjukkan bahwa model belajar secara cukup hati-hati karena learning rate yang digunakan kecil. Jumlah tree sebesar 201 membuat model tetap cukup kuat tetapi tidak terlalu berat. Kedalaman tree sebesar 6 menunjukkan bahwa model memiliki kompleksitas sedang, sehingga mampu menangkap interaksi antar fitur tanpa terlalu mudah overfit. Nilai subsample dan colsample_bytree yang berada di bawah 1 juga membantu model melakukan generalisasi lebih baik karena setiap tree tidak selalu menggunakan seluruh data dan seluruh fitur.

Berdasarkan confusion matrix, dua model XGBoost terbaik yang dibandingkan adalah XGBoost - Original + Feature Engineering dan XGBoost - Original + Loan Flags. Pada model Original + Feature Engineering, jumlah prediksi benar adalah 3.224 dari 4.577 data test. Model ini berhasil memprediksi benar 562 data Good, 894 data Poor, dan 1.768 data Standard. Sementara itu, pada model Original + Loan Flags, jumlah prediksi benar adalah 3.240 dari 4.577 data test. Model ini berhasil memprediksi benar 566 data Good, 896 data Poor, dan 1.778 data Standard.

Jika dibandingkan, model XGBoost - Original + Loan Flags memberikan hasil sedikit lebih baik dibanding XGBoost - Original + Feature Engineering. Peningkatan terlihat pada seluruh kelas, yaitu kelas Good naik dari 562 menjadi 566 prediksi benar, kelas Poor naik dari 894 menjadi 896 prediksi benar, dan kelas Standard naik dari 1.768 menjadi 1.778 prediksi benar. Secara total, model Original + Loan Flags menghasilkan 16 prediksi benar lebih banyak dibanding model Original + Feature Engineering.

Dari sisi pola kesalahan, kedua model masih memiliki tantangan dalam membedakan kelas Good dan Poor dari kelas Standard. Hal ini cukup masuk akal karena kelas Standard merupakan kelas tengah dan juga kelas mayoritas. Banyak customer yang karakteristiknya berada di antara kategori baik dan buruk, sehingga model lebih sering salah memprediksi data Good atau Poor sebagai Standard. Namun, kesalahan ekstrem tetap perlu diperhatikan, terutama ketika customer aktual Poor diprediksi sebagai Good, karena dalam konteks credit scoring kesalahan ini dapat berdampak pada risiko bisnis.

Pada model Original + Loan Flags, jumlah actual Poor yang salah diprediksi sebagai Good adalah 93 data, sedikit lebih rendah dibanding model Original + Feature Engineering yang memiliki 95 data. Meskipun selisihnya kecil, hasil ini tetap menunjukkan bahwa model Original + Loan Flags sedikit lebih baik dalam mengurangi salah prediksi yang berisiko tinggi.

Secara keseluruhan, hasil modeling menunjukkan bahwa XGBoost merupakan model yang paling sesuai untuk dataset ini karena mampu menangani fitur campuran, hubungan non-linear, outlier valid, dan interaksi antar fitur. Dari dua pendekatan feature set terbaik, model XGBoost - Original + Loan Flags dipilih sebagai model final karena memberikan hasil confusion matrix yang sedikit lebih baik, jumlah prediksi benar lebih tinggi, dan performa yang lebih stabil pada data test.

Model final ini dinilai paling layak digunakan untuk tahap deployment karena sudah melalui proses preprocessing yang konsisten, hyperparameter tuning, validasi dengan cross-validation yang mempertimbangkan Customer_ID, serta evaluasi pada data test. Dengan demikian, model tidak hanya dipilih berdasarkan performa training, tetapi juga berdasarkan kemampuannya melakukan prediksi pada data baru secara lebih realistis.

# a. [LO 1, 2, 3, 4, 5 – 25 poin] Membuat model berbasis machine learning berdasarkan data dan problem yang ingin diselesaikan. Pada bagian ini anda perlu untuk menjelaskan:
## i. fitur yang digunakan,
## ii. pre processing yang dilakukan,
## iii. model machine learning, dan
## iv. evaluasi metrik yang digunakan.
## v. Anda juga perlu menjelaskan eksperimen apa yang dilakukan untuk mendapat model terbaik. Anda akan mencoba beberapa model machine learning dalam setiap kali proses training.

## a. Model Machine Learning Berdasarkan Data dan Problem yang Ingin Diselesaikan

Project ini bertujuan untuk membangun model machine learning yang dapat memprediksi `Credit_Score` customer ke dalam tiga kelas, yaitu `Good`, `Poor`, dan `Standard`. Berdasarkan hasil EDA, problem ini termasuk **multi-class classification** karena target memiliki lebih dari dua kelas. Selain itu, distribusi target tidak sepenuhnya seimbang karena kelas `Standard` menjadi kelas dengan jumlah data paling besar dibanding kelas lainnya. Oleh karena itu, model tidak hanya dievaluasi menggunakan accuracy, tetapi juga menggunakan metrik yang lebih adil untuk setiap kelas, terutama `f1_macro`.

### i. Fitur yang Digunakan

Fitur yang digunakan dalam modeling dipilih berdasarkan hasil data cleaning, EDA, validasi outlier, dan logika bisnis credit scoring. Kolom identitas seperti `ID`, `Customer_ID`, `Name`, `SSN`, dan `Unnamed: 0` tidak digunakan sebagai fitur akhir karena hanya berfungsi sebagai identitas dan tidak memiliki makna prediktif yang valid untuk customer baru. Namun, `Customer_ID` tetap digunakan selama proses cleaning dan splitting untuk memperbaiki data berdasarkan riwayat customer yang sama serta mencegah customer yang sama masuk ke train dan test sekaligus.

Pada project ini, terdapat dua pendekatan feature set utama. Pendekatan pertama adalah **Original + Feature Engineering**. Feature set ini menggunakan fitur original terpilih seperti `Age`, `Monthly_Inhand_Salary`, `Num_Bank_Accounts`, `Num_Credit_Card`, `Interest_Rate`, `Num_of_Loan`, `Delay_from_due_date`, `Num_of_Delayed_Payment`, `Changed_Credit_Limit`, `Num_Credit_Inquiries`, `Outstanding_Debt`, `Credit_Utilization_Ratio`, dan `Credit_History_Age`. Selain itu, pendekatan ini juga menambahkan fitur hasil rekayasa seperti `Credit_History_Age_Missing`, loan flags dari `Type_of_Loan`, ratio features, serta hasil pemisahan `Payment_Behaviour`.

Ratio features yang digunakan pada Model 1 adalah `EMI_to_Salary_Ratio`, `Investment_to_Salary_Ratio`, `Balance_to_Salary_Ratio`, dan `Debt_to_Salary_Ratio`. Fitur ini dibuat agar model dapat memahami beban keuangan customer secara lebih proporsional terhadap pendapatan bulanannya. Selain itu, `Payment_Behaviour` dipisahkan menjadi `Spending_Level` dan `Payment_Value_Level` agar informasi tingkat pengeluaran dan nilai pembayaran dapat dibaca model secara lebih jelas.

Pendekatan kedua adalah **Original + Loan Flags**. Feature set ini lebih mempertahankan fitur original yang lengkap, seperti `Age`, `Annual_Income`, `Monthly_Inhand_Salary`, `Num_Bank_Accounts`, `Num_Credit_Card`, `Interest_Rate`, `Num_of_Loan`, `Delay_from_due_date`, `Num_of_Delayed_Payment`, `Changed_Credit_Limit`, `Num_Credit_Inquiries`, `Outstanding_Debt`, `Credit_Utilization_Ratio`, `Credit_History_Age`, `Total_EMI_per_month`, `Amount_invested_monthly`, dan `Monthly_Balance`. Fitur kategorikal yang digunakan adalah `Month`, `Occupation`, `Credit_Mix`, `Payment_of_Min_Amount`, dan `Payment_Behaviour`.

Kedua pendekatan sama-sama menggunakan loan flags dari `Type_of_Loan`, yaitu `Has_Payday_Loan`, `Has_Mortgage_Loan`, `Has_Student_Loan`, `Has_Credit_Builder_Loan`, `Has_Debt_Consolidation_Loan`, `Has_Personal_Loan`, `Has_Auto_Loan`, `Has_Home_Equity_Loan`, dan `Has_Not_Specified_Loan`. Jadi, perbedaan utama kedua feature set bukan pada ada atau tidaknya loan flags, tetapi pada strategi penggunaan fitur. Model **Original + Feature Engineering** lebih banyak menggunakan fitur hasil transformasi seperti ratio features dan split payment behaviour, sedangkan model **Original + Loan Flags** lebih mempertahankan fitur original yang lengkap dan menambahkan loan flags sebagai informasi jenis pinjaman.

### ii. Preprocessing yang Dilakukan

Preprocessing yang dilakukan dalam project ini cukup panjang karena data tidak langsung siap digunakan untuk modeling. Proses preprocessing diawali dengan pemeriksaan struktur data, tipe data, missing value, duplikasi, format kolom, nilai ekstrem, dan inkonsistensi antar kolom. Tujuan tahap ini adalah memastikan data yang masuk ke model sudah bersih, valid secara format, dan masuk akal secara konteks bisnis credit scoring.

Tahap pertama adalah membersihkan kolom teknis dan identitas. Kolom seperti `Unnamed: 0`, `ID`, `Customer_ID`, `Name`, dan `SSN` diperiksa terlebih dahulu. Kolom-kolom ini tidak digunakan sebagai fitur modeling karena bersifat identitas. Namun, `Customer_ID` tetap dipertahankan selama preprocessing karena sangat berguna untuk memperbaiki nilai tidak valid atau missing berdasarkan riwayat customer yang sama pada bulan lain. Dengan cara ini, proses cleaning tidak dilakukan sembarangan, tetapi menggunakan referensi dari data customer yang sama.

Tahap berikutnya adalah perbaikan format data. Beberapa kolom numerik pada dataset awal masih memiliki format yang tidak konsisten, misalnya mengandung underscore `_`, string khusus, atau format teks. Kolom `Age`, `Annual_Income`, `Num_of_Loan`, `Num_of_Delayed_Payment`, dan `Outstanding_Debt` dibersihkan dari karakter yang tidak diperlukan, kemudian dikonversi menjadi tipe numerik. Kolom `Credit_History_Age` juga diubah dari format teks seperti tahun dan bulan menjadi total bulan agar dapat digunakan sebagai fitur numerik.

Setelah format diperbaiki, dilakukan validasi nilai tidak realistis. Pada kolom `Age`, nilai yang terlalu kecil atau terlalu besar diperiksa, lalu nilai tidak valid diperbaiki menggunakan referensi usia valid dari `Customer_ID` yang sama. Pada kolom `Num_Bank_Accounts`, nilai negatif dan nilai yang terlalu ekstrem diperbaiki menggunakan modus dari customer yang sama. Pada kolom `Num_Credit_Card`, nilai ekstrem juga dicek dan diperbaiki berdasarkan referensi customer yang sama. Pada kolom `Interest_Rate`, nilai yang terlalu tinggi dibanding batas wajar diperbaiki menggunakan referensi valid dari customer yang sama.

Kolom `Num_of_Loan` juga melalui proses cleaning khusus. Nilai dengan underscore dibersihkan dan dikonversi menjadi numerik. Nilai negatif atau nilai yang terlalu besar dianggap tidak valid, kemudian diperbaiki menggunakan modus dari `Customer_ID` yang sama jika tersedia. Selain itu, hubungan antara `Num_of_Loan` dan `Type_of_Loan` juga diperiksa. `Type_of_Loan` diparsing menjadi daftar loan, lalu dibuat kolom helper seperti `Loan_List_From_Type`, `Loan_Item_Count`, dan `Unique_Loan_Type_Count` untuk memvalidasi apakah jumlah loan sesuai dengan isi `Type_of_Loan`. Jika `Num_of_Loan` missing tetapi `Type_of_Loan` tersedia, jumlah loan dapat diperbaiki berdasarkan jumlah item pada `Type_of_Loan`.

Kolom `Type_of_Loan` tidak digunakan secara langsung sebagai teks panjang. Untuk customer dengan `Num_of_Loan = 0` dan `Type_of_Loan` missing, nilai `Type_of_Loan` diisi sebagai `No Loan`. Setelah itu, `Type_of_Loan` diubah menjadi loan flags agar setiap jenis loan dapat dibaca model sebagai fitur binary. Pendekatan ini lebih baik dibanding memasukkan teks mentah karena satu customer dapat memiliki lebih dari satu jenis pinjaman dalam satu kolom.

Kolom kategorikal juga dibersihkan. Pada `Occupation`, nilai placeholder seperti `_______` dianggap tidak valid dan diperbaiki menggunakan modus occupation dari customer yang sama. Pada `Credit_Mix`, nilai `_` dianggap tidak valid dan diperbaiki menggunakan referensi kategori valid dari customer yang sama. Pada `Payment_Behaviour`, nilai tidak valid seperti `!@9#%8` diperlakukan sebagai missing, lalu diperbaiki menggunakan modus dari customer yang sama jika tersedia. Pada Model 1, `Payment_Behaviour` kemudian dipecah menjadi `Spending_Level` dan `Payment_Value_Level`.

Beberapa kolom keuangan juga diperbaiki berdasarkan hubungan antar kolom. Pada `Annual_Income`, nilai ekstrem di atas threshold tertentu diperiksa dan diganti menggunakan median valid dari customer yang sama jika tersedia. Missing value pada `Annual_Income` juga dapat diperbaiki menggunakan `Monthly_Inhand_Salary × 12`. Sebaliknya, missing value pada `Monthly_Inhand_Salary` diperbaiki menggunakan median salary dari customer yang sama, atau menggunakan `Annual_Income / 12` jika referensi customer tidak tersedia. Hal ini dilakukan karena pendapatan tahunan dan pendapatan bulanan memiliki hubungan logis.

Kolom `Total_EMI_per_month` diperiksa menggunakan hubungan dengan `Num_of_Loan`, `Type_of_Loan`, dan pendapatan customer. Jika customer memiliki `Num_of_Loan = 0` dan riwayatnya konsisten tidak memiliki loan, tetapi nilai EMI positif, maka nilai EMI dapat dianggap error dan diperbaiki menjadi 0. Nilai EMI yang terlalu tinggi diperbaiki menggunakan median EMI valid dari customer yang sama jika tersedia. Jika tidak ada referensi valid, nilai tersebut dijadikan missing. Missing value pada EMI juga diperbaiki dengan aturan: jika `Num_of_Loan = 0`, EMI diisi 0; jika masih missing, digunakan median EMI dari customer yang sama; jika tetap tidak ada referensi, baris dapat dihapus.

Kolom `Amount_invested_monthly` juga dibersihkan karena terdapat nilai khusus seperti `__10000__`. Nilai tersebut dianggap tidak valid dan diperbaiki menggunakan rata-rata atau median investasi customer yang sama. Missing value yang masih tersisa diperbaiki menggunakan median dari customer yang sama, dan jika tetap tidak ada referensi maka baris dihapus. Kolom `Monthly_Balance` juga diperbaiki menggunakan median customer yang sama, lalu sisa missing yang tidak bisa diperbaiki dihapus.

Kolom `Credit_History_Age` diperbaiki dengan validasi logis terhadap `Age`. Jika `Credit_History_Age` lebih panjang daripada umur customer dalam bulan, maka nilai tersebut dianggap tidak valid. Nilai tidak valid diperbaiki menggunakan median credit history valid dari customer yang sama. Jika tetap tidak ada referensi, nilai dibuat missing. Selain itu, dibuat fitur `Credit_History_Age_Missing` untuk menangkap informasi apakah credit history customer sebelumnya missing atau tidak.

Setelah data cleaning selesai, tahap preprocessing untuk modeling dilakukan menggunakan `Pipeline` dan `ColumnTransformer`. Untuk fitur numerik, digunakan `SimpleImputer(strategy="median")` karena median lebih tahan terhadap outlier. Untuk fitur binary, digunakan `SimpleImputer(strategy="most_frequent")`. Untuk fitur kategorikal, digunakan `SimpleImputer(strategy="constant", fill_value="Missing")` dan `OneHotEncoder(handle_unknown="ignore")`. Penggunaan `handle_unknown="ignore"` penting agar model tetap dapat melakukan inference walaupun data baru memiliki kategori yang belum muncul saat training.

Preprocessing dibedakan menjadi dua jenis, yaitu preprocessing untuk model linear dan preprocessing untuk model tree-based. Untuk Logistic Regression, fitur numerik menggunakan `RobustScaler` karena model linear sensitif terhadap skala fitur, sedangkan dataset memiliki fitur numerik dengan skala yang sangat berbeda dan banyak outlier valid. Untuk Random Forest, XGBoost, dan LightGBM, scaling tidak digunakan karena model tree-based tidak bergantung pada jarak antar nilai fitur. Namun, imputer dan encoder tetap digunakan agar data input model konsisten.

Data kemudian dipisahkan menggunakan `StratifiedGroupKFold`. Teknik ini digunakan karena target memiliki distribusi kelas yang tidak seimbang dan satu `Customer_ID` dapat muncul pada beberapa bulan. Dengan menggunakan group-based split, customer yang sama tidak masuk ke train dan test secara bersamaan. Hal ini penting untuk mencegah data leakage dan membuat evaluasi model lebih realistis. Target `Credit_Score` juga diubah menjadi format numerik menggunakan `LabelEncoder` agar dapat digunakan oleh model machine learning.

### iii. Model Machine Learning yang Digunakan

Pada tahap modeling, beberapa algoritma machine learning dibandingkan untuk mencari model terbaik. Model yang digunakan adalah Logistic Regression, Random Forest, XGBoost, dan LightGBM.

Logistic Regression digunakan sebagai baseline model karena sederhana, cepat dilatih, dan mudah diinterpretasikan. Model ini penting sebagai pembanding awal untuk melihat apakah fitur yang sudah dibersihkan dan diproses memiliki sinyal yang cukup kuat untuk memprediksi `Credit_Score`.

Random Forest digunakan karena mampu menangkap hubungan non-linear antar fitur dan lebih tahan terhadap outlier dibanding model linear. Model ini cocok untuk data tabular yang berisi campuran fitur numerik, kategorikal hasil encoding, binary loan flags, dan engineered features.

XGBoost digunakan karena merupakan boosting model yang kuat untuk data tabular. Model ini membangun tree secara bertahap, di mana tree berikutnya berusaha memperbaiki kesalahan tree sebelumnya. XGBoost cocok untuk kasus credit scoring karena risiko customer tidak hanya dipengaruhi oleh satu fitur, tetapi oleh kombinasi banyak fitur seperti payment delay, outstanding debt, credit utilization, income, credit mix, payment behaviour, dan jenis loan.

LightGBM digunakan sebagai model boosting lain yang efisien dan cepat untuk data tabular. Model ini juga mampu menangkap hubungan non-linear dan digunakan sebagai pembanding terhadap XGBoost.

Berdasarkan hasil modeling, XGBoost menjadi model terbaik pada kedua pendekatan feature set. Pada pendekatan **Original + Feature Engineering**, XGBoost memperoleh performa terbaik dibanding Logistic Regression, Random Forest, dan LightGBM. Pada pendekatan **Original + Loan Flags**, XGBoost juga menjadi model terbaik dengan hasil yang sedikit lebih tinggi dibanding pendekatan pertama.

### iv. Evaluasi Metrik yang Digunakan

Evaluasi model dilakukan menggunakan beberapa metrik klasifikasi, yaitu accuracy, balanced accuracy, precision macro, recall macro, f1 macro, f1 weighted, ROC-AUC OVR macro, dan ROC-AUC OVR weighted. Namun, metrik utama yang digunakan untuk memilih model terbaik adalah `f1_macro`.

`f1_macro` dipilih karena target `Credit_Score` memiliki tiga kelas dan distribusinya tidak sepenuhnya seimbang. Jika hanya menggunakan accuracy, model dapat terlihat baik hanya karena sering memprediksi kelas mayoritas, yaitu `Standard`. Dengan `f1_macro`, setiap kelas diberi bobot yang sama sehingga performa pada kelas `Good`, `Poor`, dan `Standard` tetap diperhatikan secara seimbang.

Selain metrik numerik, confusion matrix juga digunakan untuk menganalisis pola prediksi model. Confusion matrix membantu melihat berapa banyak data aktual `Good`, `Poor`, dan `Standard` yang diprediksi benar atau salah. Evaluasi ini penting karena dalam credit scoring, kesalahan prediksi antar kelas memiliki dampak bisnis yang berbeda. Misalnya, customer aktual `Poor` yang diprediksi sebagai `Good` merupakan kesalahan yang lebih berisiko dibanding customer `Standard` yang diprediksi sebagai kelas lain.

Pada pendekatan **Original + Feature Engineering**, XGBoost menghasilkan accuracy sebesar 0.7044, balanced accuracy sebesar 0.7005, dan f1 macro sebesar 0.6860. Pada pendekatan **Original + Loan Flags**, XGBoost menghasilkan accuracy sebesar 0.7079, balanced accuracy sebesar 0.7040, dan f1 macro sebesar 0.6898. Hasil ini menunjukkan bahwa kedua model XGBoost memiliki performa yang cukup mirip, tetapi pendekatan **Original + Loan Flags** sedikit lebih unggul.

Berdasarkan confusion matrix, XGBoost dengan pendekatan **Original + Feature Engineering** menghasilkan 3.224 prediksi benar dari 4.577 data test. Sementara itu, XGBoost dengan pendekatan **Original + Loan Flags** menghasilkan 3.240 prediksi benar dari 4.577 data test. Artinya, model **Original + Loan Flags** menghasilkan 16 prediksi benar lebih banyak dibanding model **Original + Feature Engineering**.

### v. Eksperimen yang Dilakukan untuk Mendapatkan Model Terbaik

Beberapa eksperimen dilakukan untuk mendapatkan model terbaik. Eksperimen pertama adalah membandingkan dua pendekatan feature set, yaitu **Original + Feature Engineering** dan **Original + Loan Flags**. Pendekatan pertama menggunakan fitur original terpilih, ratio features, missing indicator, loan flags, dan split `Payment_Behaviour`. Pendekatan kedua menggunakan fitur original yang lebih lengkap dan loan flags. Tujuannya adalah melihat apakah model lebih baik menggunakan fitur hasil transformasi yang lebih ringkas atau fitur original yang lebih lengkap.

Eksperimen kedua adalah membandingkan beberapa algoritma machine learning, yaitu Logistic Regression, Random Forest, XGBoost, dan LightGBM. Perbandingan ini dilakukan agar model final tidak dipilih hanya berdasarkan satu algoritma. Logistic Regression digunakan sebagai baseline, sedangkan Random Forest, XGBoost, dan LightGBM digunakan untuk menangkap pola non-linear dan interaksi antar fitur yang lebih kompleks.

Eksperimen ketiga adalah melakukan hyperparameter tuning menggunakan `RandomizedSearchCV`. Tuning dilakukan agar setiap model memiliki kesempatan menemukan kombinasi parameter terbaik. Tuning dilakukan dengan cross-validation berbasis `StratifiedGroupKFold`, sehingga distribusi target tetap dijaga dan customer yang sama tidak masuk ke train dan validation fold sekaligus. Metrik utama tuning adalah `f1_macro`.

Pada model final XGBoost, hyperparameter terbaik yang diperoleh adalah `learning_rate = 0.0173`, `n_estimators = 201`, `max_depth = 6`, `subsample = 0.8703`, `colsample_bytree = 0.8173`, `reg_alpha = 0.0099`, dan `reg_lambda = 0.0421`. Kombinasi ini menunjukkan bahwa model belajar secara hati-hati dengan learning rate kecil, memiliki kompleksitas sedang dengan `max_depth = 6`, dan menggunakan sebagian data serta sebagian fitur pada setiap tree melalui `subsample` dan `colsample_bytree`. Hal ini membantu model mengurangi risiko overfitting dan meningkatkan generalisasi.

Berdasarkan hasil evaluasi seluruh kandidat, model terbaik secara keseluruhan adalah **XGBoost - Original + Loan Flags** dengan f1 macro sebesar 0.6898. Model ini mengungguli **XGBoost - Original + Feature Engineering** yang memiliki f1 macro sebesar 0.6860. Meskipun selisih performanya tidak terlalu besar, model **XGBoost - Original + Loan Flags** dipilih sebagai model final karena memiliki f1 macro paling tinggi, jumlah prediksi benar paling banyak pada confusion matrix, dan performa test yang sedikit lebih stabil. Model final kemudian disimpan sebagai artifact untuk digunakan pada tahap deployment.
